Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
#@title Imports (run this first)

import numpy as np

# Tensor support rank decompositions (algorithms for boolean matrix multiplication)

This section contains several tensor decompositions with low support rank, discovered by AlphaEvolve, and the code to verify their correctness. A support rank decomposition is a decomposition that reconstructs the support of the tensor, but may not reconstruct the values of the tensor exactly. More specifically, a support rank decomposition is valid if the constructed tensor is zero where the target tensor is zero and non-zero where the target tensor is non-zero. The support rank of the decomposition is related to the number of scalar multiplications needed to perform boolean matrix multiplication.

In [ ]:
#@title Verification function

def verify_tensor_support_rank_decomposition(decomposition: tuple[np.ndarray, np.ndarray, np.ndarray], n: int, m: int, p: int, rank: int):
  """Verifies the correctness of the tensor support rank decomposition.

  Args:
    decomposition: a tuple of 3 factor matrices with the same number of columns.
      (The number of columns specifies the support rank of the decomposition.)
      To construct a tensor, we take the outer product of the i-th column of the
      three factor matrices, for 1 <= i <= rank, and add up all these outer
      products.
    n: the first parameter of the matrix multiplication tensor.
    m: the second parameter of the matrix multiplication tensor.
    p: the third parameter of the matrix multiplication tensor.
    rank: the expected rank of the decomposition.

  Raises:
    AssertionError: If the decomposition does not have the correct rank, or if
    the decomposition does not reconstruct the support of 3D tensor which
    corresponds to multiplying an n x m matrix by an m x p matrix.
  """
  # Check that each factor matrix has the correct shape.
  factor_matrix_1, factor_matrix_2, factor_matrix_3 = decomposition
  assert factor_matrix_1.shape == (n * m, rank), f'Expected shape of factor matrix 1 is {(n * m, rank)}. Actual shape is {factor_matrix_1.shape}.'
  assert factor_matrix_2.shape == (m * p, rank), f'Expected shape of factor matrix 1 is {(m * p, rank)}. Actual shape is {factor_matrix_2.shape}.'
  assert factor_matrix_3.shape == (p * n, rank), f'Expected shape of factor matrix 1 is {(p * n, rank)}. Actual shape is {factor_matrix_3.shape}.'

  # Form the matrix multiplication tensor <n, m, p>.
  matmul_tensor = np.zeros((n * m, m * p, p * n), dtype=np.int32)
  for i in range(n):
    for j in range(m):
      for k in range(p):
        matmul_tensor[i * m + j][j * p + k][k * n + i] = 1

  # Check that the tensor is correctly constructed.
  constructed_tensor = np.einsum('il,jl,kl -> ijk', *decomposition)
  # Verify that constructed tensor is zero where matmul tensor is zero.
  assert np.array_equal(constructed_tensor == 0, matmul_tensor == 0), f'Tensor constructed by decomposition does not match the matrix multiplication tensor <{(n,m,p)}>: {constructed_tensor}.'
  # Verify that constructed tensor is non-zero (but not necessarily equal) where
  # matmul tensor is non-zero.
  assert np.array_equal(constructed_tensor != 0, matmul_tensor != 0), f'Tensor constructed by decomposition does not match the matrix multiplication tensor <{(n,m,p)}>: {constructed_tensor}.'
  print(f'Verified a decomposition of support rank {rank} for matrix multiplication tensor <{n},{m},{p}>.')

  # Print the set of values used in the decomposition.
  np.set_printoptions(linewidth=100)
  print('This decomposition uses these factor entries:\n', np.array2string(np.unique(np.vstack((factor_matrix_1, factor_matrix_2, factor_matrix_3))), separator=', '))

## Rank-24 support rank decomposition of <2,2,7> over Z


In [ ]:
#@title Data

decomposition_227 = (np.array([[-1., -1.,  1., -1.,  0., -0.,  1.,  1., -1., -0.,  1.,  1.,  1.,
         -1., -0.,  1.,  1., -1.,  1., -1., -1.,  1., -1., -1.],
        [-1.,  1., -0.,  1., -0.,  1.,  0., -1.,  1., -1.,  1.,  1., -1.,
         -1.,  0.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1.],
        [ 1.,  1., -1.,  1., -1., -0.,  1.,  1., -1., -0.,  1.,  1.,  1.,
         -1., -1.,  0., -1.,  0., -1., -1.,  1., -1., -1.,  1.],
        [ 1.,  1., -0., -1., -1., -1., -0.,  1., -1., -1., -1.,  1., -1.,
         -1.,  1.,  0.,  1., -0., -1.,  1., -1., -1.,  1., -1.]],
       dtype=np.float32),
 np.array([[ 0.,  1., -0.,  0., -1.,  0.,  0.,  1., -1.,  0.,  1., -0., -0.,
          0., -1., -1.,  1.,  1., -0.,  0.,  0., -1.,  1.,  1.],
        [ 0.,  0.,  0.,  1.,  0.,  2.,  0.,  1., -1.,  0., -1., -0., -0.,
          0., -0.,  0., -0.,  0., -1., -0.,  1.,  0., -1., -0.],
        [-0., -1.,  0.,  0.,  0., -0.,  0., -0., -0., -2., -0., -0.,  1.,
         -1.,  0.,  0.,  1.,  0.,  0., -1., -0.,  1.,  0.,  1.],
        [ 1., -1., -2., -1.,  2., -0., -2.,  1.,  1.,  0.,  1.,  1., -1.,
          0.,  2., -2.,  1.,  2.,  0., -1.,  1., -1., -1., -1.],
        [ 1., -1., -2., -1.,  0.,  0.,  2., -1., -1., -0., -1., -1.,  1.,
         -0., -0., -0.,  1., -0.,  0.,  1.,  1., -1.,  1., -1.],
        [ 0.,  1., -0.,  0., -1.,  0.,  0., -1., -1., -0., -1., -0.,  0.,
         -0., -1.,  1., -1., -1.,  0.,  0., -0.,  1.,  1.,  1.],
        [-1., -1., -2.,  1.,  0.,  0., -2.,  1.,  1., -0.,  1., -1.,  1.,
         -0., -0., -0.,  1.,  0., -0.,  1., -1., -1., -1., -1.],
        [-0.,  1.,  0.,  0., -1.,  0.,  0.,  1., -1., -0., -1.,  0.,  0.,
         -0.,  1.,  1., -1.,  1., -0., -0.,  0., -1., -1., -1.],
        [-0.,  1.,  2., -1.,  0.,  0.,  0., -0., -0.,  0., -0., -0.,  0.,
         -0.,  0.,  0., -1.,  0., -1., -0., -1.,  1.,  0.,  1.],
        [ 0.,  0., -0.,  0.,  0.,  0., -2.,  1.,  1., -0.,  1., -0., -1.,
         -1.,  0., -0., -0., -0.,  0.,  1.,  0.,  0., -1.,  0.],
        [-1.,  1., -0., -1., -2.,  2., -0.,  1., -1.,  2., -1., -1., -1.,
          0.,  2.,  2., -1.,  2.,  0., -1.,  1., -1., -1., -1.],
        [-1., -1., -0., -1.,  0.,  2., -0.,  1., -1., -2., -1.,  1.,  1.,
          0.,  0., -0.,  1., -0.,  0.,  1.,  1.,  1., -1.,  1.],
        [-0.,  1., -0.,  0., -1., -0.,  0., -1., -1., -0.,  1.,  0.,  0.,
         -0.,  1., -1.,  1., -1.,  0., -0.,  0.,  1., -1., -1.],
        [ 1.,  1.,  0.,  1.,  0.,  2.,  0.,  1., -1.,  2., -1.,  1.,  1.,
         -0., -0., -0., -1., -0., -0.,  1., -1., -1., -1., -1.]],
       dtype=np.float32),
 np.array([[-0., -1., -0., -0.,  0.,  2.,  0.,  1.,  1., -2.,  1.,  0.,  0.,
          0.,  0., -0.,  1.,  0.,  0., -0.,  0., -1., -1., -1.],
        [ 0.,  1., -0.,  0., -0.,  2., -0.,  1.,  1.,  2.,  1.,  0., -0.,
          0.,  0., -0., -1., -0.,  0.,  0.,  0.,  1., -1.,  1.],
        [ 0.,  1., -0., -1., -2., -0.,  0.,  1.,  1.,  0., -1., -2., -1.,
          0.,  2., -2., -1.,  2., -2.,  1., -1., -1.,  1., -1.],
        [ 0.,  1.,  0.,  1., -2.,  0., -0.,  1.,  1., -0., -1., -2., -1.,
         -0.,  2., -2., -1.,  2.,  2.,  1.,  1., -1.,  1., -1.],
        [ 2., -1.,  0.,  1.,  2., -0., -0.,  1., -1.,  0., -1.,  0., -1.,
         -2., -2., -2., -1.,  2.,  0., -1., -1., -1., -1.,  1.],
        [-2.,  1.,  0., -1., -2., -0., -0., -1.,  1., -0.,  1., -0., -1.,
         -2.,  2.,  2.,  1., -2.,  0., -1.,  1.,  1.,  1., -1.],
        [ 0.,  0., -1., -0.,  0., -1., -1., -1., -0.,  1., -1., -0.,  0.,
          0.,  0., -2., -1., -2.,  0.,  0.,  0.,  1., -0., -0.],
        [ 0., -1., -1., -0.,  2., -1.,  1.,  0., -1., -1., -0., -0.,  0.,
          0.,  2.,  0., -0., -0.,  0.,  0.,  0., -0.,  1., -1.],
        [ 1., -1.,  0., -1.,  2., -0., -0.,  0., -1.,  0.,  0.,  1., -0.,
         -1.,  2., -0.,  0.,  0.,  1.,  1.,  0.,  0.,  1., -1.],
        [ 1., -0., -0., -1.,  0.,  0.,  0., -1.,  0., -0., -1., -1., -0.,
          1.,  0., -2., -1., -2.,  1., -1., -0.,  1., -0.,  0.],
        [ 0., -1., -2.,  0.,  0., -0., -2., -1.,  1., -0., -1.,  0., -0.,
          0., -0., -0., -1., -0.,  0.,  0.,  0.,  1., -1., -1.],
        [ 0., -1., -2., -0., -0., -0.,  2.,  1., -1.,  0.,  1., -0.,  0.,
          0., -0.,  0., -1.,  0.,  0., -0.,  0.,  1.,  1., -1.],
        [-1.,  0.,  1., -0.,  0.,  1.,  1.,  0., -0., -1., -0.,  1., -1.,
          1., -0.,  0.,  0., -0.,  1., -0., -1., -0.,  0.,  0.],
        [-1.,  0.,  1.,  0.,  0.,  1., -1., -0., -0.,  1., -0., -1.,  1.,
         -1., -0., -0.,  0., -0.,  1.,  0., -1., -0., -0.,  0.]],
       dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 2
p = 7
rank = 24

verify_tensor_support_rank_decomposition(decomposition_227, n, m, p, rank)

## Rank-19 support rank decomposition of <2,3,4> over Z


In [ ]:
#@title Data

decomposition_234 = (np.array([[-1., -0., -1., -0., -1.,  1., -1.,  0., -0.,  1.,  0., -0.,  0.,
         -1., -0., -1.,  0., -1., -1.],
        [-0., -1., -0.,  0.,  1., -1.,  1., -0., -1., -1., -0., -1.,  1.,
         -0.,  0., -0.,  1.,  1., -0.],
        [ 0.,  1., -1., -0.,  0.,  1.,  0.,  0.,  1., -0.,  0., -0., -1.,
         -1.,  1., -1., -1.,  0., -1.],
        [ 1.,  0.,  1.,  0.,  1., -1., -0., -0.,  0., -1.,  1.,  0., -0.,
          1.,  0., -0.,  1.,  1.,  1.],
        [-0.,  0.,  1., -1., -0., -1.,  1., -0., -1., -1., -0., -1.,  1.,
         -0.,  0., -0.,  1.,  1., -0.],
        [ 0.,  1., -1., -0.,  0.,  1.,  0., -1.,  1.,  1.,  0., -0.,  0.,
          0.,  1., -1., -1.,  0., -1.]], dtype=np.float32),
 np.array([[ 1.,  0., -0.,  0.,  0., -1., -1., -0., -1., -0., -1.,  1., -0.,
          0., -1.,  1.,  1., -1.,  1.],
        [-1., -0., -0., -0., -0.,  1., -1.,  0.,  1., -0.,  1.,  1.,  0.,
          0., -1.,  1., -1., -1.,  1.],
        [-1.,  0.,  0., -0., -0., -1., -1.,  0., -1., -0.,  1.,  1., -0.,
         -0.,  1., -1.,  1., -1., -1.],
        [-1.,  0., -0., -0., -0., -1.,  1.,  0., -1.,  0.,  1., -1., -0.,
         -0., -1.,  1.,  1.,  1.,  1.],
        [-1.,  1., -1.,  1.,  1., -1., -0., -0.,  1., -0., -0., -1.,  0.,
         -0., -1., -0., -0., -1., -1.],
        [ 1.,  1.,  1.,  1., -1.,  1., -0., -0.,  1.,  0., -0., -1.,  0.,
          0., -1., -0.,  0.,  1.,  1.],
        [-1.,  1.,  1., -1.,  1.,  1.,  0.,  0.,  1., -0.,  0.,  1., -0.,
          0., -1., -0.,  0., -1.,  1.],
        [ 1.,  1., -1., -1., -1., -1.,  0.,  0.,  1.,  0.,  0.,  1., -0.,
         -0., -1.,  0., -0.,  1., -1.],
        [-1.,  0.,  0., -0., -0., -1.,  0.,  1., -1.,  1.,  0.,  1.,  1.,
          1.,  1.,  0., -0., -1., -1.],
        [-1., -0.,  0.,  0., -0., -1., -0., -1.,  1.,  1., -0., -1., -1.,
          1., -1., -0.,  0., -1., -1.],
        [ 1.,  0., -0.,  0.,  0., -1.,  0., -1., -1.,  1., -0.,  1.,  1.,
         -1., -1., -0., -0., -1.,  1.],
        [ 1., -0., -0., -0., -0., -1.,  0.,  1.,  1.,  1.,  0., -1., -1.,
         -1.,  1.,  0.,  0., -1.,  1.]], dtype=np.float32),
 np.array([[ 1., -1.,  1.,  1.,  1.,  1., -1., -1., -0.,  1.,  1.,  1.,  1.,
          1., -1.,  1., -1., -0.,  0.],
        [ 0.,  1., -1., -1., -1.,  0.,  1., -1., -1.,  1., -1.,  0.,  1.,
          1., -0., -1.,  1., -1.,  1.],
        [-1., -1., -1.,  1., -1., -1., -1., -1.,  0., -1., -1.,  1.,  1.,
         -1., -1.,  1.,  1., -0.,  0.],
        [ 0.,  1.,  1., -1.,  1.,  0., -1., -1., -1., -1., -1., -0.,  1.,
         -1.,  0.,  1.,  1.,  1., -1.],
        [ 1., -1., -1., -1.,  1., -1.,  1., -1., -0., -1.,  1., -1., -1.,
          1., -1.,  1.,  1., -0., -0.],
        [ 0.,  1.,  1.,  1., -1., -0.,  1.,  1., -1.,  1.,  1., -0.,  1.,
         -1.,  0.,  1.,  1., -1., -1.],
        [ 1.,  1., -1.,  1.,  1., -1., -1.,  1.,  0., -1.,  1.,  1.,  1.,
          1.,  1., -1.,  1., -0., -0.],
        [ 0., -1.,  1., -1., -1., -0.,  1., -1.,  1.,  1., -1.,  0., -1.,
         -1.,  0.,  1., -1., -1., -1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 3
p = 4
rank = 19

verify_tensor_support_rank_decomposition(decomposition_234, n, m, p, rank)

## Rank-24 support rank decomposition of <2,3,5> over Z


In [ ]:
#@title Data

decomposition_235 = (np.array([[ 0.,  0.,  0., -0., -0.,  1.,  0.,  1., -0.,  0., -0., -1.,  1.,
         -1., -0.,  1.,  0., -0.,  1., -0., -1.,  1.,  1.,  1.],
        [ 1., -0.,  1., -1.,  1., -0., -0.,  0.,  0., -1., -0.,  1., -0.,
         -0.,  0., -1., -0.,  0., -1.,  1.,  1., -1.,  0.,  0.],
        [-1.,  0., -1., -1., -1.,  1., -0.,  1., -0.,  1.,  1.,  1., -0.,
         -1.,  0., -1.,  0.,  0.,  1., -0.,  1., -1.,  1., -1.],
        [-0.,  1.,  1.,  0., -0., -1.,  0., -1.,  0.,  0.,  0., -0.,  0.,
          1.,  1., -1., -1., -0., -1.,  0.,  1.,  0.,  0., -1.],
        [ 0.,  0., -1.,  1., -1., -0.,  1., -0., -1.,  1., -0., -1.,  0.,
         -1., -1., -0., -0., -0.,  1.,  0., -1.,  0., -0., -0.],
        [ 0.,  1., -1., -1.,  1., -1., -1.,  1.,  0.,  1., -0.,  1.,  0.,
         -1., -1., -1., -0.,  1.,  1., -0.,  1.,  0., -0., -1.]],
       dtype=np.float32),
 np.array([[ 0.,  0., -1., -1., -0.,  0.,  0.,  0.,  0., -1., -0.,  1., -1.,
          0., -0., -0.,  1., -0.,  1.,  0., -1.,  0., -0.,  0.],
        [ 0.,  1., -1., -1.,  0.,  0.,  0., -1., -0., -1., -1.,  1., -1.,
          0.,  0., -0.,  1., -1.,  1., -0., -1., -0.,  1., -1.],
        [-0., -0.,  1., -1.,  0.,  0., -0.,  0., -0.,  1., -0.,  1., -1.,
         -0.,  0.,  0., -1., -0., -1., -0., -1., -0.,  0.,  0.],
        [-0.,  1., -0., -0., -0., -1., -0.,  0., -0.,  0.,  1., -0.,  0.,
         -0., -0.,  0.,  0., -1.,  0., -0., -0., -0., -1., -0.],
        [-0.,  1.,  1., -1.,  0., -1.,  0.,  0.,  0.,  1.,  1.,  1.,  1.,
         -0.,  0.,  0.,  1., -1., -1.,  0., -1., -0., -1.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0., -0., -1.,  1.,  0., -0., -0.,  0.,
         -1.,  0., -1.,  0.,  0.,  1., -1., -1., -0., -0., -1.],
        [-1.,  0.,  0., -0.,  1., -0.,  1., -1., -1., -0., -1., -0.,  0.,
         -1.,  0.,  1., -0.,  1.,  1., -1.,  1., -0., -0.,  1.],
        [ 0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  1.,  0., -0.,  0., -0.,
         -1.,  0.,  1.,  0., -0.,  1.,  1.,  1.,  0., -0.,  1.],
        [-1., -0.,  0., -0.,  1.,  0.,  1.,  0., -0.,  0., -1.,  0.,  0.,
          0.,  0.,  0.,  0.,  1., -0., -0., -0., -0.,  0., -0.],
        [-1.,  0.,  0., -1., -0., -0., -1., -1.,  1., -1., -1.,  0.,  0.,
         -1.,  0., -1., -0., -1.,  1., -1., -1.,  0., -0., -1.],
        [-0.,  0., -1.,  1.,  0., -0., -0., -1.,  1., -1., -0., -1.,  1.,
         -1.,  1.,  1.,  1.,  0.,  1.,  1.,  1.,  1., -0.,  1.],
        [-0., -0.,  1., -1., -0., -1.,  0.,  0., -1.,  1.,  1.,  1., -1.,
          1., -1., -1., -1., -1., -1., -1., -1., -1.,  0.,  0.],
        [-0., -0.,  1.,  1., -0.,  0.,  0.,  1., -1.,  1.,  0., -1.,  1.,
          1., -1.,  1., -1., -0., -1.,  1.,  1.,  1.,  0.,  1.],
        [-0., -0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0., -0.,
          0., -0., -0.,  0.,  1., -0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1., -0.,  1.,  0.,  0., -1.,  1.,  0., -1., -1.,  1.,
         -1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -0.,  1.]],
       dtype=np.float32),
 np.array([[ 1.,  1.,  1., -1.,  1., -1.,  1.,  0., -0., -0.,  0.,  1., -0.,
          1.,  1.,  1., -0.,  0., -1., -0., -0., -1.,  1., -1.],
        [ 1.,  1.,  1., -0.,  1., -1.,  1.,  1.,  0.,  1.,  0.,  1., -0.,
          1.,  1.,  1., -0., -0.,  0., -0.,  1., -1.,  1.,  0.],
        [-0.,  1., -0.,  0., -0., -1., -0.,  0., -0., -0.,  0.,  0., -0.,
         -0.,  0.,  1., -0., -0., -0.,  1., -0.,  0.,  1., -1.],
        [ 0., -1.,  0., -0.,  0.,  1.,  0., -1., -1.,  0.,  0., -0.,  0.,
         -1.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -1., -0.],
        [-0., -0.,  1.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,  0., -1.,
          1.,  1., -0.,  0., -0., -1.,  1.,  0.,  1., -0.,  0.],
        [-0.,  0., -0., -0.,  0., -0., -0., -0., -1.,  0.,  0., -1., -0.,
         -0.,  1., -1.,  1., -0.,  0., -0., -1.,  1.,  0., -0.],
        [-1.,  1.,  0., -1.,  1., -1.,  1.,  0., -0., -0., -2.,  1., -1.,
          0., -0.,  1.,  0., -0., -0.,  1., -0.,  0., -1., -1.],
        [ 1., -1.,  1., -0.,  1., -1., -1.,  1.,  1.,  1.,  0.,  0.,  0.,
          1.,  0., -0., -1., -2., -0., -0.,  0., -0.,  1., -0.],
        [-1., -0.,  0.,  1., -1., -0., -1.,  0., -0.,  0., -0., -1.,  1.,
          0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,  0., -0.],
        [-1.,  0., -1.,  0., -1.,  0., -1., -0., -0., -1., -0.,  0.,  0.,
          0.,  0.,  0.,  1., -0.,  0.,  0., -0., -0.,  0., -0.]],
       dtype=np.float32))

In [ ]:
#@title Verification
n = 2
m = 3
p = 5
rank = 24

verify_tensor_support_rank_decomposition(decomposition_235, n, m, p, rank)

## Rank-34 support rank decomposition of <2,3,7> over Z


In [ ]:
#@title Data

decomposition_237 = (np.array([[-1., -1., -1., -1., -1.,  0., -0.,  1., -0.,  0., -0.,  0.,  0.,
          1.,  0., -1., -1.,  0.,  0., -1., -1.,  0.,  1.,  1.,  0.,  0.,
         -1., -0.,  1., -0.,  1.,  0.,  0.,  1.],
        [-1., -0.,  1.,  1., -0.,  1., -0., -0.,  1., -1., -1.,  0., -0.,
         -1., -1., -0., -0.,  0., -1., -0.,  1., -0., -0.,  1., -0.,  0.,
         -0.,  0., -1., -1.,  1., -1.,  1., -0.],
        [ 0.,  0.,  0.,  1., -0., -1., -1.,  1.,  1.,  0.,  1., -0.,  1.,
         -1.,  0., -1.,  1., -1.,  1.,  0.,  1., -0.,  1.,  0.,  0., -1.,
         -0., -1., -1.,  0.,  0., -0., -0.,  0.],
        [-1.,  0., -1., -1., -1.,  0.,  0., -1., -0.,  0., -0., -1., -1.,
         -0.,  0.,  1., -1., -0., -0.,  1., -1., -0.,  1., -1., -0., -0.,
         -1.,  0.,  1.,  1., -1., -0.,  0.,  1.],
        [-1., -0.,  1.,  1.,  0., -1.,  0.,  0.,  1., -1., -1., -0., -0.,
         -1., -1., -0.,  0.,  0., -1.,  0., -0.,  1., -0.,  1., -0., -1.,
         -0., -0., -1., -1., -1., -0.,  1., -1.],
        [ 0., -0., -0.,  1.,  0.,  1.,  0.,  1.,  1.,  0., -1., -0.,  1.,
         -1., -1.,  1.,  1.,  1.,  1.,  0.,  1., -0., -1., -0., -1., -1.,
          1., -1.,  0., -0., -0.,  0., -0., -0.]], dtype=np.float32),
 np.array([[ 0.,  1.,  1., -1., -1.,  0., -0.,  0.,  1., -0., -0.,  0.,  1.,
          1., -0., -0.,  1., -0., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
         -0., -1.,  0.,  1.,  0.,  0., -1., -0.],
        [-0.,  1., -1.,  1., -1., -0.,  0.,  0., -1.,  0.,  0., -0.,  1.,
         -1.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
          0., -1., -0., -1.,  0.,  0.,  1.,  0.],
        [-1., -0., -0., -0.,  0., -0., -0., -0., -0., -1., -0.,  2.,  0.,
          0., -0.,  1.,  0., -1., -0., -1.,  0.,  0.,  1., -1.,  0.,  0.,
         -0., -0., -0., -0.,  0., -0.,  0.,  0.],
        [-0.,  1., -1., -1., -1., -0.,  0., -0.,  1.,  0., -0.,  0., -1.,
          1., -0.,  0., -1.,  0., -0., -0., -0., -0., -0., -0.,  0.,  0.,
          0.,  1.,  0., -1.,  0., -0.,  1.,  0.],
        [-0., -1., -1., -1.,  1., -0., -0.,  0.,  1., -0.,  0.,  0.,  1.,
          1., -0.,  0.,  1., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,
         -0., -1.,  0., -1.,  0.,  0.,  1., -0.],
        [-1., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -1.,  0.,  2.,  0.,
         -0.,  0., -1.,  0.,  1.,  0.,  1.,  0.,  0., -1., -1.,  0.,  0.,
          0., -0., -0.,  0., -0., -0.,  0.,  0.],
        [ 1., -0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1., -0.,  2.,  0.,
          0., -0.,  1., -0., -1., -0.,  1., -0., -0.,  1.,  1., -0., -0.,
         -0.,  0.,  0., -0.,  0.,  0., -0., -0.],
        [-0., -0., -1., -1.,  1.,  0., -0., -0.,  1.,  0.,  0.,  0., -0.,
         -0., -0., -0.,  1., -0., -0.,  0.,  1.,  0., -0., -0.,  0., -1.,
          0., -1.,  0.,  0.,  0., -1.,  1.,  1.],
        [ 0., -0.,  1., -1., -1.,  0., -0., -0., -1.,  0., -0., -0., -0.,
         -0.,  0.,  0.,  1.,  0., -0.,  0.,  1.,  0.,  0.,  0., -0.,  1.,
         -0.,  1.,  0., -0.,  0., -1.,  1., -1.],
        [-0., -0., -0., -0., -0., -1., -0.,  0., -0.,  1., -1.,  0., -0.,
         -0., -0.,  0., -0.,  1.,  0.,  1.,  0.,  2.,  0.,  1., -0.,  0.,
         -0.,  0., -0.,  0., -1., -0., -0., -0.],
        [-0.,  0.,  1.,  1., -1.,  0., -0., -0.,  1.,  0., -0., -0.,  0.,
          0.,  0.,  0., -1., -0., -0.,  0., -1.,  0.,  0.,  0.,  0., -1.,
          0., -1., -0., -0.,  0., -1.,  1., -1.],
        [ 0., -0.,  1., -1., -1., -0., -0., -0.,  1., -0., -0., -0., -0.,
         -0., -0., -0.,  1., -0.,  0.,  0.,  1., -0., -0., -0., -0., -1.,
         -0., -1.,  0., -0.,  0.,  1., -1., -1.],
        [-0., -0.,  0., -0.,  0.,  1.,  0.,  0.,  0.,  1.,  1., -0., -0.,
          0.,  0., -0.,  0., -1.,  0., -1., -0.,  2.,  0., -1.,  0.,  0.,
         -0., -0., -0., -0.,  1.,  0.,  0., -0.],
        [ 0., -0.,  0.,  0., -0., -1., -0., -0.,  0., -1., -1.,  0., -0.,
         -0.,  0.,  0.,  0.,  1.,  0., -1., -0.,  2.,  0., -1., -0.,  0.,
         -0., -0., -0.,  0.,  1.,  0.,  0., -0.],
        [ 0., -0.,  1.,  1., -1., -0.,  1., -0.,  1., -0.,  0.,  0.,  0.,
          0.,  1., -0., -1.,  0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,
         -1., -1.,  1., -0.,  0.,  0.,  1.,  0.],
        [-0.,  0., -1., -1., -1.,  0.,  1., -0., -1., -0.,  0., -0.,  0.,
         -0., -1., -0., -1., -0., -0.,  0., -0., -0., -0.,  0., -0.,  0.,
         -1., -1., -1.,  0.,  0.,  0., -1., -0.],
        [-0.,  0., -0.,  0., -0., -0.,  0., -1., -0.,  1.,  1., -0.,  0.,
         -0., -0., -1., -0.,  1.,  1.,  1., -0.,  0., -0.,  0.,  2.,  0.,
         -0.,  0., -0.,  0., -0., -0., -0.,  0.],
        [ 0.,  0.,  1.,  1., -1.,  0., -1.,  0., -1., -0.,  0.,  0.,  0.,
          0., -1.,  0., -1., -0.,  0., -0.,  0., -0., -0.,  0., -0., -0.,
         -1.,  1.,  1., -0., -0.,  0., -1.,  0.],
        [-0.,  0.,  1.,  1.,  1.,  0.,  1., -0., -1., -0., -0.,  0., -0.,
          0., -1.,  0.,  1.,  0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,
          1., -1.,  1., -0.,  0., -0., -1., -0.],
        [-0., -0.,  0.,  0.,  0., -0.,  0.,  1., -0.,  1.,  1.,  0., -0.,
         -0., -0.,  1., -0., -1.,  1., -1.,  0.,  0., -0.,  0.,  2.,  0.,
          0.,  0.,  0., -0., -0.,  0., -0.,  0.],
        [ 0.,  0.,  0., -0.,  0.,  0.,  0., -1., -0.,  1.,  1.,  0.,  0.,
         -0.,  0., -1.,  0., -1.,  1.,  1., -0.,  0.,  0.,  0., -2., -0.,
          0., -0., -0., -0.,  0.,  0.,  0.,  0.]], dtype=np.float32),
 np.array([[ 0., -1., -1., -0.,  0.,  0., -1.,  0.,  1., -0.,  0.,  0.,  1.,
          1.,  1., -0., -1., -0., -0.,  0.,  1.,  0.,  0., -0.,  0., -1.,
          1.,  0., -1., -1., -0.,  1.,  0.,  1.],
        [ 0.,  1.,  0.,  1.,  1., -0.,  1., -0., -0.,  0.,  0., -0., -1.,
         -1., -1.,  0., -0., -0.,  0., -0.,  1.,  0., -0., -0.,  0., -1.,
         -1.,  1.,  1.,  1.,  0.,  1., -1.,  1.],
        [-0., -1.,  1.,  0., -0., -0., -1.,  0., -1.,  0., -0., -0.,  1.,
         -1., -1.,  0., -1.,  0., -0.,  0.,  1.,  0., -0.,  0., -0.,  1.,
          1., -0.,  1.,  1.,  0.,  1., -0., -1.],
        [ 0., -1.,  0.,  1., -1., -0., -1.,  0., -0.,  0., -0., -0.,  1.,
         -1., -1.,  0., -0.,  0., -0., -0.,  1.,  0., -0.,  0., -0.,  1.,
          1., -1.,  1.,  1.,  0.,  1., -1., -1.],
        [-1., -0., -0.,  0., -0.,  0.,  0.,  1., -0.,  2., -2., -0., -0.,
          0., -0.,  1.,  0.,  2., -0., -0., -0.,  1., -1.,  1.,  1., -0.,
          0.,  0., -0., -0.,  1.,  0.,  0.,  0.],
        [-1.,  0., -0., -0.,  0., -0., -0., -1.,  0., -0., -0., -0.,  0.,
         -0.,  0., -1., -0.,  0.,  0., -0.,  0., -1., -1., -1., -1.,  0.,
         -0., -0.,  0., -0., -1., -0.,  0.,  0.],
        [-0., -1.,  1., -0.,  0.,  0., -1.,  0.,  1., -0., -0., -0., -1.,
          1.,  1.,  0.,  1.,  0.,  0., -0., -1., -0., -0.,  0.,  0., -1.,
         -1., -0.,  1.,  1., -0.,  1.,  0., -1.],
        [-0., -1., -0., -1., -1.,  0.,  1.,  0.,  0., -0.,  0., -0., -1.,
          1., -1.,  0.,  0., -0.,  0., -0., -1., -0., -0.,  0.,  0., -1.,
          1.,  1., -1.,  1., -0.,  1., -1., -1.],
        [-0.,  1.,  1., -0., -0.,  0.,  1.,  0.,  1.,  0., -0.,  0.,  1.,
          1.,  1.,  0., -1.,  0.,  0.,  0.,  1., -0., -0., -0.,  0., -1.,
          1.,  0.,  1.,  1., -0., -1.,  0., -1.],
        [ 0.,  1., -0., -1.,  1.,  0., -1.,  0.,  0.,  0., -0., -0.,  1.,
          1., -1.,  0.,  0.,  0., -0., -0., -1.,  0.,  0.,  0.,  0.,  1.,
         -1., -1., -1.,  1., -0.,  1., -1.,  1.],
        [ 0., -0., -0., -0., -0.,  1.,  0.,  1.,  0., -2.,  1., -1., -0.,
         -0.,  0., -1.,  0.,  0.,  1., -2.,  0., -1., -1., -2.,  0., -0.,
         -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.],
        [-0., -0.,  0.,  0., -0., -1., -0.,  1., -0.,  0., -1.,  1., -0.,
         -0., -0.,  1.,  0.,  0.,  1., -0.,  0.,  1.,  1., -0.,  0., -0.,
          0.,  0.,  0.,  0., -0.,  0., -0., -0.],
        [ 1., -0., -0.,  0.,  0.,  1.,  0., -0.,  0., -0., -1.,  1., -0.,
          0., -0.,  2.,  0.,  2.,  1.,  2., -0., -0., -0.,  1.,  1., -0.,
         -0., -0., -0.,  0., -1.,  0., -0.,  0.],
        [ 1., -0.,  0.,  0., -0., -1.,  0., -0., -0.,  0., -1.,  1., -0.,
          0.,  0., -0., -0.,  0.,  1., -0., -0.,  0., -0.,  1.,  1., -0.,
         -0.,  0., -0.,  0.,  1., -0., -0., -0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 3
p = 7
rank = 34

verify_tensor_support_rank_decomposition(decomposition_237, n, m, p, rank)

## Rank-38 support rank decomposition of <2,3,8> over Z


In [ ]:
#@title Data

decomposition_238 = (np.array([[-1.,  0.,  1.,  0.,  0., -1., -1.,  1.,  0., -0., -1.,  1., -0.,
         -1., -1.,  0., -1.,  1.,  1., -1., -0., -0.,  0.,  1., -0., -0.,
         -0., -0.,  0., -1.,  1., -0., -1., -1., -1., -0.,  1., -0.],
        [-0.,  1., -1.,  1.,  0.,  0.,  1.,  0.,  0., -0., -1., -0.,  1.,
          1.,  0., -1.,  0.,  1., -1., -0., -1., -1.,  0., -1.,  0.,  0.,
          1.,  1., -0., -0.,  1.,  0., -1.,  0., -0.,  1.,  1., -1.],
        [ 0.,  0.,  1.,  0.,  0., -1., -1., -0.,  0., -0.,  0., -0., -0.,
         -1., -1., -1., -1., -0., -0.,  0.,  1., -0., -1.,  1.,  1.,  1.,
          1.,  1., -1., -1., -0.,  1., -1., -1.,  0.,  1., -0., -0.],
        [ 1., -0., -1.,  1., -1.,  0.,  1., -1.,  0., -0., -1.,  1., -0.,
         -0., -1.,  0., -1., -0., -1.,  1.,  0.,  0.,  0., -1.,  0.,  0.,
          1., -0.,  1.,  1.,  1.,  0., -1., -1., -0., -0.,  1.,  0.],
        [ 1., -0., -1.,  1., -0., -0.,  1.,  0.,  1.,  0.,  1.,  0., -1.,
          1.,  1.,  1., -0., -1., -1., -0., -1., -1., -0.,  0.,  0.,  0.,
         -1., -1., -0., -0.,  0., -1.,  1., -0., -0.,  0., -1., -1.],
        [-0., -0., -1., -0.,  0., -1., -0., -1.,  0.,  1.,  0., -0., -0.,
          1., -1.,  0.,  0., -0.,  0., -0., -1.,  0., -1., -1., -1.,  0.,
          1.,  1.,  1.,  1., -0., -1., -1., -1., -0.,  1.,  1., -1.]],
       dtype=np.float32),
 np.array([[ 0.,  0.,  1.,  1., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,
          1.,  0.,  0.,  0.,  0.,  1.,  1., -1., -1.,  0.,  0., -1., -0.,
         -0.,  0.,  1.,  1.,  0.,  0.,  0., -0.,  1.,  0.,  0., -0.],
        [ 0.,  0.,  0., -0., -1., -1.,  0.,  0.,  0.,  0., -1.,  1.,  1.,
          0.,  0.,  0.,  0.,  1., -0., -0., -0.,  0., -1., -0., -0.,  0.,
         -1.,  1., -0., -0.,  0., -0.,  1.,  1., -0., -0.,  0.,  0.],
        [-0., -0., -0.,  0., -1., -1.,  0.,  0.,  0.,  0.,  1.,  1., -1.,
         -0.,  0.,  0.,  0., -1.,  0.,  0.,  0., -0., -1.,  0.,  0.,  0.,
          1., -1., -0., -0.,  0., -0., -1.,  1.,  0.,  0., -0.,  0.],
        [-0.,  0., -1., -1.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,
         -1., -0.,  0.,  0., -0., -1.,  1.,  1.,  1.,  0., -0., -1., -0.,
         -0.,  0.,  1.,  1.,  0., -0.,  0., -0.,  1., -0.,  0.,  0.],
        [-0.,  0.,  1., -1., -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0.,
          1.,  0., -0., -0., -0., -1.,  1., -1.,  1.,  0., -0.,  1., -0.,
         -0.,  0., -1., -1.,  0., -0.,  0., -0.,  1.,  0., -0., -0.],
        [-0.,  0.,  0., -0.,  1., -1.,  0.,  0., -0., -0.,  1., -1., -1.,
          0.,  0.,  0., -0., -1.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0.,
         -1.,  1., -0., -0., -0., -0.,  1.,  1.,  0., -0.,  0., -0.],
        [-0., -0.,  1., -1.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,
          1., -0.,  0.,  0., -0., -1., -1., -1.,  1., -0.,  0., -1.,  0.,
          0., -0.,  1.,  1.,  0.,  0., -0.,  0., -1., -0.,  0., -0.],
        [-0.,  0.,  0.,  0., -1.,  1., -0., -0.,  0.,  0.,  1.,  1., -1.,
          0., -0.,  0.,  0., -1.,  0.,  0., -0., -0.,  1., -0., -0., -0.,
         -1.,  1.,  0.,  0.,  0., -0.,  1., -1., -0., -0.,  0., -0.],
        [ 1., -1.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
         -0.,  0.,  0.,  0.,  0., -1., -1., -1.,  1., -0., -1., -1.,  0.,
         -0., -0., -0.,  1.,  0., -1.,  0.,  0.,  0., -0., -0., -0.],
        [ 0.,  0., -0.,  0., -0., -0., -0.,  0.,  1.,  0., -1., -1., -1.,
          0.,  1.,  0.,  0.,  0., -0.,  0., -0., -0., -1.,  0., -0., -0.,
         -0., -1.,  0., -0.,  1., -0.,  1., -1.,  0., -1., -0., -0.],
        [-0.,  0.,  0., -0., -0., -0.,  0., -0.,  1.,  0.,  1.,  1., -1.,
         -0.,  1.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0., -0., -0.,
          0.,  1.,  0.,  0., -1.,  0.,  1., -1., -0.,  1.,  0., -0.],
        [ 1.,  1., -1., -0., -0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,
          0.,  0.,  0.,  0.,  0., -1., -1., -1., -1., -0.,  1., -1., -0.,
          0., -0.,  0., -1., -0., -1.,  0., -0.,  0., -0.,  0.,  0.],
        [ 1.,  1.,  1., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
         -0.,  0.,  0., -0.,  0., -1., -1.,  1., -1.,  0., -1.,  1., -0.,
          0.,  0., -0.,  1., -0.,  1.,  0., -0.,  0.,  0., -0.,  0.],
        [ 0., -0., -0., -0.,  0., -0., -0.,  0., -1., -0.,  1.,  1.,  1.,
         -0.,  1., -0.,  0., -0., -0., -0., -0., -0., -1.,  0., -0.,  0.,
          0., -1.,  0., -0., -1., -0.,  1., -1., -0., -1.,  0., -0.],
        [-1.,  1.,  1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
         -0., -0.,  0.,  0., -0.,  1.,  1., -1., -1., -0., -1., -1., -0.,
         -0., -0., -0.,  1., -0., -1.,  0.,  0.,  0., -0., -0.,  0.],
        [ 0.,  0., -0.,  0.,  0., -0.,  0., -0., -1., -0., -1., -1.,  1.,
          0.,  1., -0.,  0., -0.,  0., -0., -0.,  0.,  1.,  0., -0., -0.,
          0.,  1.,  0.,  0.,  1., -0.,  1., -1.,  0.,  1., -0., -0.],
        [-0.,  0.,  1.,  0., -0., -0., -1., -1.,  0., -0.,  0.,  0.,  0.,
         -0., -0.,  0., -0., -0., -1., -1.,  1., -1., -0.,  0.,  1., -1.,
          0., -0.,  0.,  1., -0., -0., -0.,  0.,  0., -0., -0., -1.],
        [ 0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  1.,  1., -1.,  1.,
         -0., -0.,  1.,  1.,  0., -0., -0., -0.,  0.,  1., -0., -0.,  0.,
         -0., -1.,  0.,  0.,  0., -0.,  1.,  1.,  0., -0., -1.,  0.],
        [ 0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0., -1.,  1.,  1.,  1.,
         -0., -0.,  1., -1.,  0.,  0.,  0.,  0., -0., -1.,  0., -0.,  0.,
         -0., -1.,  0.,  0.,  0., -0.,  1., -1., -0.,  0., -1., -0.],
        [-0., -0.,  1.,  0.,  0.,  0., -1.,  1.,  0.,  0.,  0., -0.,  0.,
         -0.,  0.,  0.,  0.,  0., -1.,  1.,  1., -1.,  0.,  0., -1.,  1.,
         -0., -0., -0., -1.,  0.,  0.,  0.,  0., -0., -0., -0., -1.],
        [ 0., -0.,  1.,  0., -0., -0., -1., -1.,  0.,  0.,  0., -0.,  0.,
         -0., -0.,  0.,  0., -0., -1., -1., -1.,  1., -0.,  0., -1.,  1.,
          0., -0., -0.,  1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1.],
        [-0., -0.,  0., -0.,  0.,  0., -0.,  0., -0., -1., -1., -1.,  1.,
          0.,  0.,  1.,  1., -0., -0.,  0., -0.,  0., -1.,  0., -0.,  0.,
         -0., -1., -0., -0., -0., -0., -1.,  1.,  0.,  0.,  1.,  0.],
        [ 0.,  0.,  1.,  0.,  0.,  0., -1.,  1., -0., -0.,  0.,  0., -0.,
         -0., -0., -0., -0.,  0., -1.,  1., -1.,  1., -0., -0.,  1., -1.,
         -0.,  0.,  0., -1.,  0., -0.,  0., -0.,  0., -0., -0.,  1.],
        [ 0., -0., -0., -0., -0., -0.,  0., -0.,  0.,  1., -1.,  1.,  1.,
         -0., -0.,  1., -1., -0.,  0., -0., -0.,  0.,  1.,  0.,  0., -0.,
         -0., -1.,  0.,  0., -0., -0., -1., -1., -0., -0.,  1.,  0.]],
       dtype=np.float32),
 np.array([[-1., -1.,  0.,  1., -0.,  0.,  1.,  1.,  0.,  0.,  0.,  0., -0.,
         -1., -0., -0., -0.,  0.,  1., -0.,  1.,  0.,  0.,  1.,  0., -1.,
         -0., -0.,  1., -1., -0., -1.,  0., -0., -1.,  0., -0.,  1.],
        [ 1.,  1., -1.,  1., -0., -0.,  1.,  1.,  0., -0.,  0.,  0., -0.,
         -1., -0., -0.,  0.,  0., -0.,  1., -0., -1.,  0., -1., -1., -1.,
          0.,  0.,  1.,  0., -0.,  1.,  0., -0., -1., -0., -0.,  1.],
        [ 0., -0., -0., -0., -1.,  1., -0., -0.,  1., -1.,  0.,  1., -1.,
          0.,  1., -1., -1.,  1., -0., -0., -0., -0., -1.,  0.,  0., -0.,
          1., -0., -0., -0.,  1.,  0., -1., -0.,  0., -1.,  1., -0.],
        [ 0.,  0., -0.,  0., -1.,  1.,  0.,  0.,  1., -1., -1., -0.,  0.,
         -0.,  1., -1., -1.,  1., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,
          1.,  1., -0.,  0.,  1., -0.,  0.,  1., -0., -1.,  1.,  0.],
        [ 0., -0., -0., -0.,  1., -1., -0., -0.,  1.,  1.,  0., -1., -1.,
         -0.,  1., -1.,  1.,  1., -0.,  0.,  0., -0.,  1., -0., -0.,  0.,
          1., -0.,  0.,  0., -1., -0., -1.,  0.,  0.,  1.,  1.,  0.],
        [-0.,  0., -0.,  0., -1.,  1.,  0., -0.,  1., -1.,  1.,  0., -0.,
         -0.,  1.,  1., -1., -1.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,
         -1., -1., -0., -0., -1., -0., -0.,  1.,  0.,  1., -1.,  0.],
        [ 1., -1., -0., -1., -0., -0., -1.,  1., -0.,  0., -0.,  0.,  0.,
          1.,  0., -0., -0.,  0., -1., -0., -1., -0.,  0.,  1.,  0., -1.,
          0.,  0.,  1., -1.,  0.,  1., -0., -0., -1., -0.,  0., -1.],
        [ 1., -1.,  1., -1., -0., -0., -1.,  1.,  0., -0., -0.,  0., -0.,
          1., -0.,  0.,  0.,  0.,  0.,  1.,  0.,  1.,  0.,  1., -1., -1.,
         -0., -0.,  1.,  0.,  0.,  1., -0., -0., -1.,  0.,  0., -1.],
        [ 1., -1.,  0., -1., -0., -0., -1., -1.,  0.,  0., -0.,  0., -0.,
         -1.,  0.,  0., -0.,  0., -1., -0.,  1., -0.,  0., -1., -0., -1.,
         -0.,  0., -1.,  1.,  0., -1., -0.,  0., -1.,  0.,  0.,  1.],
        [-1.,  1.,  1.,  1.,  0., -0., -1., -1.,  0.,  0.,  0., -0., -0.,
          1., -0., -0.,  0., -0., -0., -1.,  0., -1., -0.,  1., -1., -1.,
         -0., -0.,  1.,  0., -0.,  1.,  0., -0.,  1., -0., -0.,  1.],
        [-0., -0.,  0.,  0.,  1.,  1.,  0.,  0., -1., -1., -0., -1.,  1.,
          0.,  1.,  1.,  1., -1.,  0.,  0., -0., -0., -1.,  0., -0.,  0.,
          1., -0.,  0., -0., -1.,  0., -1., -0., -0., -1.,  1., -0.],
        [-0., -0., -0., -0.,  1.,  1., -0.,  0., -1.,  1.,  1.,  0., -0.,
          0.,  1., -1., -1., -1.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,
          1.,  1.,  0., -0., -1., -0.,  0.,  1., -0., -1., -1., -0.],
        [ 1.,  1., -0., -1.,  0., -0., -1.,  1., -0., -0., -0., -0.,  0.,
         -1., -0.,  0., -0.,  0., -1., -0.,  1.,  0.,  0.,  1., -0.,  1.,
         -0., -0.,  1., -1.,  0., -1.,  0., -0.,  1., -0., -0.,  1.],
        [ 1.,  1.,  1.,  1., -0., -0., -1.,  1.,  0., -0.,  0.,  0.,  0.,
          1., -0.,  0.,  0., -0.,  0.,  1., -0., -1., -0.,  1.,  1.,  1.,
         -0., -0., -1.,  0., -0., -1.,  0., -0., -1.,  0., -0.,  1.],
        [-0.,  0.,  0.,  0., -1., -1.,  0., -0., -1.,  1.,  0.,  1.,  1.,
         -0.,  1.,  1., -1., -1.,  0., -0.,  0.,  0.,  1.,  0.,  0., -0.,
          1.,  0., -0., -0.,  1., -0., -1., -0.,  0.,  1.,  1.,  0.],
        [ 0., -0.,  0.,  0., -1., -1.,  0.,  0.,  1., -1.,  1.,  0.,  0.,
          0., -1., -1.,  1., -1.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
          1.,  1., -0., -0., -1., -0., -0., -1., -0., -1., -1.,  0.]],
       dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 3
p = 8
rank = 38

verify_tensor_support_rank_decomposition(decomposition_238, n, m, p, rank)

## Rank-24 support rank decomposition of <2,4,4> over Z


In [ ]:
#@title Data

decomposition_244 = (np.array([[ 1.,  1., -1.,  1., -1.,  1.,  1.,  1., -0., -0.,  0., -1., -1.,
          0.,  1.,  1., -1., -1., -1.,  1.,  1.,  1., -1., -1.],
        [-1.,  1., -1.,  1.,  1.,  1.,  1., -1.,  0.,  0.,  0., -1.,  1.,
         -0.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1.,  1., -1.],
        [ 1., -1., -1.,  1., -1., -1., -1.,  1.,  0.,  0.,  0., -1.,  1.,
          0.,  1., -1.,  1., -1., -1.,  1., -1., -1.,  1.,  1.],
        [-1., -1., -1.,  1.,  1., -1., -1., -1.,  0., -0., -0., -1., -1.,
         -0.,  1., -1., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1.],
        [-1., -1.,  1., -1.,  1.,  1., -1., -1., -1., -1.,  1.,  0.,  1.,
          1., -1., -0.,  1., -1., -0.,  1., -0.,  1.,  1.,  1.],
        [ 1.,  1., -1., -1.,  1., -1., -1.,  1.,  1., -1.,  1., -0., -1.,
         -1., -1.,  0.,  1., -1.,  0., -1., -0.,  1., -1.,  1.],
        [-1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,  1.,  1., -0., -1.,
          1., -1., -0., -1.,  1., -0., -1., -0.,  1.,  1., -1.],
        [-1.,  1.,  1., -1., -1.,  1.,  1.,  1.,  1., -1., -1., -0., -1.,
          1.,  1., -0.,  1., -1.,  0., -1.,  0., -1.,  1.,  1.]],
       dtype=np.float32),
 np.array([[-0.,  0., -0.,  1., -0., -0., -1., -1.,  1.,  1.,  1.,  1.,  0.,
         -1.,  0., -1.,  0.,  0.,  1., -0., -1.,  0.,  1.,  0.],
        [-0., -0.,  0.,  0., -0.,  1.,  0., -0., -1., -1.,  1., -1., -0.,
         -1.,  0., -1., -0.,  1., -1., -1., -1., -1., -0., -0.],
        [ 1., -0.,  0.,  0.,  0., -0., -0.,  0.,  1., -1., -1.,  1., -1.,
         -1.,  1., -1.,  0., -0., -1., -0.,  1., -0.,  0., -1.],
        [ 0., -1.,  1., -0.,  1.,  0.,  0., -0.,  1., -1.,  1.,  1., -0.,
          1., -0.,  1.,  1., -0., -1., -0., -1.,  0., -0.,  0.],
        [-0.,  0., -0., -1.,  0.,  0.,  1., -1.,  1., -1., -1., -1., -0.,
         -1., -0.,  1., -0.,  0.,  1.,  0., -1.,  0.,  1., -0.],
        [-0., -0.,  0.,  0., -0.,  1., -0.,  0., -1.,  1., -1., -1.,  0.,
         -1., -0., -1.,  0., -1.,  1., -1.,  1.,  1.,  0.,  0.],
        [-1.,  0., -0.,  0.,  0.,  0.,  0.,  0., -1., -1., -1.,  1.,  1.,
          1.,  1., -1.,  0.,  0.,  1., -0., -1., -0.,  0., -1.],
        [ 0., -1.,  1., -0., -1.,  0., -0.,  0.,  1.,  1., -1.,  1.,  0.,
          1., -0.,  1., -1.,  0.,  1., -0.,  1., -0.,  0., -0.],
        [ 0.,  0., -0.,  1.,  0.,  0.,  1., -1.,  1.,  1., -1.,  1.,  0.,
          1., -0.,  1.,  0.,  0.,  1., -0.,  1., -0., -1., -0.],
        [-0.,  0.,  0., -0., -0.,  1.,  0., -0.,  1.,  1.,  1.,  1., -0.,
         -1.,  0., -1.,  0., -1.,  1.,  1., -1., -1., -0.,  0.],
        [ 1.,  0.,  0., -0.,  0.,  0.,  0., -0., -1.,  1., -1.,  1.,  1.,
         -1.,  1.,  1., -0., -0., -1., -0., -1., -0., -0.,  1.],
        [ 0.,  1.,  1., -0.,  1., -0.,  0., -0., -1.,  1.,  1.,  1., -0.,
          1.,  0., -1., -1., -0., -1., -0.,  1.,  0., -0., -0.],
        [-0., -0., -0.,  1.,  0.,  0.,  1.,  1., -1.,  1., -1.,  1.,  0.,
         -1., -0.,  1.,  0., -0., -1., -0., -1.,  0.,  1., -0.],
        [-0.,  0., -0., -0.,  0.,  1., -0., -0.,  1., -1., -1.,  1.,  0.,
         -1., -0., -1., -0.,  1., -1.,  1.,  1.,  1., -0., -0.],
        [-1.,  0., -0., -0.,  0.,  0., -0.,  0.,  1.,  1., -1.,  1., -1.,
          1.,  1.,  1.,  0.,  0.,  1., -0.,  1., -0.,  0.,  1.],
        [-0., -1., -1., -0.,  1.,  0., -0.,  0.,  1.,  1.,  1., -1.,  0.,
         -1., -0.,  1., -1., -0., -1.,  0.,  1., -0.,  0., -0.]],
       dtype=np.float32),
 np.array([[-1.,  1., -1., -1.,  1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1.,
          1.,  1.,  1., -1.,  1.,  1.,  1.,  1., -1.,  1.,  1.],
        [-1.,  1., -1.,  1.,  1.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,
          1.,  1.,  1., -1.,  1.,  1.,  1.,  1., -1., -1.,  1.],
        [-1., -1., -1.,  1.,  1.,  1.,  1., -1., -1., -1., -1.,  1.,  1.,
          1.,  1., -1.,  1., -1.,  1., -1., -1., -1.,  1., -1.],
        [ 1.,  1.,  1., -1., -1.,  1., -1.,  1.,  1.,  1.,  1., -1., -1.,
         -1., -1.,  1., -1., -1., -1., -1.,  1., -1., -1.,  1.],
        [ 1., -1.,  1., -1.,  1., -1.,  1., -1., -1.,  1., -1., -1.,  1.,
         -1.,  1., -1., -1.,  1.,  1., -1.,  1., -1., -1.,  1.],
        [-1., -1.,  1., -1.,  1., -1.,  1., -1., -1.,  1., -1., -1., -1.,
         -1., -1., -1., -1.,  1.,  1., -1.,  1., -1., -1., -1.],
        [-1., -1., -1., -1., -1.,  1., -1., -1., -1.,  1.,  1., -1.,  1.,
          1., -1.,  1., -1.,  1.,  1., -1., -1.,  1.,  1.,  1.],
        [-1.,  1.,  1., -1.,  1.,  1., -1., -1., -1.,  1.,  1., -1.,  1.,
          1., -1.,  1.,  1.,  1.,  1., -1., -1.,  1.,  1.,  1.]],
       dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 4
p = 4
rank = 24

verify_tensor_support_rank_decomposition(decomposition_244, n, m, p, rank)

## Rank-31 support rank decomposition of <2,4,5> over Z


In [ ]:
#@title Data

decomposition_245 = (np.array([[-1.,  0., -1., -1., -1., -1., -0., -1.,  1., -1.,  1.,  0.,  1.,
         -1., -1.,  1.,  1.,  1.,  1.,  0., -1., -0., -1.,  1.,  0., -0.,
          1., -1., -1.,  0.,  1.],
        [ 1., -0., -1., -1.,  1.,  1.,  0., -1., -1., -1., -1.,  0.,  1.,
         -1., -1., -1., -1.,  1.,  1.,  0., -1.,  0., -1., -1., -0., -0.,
          1., -1., -1.,  0., -1.],
        [ 1., -0.,  1., -1., -1., -1., -0.,  1.,  1., -1.,  1., -0., -1.,
         -1.,  1., -1.,  1.,  1., -1.,  0.,  1.,  0.,  1.,  1.,  0., -0.,
          1., -1., -1.,  0.,  1.],
        [-1., -0., -1.,  1., -1.,  1., -0., -1.,  1.,  1.,  1.,  0.,  1.,
          1., -1.,  1.,  1., -1., -1., -0.,  1.,  0., -1.,  1.,  0.,  0.,
          1.,  1., -1., -0., -1.],
        [ 1.,  1.,  1.,  1.,  1., -0.,  1.,  1.,  1.,  1., -1.,  1., -1.,
         -1.,  1., -0., -1., -1., -1., -1., -0., -1., -1., -1.,  1., -1.,
          0.,  1.,  1., -1., -1.],
        [-1., -1.,  1., -1., -1.,  0.,  1.,  1.,  1.,  1.,  1.,  1., -1.,
          1., -1., -0.,  1., -1., -1., -1.,  0.,  1.,  1., -1.,  1.,  1.,
         -0.,  1.,  1.,  1.,  1.],
        [ 1., -1.,  1., -1., -1.,  0.,  1.,  1.,  1., -1., -1.,  1.,  1.,
         -1.,  1.,  0.,  1.,  1., -1.,  1.,  0.,  1.,  1.,  1., -1., -1.,
          0.,  1., -1., -1.,  1.],
        [ 1., -1.,  1., -1.,  1., -0.,  1., -1., -1.,  1., -1., -1., -1.,
         -1.,  1., -0.,  1.,  1.,  1.,  1.,  0., -1.,  1., -1.,  1.,  1.,
          0., -1.,  1., -1.,  1.]], dtype=np.float32),
 np.array([[-1., -1.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0.,  0., -1.,  0.,
         -0., -0.,  1.,  0., -0., -1., -0., -1., -0., -0.,  0.,  1.,  0.,
          1., -0., -1., -1., -1.],
        [-1.,  0.,  0.,  0.,  2., -0., -1.,  0., -1., -1., -0.,  0.,  1.,
         -1.,  0., -0.,  0.,  0.,  1., -1.,  0., -1.,  1.,  1., -2.,  1.,
          0., -0., -1., -0., -1.],
        [-1., -0.,  1.,  0., -2., -0.,  1., -0.,  1.,  1., -0., -0., -1.,
         -1.,  0.,  0.,  0.,  2., -1.,  1.,  0.,  1.,  1., -1.,  2., -1.,
         -0., -0.,  1., -0., -1.],
        [ 1., -0.,  1.,  0.,  0., -0.,  1.,  0., -1.,  1.,  0., -0., -1.,
          1., -0., -0., -0.,  0.,  1.,  1., -0.,  1.,  1., -1.,  2., -1.,
         -0.,  0.,  1.,  0., -1.],
        [ 1., -0.,  0., -0., -0., -0., -1., -0.,  1., -1.,  0.,  0.,  1.,
          1., -0., -0., -0., -2., -1., -1., -0., -1.,  1.,  1., -2.,  1.,
          0., -0., -1.,  0., -1.],
        [-1., -1.,  0., -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  1.,  0.,
         -0.,  0.,  1., -0., -0.,  1.,  0.,  1.,  0., -0., -0., -1., -0.,
         -1., -0.,  1., -1., -1.],
        [-1.,  0., -0.,  1.,  2., -0.,  1., -1., -1.,  0.,  0., -2., -0.,
         -0.,  1.,  0., -0., -0.,  1.,  1.,  0., -1., -0.,  1., -0.,  1.,
          0., -1., -1.,  0., -1.],
        [ 1.,  0.,  1., -1.,  2., -0.,  1., -1., -1., -0., -0., -2., -0.,
         -0., -1.,  0., -0.,  2.,  1.,  1.,  0., -1., -0.,  1.,  0.,  1.,
          0., -1., -1., -0.,  1.],
        [-1., -0., -1., -1.,  0., -0., -1.,  1.,  1., -0.,  0.,  2.,  0.,
          0.,  1.,  0.,  0.,  0., -1., -1., -0.,  1.,  0.,  1.,  0., -1.,
          0.,  1., -1., -0.,  1.],
        [ 1.,  0.,  0.,  1.,  0.,  0., -1.,  1.,  1.,  0.,  0.,  2.,  0.,
         -0., -1., -0.,  0., -2., -1., -1.,  0.,  1., -0.,  1.,  0., -1.,
         -0.,  1., -1., -0., -1.],
        [ 1., -1., -0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0.,  1., -0.,
          0.,  0., -1.,  0., -0.,  1., -0.,  1., -0.,  0., -0.,  1.,  0.,
          1., -0., -1.,  1., -1.],
        [-1.,  2., -0.,  1.,  2., -0.,  1., -0., -0., -0., -1.,  0.,  1.,
         -0., -0.,  0.,  1., -0.,  1., -1., -0., -1.,  1., -0., -0., -1.,
         -0., -1., -1.,  0., -1.],
        [-1.,  2.,  1.,  1.,  2.,  0.,  1.,  0., -0., -0., -1., -0., -1.,
          0., -0., -0.,  1., -2., -1., -1., -0., -1.,  1.,  0.,  0., -1.,
         -0.,  1.,  1., -0., -1.],
        [ 1.,  2.,  1.,  1.,  0., -0.,  1.,  0.,  0.,  0.,  1.,  0., -1.,
          0., -0., -0.,  1.,  0.,  1., -1.,  0., -1.,  1., -0.,  0., -1.,
          0., -1.,  1., -0., -1.],
        [-1., -2., -0., -1.,  0.,  0., -1., -0., -0.,  0., -1.,  0., -1.,
         -0., -0., -0., -1.,  2.,  1.,  1., -0.,  1., -1., -0., -0.,  1.,
          0., -1.,  1., -0.,  1.],
        [-1.,  1.,  0.,  0., -0., -1., -0.,  0.,  0.,  0.,  0.,  1., -0.,
          0., -0.,  1.,  0., -0.,  1.,  0.,  1.,  0.,  0., -0.,  1.,  0.,
          1.,  0., -1., -1.,  1.],
        [ 1.,  0.,  0., -0., -2.,  0.,  1.,  1., -0.,  1.,  1.,  0.,  0.,
          1., -1., -0., -1.,  0., -1., -1., -0.,  1., -0., -0., -0.,  1.,
          0.,  0.,  1., -2.,  1.],
        [-1.,  0., -1.,  0.,  2.,  0., -1.,  1.,  0.,  1., -1.,  0.,  0.,
         -1.,  1.,  0.,  1.,  2., -1.,  1., -0., -1., -0.,  0.,  0., -1.,
          0.,  0.,  1.,  2., -1.],
        [-1., -0., -1., -0.,  0.,  0., -1.,  1.,  0., -1., -1., -0., -0.,
         -1.,  1.,  0., -1., -0., -1.,  1., -0., -1.,  0.,  0.,  0., -1.,
          0.,  0., -1.,  2.,  1.],
        [-1., -0., -0., -0., -0.,  0., -1., -1.,  0.,  1., -1.,  0., -0.,
         -1.,  1., -0., -1.,  2.,  1.,  1.,  0., -1.,  0.,  0.,  0., -1.,
          0., -0.,  1.,  2.,  1.]], dtype=np.float32),
 np.array([[-0., -0., -0.,  0., -0.,  1., -0.,  0., -0.,  0., -0.,  0., -0.,
          0., -0., -1.,  0., -0.,  0.,  0., -1.,  0.,  0.,  0., -0., -0.,
         -1., -0., -0., -0.,  0.],
        [ 0.,  1., -0.,  0., -0., -0.,  1., -0.,  0.,  0., -0.,  1.,  0.,
         -0.,  0., -0., -0., -0.,  0., -1., -0., -1.,  0.,  0., -1., -1.,
          0., -0.,  0., -1., -0.],
        [ 1., -1.,  0.,  1., -0.,  1., -0., -1., -1.,  1.,  1.,  1.,  1.,
          1.,  1., -1., -1., -0.,  1.,  0.,  1., -0., -1.,  1., -1.,  0.,
          1., -1., -1.,  1., -1.],
        [ 0.,  0., -2., -1., -1., -0.,  1., -1.,  1.,  1.,  1.,  0.,  1.,
         -1., -1., -0., -1.,  1.,  0., -1.,  0.,  1.,  1., -1., -0.,  1.,
          0., -1., -0., -0., -0.],
        [ 1., -1.,  0.,  1., -0.,  1.,  0.,  1.,  1., -1.,  1., -1., -1.,
          1.,  1., -1., -1., -0., -1., -0., -1., -0., -1., -1.,  1., -0.,
         -1.,  1.,  1.,  1., -1.],
        [ 0., -0.,  2.,  1., -1., -0.,  1.,  1.,  1., -1.,  1., -0., -1.,
          1.,  1., -0., -1., -1.,  0., -1., -0., -1., -1., -1.,  0., -1.,
          0.,  1., -0.,  0., -0.],
        [-1., -1.,  0.,  1., -0.,  1., -0., -1., -1., -1., -1.,  1., -1.,
         -1., -1.,  1., -1., -0.,  1., -0.,  1., -0., -1., -1.,  1., -0.,
         -1., -1.,  1., -1., -1.],
        [ 0., -0.,  2., -1.,  1., -0., -1.,  1., -1.,  1., -1., -0., -1.,
         -1.,  1.,  0.,  1.,  1., -0., -1.,  0., -1., -1.,  1.,  0.,  1.,
          0., -1.,  0.,  0., -0.],
        [ 1.,  1., -0., -1.,  0., -1.,  0., -1., -1., -1.,  1.,  1., -1.,
          1.,  1., -1.,  1.,  0.,  1., -0.,  1.,  0.,  1., -1.,  1., -0.,
         -1., -1.,  1.,  1.,  1.],
        [ 0., -0.,  2., -1., -1., -0., -1.,  1.,  1.,  1.,  1.,  0., -1.,
         -1.,  1., -0., -1.,  1.,  0., -1.,  0.,  1., -1., -1.,  0., -1.,
          0., -1.,  0., -0., -0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 4
p = 5
rank = 31

verify_tensor_support_rank_decomposition(decomposition_245, n, m, p, rank)

## Rank-37 support rank decomposition of <2,4,6> over C


In [ ]:
#@title Data

decomposition_246 = (np.array([[ 1.-0.j,  0.-0.j,  0.-0.j,  0.+1.j,  1.+0.j, -0.+0.j, -0.+0.j,
          0.-0.j, -1.+0.j, -0.-0.j,  1.-0.j,  0.-0.j,  0.+1.j,  0.+0.j,
          1.-0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.-0.j,  1.-0.j,  0.+0.j,
          1.-0.j, -0.-0.j,  0.+0.j, -0.-0.j,  0.+0.j,  1.-0.j,  0.+1.j,
          0.-1.j,  0.-0.j,  0.+0.j, -1.-0.j, -0.+0.j,  1.-0.j, -0.+0.j,
         -0.+0.j,  0.+0.j],
        [ 1.+0.j, -1.-0.j, -1.-0.j,  0.-0.j,  1.+0.j,  1.-0.j,  1.-0.j,
          0.+0.j,  0.+0.j,  0.-0.j, -0.-0.j, -1.-0.j, -0.-0.j,  0.-1.j,
          1.+0.j,  0.+0.j, -0.+0.j,  0.-1.j,  0.+1.j,  0.-0.j, -0.+0.j,
         -0.-0.j,  0.-0.j, -0.-0.j,  0.+0.j,  0.-1.j,  1.+0.j,  0.-0.j,
          0.-1.j, -1.-0.j,  0.+0.j, -1.-0.j,  1.-0.j,  1.+0.j,  1.+0.j,
         -0.-0.j,  0.+0.j],
        [-1.+0.j, -0.-0.j, -0.-0.j,  0.+0.j, -1.+0.j,  0.+0.j, -1.+0.j,
         -0.-0.j, -0.-0.j,  0.-1.j,  0.+0.j,  1.+0.j,  0.+0.j,  0.-0.j,
          0.+0.j,  0.-0.j,  0.-0.j,  0.-0.j, -0.+0.j,  0.+0.j,  0.+1.j,
          0.+0.j,  0.+0.j,  0.+1.j, -0.-1.j,  0.+1.j, -1.+0.j, -0.-1.j,
         -0.+1.j,  1.+0.j,  0.-0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
         -1.+0.j,  1.+0.j],
        [-0.-0.j,  0.-0.j,  0.-0.j,  0.+0.j, -0.-0.j, -0.+0.j,  1.+0.j,
         -1.-0.j, -0.+0.j, -0.-0.j,  1.+0.j, -1.-0.j,  0.+0.j,  0.-1.j,
          0.-0.j,  0.-1.j,  0.-1.j,  0.+0.j, -0.-0.j, -0.-0.j,  0.-1.j,
          0.-0.j,  0.+1.j,  0.-0.j, -0.-0.j, -0.+0.j, -0.-0.j,  0.-0.j,
         -0.+0.j, -1.-0.j,  0.-1.j, -1.-0.j,  1.+0.j, -0.-0.j,  1.+0.j,
         -0.-0.j,  0.+0.j],
        [ 1.-0.j, -1.+0.j,  0.-0.j,  0.+1.j,  1.-0.j, -0.+0.j,  1.-0.j,
         -1.-0.j, -1.+0.j, -0.-0.j,  1.-0.j,  0.-0.j,  0.+1.j,  0.+0.j,
          1.-0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.-0.j,  1.-0.j,  0.+0.j,
         -0.+0.j, -0.-0.j,  0.+0.j, -0.-0.j,  0.+0.j,  1.-0.j,  0.+1.j,
          0.+0.j,  0.-0.j,  0.-0.j, -1.-0.j, -0.+0.j,  1.-0.j, -0.+0.j,
         -0.+0.j, -1.-0.j],
        [ 1.+0.j, -1.-0.j, -1.-0.j,  0.-0.j, -0.-0.j,  1.-0.j,  1.-0.j,
          0.-0.j, -1.-0.j, -0.+1.j, -0.+0.j, -1.-0.j,  0.-0.j,  0.-1.j,
          1.+0.j,  0.+0.j,  0.+0.j,  0.-1.j,  0.+0.j, -0.+0.j,  0.+0.j,
          0.+0.j,  0.-0.j, -0.+0.j, -0.-0.j,  0.-1.j,  1.+0.j,  0.-0.j,
          0.-1.j,  0.-0.j,  0.-1.j, -1.-0.j,  1.-0.j,  1.+0.j,  1.+0.j,
          0.+0.j,  0.+0.j],
        [ 0.+1.j,  0.+0.j,  0.+0.j, -1.+0.j,  0.+1.j,  0.-0.j,  0.+1.j,
          0.+0.j,  0.+0.j, -1.-0.j,  0.-0.j,  0.-1.j,  0.+0.j, -0.-0.j,
         -0.-0.j, -0.-0.j,  1.+0.j,  1.+0.j,  0.+0.j,  0.-0.j,  1.-0.j,
          0.-0.j,  0.+0.j, -0.-0.j, -1.+0.j,  1.-0.j, -0.-0.j, -1.+0.j,
          1.+0.j,  0.-1.j, -0.-0.j,  0.-1.j,  0.+1.j,  0.+1.j,  0.-0.j,
          0.+1.j,  0.-1.j],
        [ 0.-0.j, -0.+0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.+1.j,  0.+1.j,
         -0.-1.j,  0.+0.j,  0.+0.j,  0.+1.j,  0.+0.j,  0.+0.j,  1.+0.j,
          0.-0.j,  1.-0.j,  1.+0.j, -0.-0.j,  0.+0.j,  0.+1.j,  1.-0.j,
          0.+0.j,  0.-0.j, -0.+0.j, -1.+0.j, -0.-0.j,  0.-0.j,  0.+0.j,
         -0.-0.j,  0.-1.j,  1.+0.j, -0.-1.j,  0.+1.j,  0.+1.j,  0.+1.j,
         -0.-0.j, -0.+0.j]], dtype=np.complex64),
 np.array([[ 0.+1.j,  1.-0.j,  1.-0.j,  0.+0.j,  0.-0.j, -0.-0.j,  0.+0.j,
          0.-0.j,  0.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+1.j,  0.-0.j,
          1.-0.j,  0.+0.j, -0.-0.j, -0.-0.j,  0.-0.j,  0.-0.j,  0.-0.j,
         -1.+0.j,  0.-0.j,  0.-0.j,  0.-0.j,  0.-1.j,  0.+0.j,  0.+1.j,
         -1.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+0.j,
          1.-0.j, -1.+0.j],
        [-0.-1.j, -1.+0.j, -1.+0.j,  0.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,
          0.-0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+1.j,  0.-0.j,
         -1.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-0.j,
         -1.+0.j,  0.-0.j,  0.-0.j,  0.-0.j,  0.+1.j,  0.-0.j,  0.+1.j,
          1.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,
          1.-0.j, -1.+0.j],
        [ 0.+1.j,  0.+0.j,  0.+0.j, -0.-0.j,  0.+0.j,  0.-0.j,  1.+0.j,
          1.-0.j,  0.+0.j,  0.+0.j,  1.-0.j,  0.+0.j,  0.-1.j,  0.-0.j,
          0.+0.j,  0.+1.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-0.j,
          1.+0.j,  0.-0.j, -0.+0.j,  0.+0.j,  0.-1.j,  0.-0.j, -0.+0.j,
         -1.-0.j,  0.+0.j,  0.-0.j,  1.-0.j,  1.-0.j,  0.+0.j,  0.+0.j,
          0.+0.j, -0.-0.j],
        [ 0.+0.j, -1.-0.j, -1.+0.j, -0.+0.j,  0.+0.j,  0.-0.j, -1.-0.j,
         -1.+0.j,  0.+0.j,  0.+0.j, -1.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,
         -1.+0.j,  0.-1.j, -0.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-0.j,
          0.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j, -0.+1.j,
         -0.-0.j,  0.+0.j,  0.+0.j, -1.+0.j, -1.-0.j,  0.+0.j,  0.+0.j,
          1.+0.j, -1.+0.j],
        [ 0.+1.j,  0.+0.j,  0.+0.j, -0.+0.j,  0.+0.j,  0.-0.j,  1.+0.j,
         -1.+0.j,  0.+0.j,  0.+0.j, -1.-0.j,  0.+0.j,  0.+1.j,  0.-0.j,
          0.+0.j, -0.-1.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-0.j,
         -1.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-1.j,  0.-0.j,  0.-0.j,
         -1.-0.j,  0.+0.j,  0.-0.j,  1.-0.j,  1.-0.j,  0.+0.j,  0.+0.j,
         -0.-0.j,  0.-0.j],
        [ 0.+0.j, -1.-0.j, -1.+0.j,  0.-0.j,  0.+0.j,  0.+0.j, -1.-0.j,
          1.-0.j,  0.+0.j,  0.+0.j,  1.+0.j, -0.+0.j,  0.+0.j,  0.+0.j,
         -1.+0.j,  0.+1.j,  0.+0.j,  0.+0.j,  0.-0.j, -0.-0.j,  0.-0.j,
          0.+0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-0.j,  0.-1.j,
         -0.-0.j,  0.+0.j,  0.-0.j, -1.+0.j, -1.+0.j,  0.+0.j, -0.-0.j,
         -1.-0.j,  1.+0.j],
        [ 1.-0.j,  0.-0.j,  0.+1.j, -0.-0.j,  0.+1.j,  0.-0.j,  0.+0.j,
          0.+0.j,  0.-1.j,  1.+0.j,  0.-0.j,  0.-0.j, -1.-0.j,  0.-0.j,
          0.-1.j,  0.+0.j, -0.+0.j,  0.+0.j,  1.+0.j,  0.-0.j,  0.-0.j,
          0.+0.j,  0.-0.j,  0.-0.j,  0.+0.j,  1.+0.j,  0.-0.j, -1.-0.j,
         -0.-0.j,  0.-0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-0.j,
          0.-1.j,  0.+0.j],
        [ 0.+1.j,  0.+0.j, -1.+0.j,  0.-0.j, -1.+0.j, -0.-0.j,  0.+0.j,
          0.+0.j, -1.-0.j, -0.-1.j,  0.+0.j, -0.+0.j, -0.+1.j,  0.-0.j,
         -1.+0.j,  0.-0.j, -0.+0.j, -0.+0.j, -0.+1.j,  0.+0.j,  0.-0.j,
          0.+0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-1.j,  0.+0.j,  0.-1.j,
          0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.+0.j,
         -1.-0.j,  0.+0.j],
        [ 0.+1.j,  0.+0.j,  0.+0.j,  0.-0.j, -1.+0.j,  0.-0.j,  0.+0.j,
          0.-0.j, -1.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+1.j,  0.+1.j,
         -1.+0.j,  0.+1.j, -0.+0.j, -0.-0.j,  0.-0.j,  0.-0.j,  0.+1.j,
          0.+0.j,  0.-0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -0.-1.j,
          0.+0.j, -1.+0.j,  0.+1.j,  0.+0.j, -1.+0.j,  0.+0.j,  0.+0.j,
          0.-0.j,  0.+0.j],
        [ 0.+0.j,  0.+0.j,  0.-1.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,
          0.-0.j,  0.+0.j,  1.+0.j,  0.-0.j,  0.-0.j,  0.-0.j, -1.-0.j,
          0.+0.j, -1.-0.j, -0.-0.j,  0.-0.j, -1.-0.j,  0.-0.j, -1.-0.j,
          0.-0.j, -0.-0.j,  0.+0.j,  0.-0.j,  1.+0.j, -0.-0.j, -0.-0.j,
         -0.+0.j,  0.-1.j, -1.-0.j,  0.+0.j,  0.-1.j,  0.+0.j,  0.+0.j,
          0.-1.j,  0.-0.j],
        [ 1.-0.j,  0.+0.j,  0.+0.j,  0.-0.j,  0.+1.j,  0.-0.j,  0.+0.j,
          0.+0.j,  0.-1.j, -0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j, -1.-0.j,
          0.-1.j, -1.+0.j,  0.+0.j, -0.+0.j, -0.-0.j,  0.-0.j,  1.+0.j,
          0.+0.j,  0.-0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j, -1.-0.j,
         -0.-0.j,  0.+1.j, -1.+0.j,  0.+0.j,  0.+1.j,  0.-0.j,  0.+0.j,
          0.+0.j,  0.+0.j],
        [-0.-0.j,  0.-0.j,  0.+1.j,  0.-0.j,  0.-0.j,  0.-0.j,  0.-0.j,
          0.-0.j,  0.+0.j,  1.-0.j,  0.-0.j,  0.+0.j,  0.-0.j, -1.-0.j,
          0.+0.j, -1.+0.j,  0.+0.j,  0.-0.j,  1.+0.j,  0.-0.j,  1.-0.j,
          0.+0.j,  0.-0.j, -0.-0.j,  0.-0.j,  1.-0.j, -0.-0.j,  0.+0.j,
         -0.+0.j,  0.+1.j, -1.-0.j,  0.+0.j,  0.+1.j,  0.+0.j,  0.+0.j,
          0.-1.j,  0.+0.j],
        [ 1.-0.j,  0.+0.j,  0.-1.j,  1.-0.j,  0.+0.j, -0.-0.j,  0.+0.j,
          0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j,  0.-0.j,
          0.-1.j,  0.+0.j,  0.+0.j,  1.-0.j,  0.-0.j,  0.+0.j, -0.-0.j,
          0.-0.j,  0.+0.j,  1.+0.j, -0.+0.j, -1.-0.j, -1.+0.j, -1.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+0.j,
          0.+1.j,  0.+0.j],
        [-1.+0.j,  0.-0.j,  0.+1.j,  1.-0.j,  0.+0.j, -0.-0.j,  0.-0.j,
          0.-0.j,  0.-0.j,  0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j, -0.-0.j,
          0.+1.j,  0.+0.j,  0.+0.j, -1.+0.j,  0.-0.j,  0.+0.j, -0.-0.j,
          0.-0.j,  0.-0.j,  1.+0.j,  0.+0.j,  1.-0.j,  1.-0.j, -1.+0.j,
         -0.+0.j,  0.+0.j, -0.-0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.-0.j,
          0.+1.j,  0.+0.j],
        [-1.-0.j,  0.-0.j,  0.+0.j,  1.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,
          0.+0.j,  0.-0.j,  0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j,  1.-0.j,
          0.+1.j, -1.+0.j, -1.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  1.-0.j,
          0.+0.j,  0.-0.j, -0.-0.j,  0.+0.j,  0.-0.j,  1.-0.j, -1.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-1.j,  0.+0.j,  0.+1.j,
          0.-0.j,  0.+0.j],
        [ 0.-0.j,  0.+0.j,  0.+1.j, -0.-0.j,  0.+0.j,  0.+0.j,  0.-0.j,
          0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  1.-0.j,
          0.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,  0.-0.j,  0.+0.j,  1.-0.j,
          0.-0.j, -0.-0.j,  1.+0.j, -0.+0.j,  1.-0.j, -0.-0.j,  0.+0.j,
         -0.+0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-1.j,  0.+0.j,  0.+1.j,
          0.+1.j,  0.+0.j],
        [-1.-0.j,  0.-0.j,  0.+0.j, -1.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,
          0.-0.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  1.-0.j,  1.-0.j,
          0.+1.j,  1.+0.j,  1.-0.j,  0.-0.j,  0.-0.j,  0.-0.j, -1.-0.j,
          0.-0.j, -0.-0.j,  0.+0.j,  0.-0.j, -0.-0.j,  1.+0.j,  1.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.-1.j,  0.+0.j,  0.+1.j,
          0.+0.j,  0.-0.j],
        [-0.+0.j,  0.-0.j,  0.-1.j, -0.-0.j,  0.+0.j, -0.-0.j,  0.-0.j,
          0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.+0.j,  0.-0.j, -1.+0.j,
          0.+0.j, -1.+0.j, -1.+0.j,  1.-0.j,  0.-0.j,  0.-0.j,  1.-0.j,
          0.+0.j,  0.-0.j,  1.+0.j, -0.+0.j, -1.-0.j,  0.-0.j,  0.-0.j,
         -0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  0.+1.j,  0.+0.j, -0.-1.j,
          0.+1.j,  0.+0.j],
        [ 0.-1.j,  0.-0.j,  1.+0.j,  0.-0.j, -0.-0.j, -1.-0.j,  0.-0.j,
          0.-0.j,  0.-0.j,  0.+0.j, -1.-0.j, -0.-0.j,  0.-1.j,  0.-1.j,
          0.-0.j,  0.-0.j,  0.+0.j,  0.+0.j,  0.-0.j, -1.-0.j,  0.-1.j,
          0.-0.j,  0.-0.j,  0.-0.j,  0.-1.j,  0.-0.j,  0.-0.j,  0.+0.j,
         -0.-0.j,  0.+0.j,  0.+0.j,  1.+0.j,  0.-0.j,  1.+0.j,  0.-0.j,
          1.+0.j,  0.+0.j],
        [ 0.+1.j,  0.+0.j, -1.+0.j,  0.-0.j,  0.+0.j,  1.+0.j,  0.+0.j,
          0.-0.j,  0.+0.j,  0.+0.j, -1.-0.j,  0.+0.j,  0.-1.j,  0.+1.j,
          0.+0.j,  0.-0.j, -0.-0.j, -0.+0.j,  0.-0.j, -1.-0.j,  0.-1.j,
          0.-0.j,  0.-0.j,  0.-0.j,  0.-1.j,  0.-0.j,  0.+0.j, -0.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j, -1.+0.j, -0.-0.j, -1.+0.j,  0.+0.j,
          1.-0.j,  0.+0.j],
        [ 0.+1.j,  0.+0.j,  0.+0.j, -0.-0.j,  0.+0.j, -0.-0.j,  0.+0.j,
          0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j, -1.+0.j,  0.-1.j,  0.-0.j,
          0.+0.j,  0.+1.j,  0.+0.j,  0.+0.j,  0.-0.j, -1.+0.j,  0.+0.j,
         -0.+0.j, -0.+1.j,  0.-0.j,  0.+0.j,  0.+1.j,  0.+0.j,  0.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j, -1.+0.j,  1.-0.j, -1.+0.j,  0.+0.j,
         -0.-0.j,  0.+0.j],
        [ 0.-0.j,  0.+0.j, -1.+0.j, -0.-0.j,  0.+0.j,  1.+0.j,  0.+0.j,
          0.-0.j,  0.+0.j,  0.+0.j,  0.-0.j, -1.+0.j,  0.+0.j,  0.+1.j,
          0.+0.j,  0.+1.j,  0.-0.j,  0.+0.j,  0.-0.j,  0.-0.j,  0.-1.j,
          0.-0.j, -0.+1.j,  0.-0.j, -0.-1.j,  0.+1.j,  0.-0.j, -0.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  1.+0.j,  0.+0.j,  0.+0.j,
          1.-0.j,  0.+0.j],
        [ 0.+1.j,  0.+0.j,  0.+0.j,  0.-0.j,  0.-0.j, -0.-0.j,  0.+0.j,
          0.-0.j, -0.+0.j,  0.+0.j,  1.+0.j, -1.+0.j,  0.+1.j,  0.-0.j,
          0.+0.j,  0.-1.j, -0.+0.j, -0.+0.j,  0.-0.j,  1.+0.j,  0.-0.j,
          0.-0.j,  0.-1.j,  0.+0.j,  0.-0.j,  0.+1.j,  0.-0.j,  0.-0.j,
         -0.-0.j,  0.+0.j,  0.+0.j, -1.+0.j,  1.+0.j, -1.+0.j,  0.+0.j,
          0.+0.j,  0.-0.j],
        [ 0.-0.j,  0.+0.j, -1.+0.j,  0.-0.j,  0.+0.j,  1.-0.j,  0.+0.j,
          0.-0.j,  0.+0.j, -0.+0.j, -0.+0.j, -1.+0.j,  0.-0.j,  0.+1.j,
          0.+0.j,  0.-1.j,  0.+0.j,  0.+0.j,  0.-0.j, -0.+0.j,  0.+1.j,
          0.+0.j,  0.-1.j, -0.+0.j,  0.+1.j,  0.+1.j,  0.-0.j, -0.+0.j,
         -0.-0.j,  0.+0.j,  0.-0.j,  0.+0.j,  1.+0.j,  0.+0.j,  0.+0.j,
         -1.+0.j,  0.+0.j]], dtype=np.complex64),
 np.array([[ 0.+0.j, -1.+0.j,  0.-0.j,  1.-0.j,  1.+0.j,  1.-0.j, -1.-0.j,
         -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j,  1.-0.j,  0.-0.j, -1.+0.j,
         -1.+0.j,  0.-0.j, -1.+0.j, -1.+0.j,  1.-0.j,  1.+0.j, -1.+0.j,
          1.+0.j,  1.+0.j,  1.-0.j, -1.+0.j, -1.+0.j,  0.+1.j,  1.+0.j,
          0.+1.j,  1.+0.j,  1.+0.j, -1.+0.j,  0.+0.j, -1.-0.j, -1.-0.j,
          0.+0.j,  1.-0.j],
        [ 0.-1.j, -1.+0.j,  1.+0.j, -1.-0.j,  1.+0.j, -1.+0.j, -1.-0.j,
         -1.+0.j, -1.+0.j, -1.-0.j, -0.-0.j, -1.+0.j, -1.-0.j, -0.-0.j,
          0.+0.j, -1.-0.j,  1.+0.j,  1.-0.j,  1.+0.j, -1.-0.j, -0.-0.j,
          1.-0.j, -1.-0.j, -1.+0.j,  1.-0.j,  0.-0.j,  0.-1.j,  0.+0.j,
          0.+1.j,  1.+0.j,  1.-0.j,  0.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
         -1.+0.j,  1.+0.j],
        [ 0.-0.j, -1.+0.j,  0.-0.j, -1.+0.j, -1.-0.j,  1.-0.j, -1.+0.j,
          1.-0.j, -1.-0.j, -1.+0.j,  1.+0.j,  1.-0.j,  0.+0.j, -1.+0.j,
         -1.+0.j,  0.+0.j,  1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j,  1.-0.j,
         -1.+0.j, -1.-0.j, -1.-0.j,  1.-0.j, -1.+0.j,  0.+1.j, -1.-0.j,
          0.+1.j, -1.+0.j,  1.-0.j, -1.+0.j,  0.-0.j, -1.+0.j, -1.+0.j,
          0.-0.j, -1.+0.j],
        [ 0.-1.j, -1.-0.j,  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.-0.j,
          1.+0.j,  1.-0.j,  1.+0.j,  0.+0.j, -1.-0.j,  1.+0.j, -0.+0.j,
          0.+0.j,  1.+0.j, -1.-0.j,  1.+0.j,  1.+0.j,  1.+0.j,  0.-0.j,
         -1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j,  0.-0.j,  0.-1.j,  0.-0.j,
          0.+1.j,  1.-0.j, -1.-0.j,  0.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
          1.+0.j, -1.-0.j],
        [-0.+0.j,  1.+0.j,  0.-0.j,  1.-0.j,  1.-0.j,  1.-0.j,  1.+0.j,
         -1.-0.j,  1.-0.j, -1.-0.j, -1.-0.j,  1.-0.j, -0.-0.j, -1.+0.j,
          1.+0.j,  0.-0.j,  1.+0.j, -1.-0.j, -1.+0.j,  1.+0.j,  1.-0.j,
         -1.-0.j, -1.+0.j, -1.-0.j,  1.-0.j, -1.+0.j,  0.-1.j,  1.+0.j,
          0.+1.j, -1.+0.j,  1.-0.j,  1.-0.j,  0.+0.j,  1.+0.j, -1.+0.j,
          0.+0.j,  1.+0.j],
        [ 0.-1.j,  1.-0.j, -1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
         -1.+0.j,  1.-0.j, -1.-0.j,  0.+0.j,  1.+0.j,  1.+0.j,  0.-0.j,
         -0.-0.j, -1.+0.j,  1.+0.j, -1.+0.j, -1.-0.j,  1.+0.j, -0.+0.j,
         -1.+0.j, -1.-0.j, -1.-0.j,  1.-0.j,  0.+0.j,  0.-1.j,  0.-0.j,
          0.+1.j, -1.-0.j,  1.-0.j,  0.-0.j, -1.-0.j,  1.+0.j, -1.-0.j,
         -1.-0.j,  1.-0.j],
        [-0.-0.j, -1.-0.j, -0.-0.j, -1.+0.j, -1.-0.j,  1.-0.j,  1.-0.j,
         -1.+0.j, -1.+0.j,  1.+0.j, -1.+0.j, -1.-0.j,  0.+0.j, -1.+0.j,
         -1.+0.j,  0.+0.j,  1.+0.j,  1.-0.j,  1.+0.j,  1.-0.j,  1.+0.j,
          1.-0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.-0.j,  0.+1.j, -1.+0.j,
         -0.-1.j, -1.-0.j,  1.+0.j,  1.-0.j,  0.+0.j,  1.-0.j, -1.-0.j,
         -0.+0.j, -1.+0.j],
        [ 0.-1.j,  1.-0.j, -1.-0.j,  1.-0.j,  1.-0.j,  1.-0.j, -1.-0.j,
          1.-0.j,  1.-0.j, -1.-0.j,  0.+0.j, -1.-0.j,  1.+0.j, -0.+0.j,
         -0.-0.j,  1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j,  1.-0.j,  0.+0.j,
         -1.+0.j,  1.+0.j, -1.-0.j,  1.-0.j,  0.-0.j,  0.-1.j,  0.+0.j,
          0.+1.j,  1.-0.j, -1.+0.j,  0.+0.j,  1.+0.j,  1.-0.j,  1.+0.j,
         -1.-0.j,  1.+0.j],
        [ 0.-0.j, -1.-0.j, -0.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.-0.j,
         -1.+0.j, -1.+0.j,  1.+0.j, -1.-0.j, -1.-0.j,  0.+0.j,  1.-0.j,
         -1.-0.j,  0.+0.j,  1.-0.j,  1.+0.j, -1.-0.j,  1.+0.j,  1.-0.j,
         -1.-0.j, -1.-0.j, -1.+0.j,  1.-0.j,  1.-0.j,  0.+1.j,  1.+0.j,
         -0.-1.j, -1.-0.j, -1.+0.j, -1.+0.j, -0.+0.j, -1.-0.j,  1.+0.j,
          0.-0.j,  1.-0.j],
        [ 0.-1.j,  1.-0.j, -1.-0.j, -1.-0.j,  1.+0.j,  1.-0.j,  1.+0.j,
          1.-0.j, -1.+0.j,  1.+0.j, -0.-0.j,  1.+0.j, -1.-0.j,  0.+0.j,
          0.-0.j,  1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j,  0.+0.j,
          1.-0.j,  1.+0.j,  1.-0.j, -1.+0.j,  0.+0.j,  0.-1.j,  0.+0.j,
          0.+1.j, -1.-0.j, -1.+0.j, -0.-0.j, -1.-0.j,  1.+0.j, -1.-0.j,
          1.+0.j, -1.+0.j],
        [ 0.-0.j,  1.+0.j,  0.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j,
         -1.-0.j,  1.-0.j, -1.-0.j, -1.-0.j,  1.-0.j,  0.-0.j,  1.-0.j,
          1.+0.j,  0.+0.j,  1.-0.j, -1.+0.j,  1.+0.j,  1.+0.j,  1.-0.j,
          1.+0.j,  1.+0.j,  1.+0.j,  1.-0.j, -1.+0.j,  0.-1.j, -1.+0.j,
          0.+1.j, -1.-0.j, -1.-0.j, -1.+0.j,  0.-0.j, -1.-0.j,  1.+0.j,
          0.+0.j, -1.+0.j],
        [-0.+1.j, -1.+0.j,  1.-0.j,  1.-0.j, -1.+0.j, -1.+0.j,  1.-0.j,
          1.-0.j,  1.+0.j, -1.-0.j,  0.+0.j,  1.+0.j,  1.+0.j,  0.+0.j,
          0.+0.j,  1.-0.j, -1.+0.j,  1.-0.j,  1.-0.j,  1.-0.j,  0.+0.j,
         -1.+0.j,  1.-0.j, -1.+0.j,  1.-0.j, -0.+0.j,  0.+1.j,  0.+0.j,
         -0.-1.j, -1.+0.j, -1.+0.j,  0.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
         -1.+0.j,  1.-0.j]], dtype=np.complex64))

In [ ]:
#@title Verification

n = 2
m = 4
p = 6
rank = 37

verify_tensor_support_rank_decomposition(decomposition_246, n, m, p, rank)

## Rank-42 support rank decomposition of <2,4,7> over Z


In [ ]:
#@title Data

decomposition_247 = (np.array([[-0., -1., -1., -1.,  1.,  1.,  1., -0., -0., -1.,  1., -1.,  1.,
         1.,  1., -1.,  1., -0.,  1.,  1., -1.,  1.,  1.,  1.,  1., -0.,
        -1., -1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1., -1., -1.],
       [-0., -1.,  1.,  1.,  1., -1.,  1.,  0., -0.,  1.,  1.,  1., -1.,
        -1., -1.,  1.,  1.,  0.,  1.,  1.,  1., -1.,  1.,  1., -1., -0.,
        -1.,  1.,  1., -0., -1., -1.,  1., -1.,  1.,  1., -1.,  1.,  1.,
        -1., -1., -1.],
       [-0., -1.,  1., -1.,  1.,  1.,  1., -0., -0., -1.,  1., -1., -1.,
         1., -1.,  1., -1., -0.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  0.,
        -1., -1., -1., -0., -1., -1.,  1.,  1., -1., -1.,  1., -1.,  1.,
         1., -1.,  1.],
       [-0., -1.,  1.,  1.,  1., -1.,  1.,  0., -0.,  1.,  1.,  1.,  1.,
        -1.,  1.,  1.,  1.,  0.,  1.,  1., -1.,  1.,  1., -1., -1.,  0.,
         1.,  1., -1.,  0.,  1.,  1.,  1., -1., -1., -1., -1., -1.,  1.,
        -1., -1., -1.],
       [ 1., -1.,  1., -1.,  1.,  1., -0., -1.,  1.,  1., -0.,  1.,  1.,
        -1., -0., -1.,  1., -1.,  1., -1., -1., -1., -1., -1.,  1., -1.,
        -1.,  1.,  1., -1., -1.,  1., -1.,  0.,  0., -1.,  0., -1.,  1.,
         1.,  1.,  1.],
       [ 1.,  1.,  1.,  1., -1.,  1.,  0., -1., -1.,  1., -0.,  1.,  1.,
         1., -0., -1., -1.,  1., -1.,  1.,  1., -1.,  1., -1., -1.,  1.,
         1.,  1.,  1., -1.,  1.,  1., -1., -0., -0.,  1.,  0.,  1.,  1.,
         1.,  1., -1.],
       [-1., -1., -1.,  1., -1.,  1., -0., -1.,  1., -1.,  0.,  1., -1.,
         1.,  0.,  1., -1.,  1.,  1.,  1.,  1., -1., -1.,  1.,  1.,  1.,
        -1.,  1.,  1.,  1., -1., -1., -1., -0., -0.,  1.,  0.,  1., -1.,
        -1., -1., -1.],
       [-1.,  1., -1., -1.,  1., -1.,  0., -1., -1., -1.,  0.,  1., -1.,
        -1.,  0.,  1.,  1., -1.,  1., -1., -1., -1., -1.,  1., -1., -1.,
         1., -1.,  1.,  1.,  1.,  1., -1.,  0.,  0.,  1.,  0., -1., -1.,
        -1., -1.,  1.]], dtype=np.float32), np.array([[ 0.,  0.,  1.,  1.,  0., -1., -0., -0.,  0.,  0., -0., -0.,  0.,
         1., -0., -1., -1., -0.,  1., -0., -0., -0.,  1., -0.,  0., -0.,
        -0., -1.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,
        -0., -1.,  1.],
       [-0., -1., -1., -1.,  0., -1., -1.,  1.,  1., -0.,  0., -1., -1.,
         1., -1., -1., -1., -0.,  1.,  0., -0.,  0., -1.,  0.,  0., -1.,
        -0.,  1., -0.,  1.,  0., -0., -0., -0., -1., -0.,  1., -1., -1.,
        -0., -1., -1.],
       [-0., -0., -1.,  0., -0.,  1.,  0., -0.,  0., -0.,  0.,  0., -0.,
        -1., -0., -0.,  1.,  0.,  0.,  0.,  0., -1., -1.,  0., -0., -0.,
        -1., -0.,  1., -0.,  1., -1., -0.,  0.,  0.,  1.,  0.,  0., -1.,
         0., -0.,  0.],
       [ 0.,  1., -0.,  1., -0., -0.,  1., -1., -1.,  0., -0.,  1.,  1.,
         0.,  1.,  1., -0.,  0., -1., -0.,  0.,  1., -0., -0., -0.,  1.,
        -1., -1., -1., -1.,  1., -1.,  0.,  0.,  1., -1., -1.,  1., -0.,
         0.,  1.,  1.],
       [-0., -1., -1.,  0.,  0., -1., -1.,  1.,  1., -0.,  0., -1., -1.,
         1., -1.,  0., -1., -0., -0.,  0., -0., -1., -1.,  0.,  0., -1.,
         1., -0.,  1.,  1., -1.,  1., -0., -0., -1.,  1.,  1., -1., -1.,
        -0.,  0.,  0.],
       [-0., -0.,  0., -1., -0., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,
         0.,  0.,  1., -0.,  0., -1.,  0., -0., -1.,  0., -0., -0., -0.,
        -1.,  1.,  1., -0.,  1., -1.,  0.,  0., -0.,  1.,  0.,  0.,  0.,
        -0.,  1., -1.],
       [-0., -1.,  0.,  0.,  0.,  0., -1.,  1.,  1., -0.,  0., -1., -1.,
        -0., -1.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,  0., -1.,
        -0., -0., -0.,  1.,  0., -0., -0., -0., -1., -0.,  1., -1.,  0.,
        -0.,  0.,  0.],
       [ 1., -1., -1., -1.,  0., -1., -1., -0., -0., -0., -1., -1., -1.,
         1.,  0., -1., -1.,  1.,  1.,  0.,  1.,  0., -1., -1., -1., -1.,
        -0.,  1., -0.,  1.,  0., -0.,  1.,  1.,  0., -0.,  1., -1., -1.,
        -0., -1., -1.],
       [ 1.,  0., -1., -1., -0.,  1.,  0., -1.,  1., -0., -1., -0.,  0.,
        -1., -1.,  1.,  1., -1., -1.,  0., -1.,  0., -1., -1.,  1., -0.,
         0.,  1., -0., -0., -0.,  0.,  1., -1.,  1., -0.,  0., -0., -1.,
         0.,  1., -1.],
       [ 0.,  0.,  1.,  0., -0., -1.,  0.,  0.,  0.,  0.,  0.,  0., -0.,
        -0.,  0., -0.,  1.,  0., -0.,  0., -1.,  1., -1.,  1., -1., -0.,
        -1., -0.,  0.,  0., -0.,  1.,  1., -0.,  0.,  1., -0., -0.,  0.,
        -0.,  0., -0.],
       [-1., -0., -0.,  1.,  0.,  0., -0.,  1., -1.,  0.,  1.,  0., -0.,
         1.,  1., -1.,  0.,  1.,  1., -0., -0.,  1., -0., -0.,  0.,  0.,
         1., -1.,  0.,  0.,  0.,  1.,  0.,  1., -1., -1., -0.,  0.,  1.,
        -0., -1.,  1.],
       [-0.,  1., -1., -0., -0., -1.,  1.,  1., -1., -0., -0., -1., -1.,
        -0., -1., -0.,  1.,  0., -0., -0., -1., -1.,  1., -1., -1.,  1.,
        -1.,  0.,  0.,  1., -0.,  1., -1., -0.,  1., -1.,  1.,  1., -0.,
        -0.,  0.,  0.],
       [-1.,  1., -0.,  1., -0., -0.,  1.,  0.,  0.,  0.,  1.,  1.,  1.,
        -1., -0.,  1., -0., -1., -1., -0.,  0.,  1., -0., -0., -0.,  1.,
        -1., -1.,  0., -1., -0., -1.,  0., -1., -0., -1., -1.,  1.,  1.,
         0.,  1.,  1.],
       [-0.,  1.,  0., -0., -0.,  0.,  1.,  1., -1., -0., -0., -1., -1.,
        -0., -1., -0., -0.,  0., -0., -0.,  0.,  0., -0.,  0.,  0.,  1.,
         0.,  0.,  0.,  1., -0., -0.,  0., -0.,  1.,  0.,  1.,  1., -0.,
        -0.,  0.,  0.],
       [ 0.,  0.,  1., -0., -1.,  1.,  0.,  0., -0., -1.,  0.,  0.,  0.,
        -0., -0., -1., -1., -0., -1., -1., -0., -0., -1., -0., -0., -0.,
        -0.,  1., -0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,
        -1., -0.,  1.],
       [-0.,  1., -1., -0., -1.,  1.,  1., -1., -1.,  1., -0.,  1., -1.,
         0., -1., -1., -1., -0., -1.,  1., -0.,  0.,  1.,  0., -0., -1.,
        -0., -1.,  0.,  1., -0.,  0.,  0.,  0., -1.,  0., -1., -1., -0.,
        -1., -0., -1.],
       [ 1.,  0., -1.,  0.,  1.,  1., -1., -0.,  0.,  1., -1., -0., -0.,
        -1., -0., -0.,  1., -1.,  0., -1.,  0., -1., -1.,  0., -0.,  1.,
        -1., -0.,  1.,  1.,  1., -1., -0., -1.,  0.,  1., -1., -0., -1.,
        -1., -0.,  0.],
       [-1.,  1., -0., -0., -0., -0., -0., -1., -1.,  0.,  1.,  1.,  1.,
        -1.,  1.,  1., -0., -1., -1., -0.,  0.,  1., -0., -0., -0., -0.,
        -1., -1., -1.,  0.,  1., -1.,  0., -1.,  1., -1.,  0.,  1.,  1.,
         0., -0.,  1.],
       [ 1., -1., -1.,  0., -1., -1.,  0.,  1.,  1.,  1., -1., -1., -1.,
         1., -1.,  0., -1.,  1., -0., -1., -0., -1., -1.,  0.,  0.,  0.,
         1., -0.,  1., -0., -1.,  1., -0.,  1., -1.,  1., -0., -1., -1.,
         1.,  0.,  0.],
       [ 1., -0.,  0.,  0., -0., -0., -1.,  0., -0., -0., -1., -0., -0.,
        -1.,  0.,  1., -0., -1., -1.,  0.,  0., -1.,  0.,  0., -0.,  1.,
        -1.,  1.,  1.,  1.,  1., -1., -0., -1., -0.,  1., -1.,  0., -1.,
         0., -0., -1.],
       [-0.,  1.,  0., -0.,  0., -0.,  1., -1., -1., -0., -0.,  1., -1.,
         0., -1.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0., -0., -1.,
         0.,  0.,  0.,  1., -0.,  0.,  0.,  0., -1.,  0., -1., -1., -0.,
         0., -0.,  0.],
       [ 1.,  1., -1., -0., -1.,  1.,  1.,  0.,  0.,  1.,  1.,  1., -1.,
         0.,  0., -1., -1.,  1., -1.,  1.,  1.,  0.,  1., -1.,  1., -1.,
        -0., -1.,  0.,  1., -0.,  0., -1., -1.,  0.,  0., -1., -1., -0.,
        -1., -0., -1.],
       [-1.,  0.,  1.,  0., -1.,  1.,  0., -1.,  1., -1., -1., -0., -0.,
         0.,  1., -1., -1.,  1., -1., -1.,  1., -0., -1.,  1.,  1.,  0.,
        -0.,  1., -0.,  0.,  0.,  0.,  1., -1., -1., -0.,  0.,  0.,  0.,
        -1., -0.,  1.],
       [ 1., -0., -1., -0., -1.,  1.,  1., -0., -0.,  1.,  1., -0.,  0.,
         0., -0.,  0., -1.,  1.,  0.,  1.,  1., -1.,  1., -1.,  1., -1.,
         1.,  0.,  0.,  1., -0., -1., -1., -1., -0., -1., -1.,  0., -0.,
        -1., -0.,  0.],
       [-0.,  0.,  0.,  0., -0., -0., -1., -1.,  1., -0.,  0., -0.,  0.,
         0., -1.,  1., -0.,  0., -1.,  0.,  0., -1.,  0.,  0., -0.,  1.,
        -1.,  1., -0.,  1., -0., -1., -0.,  0.,  1.,  1., -1., -0.,  0.,
         0., -0., -1.],
       [-1., -1.,  1.,  0., -1.,  1.,  0., -1.,  1., -1., -1.,  1.,  1.,
         0.,  1.,  0., -1.,  1.,  0., -1.,  1.,  1., -1.,  1.,  1.,  0.,
         1., -0., -0.,  0.,  0., -1.,  1., -1., -1.,  1.,  0., -1.,  0.,
        -1., -0., -0.],
       [-0., -1.,  0.,  0.,  0.,  0.,  0., -0., -0., -0.,  0., -1., -1.,
        -0.,  0., -1.,  0., -0.,  1.,  0., -0., -1.,  0.,  0.,  0.,  0.,
         1.,  1., -0., -0.,  0.,  1., -0., -0.,  0.,  1., -0., -1.,  0.,
        -0.,  0., -1.],
       [-0., -1.,  0.,  0., -0., -0., -1., -1.,  1., -0.,  0.,  1., -1.,
         0., -1., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  1.,
         0., -0., -0.,  1., -0.,  0., -0.,  0.,  1., -0., -1.,  1.,  0.,
         0., -0.,  0.]], dtype=np.float32), np.array([[-1., -1.,  1.,  1., -1.,  0., -1., -1.,  1.,  1.,  1.,  1., -1.,
        -1.,  1., -1.,  1.,  1., -0., -1., -1.,  1., -0.,  1., -1., -1.,
        -1., -0., -1.,  1., -1.,  0., -1.,  1., -1.,  0., -1.,  1., -1.,
        -1., -1.,  1.],
       [ 1., -1.,  0., -1., -1., -1., -1.,  1.,  1., -1., -1., -1., -1.,
        -1.,  1.,  0., -0.,  1.,  1.,  1.,  1.,  0., -1.,  1.,  1.,  1.,
         0., -1.,  1.,  1., -1., -1., -1.,  1.,  1., -1.,  1., -1.,  1.,
        -1., -1.,  0.],
       [ 1., -1., -1., -1., -1.,  0., -1.,  1.,  1., -1., -1., -1., -1.,
        -1.,  1., -1.,  1.,  1., -0.,  1., -1., -1.,  0., -1., -1.,  1.,
        -1.,  0.,  1.,  1., -1.,  0.,  1.,  1.,  1.,  0.,  1., -1.,  1.,
        -1., -1., -1.],
       [-1., -1., -0.,  1., -1., -1., -1., -1.,  1.,  1.,  1.,  1., -1.,
        -1.,  1.,  0., -0.,  1.,  1., -1.,  1., -0.,  1., -1.,  1., -1.,
         0.,  1., -1.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,  1., -1.,
        -1., -1., -0.],
       [-1.,  1.,  1., -1., -1.,  0.,  1.,  1., -1.,  1.,  1., -1.,  1.,
        -1., -1.,  1.,  1.,  1.,  0.,  1., -1., -1., -0.,  1.,  1.,  1.,
         1.,  0.,  1., -1.,  1., -0.,  1.,  1.,  1.,  0.,  1., -1., -1.,
         1.,  1., -1.],
       [ 1.,  1.,  0.,  1.,  1., -1.,  1., -1., -1.,  1., -1.,  1.,  1.,
        -1., -1., -0., -0.,  1., -1.,  1., -1., -0., -1., -1.,  1., -1.,
        -0.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,  1., -1.,  1.,  1.,
        -1.,  1., -0.],
       [ 1., -1., -1.,  1., -1., -0., -1.,  1.,  1., -1.,  1., -1.,  1.,
        -1., -1.,  1.,  1.,  1.,  0., -1., -1., -1., -0., -1., -1., -1.,
        -1., -0.,  1., -1., -1.,  0.,  1., -1., -1.,  0.,  1.,  1.,  1.,
         1.,  1.,  1.],
       [-1., -1., -0.,  1., -1.,  1., -1., -1.,  1.,  1., -1.,  1.,  1.,
         1., -1.,  0., -0.,  1.,  1.,  1., -1.,  0., -1.,  1., -1.,  1.,
        -0.,  1.,  1., -1.,  1.,  1., -1., -1.,  1., -1., -1., -1.,  1.,
         1., -1., -0.],
       [ 1.,  1., -1.,  1., -1.,  0.,  1., -1., -1., -1., -1.,  1.,  1.,
        -1., -1.,  1.,  1.,  1.,  0., -1., -1.,  1.,  0., -1.,  1., -1.,
         1., -0., -1., -1.,  1., -0., -1.,  1., -1., -0., -1.,  1.,  1.,
         1.,  1.,  1.],
       [-1.,  1., -0., -1.,  1., -1.,  1.,  1., -1., -1.,  1., -1.,  1.,
        -1., -1., -0., -0.,  1., -1., -1., -1.,  0.,  1.,  1.,  1.,  1.,
        -0., -1.,  1., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1., -1.,
        -1.,  1.,  0.],
       [ 1.,  1., -1.,  1.,  1.,  0.,  1.,  1., -1., -1.,  1., -1., -1.,
         1.,  1., -1., -1., -1.,  0., -1.,  1., -1., -0., -1.,  1., -1.,
         1.,  0.,  1.,  1.,  1.,  0.,  1.,  1., -1., -0.,  1.,  1.,  1.,
        -1., -1.,  1.],
       [ 1., -1.,  0., -1., -1.,  1., -1.,  1.,  1., -1.,  1., -1.,  1.,
         1., -1., -0., -0.,  1.,  1., -1., -1.,  0.,  1., -1., -1., -1.,
        -0., -1., -1., -1.,  1.,  1.,  1., -1., -1.,  1.,  1.,  1., -1.,
         1., -1., -0.],
       [ 1.,  1., -1., -1., -1.,  0., -1., -1., -1., -1., -1.,  1., -1.,
        -1., -1., -1.,  1.,  1., -0.,  1., -1.,  1.,  0., -1.,  1.,  1.,
         1.,  0., -1.,  1.,  1., -0., -1.,  1., -1., -0.,  1., -1.,  1.,
        -1., -1., -1.],
       [-1., -1., -0.,  1., -1., -1., -1.,  1., -1.,  1.,  1.,  1.,  1.,
        -1., -1.,  0., -0.,  1.,  1., -1., -1.,  0.,  1.,  1.,  1., -1.,
        -0.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1., -1.,
        -1., -1., -0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 4
p = 7
rank = 42

verify_tensor_support_rank_decomposition(decomposition_247, n, m, p, rank)

## Rank-47 support rank decomposition of <2,4,8> over Z


In [ ]:
#@title Data

decomposition_248 = (np.array([[-0.,  0.,  0.,  1.,  1., -0., -1., -0., -0., -1.,  0.,  0., -1.,
        -0.,  1.,  1., -0.,  0., -1., -1., -0., -0.,  1.,  0.,  0.,  1.,
         0., -1.,  0.,  1., -0., -1.,  1.,  0.,  0.,  1.,  1., -1., -0.,
        -1., -1.,  0., -1., -1., -1., -1.,  0.],
       [-1., -0., -0.,  1.,  1., -1., -1.,  0.,  0., -0.,  1., -1., -0.,
        -0., -0.,  0.,  0.,  0.,  0., -1.,  1.,  0.,  1.,  1., -1.,  0.,
         1., -1., -0.,  1.,  0., -0.,  0., -1.,  0.,  1., -0., -1., -1.,
        -1.,  0., -1.,  0., -0., -1., -1., -1.],
       [ 0., -0., -0., -0., -1.,  0.,  1.,  0.,  1., -0.,  0.,  1., -0.,
         1., -1., -0., -1.,  1.,  1., -0., -1.,  1., -0., -1.,  0.,  0.,
        -1., -0., -1., -1., -0., -0., -1.,  0.,  0.,  0.,  0., -0.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.,  0.],
       [ 1.,  0., -0.,  0., -1.,  0.,  1., -0.,  1.,  1., -1.,  0.,  1.,
         1.,  0., -1., -1.,  1.,  0.,  0.,  0., -0., -1., -0.,  1., -1.,
        -0., -0., -1., -0.,  1.,  0., -1.,  1., -0., -0., -1., -0.,  1.,
         1., -0., -0.,  0.,  0.,  1.,  1.,  1.],
       [-1.,  0.,  1.,  1.,  1.,  0., -1.,  0., -1., -0., -0., -1., -1.,
         0.,  1.,  1., -0., -0., -1., -1.,  0., -0.,  1.,  0., -0.,  1.,
        -0.,  0.,  0.,  1.,  0., -1.,  1.,  0.,  0.,  1.,  1., -1.,  0.,
         0., -1.,  0.,  0., -1., -1., -1.,  0.],
       [-1.,  1.,  0.,  1.,  1., -1., -1.,  0.,  0., -0., -0., -1., -0.,
         0.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0.,  1.,  1., -1.,  1.,
         1., -1., -0.,  1., -0., -0.,  0., -1., -0.,  1., -0., -1., -1.,
        -1., -0., -1.,  0., -1., -1., -0., -1.],
       [-0.,  0.,  0.,  0.,  0., -0.,  1.,  0.,  1.,  0.,  0.,  1.,  0.,
         1., -1., -0., -1.,  0.,  1., -0., -1.,  1.,  0., -1.,  1.,  0.,
        -1., -0., -1., -1., -0.,  0., -1., -0.,  1., -1., -1., -0.,  1.,
         1.,  0., -0.,  1.,  1.,  1.,  1.,  0.],
       [-1., -0.,  0.,  1.,  1.,  0., -1.,  1., -1., -1.,  1., -0.,  0.,
        -1., -0.,  1.,  1., -1., -1., -0.,  0.,  0.,  1.,  1., -1.,  1.,
         0., -0.,  0., -0., -1.,  0.,  1., -1., -0.,  0.,  1., -0., -1.,
        -1., -0., -0., -0.,  0., -0., -1., -0.]], dtype=np.float32), np.array([[-1.,  0.,  1., -0.,  0., -1.,  1., -0.,  1., -1., -0.,  1., -0.,
        -1.,  1.,  1.,  0., -0., -0.,  0.,  0., -1., -1.,  0.,  0., -0.,
         1., -1., -0., -1.,  1.,  1., -1., -1., -0., -0.,  0.,  1., -1.,
        -1., -0.,  0.,  1.,  0., -0., -0., -0.],
       [-1., -0., -1., -0., -0., -1.,  1.,  0., -1.,  1., -0.,  1.,  0.,
         1., -1., -1., -0.,  0., -0.,  0.,  0.,  1., -1.,  0.,  0., -0.,
         1., -1.,  0., -1., -1., -1.,  1., -1., -0.,  0., -0.,  1., -1.,
        -1.,  0.,  0., -1., -0.,  0.,  0., -0.],
       [ 1., -0.,  1.,  0., -0., -1.,  1., -0.,  1.,  1., -0., -1., -0.,
        -1., -1., -1.,  0.,  0., -0., -0.,  0.,  1.,  1.,  0.,  0.,  0.,
        -1., -1.,  0.,  1., -1.,  1., -1.,  1., -0.,  0., -0.,  1., -1.,
        -1.,  0., -0., -1.,  0., -0.,  0., -0.],
       [-1.,  0.,  1., -0.,  0., -1., -1., -0., -1., -1., -0., -1., -0.,
         1., -1.,  1., -0.,  0.,  0.,  0.,  0.,  1., -1., -0., -0.,  0.,
        -1., -1., -0.,  1.,  1.,  1.,  1., -1.,  0.,  0.,  0.,  1.,  1.,
         1.,  0., -0., -1.,  0., -0.,  0., -0.],
       [ 1., -0.,  1., -0.,  0., -1., -1., -0., -1.,  1.,  0.,  1., -0.,
         1.,  1., -1.,  0.,  0.,  0., -0.,  0., -1.,  1.,  0.,  0., -0.,
         1., -1.,  0., -1., -1.,  1.,  1.,  1., -0., -0.,  0.,  1.,  1.,
         1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.],
       [-1., -0., -1., -0.,  0., -1., -1., -0.,  1.,  1., -0., -1., -0.,
        -1.,  1., -1.,  0.,  0.,  0.,  0., -0., -1., -1.,  0.,  0.,  0.,
        -1., -1., -0.,  1., -1., -1., -1., -1., -0.,  0., -0.,  1.,  1.,
         1.,  0., -0.,  1.,  0.,  0., -0., -0.],
       [-1.,  0.,  1.,  0.,  0.,  1., -1.,  0.,  1.,  1., -0.,  1.,  0.,
        -1., -1., -1.,  0., -0.,  0.,  0., -0.,  1., -1.,  0., -0., -0.,
         1.,  1., -0., -1., -1.,  1., -1., -1.,  0., -0., -0., -1.,  1.,
         1.,  0., -0., -1.,  0., -0., -0.,  0.],
       [ 1., -0., -1.,  0., -0., -1., -1.,  0.,  1., -1.,  0.,  1., -0.,
        -1., -1.,  1., -0.,  0.,  0.,  0.,  0.,  1.,  1.,  0., -0., -0.,
         1., -1.,  0., -1.,  1., -1., -1.,  1., -0.,  0.,  0.,  1.,  1.,
         1.,  0.,  0., -1.,  0.,  0., -0.,  0.],
       [-0., -1.,  0., -0., -0., -1., -1.,  0.,  0., -0.,  1., -0., -0.,
         1.,  1.,  1., -1.,  0.,  0., -1.,  1.,  1.,  1.,  0.,  0., -1.,
         1.,  0.,  0.,  1., -1.,  1., -1., -1., -0., -0., -0., -1., -1.,
         0.,  0., -0.,  0., -1., -0., -1., -0.],
       [-0., -1.,  0.,  0., -0., -1.,  1., -0.,  0., -0., -1., -0.,  0.,
         1.,  1.,  1., -1.,  0., -0.,  1., -1., -1.,  1.,  0., -0., -1.,
        -1., -0.,  0.,  1.,  1., -1.,  1.,  1., -0.,  0., -0.,  1., -1.,
        -0., -0., -0.,  0., -1., -0.,  1., -0.],
       [-0.,  1.,  0., -0.,  0.,  1.,  1., -0., -0.,  0.,  1., -0., -0.,
        -1.,  1.,  1.,  1.,  0., -0.,  1.,  1.,  1.,  1.,  0., -0., -1.,
         1.,  0.,  0.,  1., -1., -1.,  1., -1.,  0.,  0.,  0.,  1.,  1.,
        -0.,  0.,  0.,  0., -1.,  0.,  1.,  0.],
       [ 0., -1., -0., -0., -0., -1.,  1.,  0.,  0., -0.,  1., -0., -0.,
        -1., -1.,  1.,  1.,  0., -0., -1., -1., -1.,  1., -0.,  0., -1.,
        -1., -0.,  0., -1., -1.,  1.,  1., -1., -0.,  0., -0., -1.,  1.,
         0.,  0.,  0., -0.,  1., -0.,  1.,  0.],
       [ 0.,  1.,  0.,  0.,  0.,  1., -1., -0., -0., -0.,  1., -0.,  0.,
         1., -1.,  1., -1.,  0., -0.,  1., -1., -1.,  1.,  0.,  0., -1.,
        -1.,  0.,  0., -1., -1., -1., -1., -1.,  0., -0.,  0.,  1., -1.,
        -0.,  0., -0.,  0.,  1.,  0., -1.,  0.],
       [ 0., -1.,  0.,  0.,  0., -1., -1., -0.,  0., -0., -1.,  0., -0.,
        -1., -1.,  1.,  1., -0.,  0.,  1.,  1.,  1.,  1., -0.,  0., -1.,
         1.,  0., -0., -1.,  1., -1., -1.,  1.,  0., -0., -0.,  1.,  1.,
         0.,  0.,  0.,  0.,  1., -0., -1., -0.],
       [-0.,  1.,  0.,  0.,  0.,  1., -1., -0.,  0.,  0., -1.,  0., -0.,
        -1.,  1.,  1.,  1., -0., -0., -1., -1., -1.,  1., -0., -0., -1.,
        -1.,  0., -0.,  1.,  1.,  1., -1.,  1.,  0., -0., -0., -1.,  1.,
        -0., -0., -0., -0., -1., -0., -1.,  0.],
       [-0.,  1.,  0.,  0., -0.,  1.,  1.,  0., -0.,  0., -1.,  0.,  0.,
         1., -1.,  1., -1., -0.,  0., -1.,  1.,  1.,  1., -0., -0., -1.,
         1., -0.,  0., -1.,  1.,  1.,  1.,  1., -0., -0.,  0., -1., -1.,
        -0., -0., -0., -0.,  1.,  0.,  1.,  0.],
       [-0., -0.,  0., -0., -1.,  1., -1., -0., -0., -0., -0.,  0.,  0.,
        -1., -1.,  1., -0., -1., -0., -0.,  0., -1., -1.,  0.,  1.,  0.,
         1.,  0., -0.,  1., -1.,  1.,  1.,  1.,  1., -1.,  1.,  1., -1.,
         0.,  1.,  1., -0.,  0., -0., -0.,  0.],
       [-0.,  0.,  0., -0., -1.,  1., -1.,  0., -0.,  0.,  0.,  0., -0.,
         1.,  1., -1., -0.,  1.,  0.,  0.,  0.,  1., -1.,  0.,  1., -0.,
         1., -0.,  0.,  1.,  1., -1., -1.,  1., -1., -1., -1.,  1., -1.,
         0., -1.,  1.,  0., -0., -0.,  0.,  0.],
       [-0., -0., -0.,  0.,  1.,  1.,  1., -0., -0., -0.,  0.,  0., -0.,
         1., -1.,  1.,  0.,  1.,  0., -0.,  0.,  1.,  1.,  0.,  1.,  0.,
         1.,  0., -0., -1.,  1.,  1.,  1.,  1., -1.,  1.,  1., -1., -1.,
         0.,  1.,  1.,  0., -0.,  0.,  0.,  0.],
       [-0.,  0.,  0., -0.,  1., -1.,  1.,  0., -0.,  0.,  0., -0., -0.,
         1.,  1.,  1.,  0.,  1., -0.,  0.,  0., -1.,  1.,  0.,  1.,  0.,
        -1.,  0., -0.,  1.,  1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1.,
        -0., -1., -1.,  0.,  0., -0., -0., -0.],
       [-0.,  0., -0.,  0., -1., -1., -1., -0., -0., -0.,  0.,  0., -0.,
        -1.,  1.,  1., -0., -1.,  0., -0., -0.,  1., -1.,  0.,  1., -0.,
        -1.,  0.,  0., -1., -1., -1.,  1.,  1., -1.,  1.,  1., -1., -1.,
        -0., -1., -1., -0., -0., -0.,  0., -0.],
       [ 0.,  0.,  0.,  0., -1.,  1., -1., -0., -0., -0.,  0.,  0., -0.,
         1.,  1.,  1., -0.,  1., -0.,  0.,  0., -1., -1., -0., -1., -0.,
         1., -0., -0., -1.,  1., -1.,  1., -1.,  1.,  1.,  1., -1.,  1.,
        -0., -1.,  1.,  0., -0., -0.,  0., -0.],
       [ 0.,  0.,  0.,  0., -1., -1., -1., -0., -0.,  0.,  0., -0., -0.,
         1., -1.,  1.,  0.,  1., -0., -0.,  0.,  1., -1.,  0., -1., -0.,
        -1.,  0.,  0.,  1.,  1.,  1.,  1., -1., -1., -1.,  1.,  1.,  1.,
         0.,  1., -1., -0.,  0., -0., -0.,  0.],
       [ 0., -0.,  0.,  0.,  1.,  1.,  1., -0., -0., -0., -0.,  0., -0.,
        -1.,  1.,  1.,  0., -1., -0.,  0., -0.,  1.,  1.,  0., -1., -0.,
         1., -0., -0.,  1., -1., -1.,  1., -1., -1., -1.,  1.,  1.,  1.,
        -0., -1.,  1., -0., -0.,  0.,  0., -0.],
       [ 0.,  0.,  0.,  1.,  0., -1., -1.,  1.,  0., -0., -0.,  0.,  1.,
        -1.,  1., -1.,  0.,  0., -1.,  0., -0., -1., -1., -1.,  0., -0.,
        -1.,  0., -1.,  1., -1., -1., -1., -1., -0.,  0.,  0.,  1.,  1.,
         0.,  0., -0.,  0.,  0.,  1.,  0.,  1.],
       [-0., -0.,  0., -1., -0.,  1.,  1.,  1.,  0.,  0., -0.,  0.,  1.,
        -1.,  1., -1., -0.,  0., -1.,  0., -0., -1.,  1.,  1., -0.,  0.,
         1., -0., -1., -1., -1., -1., -1.,  1., -0., -0.,  0., -1., -1.,
        -0., -0., -0.,  0., -0., -1.,  0., -1.],
       [ 0.,  0.,  0., -1., -0., -1.,  1., -1.,  0., -0., -0., -0.,  1.,
         1.,  1., -1.,  0., -0., -1., -0.,  0.,  1.,  1., -1.,  0.,  0.,
        -1., -0.,  1., -1.,  1., -1., -1., -1., -0.,  0., -0., -1.,  1.,
        -0.,  0.,  0.,  0., -0., -1., -0.,  1.],
       [-0.,  0., -0., -1.,  0.,  1., -1., -1.,  0., -0.,  0.,  0., -1.,
        -1.,  1.,  1.,  0., -0., -1., -0.,  0., -1.,  1., -1., -0.,  0.,
        -1.,  0., -1.,  1.,  1.,  1., -1.,  1.,  0., -0.,  0., -1.,  1.,
         0., -0., -0., -0.,  0.,  1.,  0., -1.],
       [-0., -0., -0.,  1., -0.,  1.,  1.,  1.,  0.,  0.,  0., -0., -1.,
         1.,  1.,  1., -0., -0., -1., -0., -0.,  1., -1., -1., -0.,  0.,
        -1.,  0.,  1., -1., -1.,  1., -1.,  1.,  0.,  0., -0.,  1.,  1.,
        -0., -0., -0.,  0.,  0., -1.,  0., -1.],
       [-0.,  0.,  0.,  1., -0., -1.,  1., -1.,  0., -0., -0., -0., -1.,
        -1.,  1.,  1., -0., -0., -1., -0., -0., -1., -1.,  1., -0., -0.,
         1.,  0., -1., -1.,  1.,  1., -1., -1.,  0., -0., -0.,  1., -1.,
         0.,  0., -0.,  0.,  0., -1., -0.,  1.],
       [ 0.,  0.,  0., -1., -0., -1.,  1.,  1.,  0., -0., -0., -0., -1.,
        -1., -1.,  1.,  0.,  0.,  1., -0., -0., -1.,  1., -1., -0.,  0.,
        -1.,  0., -1., -1., -1.,  1.,  1., -1.,  0., -0.,  0., -1.,  1.,
         0.,  0., -0., -0., -0., -1.,  0.,  1.],
       [-0.,  0., -0., -1.,  0., -1., -1.,  1., -0., -0.,  0., -0., -1.,
         1.,  1.,  1., -0.,  0., -1., -0.,  0.,  1.,  1.,  1.,  0., -0.,
         1., -0.,  1.,  1., -1.,  1., -1., -1.,  0.,  0.,  0., -1., -1.,
        -0., -0.,  0., -0., -0.,  1.,  0.,  1.]], dtype=np.float32), np.array([[-1., -1., -1., -1.,  1., -1., -0., -1.,  1.,  1., -1.,  1., -1.,
        -0.,  0.,  0., -1., -1., -1., -1., -1., -1., -1.,  1., -1., -1.,
         0.,  1., -1., -1.,  1., -1., -1.,  0., -1., -1.,  1., -0., -1.,
         1.,  1.,  1., -1.,  1., -1., -1., -1.],
       [ 1.,  1.,  1., -1., -1.,  0., -1., -1., -1., -1.,  1., -1., -1.,
        -1.,  1.,  1.,  1.,  1., -1.,  1.,  1., -0.,  0.,  1.,  1.,  1.,
        -1., -1., -1.,  0.,  0., -0.,  0., -1.,  1.,  1., -1., -1., -0.,
        -1., -1., -1.,  1., -1., -1.,  1., -1.],
       [-1., -1.,  1., -1.,  1., -1.,  0.,  1., -1., -1.,  1.,  1.,  1.,
         0.,  0.,  0., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1., -1.,
        -0.,  1.,  1., -1., -1.,  1.,  1.,  0.,  1., -1., -1., -0., -1.,
         1., -1.,  1.,  1.,  1., -1.,  1., -1.],
       [ 1., -1., -1., -1., -1.,  0., -1.,  1.,  1.,  1.,  1., -1.,  1.,
         1., -1., -1., -1., -1.,  1.,  1.,  1., -0., -0.,  1.,  1., -1.,
        -1., -1.,  1.,  0., -0., -0.,  0., -1., -1.,  1.,  1., -1.,  0.,
        -1.,  1., -1., -1.,  1., -1.,  1., -1.],
       [-1.,  1.,  1., -1.,  1.,  1.,  0., -1., -1.,  1., -1.,  1.,  1.,
         0., -0., -0.,  1., -1.,  1.,  1., -1., -1., -1., -1.,  1., -1.,
        -0., -1., -1., -1.,  1.,  1.,  1., -0., -1., -1., -1., -0.,  1.,
        -1., -1., -1., -1.,  1., -1.,  1.,  1.],
       [-1.,  1.,  1., -1., -1.,  0., -1., -1., -1.,  1., -1.,  1.,  1.,
        -1., -1., -1.,  1.,  1.,  1.,  1., -1.,  0.,  0., -1., -1., -1.,
         1., -1., -1., -0., -0.,  0.,  0.,  1.,  1.,  1.,  1., -1., -0.,
        -1.,  1.,  1., -1.,  1., -1.,  1.,  1.],
       [-1., -1., -1., -1.,  1., -1., -0., -1., -1.,  1., -1., -1., -1.,
         0.,  0., -0.,  1., -1.,  1., -1.,  1.,  1., -1., -1.,  1., -1.,
        -0.,  1.,  1.,  1.,  1., -1.,  1., -0.,  1.,  1., -1.,  0.,  1.,
        -1.,  1.,  1.,  1., -1.,  1.,  1., -1.],
       [ 1.,  1.,  1., -1.,  1.,  0.,  1., -1.,  1., -1.,  1.,  1., -1.,
         1., -1.,  1., -1., -1.,  1.,  1., -1.,  0., -0., -1.,  1.,  1.,
         1., -1.,  1., -0.,  0., -0.,  0., -1.,  1.,  1., -1., -1., -0.,
         1.,  1.,  1., -1.,  1.,  1., -1., -1.],
       [-1.,  1.,  1., -1.,  1.,  1., -0., -1.,  1.,  1., -1., -1.,  1.,
        -0.,  0.,  0., -1., -1., -1.,  1.,  1.,  1., -1.,  1., -1., -1.,
        -0., -1.,  1.,  1.,  1.,  1., -1., -0.,  1.,  1.,  1., -0., -1.,
         1., -1., -1.,  1., -1.,  1., -1.,  1.],
       [-1.,  1.,  1., -1.,  1., -0.,  1., -1.,  1.,  1., -1., -1.,  1.,
         1.,  1., -1., -1., -1., -1.,  1.,  1.,  0.,  0.,  1., -1., -1.,
        -1., -1.,  1., -0.,  0.,  0.,  0.,  1.,  1.,  1.,  1., -1., -0.,
         1., -1., -1.,  1., -1.,  1., -1.,  1.],
       [ 1.,  1., -1.,  1., -1.,  1., -0., -1., -1.,  1., -1.,  1., -1.,
        -0.,  0., -0., -1., -1.,  1., -1.,  1.,  1.,  1.,  1., -1.,  1.,
        -0., -1.,  1., -1.,  1., -1.,  1., -0.,  1., -1., -1., -0., -1.,
         1.,  1., -1.,  1.,  1., -1.,  1.,  1.],
       [ 1., -1., -1., -1.,  1.,  0.,  1.,  1., -1.,  1.,  1.,  1.,  1.,
        -1.,  1., -1.,  1.,  1., -1.,  1., -1.,  0.,  0., -1.,  1., -1.,
         1., -1., -1.,  0., -0., -0., -0., -1., -1.,  1.,  1., -1.,  0.,
         1., -1.,  1.,  1., -1.,  1., -1., -1.],
       [-1.,  1., -1., -1.,  1.,  1.,  0.,  1.,  1., -1.,  1.,  1., -1.,
        -0., -0.,  0.,  1.,  1., -1., -1.,  1.,  1., -1., -1.,  1., -1.,
        -0., -1.,  1., -1., -1., -1., -1., -0.,  1., -1.,  1., -0.,  1.,
        -1.,  1., -1.,  1.,  1., -1., -1.,  1.],
       [ 1.,  1.,  1.,  1.,  1., -0.,  1., -1., -1.,  1.,  1., -1.,  1.,
        -1., -1., -1.,  1.,  1.,  1., -1.,  1.,  0.,  0.,  1.,  1., -1.,
        -1.,  1., -1., -0., -0.,  0.,  0., -1.,  1., -1.,  1.,  1.,  0.,
         1.,  1., -1., -1.,  1.,  1., -1., -1.],
       [ 1., -1.,  1.,  1., -1., -1.,  0., -1.,  1.,  1., -1.,  1.,  1.,
        -0.,  0., -0.,  1., -1., -1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,
        -0.,  1.,  1., -1.,  1.,  1., -1., -0.,  1., -1.,  1.,  0.,  1.,
        -1., -1.,  1.,  1.,  1., -1., -1., -1.],
       [ 1.,  1.,  1.,  1., -1., -0., -1., -1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1., -1., -1., -1., -1., -1., -1.,  0., -0., -1.,  1., -1.,
         1.,  1.,  1.,  0., -0.,  0.,  0., -1.,  1., -1.,  1.,  1.,  0.,
        -1., -1.,  1.,  1., -1., -1.,  1., -1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 4
p = 8
rank = 47

verify_tensor_support_rank_decomposition(decomposition_248, n, m, p, rank)

## Rank-38 support rank decomposition of <2,5,5> over Z


In [ ]:
#@title Data

decomposition_255 = (np.array([[-0., -1., -0., -1., -1.,  1., -0.,  0., -1., -0., -1.,  1.,  1.,
         1.,  1., -1., -1.,  1.,  0.,  1., -0., -1., -1., -0., -1.,  0.,
        -1., -0., -1.,  0.,  0.,  1., -0.,  1., -0.,  1.,  1.,  1.],
       [-0.,  1.,  0.,  1.,  1., -1.,  0.,  0.,  1.,  0.,  1.,  1.,  1.,
         1., -1., -1., -1.,  1.,  0.,  1., -0., -1., -1., -0., -1.,  0.,
        -1., -0., -1.,  0.,  0.,  1.,  0.,  1.,  0., -1.,  1., -1.],
       [ 0.,  1.,  0., -1.,  1., -1.,  0., -0.,  1.,  0., -1.,  1., -1.,
         1., -1.,  1., -1.,  1., -0.,  1., -0., -1.,  1.,  0.,  1., -0.,
         1., -0., -1., -0., -0., -1., -0., -1.,  0., -1., -1., -1.],
       [ 0., -1.,  0., -1.,  1., -1.,  0., -0., -1., -0., -1.,  1.,  1.,
         1.,  1.,  1.,  1., -1., -0.,  1.,  0., -1.,  1.,  0., -1.,  0.,
         1., -0., -1.,  0.,  0., -1., -0., -1., -0.,  1., -1.,  1.],
       [-0., -0., -1.,  0., -1., -0.,  0., -0., -0.,  1., -1., -0.,  0.,
         0.,  0., -0., -1., -0., -0., -0., -1., -0.,  0.,  1.,  1., -0.,
         0.,  0., -0.,  1.,  0.,  0., -0., -0., -2., -0.,  0.,  0.],
       [ 1., -1.,  1., -1., -0.,  1., -1.,  1.,  0., -1., -0., -1.,  1.,
         1.,  1., -1., -0.,  1.,  1.,  1., -1., -1.,  1., -1.,  0.,  1.,
        -1.,  1., -1., -0.,  1.,  0.,  1.,  1., -0.,  1.,  1., -1.],
       [ 1.,  1.,  1., -1.,  0.,  1.,  1.,  1.,  0.,  1., -0., -1., -1.,
         1.,  1.,  1., -0., -1.,  1., -1.,  1.,  1.,  1., -1.,  0.,  1.,
         1., -1.,  1.,  0., -1.,  0., -1., -1.,  0.,  1.,  1.,  1.],
       [ 1.,  1., -1., -1.,  0., -1.,  1., -1., -0., -1., -0.,  1., -1.,
         1.,  1.,  1., -0.,  1., -1., -1.,  1., -1.,  1., -1., -0.,  1.,
         1., -1., -1., -0.,  1., -0.,  1.,  1., -0., -1., -1., -1.],
       [-1.,  1.,  1.,  1.,  0.,  1.,  1.,  1., -0., -1.,  0., -1., -1.,
        -1., -1.,  1.,  0.,  1.,  1., -1.,  1., -1., -1.,  1.,  0., -1.,
        -1., -1.,  1.,  0.,  1.,  0.,  1.,  1.,  0.,  1.,  1., -1.],
       [-1.,  0., -0.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0., -0., -1.,  0., -0.,  0.,  0., -0., -0., -0.,
        -0.,  1., -0., -1., -0.,  0., -1.,  0., -0., -0., -0., -0.]],
      dtype=np.float32), np.array([[ 0., -1.,  0., -1., -0., -1.,  1.,  1., -0.,  0.,  0.,  1.,  1.,
        -0., -0., -0.,  0., -1., -0., -1., -0.,  0., -2.,  0., -0., -1.,
         2., -0.,  0., -0., -1.,  2., -0., -1.,  0., -0.,  1., -1.],
       [-0., -0.,  1.,  1.,  1., -1.,  0., -0.,  0.,  1.,  1., -0.,  1.,
         0., -0., -0.,  1., -1.,  0., -0.,  1.,  0., -0., -1., -1., -0.,
        -0.,  0., -0.,  0.,  0., -0., -0.,  0., -1., -0.,  0.,  0.],
       [-0., -1., -0.,  1., -0.,  1.,  1.,  1., -0.,  0.,  0., -1.,  1.,
         0., -0.,  0.,  0., -1., -0., -1.,  0.,  0.,  0., -0., -0., -1.,
        -0.,  0., -2.,  0., -1.,  2., -0., -1.,  0., -0.,  1., -1.],
       [ 0., -0., -0., -1.,  0.,  1., -1., -1.,  2., -0.,  0., -1., -1.,
        -0.,  1., -1.,  0., -1.,  0.,  1., -0., -0., -1.,  0., -0.,  1.,
         2.,  0.,  0., -0.,  1.,  0.,  0.,  0.,  0., -1., -0., -0.],
       [-0., -0.,  0., -1.,  0.,  1.,  1.,  1., -2., -0.,  0., -1.,  1.,
        -0., -1.,  1., -0.,  1., -0., -1., -0.,  0., -1., -0.,  0., -1.,
        -0.,  0., -2., -0., -1.,  0., -0., -0., -0.,  1., -0., -0.],
       [ 0.,  1.,  0.,  1.,  0.,  1., -1.,  1., -0.,  0., -0., -0., -1.,
         1., -0., -1.,  0.,  1.,  0.,  0.,  0.,  1., -1.,  0.,  0., -1.,
         0., -0., -2.,  0.,  1.,  2., -0.,  0., -0., -0., -0.,  1.],
       [-0., -0.,  1.,  1.,  1., -1.,  0., -0.,  0., -1.,  1., -0., -1.,
         0., -0.,  0., -1.,  1.,  0., -0., -1.,  0.,  0., -1.,  1., -0.,
        -0., -0.,  0.,  0., -0., -0.,  0.,  0., -1., -0.,  0.,  0.],
       [-0., -1.,  0.,  1.,  0.,  1.,  1., -1., -0., -0.,  0., -0.,  1.,
         1., -0.,  1., -0., -1.,  0., -0., -0., -1.,  1.,  0., -0.,  1.,
        -2., -0.,  0.,  0., -1., -2., -0.,  0., -0., -0., -2., -1.],
       [ 0., -0., -0., -1., -0.,  1.,  1., -1.,  2.,  0., -0.,  0., -1.,
        -1.,  1.,  0., -0., -1., -0., -0.,  0., -1., -0.,  0., -0.,  1.,
        -0., -0.,  2., -0., -1.,  0.,  0., -1., -0., -1., -1., -0.],
       [-0., -0.,  0.,  1., -0., -1.,  1., -1.,  2., -0.,  0.,  0., -1.,
         1.,  1., -0.,  0., -1., -0., -0., -0., -1.,  0.,  0., -0.,  1.,
         2.,  0., -0., -0., -1., -0.,  0., -1.,  0., -1.,  1., -0.],
       [ 0.,  1.,  0.,  1., -0.,  1., -1., -1., -2., -0., -0., -1., -1.,
         0.,  0.,  0., -0.,  1., -0.,  1., -0.,  0., -0., -0., -0., -1.,
        -2.,  0., -0., -0., -1.,  0.,  0.,  1., -0.,  0., -1., -1.],
       [ 0.,  0.,  1., -1.,  1., -1.,  0., -0.,  0., -1., -1.,  0.,  1.,
         0., -0., -0., -1.,  1.,  0.,  0.,  1., -0., -0.,  1., -1.,  0.,
        -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1.,  0., -0., -0.],
       [-0.,  1., -0.,  1., -0.,  1., -1., -1., -2.,  0., -0., -1.,  1.,
        -0.,  0., -0., -0., -1., -0., -1.,  0.,  0.,  0.,  0.,  0., -1.,
        -0.,  0., -2.,  0., -1., -0., -0., -1., -0., -0., -1., -1.],
       [-0., -0.,  0.,  1., -0., -1.,  1.,  1.,  0.,  0.,  0.,  1.,  1.,
        -0.,  1.,  1., -0.,  1.,  0., -1.,  0.,  0.,  1., -0.,  0.,  1.,
        -2., -0.,  0.,  0.,  1., -2., -0.,  2., -0.,  1.,  0.,  0.],
       [-0.,  0., -0., -1.,  0.,  1., -1., -1.,  0., -0.,  0., -1.,  1.,
        -0., -1., -1., -0.,  1., -0., -1.,  0.,  0., -1., -0., -0., -1.,
        -0., -0., -2., -0., -1.,  2.,  0.,  0.,  0., -1., -0.,  0.],
       [ 0., -1.,  0., -1.,  0., -1.,  1., -1.,  2.,  0., -0., -0.,  1.,
        -1.,  0.,  1.,  0., -1., -0., -0.,  0., -1., -1., -0.,  0., -1.,
         0., -0.,  2., -0.,  1.,  0., -0., -0.,  0.,  0.,  0.,  1.],
       [ 0.,  0.,  1., -1.,  1., -1., -0.,  0.,  0.,  1., -1., -0., -1.,
        -0., -0.,  0.,  1., -1.,  0., -0., -1.,  0., -0.,  1.,  1.,  0.,
        -0., -0., -0.,  0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -0.],
       [ 0.,  1.,  0.,  1., -0.,  1., -1.,  1., -2.,  0.,  0.,  0.,  1.,
         1.,  0.,  1., -0., -1.,  0., -0.,  0., -1.,  1., -0.,  0.,  1.,
        -2., -0., -0., -0., -1.,  0., -0., -0., -0., -0.,  0., -1.],
       [-0., -0., -0.,  1., -0., -1., -1.,  1., -0., -0.,  0.,  0.,  1.,
         1.,  1., -0.,  0.,  1.,  0.,  0., -0.,  1., -0., -0.,  0.,  1.,
        -0.,  0., -2.,  0., -1.,  2.,  0., -1.,  0.,  1.,  1.,  0.],
       [-0., -0., -0., -1., -0.,  1.,  1., -1., -0., -0., -0., -0.,  1.,
        -1., -1.,  2.,  0.,  1., -0.,  0., -0.,  1.,  0., -0.,  0., -1.,
        -2.,  0., -0.,  0.,  1., -2., -0.,  1.,  0., -1., -1.,  0.],
       [-1.,  0.,  1.,  0., -0.,  0.,  1.,  1.,  0.,  1., -0., -0., -0.,
         0., -0.,  0., -0.,  0., -1.,  0.,  1., -0.,  0.,  1.,  0., -1.,
         0., -1.,  0.,  0., -1.,  0.,  1.,  0., -0.,  0.,  0., -0.],
       [-0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  0., -0., -2., -0.,  0.,  0.,  0., -2., -0.,  0., -0.],
       [-1., -0.,  1.,  0., -0., -0., -1.,  1., -0., -1.,  0., -0.,  0.,
         0., -0.,  0., -0., -0., -1., -0., -1., -0., -0.,  1.,  0., -1.,
        -0.,  1.,  0.,  0.,  1.,  0., -1.,  0., -0.,  0.,  0., -0.],
       [-1., -0., -1.,  0., -0., -0., -1., -1., -0.,  1., -0.,  0.,  0.,
        -0.,  0.,  0., -0.,  0.,  1., -0., -1., -0., -0.,  1.,  0., -1.,
        -0.,  1., -0.,  0., -1., -0.,  1.,  0., -0., -0.,  0.,  0.],
       [-1., -0., -1.,  0., -0., -0.,  1., -1., -0., -1., -0., -0.,  0.,
         0.,  0., -0.,  0., -0.,  1., -0.,  1.,  0.,  0.,  1., -0., -1.,
         0., -1.,  0., -2.,  1.,  0., -1., -0.,  0.,  0., -0., -0.]],
      dtype=np.float32), np.array([[-0.,  1., -1.,  1.,  1., -1.,  1.,  1.,  1.,  1., -1.,  1., -1.,
         1.,  1., -1.,  1., -1., -0., -1., -1., -1.,  1.,  1.,  1., -1.,
        -0.,  0.,  0.,  0., -1.,  1., -0.,  1.,  0., -1., -1., -1.],
       [ 1.,  1.,  0., -0.,  0., -0.,  1.,  1.,  1., -0.,  0., -1., -0.,
         1.,  1., -1., -0.,  0.,  1., -1.,  0., -1., -1., -0., -0., -1.,
        -1., -1., -1., -0., -1.,  1., -1.,  1.,  0., -1., -1.,  1.],
       [-0., -0.,  0., -0., -1., -0., -0.,  0.,  0.,  0., -1., -0.,  0.,
        -0.,  0., -0., -1.,  0.,  0.,  0., -0., -0., -0.,  0.,  1., -0.,
        -0., -0., -0.,  0.,  0., -0.,  0.,  0., -1., -0.,  0.,  0.],
       [ 1., -0., -1.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0., -0.,
        -0., -0., -0.,  0.,  0., -1.,  0.,  1.,  0., -0., -1.,  0.,  0.,
        -0., -1., -0., -2., -0., -0.,  1.,  0., -1.,  0.,  0., -0.],
       [ 0.,  1.,  1., -1., -1.,  1.,  1., -1., -1.,  1.,  1., -1., -1.,
        -1., -1., -1.,  1., -1., -0., -1., -1., -1., -1., -1.,  1.,  1.,
        -0., -0.,  0., -0., -1.,  1.,  0.,  1., -0.,  1.,  1., -1.],
       [-1., -1., -0., -0.,  0.,  0.,  1., -1., -1.,  0., -0., -1., -0.,
         1., -1., -1.,  0.,  0., -1., -1.,  0., -1., -1.,  0., -0.,  1.,
        -1., -1., -1.,  0., -1.,  1., -1.,  1.,  0.,  1., -1., -1.],
       [ 0.,  1., -1., -1.,  1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1.,
        -1., -1., -1., -1.,  1.,  0., -1., -1.,  1., -1., -1.,  1.,  1.,
         0., -0., -0., -0.,  1.,  1.,  0., -1.,  0., -1., -1.,  1.],
       [-1.,  1.,  0., -0.,  0.,  0.,  1.,  1.,  1.,  0., -0.,  1.,  0.,
        -1.,  1., -1., -0., -0.,  1.,  1., -0.,  1., -1.,  0.,  0.,  1.,
        -1., -1.,  1.,  0.,  1.,  1.,  1.,  1., -0., -1., -1.,  1.],
       [-0., -1., -1., -1.,  1., -1., -1.,  1., -1.,  1.,  1.,  1.,  1.,
        -1., -1.,  1.,  1., -1., -0.,  1.,  1., -1., -1., -1., -1.,  1.,
         0., -0., -0., -0., -1.,  1., -0.,  1., -0., -1., -1., -1.],
       [-1., -1., -0., -0., -0.,  0., -1.,  1., -1., -0., -0.,  1., -0.,
        -1., -1., -1.,  0., -0.,  1.,  1., -0.,  1., -1.,  0., -0.,  1.,
        -1.,  1.,  1., -0., -1.,  1., -1.,  1.,  0.,  1., -1., -1.]],
      dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 5
p = 5
rank = 38

verify_tensor_support_rank_decomposition(decomposition_255, n, m, p, rank)

## Rank-46 support rank decomposition of <2,5,6> over Z


In [ ]:
#@title Data

decomposition_256 = (np.array([[ 1., -1., -1., -1.,  1., -1., -1.,  1., -1., -1.,  1.,  1., -1.,
         1., -0., -1., -1., -1., -1., -1., -1., -1.,  1.,  1.,  0.,  1.,
         1.,  1.,  1.,  1., -1., -1., -1., -0.,  1., -0., -1., -1., -1.,
         1.,  1.,  1., -1., -1.,  1.,  1.],
       [-1.,  1.,  1.,  1.,  1., -1., -1.,  1., -1., -1.,  1.,  1.,  1.,
        -1., -0., -1., -1.,  1., -1.,  1.,  1., -1.,  1.,  1.,  0.,  1.,
         1.,  1., -1.,  1., -1., -1.,  1., -0., -1., -0.,  1., -1.,  1.,
         1.,  1.,  1., -1., -1.,  1.,  1.],
       [ 1.,  1.,  1., -1., -1., -1.,  1., -1.,  1.,  1., -1.,  1., -1.,
         1., -0., -1.,  1.,  1., -1., -1.,  1., -1.,  1.,  1., -0.,  1.,
        -1., -1.,  1.,  1., -1.,  1., -1., -0., -1., -0.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1., -1.],
       [ 0., -0., -0., -0., -0.,  1.,  0., -1.,  1.,  0., -0., -0., -0.,
         0.,  0.,  0.,  1., -0.,  0., -0., -0.,  1., -1., -0.,  1., -1.,
        -1., -1.,  0., -1.,  1.,  0., -0., -1.,  0.,  0., -0.,  1., -0.,
        -0., -0., -0.,  1.,  0., -1., -0.],
       [-1., -1., -1.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,  1., -1.,
        -1., -0., -1., -1., -1., -1.,  1.,  1.,  1., -1.,  1., -0., -1.,
        -1.,  1., -1., -1.,  1.,  1.,  1., -0.,  1.,  0., -1., -1., -1.,
         1., -1.,  1., -1., -1., -1., -1.],
       [ 1., -1.,  1., -1.,  1., -1.,  1., -1., -1., -1.,  1.,  1., -1.,
         1.,  0.,  1., -1., -1., -1., -1., -1., -1.,  1.,  1., -0., -1.,
         1.,  1.,  1.,  1., -1., -1.,  1., -0.,  1.,  0., -1., -1., -1.,
         1.,  1.,  1., -1., -1.,  1.,  1.],
       [ 1., -1., -1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1., -1., -1.,
        -1.,  0.,  1.,  1., -1., -1., -1., -1.,  1., -1., -1.,  0., -1.,
        -1., -1.,  1., -1.,  1.,  1., -1., -0., -1.,  0., -1., -1., -1.,
        -1., -1., -1.,  1.,  1.,  1., -1.],
       [ 1.,  1.,  1.,  1., -1., -1.,  1., -1.,  1.,  1., -1., -1., -1.,
         1., -0., -1.,  1.,  1., -1., -1.,  1.,  1.,  1.,  1., -0.,  1.,
        -1.,  1.,  1.,  1., -1.,  1., -1., -0., -1.,  0.,  1.,  1., -1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.],
       [ 0., -1., -0., -0.,  0.,  1.,  0., -1.,  0.,  1., -0., -0., -0.,
        -0.,  1.,  0.,  1., -0., -0., -1., -0.,  1., -0., -1.,  0., -1.,
        -1., -1.,  0., -1.,  1.,  0., -0., -0., -0.,  1., -0., -1., -0.,
        -0., -0., -0.,  1.,  0.,  1., -0.],
       [-1.,  1.,  1., -1.,  1.,  1., -1., -1.,  1., -1., -1., -1.,  1.,
         1., -0.,  1., -1., -1.,  1., -1., -1., -1.,  1., -1.,  0.,  1.,
         1., -1.,  1., -1., -1., -1., -1., -0., -1.,  0.,  1.,  1.,  1.,
        -1.,  1.,  1.,  1.,  1.,  1.,  1.]], dtype=np.float32), np.array([[ 0., -1.,  1., -0., -0.,  0.,  0.,  1.,  1., -0., -0.,  0.,  0.,
        -0.,  1., -0., -0., -0., -0., -1.,  0., -0.,  1., -0., -1., -1.,
        -0.,  0., -2.,  0., -2., -0.,  1., -1., -0., -1.,  2.,  0.,  0.,
        -0., -0., -0., -2., -0., -0.,  0.],
       [-0., -0., -1.,  0., -0., -0., -1.,  0.,  0., -0., -0.,  0.,  1.,
         0.,  0., -1.,  0.,  0., -0., -0.,  1.,  0.,  0., -0., -0., -0.,
         0., -0.,  1., -0., -0., -1., -1., -0.,  0., -0., -1., -0., -0.,
         1.,  1., -0., -0.,  1.,  0., -0.],
       [-0., -1., -1.,  0.,  0., -0., -0., -1.,  1., -0.,  0., -0., -0.,
         0.,  1.,  0., -0., -0.,  0.,  1.,  0., -0., -1.,  0., -1., -1.,
        -0.,  0.,  0., -0., -0., -0.,  1.,  1., -0.,  1.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -0.],
       [-0., -0.,  1., -0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0., -1.,
         0.,  0., -1.,  0.,  0.,  0.,  0.,  1., -0., -0., -0., -0., -0.,
        -0., -0.,  1., -0.,  0.,  1., -1.,  0., -0.,  0.,  1.,  0., -0.,
         1., -1., -0., -0.,  1.,  0.,  0.],
       [-0., -0., -1., -0., -0., -0.,  1., -0.,  0.,  0.,  0.,  0., -1.,
        -0.,  0., -1., -0., -0.,  0.,  0.,  1.,  0., -0., -0., -0., -0.,
        -0.,  0., -1., -0., -0.,  1.,  1.,  0.,  0., -0., -1., -0.,  0.,
         1.,  1., -0.,  0., -1., -0., -0.],
       [-0.,  0., -1., -0.,  0., -0.,  1., -0., -0.,  0.,  0., -0., -1.,
        -0.,  0.,  1.,  0.,  0.,  0.,  0., -1., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  1., -0.,  0.,  1., -1.,  0., -0., -0., -1.,  0., -0.,
        -1.,  1.,  0.,  0.,  1., -0.,  0.],
       [-0., -1.,  0., -0.,  0., -0.,  0.,  0., -1.,  0., -0., -0., -0.,
        -1.,  1., -0.,  0.,  0.,  0., -1., -0.,  0., -1., -0.,  1., -0.,
         0., -0., -0., -0., -0.,  0.,  0.,  1., -1., -1.,  0.,  1.,  0.,
        -0.,  0., -0., -0.,  0., -1., -0.],
       [ 0.,  0., -0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,  0., -0., -1.,
        -1., -0.,  0., -0.,  0.,  1.,  0., -1., -0.,  0.,  0., -0.,  0.,
        -0., -0.,  1., -0., -0.,  1., -0., -0., -1.,  0., -1., -0., -0.,
        -1.,  1.,  0., -0.,  1.,  0.,  0.],
       [-0.,  1.,  0., -0., -0., -0., -0.,  0.,  1., -0., -0.,  0., -0.,
        -1., -1., -0., -0., -0., -0., -1.,  0.,  0., -1., -0., -1.,  0.,
        -0., -0., -0., -0., -0., -0., -0.,  1.,  1., -1., -0., -1.,  0.,
        -0.,  0., -0., -0., -0., -1., -0.],
       [-0.,  0.,  0.,  0.,  1., -0., -0., -0.,  0.,  0.,  0.,  0.,  1.,
         1., -0.,  0., -0.,  0., -1., -0., -1.,  0., -0., -0., -0.,  0.,
        -0.,  0., -1., -0.,  0.,  1.,  0.,  0., -1., -0., -1., -0.,  0.,
         1., -1., -0., -0.,  1., -0.,  0.],
       [ 0.,  0., -0., -0.,  1., -0., -0.,  0., -0.,  0.,  0.,  0.,  1.,
        -1., -0., -0.,  0.,  0., -1., -0., -1., -0.,  0., -0.,  0., -0.,
        -0., -0.,  1.,  0., -0.,  1.,  0., -0.,  1., -0.,  1., -0., -0.,
         1.,  1., -0.,  0., -1., -0., -0.],
       [ 0.,  0., -0., -0., -1., -0., -0., -0.,  0., -0., -0.,  0.,  1.,
        -1.,  0., -0., -0., -0., -1.,  0.,  1., -0.,  0., -0., -0.,  0.,
         0.,  0.,  1.,  0., -0., -1., -0., -0., -1., -0., -1.,  0.,  0.,
         1.,  1., -0., -0.,  1., -0., -0.],
       [-0.,  0., -0.,  1., -0.,  1., -0., -0., -0., -0., -0., -0.,  1.,
         0.,  0.,  0., -0., -0.,  0., -0., -1.,  1.,  0.,  0.,  0., -0.,
         1.,  1., -1., -0., -1., -0., -0., -0., -0.,  0., -1.,  0.,  1.,
        -0.,  0.,  0.,  1., -0.,  0., -0.],
       [ 0., -0.,  0., -1., -0.,  0.,  0.,  0., -0.,  0., -0.,  1., -1.,
         0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0., -0., -0.,  0., -0.,
         0.,  0.,  1.,  0., -0.,  1., -0., -0., -0.,  0.,  1.,  0., -1.,
         1., -1.,  0.,  0.,  1.,  0.,  1.],
       [-0., -0.,  0., -1.,  0.,  1., -0.,  0., -0.,  0.,  0.,  0.,  1.,
         0.,  0.,  0.,  0.,  0.,  0., -0.,  1., -1., -0.,  0., -0., -0.,
        -1.,  1.,  1., -0.,  1., -0.,  0., -0.,  0., -0., -1., -0.,  1.,
         0.,  0.,  0.,  1., -0.,  0.,  0.],
       [ 0., -0., -0., -1., -0., -0., -0.,  0.,  0., -0., -0.,  1.,  1.,
        -0.,  0., -0., -0., -0., -0.,  0.,  1.,  0.,  0., -0., -0., -0.,
         0., -0.,  1.,  0., -0., -1., -0., -0., -0., -0., -1.,  0.,  1.,
         1.,  1., -0., -0.,  1., -0., -1.],
       [ 0.,  0., -0., -1.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1.,  1.,
         0.,  0.,  0., -0., -0., -0.,  0.,  1., -0., -0.,  0.,  0.,  0.,
        -0.,  0.,  1., -0.,  0.,  1.,  0., -0.,  0., -0., -1., -0.,  1.,
        -1., -1.,  0.,  0., -1., -0.,  1.],
       [-0.,  0.,  0.,  1.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  1.,  1.,
         0.,  0., -0., -0., -0., -0., -0., -1.,  0., -0., -0.,  0.,  0.,
         0.,  0., -1., -0., -0.,  1., -0., -0., -0.,  0., -1., -0.,  1.,
         1., -1.,  0.,  0.,  1.,  0.,  1.],
       [ 0.,  0., -0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0.,
        -0.,  1.,  0., -0., -0.,  0., -0.,  0., -0., -0.,  0., -1.,  0.,
        -0.,  0.,  0.,  0.,  0., -0., -0., -1., -0., -1., -0.,  0.,  0.,
        -0., -0.,  0.,  0., -0., -0., -0.],
       [ 0., -0., -0., -0., -0.,  1., -0.,  0.,  1.,  1., -0., -0.,  0.,
        -0.,  1.,  0., -0., -0., -0., -0.,  0., -0.,  1.,  1.,  1., -0.,
        -1., -0.,  0., -0., -1.,  1.,  0.,  1.,  0., -1.,  0.,  0.,  0.,
        -1.,  1., -0., -1.,  1., -0., -0.],
       [ 0., -0., -0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0.,  0.,
        -0., -1.,  0.,  0.,  0., -0., -0., -0.,  0., -0.,  0.,  1., -0.,
         0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  0., -1.,  0., -0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  0.],
       [-0.,  0.,  0., -0.,  0.,  1., -0., -0., -1., -1.,  0.,  0., -0.,
         0., -1., -0.,  0., -0., -0., -0., -0.,  0.,  1.,  1., -1., -0.,
         1.,  0.,  0., -0., -1., -1., -0.,  1., -0., -1.,  0.,  0.,  0.,
        -1.,  1., -0.,  1., -1.,  0.,  0.],
       [-0., -0.,  0., -0.,  0.,  1.,  0., -0.,  1., -1.,  0.,  0., -0.,
         0.,  1., -0.,  0., -0., -0.,  0.,  0.,  0., -1.,  1., -1., -0.,
         1., -0., -0.,  0.,  1., -1.,  0.,  1.,  0.,  1., -0., -0., -0.,
        -1., -1.,  0., -1.,  1., -0.,  0.],
       [-0., -0.,  0.,  0., -0.,  1.,  0., -0., -1.,  1., -0.,  0.,  0.,
         0., -1.,  0., -0.,  0., -0.,  0.,  0., -0., -1.,  1.,  1.,  0.,
        -1.,  0., -0.,  0.,  1.,  1., -0.,  1., -0.,  1.,  0., -0., -0.,
        -1., -1., -0.,  1., -1.,  0., -0.],
       [ 1., -0., -0., -0., -0.,  1., -0.,  0.,  0., -0., -0.,  0.,  1.,
        -0., -0., -0.,  1.,  1., -0.,  0., -1., -0., -0.,  0.,  0.,  0.,
         1., -0.,  1.,  1.,  1., -0.,  0.,  0., -0., -0.,  1.,  0., -0.,
        -0., -0.,  0., -1.,  0., -0., -0.],
       [ 1., -0.,  0., -0.,  0., -0., -0., -0., -0., -0., -1., -0.,  1.,
         0., -0.,  0., -0.,  1., -0.,  0., -1.,  0., -0., -0., -0.,  0.,
        -0., -0.,  1., -0., -0., -1., -0., -0., -0.,  0.,  1., -0., -0.,
        -1., -1.,  1.,  0.,  1., -0.,  0.],
       [ 1.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0., -0.,  1.,
        -0.,  0.,  0., -1., -1.,  0., -0.,  1.,  0., -0.,  0., -0., -0.,
        -1.,  0.,  1.,  1.,  1.,  0.,  0.,  0.,  0., -0., -1., -0., -0.,
        -0., -0.,  0.,  1., -0., -0.,  0.],
       [ 1., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  1., -0.,  1.,
         0., -0.,  0.,  0., -1.,  0., -0.,  1., -0., -0., -0.,  0.,  0.,
         0., -0.,  1., -0., -0.,  1.,  0., -0.,  0.,  0., -1., -0.,  0.,
        -1., -1.,  1., -0., -1.,  0.,  0.],
       [ 1.,  0., -0., -0., -0.,  0., -0., -0.,  0., -0., -1.,  0.,  1.,
        -0.,  0., -0., -0., -1., -0.,  0.,  1.,  0.,  0., -0.,  0., -0.,
        -0.,  0.,  1.,  0.,  0., -1., -0., -0., -0., -0., -1., -0.,  0.,
         1.,  1., -1.,  0.,  1., -0., -0.],
       [ 1., -0., -0., -0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0.,  1.,
        -0., -0., -0.,  0.,  1., -0.,  0., -1.,  0.,  0., -0.,  0., -0.,
         0., -0.,  1.,  0.,  0.,  1.,  0., -0.,  0., -0.,  1., -0., -0.,
         1.,  1., -1., -0., -1., -0., -0.]], dtype=np.float32), np.array([[ 0.,  1., -0.,  0., -0.,  1., -0.,  1., -0., -1., -0.,  0.,  0.,
        -0., -0., -0.,  1.,  0.,  0.,  1.,  0.,  1.,  0.,  1., -1., -1.,
         1.,  1., -0.,  1., -0.,  0., -0.,  1., -0.,  0.,  0.,  1., -0.,
         0., -0., -0., -0.,  0.,  1., -0.],
       [-0.,  0., -0., -0.,  0.,  0., -0.,  1., -1., -0.,  0., -0., -0.,
         0., -1., -0., -1., -0., -0.,  0., -0., -1.,  1.,  0., -0., -1.,
         0., -1.,  0., -1., -1.,  0., -0.,  0.,  0.,  1.,  0., -1.,  0.,
         0., -0.,  0., -1., -0., -1.,  0.],
       [-1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1., -1.,
         1.,  0.,  1.,  1., -1., -1.,  1., -1.,  1., -1.,  1., -0., -1.,
         1.,  1.,  0.,  1.,  0.,  0.,  1.,  0.,  1., -0.,  0.,  1.,  1.,
         0.,  1.,  1.,  0., -1.,  1.,  1.],
       [ 1., -1.,  1.,  1., -1., -0.,  1.,  1., -1.,  1., -1.,  1.,  0.,
        -1.,  0.,  1., -1.,  1.,  1., -1.,  0., -1.,  1., -1., -0., -1.,
        -0., -1., -1., -1., -1., -1.,  1., -0., -1., -0., -1., -1., -1.,
        -1., -0., -1., -1.,  0., -1., -1.],
       [ 0., -1., -0., -0.,  0.,  1., -0.,  1.,  0.,  1.,  0., -0.,  0.,
        -0.,  0.,  0., -1., -0.,  0.,  1., -0., -1.,  0.,  1.,  1.,  1.,
        -1.,  1., -0.,  1., -0.,  0.,  0.,  1.,  0.,  0., -0., -1., -0.,
         0., -0., -0.,  0., -0.,  1., -0.],
       [-0., -0.,  0., -0., -0.,  0.,  0., -1.,  1.,  0., -0., -0., -0.,
         0.,  1., -0.,  1.,  0., -0.,  0., -0., -1.,  1.,  0.,  0., -1.,
        -0.,  1.,  0., -1., -1., -0., -0.,  0., -0.,  1., -0.,  1., -0.,
         0., -0.,  0.,  1.,  0., -1., -0.],
       [-1., -1.,  1.,  1., -1.,  1.,  1.,  1., -1.,  1., -1.,  1., -1.,
         1.,  0., -1., -1.,  1., -1.,  1.,  1., -1., -1.,  1.,  0.,  1.,
        -1.,  1.,  0.,  1.,  0., -0., -1.,  0., -1.,  0., -0., -1.,  1.,
         0.,  1.,  1., -0.,  1.,  1.,  1.],
       [ 1.,  1., -1.,  1.,  1., -0., -1., -1.,  1., -1.,  1.,  1.,  0.,
        -1., -0.,  1.,  1., -1.,  1., -1., -0., -1.,  1., -1.,  0., -1.,
         0.,  1., -1., -1., -1.,  1.,  1.,  0.,  1., -0.,  1.,  1.,  1.,
        -1., -0., -1.,  1., -0., -1.,  1.],
       [-1.,  1., -1.,  1., -1.,  1.,  1., -1.,  1.,  1.,  1., -1., -1.,
        -1., -0., -1., -1.,  1., -1., -1.,  1., -1.,  1.,  1., -0., -1.,
        -1.,  1., -0.,  1., -0., -0.,  1., -0.,  1., -0.,  0.,  1.,  1.,
         0., -1., -1.,  0., -1., -1., -1.],
       [-1., -1.,  1., -1.,  1., -0., -1.,  1., -1., -1.,  1.,  1.,  0.,
         1.,  0.,  1., -1.,  1.,  1.,  1., -0.,  1., -1., -1., -0.,  1.,
         0., -1.,  1.,  1.,  1.,  1., -1., -0., -1.,  0., -1., -1., -1.,
        -1.,  0., -1., -1.,  0.,  1.,  1.],
       [-1., -1., -1., -1.,  1.,  1.,  1., -1., -1., -1., -1.,  1., -1.,
        -1.,  0.,  1.,  1., -1., -1., -1., -1.,  1.,  1.,  1.,  0.,  1.,
         1.,  1.,  0.,  1., -0., -0., -1., -0., -1., -0.,  0., -1.,  1.,
        -0., -1., -1., -0.,  1., -1., -1.],
       [-1.,  1., -1., -1., -1., -0.,  1., -1.,  1.,  1., -1.,  1., -0.,
         1., -0.,  1.,  1., -1.,  1.,  1., -0.,  1., -1., -1., -0.,  1.,
        -0.,  1.,  1.,  1.,  1., -1., -1.,  0.,  1.,  0.,  1.,  1.,  1.,
        -1., -0., -1.,  1.,  0.,  1., -1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 2
m = 5
p = 6
rank = 46

verify_tensor_support_rank_decomposition(decomposition_256, n, m, p, rank)

## Rank-22 support rank decomposition of <3,3,3> over Z


In [ ]:
#@title Data

decomposition_333 = (np.array([[ 1., -0.,  1.,  1.,  0.,  0., -0.,  1., -0., -0., -0.,  1.,  1.,
        -0.,  1.,  1., -2.,  1., -0., -0., -0.,  1.],
       [ 2.,  1., -0., -0.,  0., -1.,  0.,  0.,  0.,  0., -1.,  0.,  2.,
        -0., -0.,  0., -1., -0., -2., -1., -1.,  1.],
       [ 1.,  0.,  0.,  1., -1., -1.,  0.,  1., -2., -1.,  1.,  1.,  1.,
         2., -1.,  1., -1., -1., -0.,  1., -1.,  1.],
       [ 1.,  0., -1., -1.,  0., -0.,  0., -1.,  0.,  0.,  0., -1., -1.,
         0., -1., -1.,  0.,  1., -0., -0.,  0., -1.],
       [-0.,  1.,  0., -0., -0., -1., -1.,  0., -0.,  0., -1.,  0., -0.,
         0., -0.,  0., -1., -0.,  0.,  0., -1.,  1.],
       [ 1.,  0., -0., -1., -1.,  1.,  1., -1., -0.,  1.,  1., -1., -1.,
         0., -1., -1.,  1.,  1.,  0., -0.,  1., -1.],
       [ 1., -0., -1., -1.,  0.,  0., -0., -1.,  0.,  0., -0.,  1.,  1.,
        -0., -1.,  1., -0., -1., -0.,  0.,  0.,  1.],
       [ 0.,  1.,  0., -0.,  0., -1.,  0.,  0.,  0.,  0., -1., -2.,  0.,
        -0.,  0., -0.,  1.,  0., -0., -1.,  1., -1.],
       [ 1.,  0.,  0.,  1., -1.,  1.,  0., -1., -0., -1.,  1.,  1.,  1.,
         2., -1.,  1., -1., -1., -0.,  1., -1.,  1.]], dtype=np.float32), np.array([[-1., -0.,  0.,  0., -1.,  0.,  0., -0.,  0., -0.,  0., -0., -1.,
         1., -1., -0., -0.,  1.,  1.,  0.,  0., -0.],
       [ 0.,  0., -1., -1.,  0.,  0., -0., -0., -0., -1.,  0., -0.,  0.,
        -0.,  0.,  1., -0., -1., -0.,  0.,  0.,  0.],
       [-0.,  0., -0.,  1.,  0., -0., -0.,  0.,  0.,  1.,  0.,  0., -0.,
        -0., -0., -1., -1.,  1.,  0., -0.,  1., -1.],
       [ 0., -0., -0., -0., -1., -0.,  1.,  0.,  0., -0., -1., -0.,  0.,
         1.,  0.,  0.,  0., -0., -1.,  1., -0.,  0.],
       [-0.,  1.,  0.,  0.,  1., -0., -1., -0.,  0.,  0.,  1.,  1.,  1.,
        -1.,  0., -1., -0.,  0.,  0., -1., -0., -1.],
       [ 0., -0., -0., -0., -0., -0., -1.,  0.,  0., -1., -0., -1., -1.,
         1., -0.,  1.,  0., -0., -0.,  1., -1.,  1.],
       [-0., -0., -0., -0., -1., -0., -0.,  0., -1., -0.,  0., -0., -0.,
         1., -0.,  0.,  0., -0.,  0., -0., -0.,  0.],
       [-0., -0.,  1.,  1., -0., -0., -0., -1., -1.,  1., -0.,  0.,  0.,
         0.,  1., -0., -0.,  0.,  0., -0.,  0., -0.],
       [ 0., -1., -1., -1., -1., -1., -0.,  1.,  1., -0., -1., -0.,  0.,
        -0., -1.,  0.,  0., -0., -0.,  0., -0.,  0.]], dtype=np.float32), np.array([[ 1., -0., -0.,  0., -0.,  0., -0.,  1.,  1.,  0., -0.,  0., -0.,
         0.,  1., -0.,  0.,  0., -1., -0., -0.,  0.],
       [-1., -2., -0.,  0., -2.,  0.,  0., -0.,  0.,  0., -2., -1.,  1.,
         1., -0., -0., -0., -0., -0., -2., -0.,  0.],
       [-0., -0.,  0., -0., -0., -0.,  0., -1., -1.,  0.,  0., -1.,  1.,
         1., -1., -0.,  0., -0., -1., -2., -0.,  0.],
       [-1.,  0., -0.,  0.,  0.,  1., -1., -1., -0.,  0., -1.,  0.,  1.,
        -0., -1.,  1.,  1., -1.,  0., -1.,  1., -0.],
       [ 1.,  1., -1., -0.,  0.,  0.,  1., -0., -0., -1.,  1.,  0., -1.,
        -0.,  1.,  0., -1.,  1.,  0.,  1.,  0., -1.],
       [ 0.,  1., -1.,  0.,  0.,  1., -0., -1.,  0.,  1., -0.,  0., -0.,
        -0.,  0., -1., -0.,  0., -0., -0., -1.,  1.],
       [-0.,  0.,  0., -0., -0., -1.,  1.,  0.,  0., -0.,  1.,  0., -0.,
         0.,  0.,  0., -1., -0.,  0.,  1., -1., -0.],
       [ 0.,  1.,  1., -1., -0.,  0.,  1.,  0.,  0., -1.,  1.,  1., -0.,
        -0.,  0., -0.,  1.,  0.,  0.,  1., -0.,  1.],
       [-0.,  1., -1.,  1.,  0.,  1., -0.,  0., -0.,  1.,  0.,  1., -0.,
        -0., -0.,  0., -0., -0.,  0.,  0., -1.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 3
p = 3
rank = 22

verify_tensor_support_rank_decomposition(decomposition_333, n, m, p, rank)

## Rank-28 support rank decomposition of <3,3,4> over Z


In [ ]:
#@title Data

decomposition_334 = (np.array([[ 2.,  1., -0., -0., -0., -0., -1., -0., -1., -0., -2., -0., -1.,
        -1., -1., -1.,  1., -1., -1.,  2.,  2.,  0., -0., -1.,  1., -0.,
        -0., -0.],
       [-0.,  0., -1., -0.,  1.,  1.,  1.,  0., -1.,  0., -0.,  1., -1.,
         1.,  1., -1., -0., -0.,  1., -0.,  0., -1., -1., -0.,  1.,  2.,
         1., -0.],
       [-2.,  0.,  0., -0.,  0., -0., -1., -0.,  1., -0., -0.,  1.,  1.,
         1.,  1., -1.,  0., -1., -1., -0., -0., -0., -0., -0.,  1., -2.,
        -0., -0.],
       [-0., -1.,  0., -1., -0., -0.,  1., -1., -1., -1., -0.,  1., -0.,
         1., -0., -1.,  1., -0., -0.,  2., -2.,  0., -1.,  0., -0.,  2.,
         1., -2.],
       [-2., -0., -1., -1., -1.,  1., -1., -1.,  1.,  1., -2., -0.,  0.,
         1., -0., -1.,  0., -1.,  0.,  0., -0.,  1., -0.,  1., -0., -0.,
        -0.,  2.],
       [ 0.,  0., -0.,  1., -0., -0.,  1., -1.,  1., -1.,  2., -0., -0.,
         1.,  0.,  1.,  0., -0., -0.,  0.,  0., -0.,  1.,  1.,  0., -0.,
         1.,  2.],
       [-0.,  1., -0., -0., -0., -0.,  1., -0., -1.,  0.,  2., -1.,  0.,
        -1.,  0.,  1.,  1., -0., -0., -2., -2.,  0., -0., -1.,  0.,  2.,
        -0., -0.],
       [ 2., -0., -1., -0., -1., -1., -1., -0., -1., -0.,  0., -0.,  0.,
        -1., -0., -1., -0., -1.,  0., -0.,  0., -1., -1.,  0., -0., -0.,
        -1., -0.],
       [-0., -0., -0., -0., -0., -0.,  1.,  0., -1.,  0., -0.,  0.,  0.,
        -1., -0.,  1., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0.]], dtype=np.float32), np.array([[ 0.,  2., -0., -0., -0.,  0., -0.,  1.,  0.,  0., -0., -2.,  0.,
        -0.,  1., -0.,  0., -0., -0.,  0., -0., -0., -0., -2.,  1., -0.,
         0., -1.],
       [-0.,  0., -0., -1., -0.,  0.,  0., -0.,  0., -1., -1., -0., -1.,
         0.,  0., -0.,  0.,  0.,  1., -0., -1., -0.,  0.,  0., -0.,  1.,
        -0., -0.],
       [-0.,  0., -0., -1., -0., -0., -0., -0.,  0., -1., -1.,  2.,  0.,
         0., -1., -0., -0.,  0., -0.,  1.,  0., -0.,  0., -0., -1., -0.,
        -0.,  0.],
       [ 0.,  0., -0., -0., -0., -0., -0., -1., -0., -0., -0., -0.,  1.,
         0., -0., -0.,  2., -0., -1., -0., -0.,  0.,  0.,  2., -0., -1.,
        -0.,  1.],
       [-1., -0., -0., -0., -0., -2., -0.,  1., -0.,  1., -0.,  0.,  1.,
        -0.,  1., -0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -2., -0.],
       [-0., -0.,  0., -0.,  2.,  0., -0.,  1.,  0.,  1., -0., -0., -0.,
         0.,  0., -0.,  0.,  2., -1., -0., -0.,  0., -0.,  0.,  1.,  0.,
        -2.,  0.],
       [-0.,  0., -2.,  1., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
        -0.,  0.,  0.,  0.,  2., -1.,  0., -0., -0.,  2., -0.,  1.,  0.,
         0., -1.],
       [-1., -0.,  0., -1., -0.,  0., -0., -0., -0., -0.,  0., -0.,  1.,
        -0.,  1., -0., -0.,  0., -0., -0.,  0., -2., -2., -0., -0.,  0.,
         0.,  1.],
       [ 1., -2., -0., -0., -0.,  2., -0.,  0.,  0., -1.,  0.,  2., -1.,
        -2.,  0., -0.,  0., -0., -0.,  0., -0., -0., -0.,  2., -1.,  0.,
         2.,  1.],
       [-0., -0., -0., -1., -2., -0., -2., -1., -0.,  0., -1., -0., -1.,
        -0.,  0., -0., -0., -2., -0., -0., -1., -0., -0., -0., -1.,  1.,
         2., -0.],
       [-0., -0., -2., -0., -0., -0., -0., -0., -0.,  1.,  1., -2., -0.,
        -0.,  1., -2., -0.,  2., -1., -1., -0., -0.,  2., -0., -0.,  0.,
         0., -1.],
       [-1., -0., -0., -1., -0., -0.,  0., -1.,  2., -0., -0., -0., -0.,
        -0.,  1., -0.,  2., -0., -1.,  0., -0., -2., -2.,  2., -0., -1.,
        -0., -0.]], dtype=np.float32), np.array([[ 1.,  1., -0., -0.,  0.,  1.,  0., -0., -0., -0., -0., -1.,  0.,
         0., -2.,  0., -0., -0., -0.,  1.,  0.,  1., -0.,  0.,  0., -0.,
         0., -0.],
       [-0.,  1., -0., -0., -1.,  1.,  0.,  2., -0., -0., -0., -0., -0.,
         0.,  0., -0., -1., -0., -0.,  0., -0.,  0., -0., -1., -0.,  0.,
        -1., -0.],
       [ 1., -0., -0., -0., -1., -0.,  0.,  2., -0., -0., -0., -1., -0.,
        -2., -2.,  0., -1., -0., -0.,  1., -0.,  1., -0., -1., -0.,  0.,
        -1., -0.],
       [-0., -0., -1., -0.,  1., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
        -0.,  0.,  0.,  1., -1., -2., -0., -1., -0.,  0., -0.,  0.,  1.,
        -0., -0.],
       [ 0., -0.,  0., -0.,  1., -1., -0.,  0., -0., -2., -1.,  0., -0.,
        -0., -0.,  0., -0.,  0., -0.,  1., -1., -0., -0., -0., -0., -0.,
         1., -0.],
       [-0., -0.,  1., -0.,  0.,  1., -2., -0., -0.,  2.,  1., -0., -0.,
        -0.,  0.,  0., -1.,  1.,  2., -1., -0., -0., -0., -0., -0., -1.,
        -1., -0.],
       [-0.,  1., -1., -0.,  1.,  0., -0., -0., -0., -0.,  0., -1., -0.,
        -0.,  0.,  0., -0., -1.,  0.,  1., -0., -0.,  0.,  0., -2., -0.,
        -0., -0.],
       [-0., -0., -1.,  2., -0., -0., -0.,  0.,  0.,  0., -1.,  0., -0.,
        -0.,  0., -0., -0.,  0.,  0.,  1., -1.,  1., -1., -0.,  0., -0.,
        -0.,  0.],
       [ 0., -1., -0., -2., -1.,  0., -0.,  0., -0., -0.,  1.,  1., -0.,
        -0.,  0.,  2., -0.,  1., -0.,  0.,  1., -1.,  1., -0.,  2.,  0.,
         0., -0.],
       [ 1., -0., -0., -0., -0.,  1., -0.,  0.,  0.,  0., -0.,  0., -2.,
        -0.,  0., -0.,  1., -0.,  0., -0., -1.,  1.,  0., -0., -0.,  1.,
        -0.,  0.],
       [ 0., -1., -1., -0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
        -0., -0., -0.,  1., -0., -0., -0.,  0.,  1., -1.,  1., -0., -0.,
        -0., -1.],
       [ 1., -1., -1.,  0.,  0.,  1., -0.,  0., -2., -0.,  0.,  0., -2.,
        -0., -0., -0., -0., -0., -0., -0., -1., -0., -1.,  1., -0.,  1.,
        -0., -1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 3
p = 4
rank = 28

verify_tensor_support_rank_decomposition(decomposition_334, n, m, p, rank)

## Rank-35 support rank decomposition of <3,3,5> over Z


In [ ]:
#@title Data

decomposition_335 = (np.array([[-0., -1.,  1.,  1., -0., -0., -0., -0., -0., -1., -0.,  0.,  0.,
        -1., -1., -0., -0., -0., -0., -0., -1., -0.,  0., -0.,  0., -0.,
         0.,  0., -0., -0.,  0.,  0.,  1., -1.,  0.],
       [ 0., -0., -0.,  0.,  0.,  0., -0., -1., -0.,  1.,  0., -0.,  0.,
        -0., -0., -0.,  1., -0.,  0., -1., -0.,  0., -0., -0., -1., -0.,
        -1., -0.,  0., -0.,  0.,  1.,  0.,  1.,  1.],
       [-0., -1.,  1., -0.,  0.,  0., -0.,  0.,  1., -0., -0.,  0.,  0.,
        -1.,  0., -0.,  1.,  1., -0.,  0.,  0., -0., -0.,  1., -1.,  0.,
        -1., -0.,  0.,  1., -0., -0., -0., -0., -0.],
       [-0., -1.,  1.,  1., -0.,  0.,  1., -0., -0., -1.,  1., -0., -1.,
        -1., -1., -0., -0., -0.,  1.,  0., -1.,  1., -0.,  0.,  0.,  0.,
        -0., -0.,  0.,  1., -0.,  0., -0.,  0., -0.],
       [ 1., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -0., -1.,  0.,
        -0.,  0.,  1.,  1., -0.,  1., -0., -1., -0.,  1.,  1., -0.,  0.,
         0., -0., -1., -0., -0.,  0.,  0., -0., -0.],
       [-0., -1.,  1.,  1.,  0.,  0., -0., -0., -0.,  0.,  1.,  0., -1.,
        -1.,  0.,  1.,  1.,  1., -0., -0.,  0.,  1., -0.,  1., -1., -1.,
         0., -0.,  0.,  1., -0., -0., -0., -0.,  0.],
       [ 0., -1., -0., -0.,  0., -1.,  1.,  0., -0., -0.,  1.,  1.,  0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0.,  1., -0., -0., -0., -0.,
         0.,  1.,  0.,  1., -0., -0.,  0.,  1.,  1.],
       [ 0., -0.,  0., -0.,  1., -0.,  0.,  1., -0., -0.,  0.,  0.,  0.,
         0.,  0., -1., -1.,  0.,  0., -0., -0.,  0., -1., -1.,  1., -0.,
         1., -1.,  1.,  0.,  0., -1., -0., -1., -1.],
       [ 0., -1.,  1., -0., -1.,  0.,  0.,  0., -0.,  0.,  1., -0.,  0.,
         0., -0.,  1.,  1.,  1.,  0.,  0., -0., -0., -0.,  1., -1.,  0.,
        -1., -0., -1.,  1.,  1.,  1., -0.,  0., -0.]], dtype=np.float32), np.array([[ 0., -0.,  1., -0., -0.,  0., -0., -1.,  0., -0.,  0., -0.,  0.,
         0.,  1., -0.,  0.,  0., -0., -0.,  0.,  0., -0.,  0.,  1.,  0.,
         0., -0., -0., -0.,  0., -0., -1.,  1.,  1.],
       [-0., -1.,  1.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  0., -0.,
        -0., -1.,  0.,  0.,  1.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,
        -0.,  0., -0., -1., -0., -0.,  1.,  0.,  0.],
       [-1.,  0., -0., -0.,  0., -0., -1.,  0., -0., -0.,  0.,  1., -0.,
         0.,  1.,  0.,  0.,  0.,  1.,  0., -1., -0., -0.,  0., -0., -0.,
         0., -0.,  0., -0.,  0.,  0.,  0.,  0., -0.],
       [-0.,  0., -1., -1., -0.,  1.,  0.,  1.,  0., -0., -0.,  0.,  1.,
         0.,  0., -0., -0., -0., -0., -0.,  0.,  1.,  0.,  0., -1.,  1.,
         0.,  0.,  0.,  0., -0.,  0.,  1., -1., -1.],
       [ 0., -0., -1.,  0.,  0.,  1., -1.,  1.,  0.,  0.,  0.,  0., -0.,
        -0.,  0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0., -1., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0., -1., -1.],
       [-0., -0., -1., -0.,  0., -0.,  0.,  1., -0., -1., -0.,  0., -0.,
        -0., -1.,  0., -0.,  0.,  0., -1.,  1., -0., -0., -0., -1.,  0.,
         0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  0.],
       [ 0.,  0.,  1., -0., -0., -0., -0.,  0., -0.,  1., -0.,  0., -0.,
         0.,  1., -0., -1.,  1., -0.,  0., -1., -0.,  0., -1.,  2., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0.],
       [ 1.,  0., -1., -0.,  0.,  0., -0., -0., -0., -1., -0., -0., -0.,
         0., -1., -0.,  0.,  0.,  0.,  0.,  1.,  0.,  1., -0., -1., -0.,
         0., -0.,  0., -0., -0., -0.,  0.,  0., -0.],
       [-1., -0., -0., -0., -1.,  0.,  0.,  1., -0., -0., -0., -0., -0.,
        -0., -0., -0., -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,
         0., -0., -1.,  0., -1., -1., -0.,  0.,  0.],
       [-0., -0., -0.,  0., -0., -1., -0., -1., -0., -0., -0.,  1.,  0.,
         0.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  1.,  0., -0.,  0.,
        -0.,  1.,  0., -0., -0., -0., -0., -0.,  1.],
       [ 0.,  0., -1., -1., -0., -0., -0.,  1.,  0.,  0., -0., -0., -0.,
        -1.,  1.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  1., -0.,
        -1.,  0.,  0., -0.,  0., -1., -0., -0.,  0.],
       [ 0., -0., -1., -1.,  0.,  0.,  0., -0., -1.,  0.,  0.,  0., -0.,
        -1.,  1., -0., -0., -1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.],
       [-0.,  0.,  1.,  1., -0., -0., -0.,  0.,  1.,  0., -0.,  0.,  0.,
         1., -1., -1., -0., -0., -0., -0., -0.,  0., -1.,  1.,  0., -1.,
         0.,  0.,  1., -0.,  0., -0.,  0., -0.,  0.],
       [ 0.,  0.,  0., -0.,  0.,  0., -0.,  1., -0., -0., -0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,  1., -1.,
        -1., -0.,  0., -0., -1., -1.,  0., -0.,  0.],
       [-0.,  0., -0., -0., -0., -0., -1., -1., -1., -0., -1., -0.,  0.,
        -0., -0., -0.,  0., -0., -0., -0.,  0., -1., -0.,  0., -1., -0.,
         1.,  0.,  0., -1.,  1.,  1.,  0.,  0., -0.]], dtype=np.float32), np.array([[-0.,  1.,  0., -1., -0., -0., -0.,  0., -1.,  0., -1., -0.,  1.,
        -1., -0., -0.,  0., -0., -0., -1.,  0.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  0.,  1.,  0., -0., -1., -0., -0.],
       [ 1.,  1., -1., -1., -1., -1.,  1., -1., -1., -1., -1.,  1.,  1.,
        -1.,  1., -1.,  1.,  1.,  1., -1.,  1., -1.,  1.,  1., -1., -1.,
        -1., -1.,  1.,  1., -1.,  1., -1., -1.,  1.],
       [ 0.,  0., -0.,  0., -0.,  1.,  0.,  1.,  0., -0., -0.,  0.,  0.,
        -0.,  0., -0.,  0.,  0.,  0.,  1., -0., -0.,  0.,  0., -0., -0.,
         0.,  0., -0., -0.,  1., -1., -0.,  0., -1.],
       [-0.,  1.,  0.,  0., -0.,  0., -0., -0., -1.,  0., -1.,  0., -0.,
         0.,  0.,  1., -1., -0., -0.,  0.,  0., -0.,  0.,  1., -0., -0.,
        -0., -0.,  0.,  1., -0., -0.,  0., -0., -0.],
       [-1.,  1., -1.,  1.,  1., -1.,  1., -1., -1., -1., -1.,  1., -1.,
        -1., -1.,  1.,  1.,  1.,  1., -1., -1., -1.,  1.,  1., -1., -1.,
        -1., -1., -1.,  1., -1.,  1.,  1., -1.,  1.],
       [-1.,  1., -1.,  1.,  1., -1.,  1., -1.,  1., -1.,  1.,  1., -1.,
        -1., -1., -1.,  1., -1.,  1., -1., -1., -1.,  1., -1., -1., -1.,
        -1., -1., -1., -1., -1.,  1.,  1., -1.,  1.],
       [ 0.,  0., -0.,  0.,  0., -0.,  0., -0., -0.,  1., -0.,  0., -0.,
         0., -0., -1.,  1., -0., -1.,  1.,  1., -0., -0., -1., -0., -0.,
        -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0.],
       [ 1.,  0., -0.,  0., -1.,  0., -0.,  0.,  0.,  0., -0.,  0., -0.,
        -0., -0., -1.,  0., -0.,  1.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  1., -0., -0.,  0.,  0.,  0.,  0.],
       [ 1.,  0., -0.,  0., -1.,  0.,  0.,  0., -0., -0., -0., -1., -0.,
        -0., -0.,  0.,  0.,  0., -0.,  0., -0., -0., -1., -0., -0., -0.,
        -0.,  1.,  1.,  0.,  0.,  0.,  0., -0., -0.],
       [-0., -0., -0., -1.,  1., -0., -0.,  0., -1., -0.,  0.,  0.,  1.,
        -1.,  0., -0.,  0., -0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,
         1.,  0.,  0., -0.,  0.,  1.,  0.,  0., -0.],
       [ 0.,  0., -0., -0.,  1.,  0.,  0.,  0., -0., -0., -0., -0., -1.,
        -0.,  0.,  1.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -0.,  1.,
         0., -0., -1.,  0., -0.,  0., -0., -0.,  0.],
       [-0., -0., -0.,  0., -1., -0.,  0.,  0., -0., -0., -1.,  0.,  1.,
        -0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  1.,  0., -0., -0.,  0.,
         0.,  0., -0., -0., -1., -0.,  0., -0., -0.],
       [ 0., -1.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  1., -0., -0.,
        -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,
        -0., -1., -0., -1.,  0., -0.,  1., -1.,  1.],
       [-0.,  0., -0.,  0., -0.,  1., -1., -0., -0., -0.,  0., -1.,  1.,
         0.,  0., -0.,  0.,  0., -1., -0.,  0.,  1.,  0.,  0., -0., -0.,
        -0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.],
       [-0.,  0.,  0., -0., -0., -1., -0.,  0., -0., -0.,  1.,  0., -1.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0., -1.,  0., -0.,  0.,  0.,
         0.,  1., -0.,  0., -0., -0.,  0., -0., -0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 3
p = 5
rank = 35

verify_tensor_support_rank_decomposition(decomposition_335, n, m, p, rank)

## Rank-47 support rank decomposition of <3,3,7> over Z


In [ ]:
#@title Data

decomposition_337 = (np.array([[ 0., -0., -0.,  1.,  0.,  0.,  0.,  1., -0.,  0.,  1., -1., -0.,
         0., -0., -0.,  1., -0.,  0., -1.,  0.,  1., -1.,  0.,  0., -0.,
         0.,  1.,  1., -0., -0.,  0.,  1., -1.,  0.,  0.,  1.,  0.,  0.,
         0., -0., -1., -0.,  0.,  1., -1., -1.],
       [ 0., -1., -0.,  1., -1.,  0.,  1.,  1., -1., -1.,  0., -0., -1.,
        -0.,  1., -1.,  0., -1., -1., -1.,  1., -0., -1.,  1., -1., -0.,
        -0., -0.,  1.,  1.,  1.,  1.,  1.,  0.,  1.,  1., -0.,  0., -1.,
         1.,  0., -0.,  1.,  1.,  1., -1., -1.],
       [ 0., -1., -0., -0., -1.,  0.,  1., -0., -1., -1.,  1., -1., -1.,
         1.,  1.,  0.,  1., -0., -1., -0.,  1.,  1.,  0.,  1., -1., -1.,
         0., -0.,  0.,  1.,  1.,  1.,  0., -1.,  1.,  1.,  1., -0., -1.,
         1.,  0., -1.,  1.,  1.,  1., -1., -0.],
       [-0., -1.,  0., -0.,  0.,  1.,  0.,  0.,  0., -1.,  0.,  0., -0.,
         1.,  1., -0.,  1., -1., -1., -0., -0.,  0.,  0., -0., -0., -0.,
        -1.,  0.,  0.,  1., -0.,  1.,  1.,  0.,  1.,  1., -0.,  0., -0.,
         0., -1.,  0.,  1.,  1.,  0., -0.,  0.],
       [ 0., -0., -0.,  0.,  0., -1., -1.,  0., -0., -0., -0., -0.,  1.,
        -0., -0., -0., -0.,  1.,  1.,  1.,  0., -1., -0., -0.,  0., -0.,
         0.,  0., -1.,  0., -1., -1., -1., -0., -1., -0., -0.,  0.,  1.,
        -1.,  0., -0., -1., -1., -1., -0., -0.],
       [ 0.,  1., -0.,  0.,  1., -0., -1., -0., -0.,  1.,  0.,  1.,  0.,
        -1., -1.,  0., -1.,  0., -0.,  1., -0., -1.,  0.,  0.,  0.,  0.,
         1.,  0., -0., -1., -1., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,
        -1.,  0., -0., -0., -1., -1., -0., -0.],
       [ 0.,  1.,  0., -1., -0., -1., -1.,  0.,  0., -0., -1.,  1., -0.,
        -1., -1., -0., -1.,  1.,  1.,  1., -1., -1.,  1.,  0.,  0., -0.,
         1., -1., -1., -1.,  0., -1., -1.,  1., -1., -1., -1., -1.,  0.,
         0.,  1., -0.,  0., -1., -1.,  1.,  1.],
       [ 1., -1.,  1., -0.,  0.,  1.,  1., -0., -1.,  0.,  1., -1., -1.,
         1.,  0.,  0.,  1., -1., -1., -1.,  0.,  1.,  0.,  0.,  0., -1.,
        -1.,  1.,  1., -0., -0.,  0.,  1.,  0.,  1., -0.,  1.,  1., -1.,
         1., -1., -1.,  1.,  1.,  1.,  0., -1.],
       [-0.,  0., -1., -1., -0.,  0., -0.,  0.,  1.,  0.,  0.,  0.,  1.,
        -0., -1., -0., -0.,  0.,  0., -0., -1., -0.,  0., -1., -0.,  1.,
         0.,  0., -0., -0.,  0., -1., -0.,  1., -1., -1., -1., -0.,  1.,
        -0.,  0.,  1., -1.,  0., -0.,  1., -0.]], dtype=np.float32), np.array([[ 1., -0.,  0.,  0.,  1.,  0., -1.,  1.,  0.,  1., -0., -1., -1.,
         1., -0.,  1., -1.,  1., -0., -0.,  1.,  0., -1., -1.,  0., -1.,
         0., -1.,  1., -1., -0., -0.,  1., -0., -1., -1., -1.,  1.,  0.,
        -1., -1.,  1.,  1.,  1.,  1.,  1.,  0.],
       [ 1., -0.,  0.,  0.,  1.,  0.,  1.,  1.,  0.,  1.,  0., -1.,  1.,
         1.,  0.,  1., -1., -1.,  0.,  0.,  1.,  0., -1., -1., -0., -1.,
         0., -1., -1., -1., -0., -0., -1.,  0.,  1., -1., -1.,  1.,  0.,
         1., -1.,  1., -1., -1., -1.,  1.,  0.],
       [ 1., -0., -0.,  0.,  1.,  0., -1., -1.,  0.,  1.,  0.,  1.,  1.,
        -1.,  0., -1.,  1., -1.,  0., -0., -1., -0.,  1.,  1., -0., -1.,
        -0., -1., -1., -1., -0.,  0., -1., -0.,  1.,  1., -1.,  1., -0.,
        -1., -1.,  1., -1.,  1.,  1., -1.,  0.],
       [-1.,  0.,  0.,  0., -1.,  0., -1., -1., -0., -1.,  0.,  1.,  1.,
        -1.,  0., -1.,  1.,  1.,  0.,  0.,  1.,  0.,  1., -1., -0., -1.,
         0.,  1.,  1.,  1.,  0., -0.,  1.,  0.,  1., -1., -1., -1., -0.,
        -1.,  1.,  1., -1.,  1.,  1.,  1.,  0.],
       [ 1.,  0., -0., -0., -1., -0.,  1., -1., -0., -1., -0., -1.,  1.,
         1., -0., -1., -1., -1., -0., -0., -1., -0.,  1.,  1., -0., -1.,
        -0., -1., -1.,  1.,  0., -0., -1.,  0.,  1.,  1., -1.,  1.,  0.,
         1., -1.,  1., -1., -1., -1., -1.,  0.],
       [ 1., -0.,  0.,  0.,  1.,  0.,  1., -1., -0.,  1., -0.,  1., -1.,
        -1.,  0., -1.,  1.,  1.,  0., -0., -1., -0.,  1.,  1., -0., -1.,
         0., -1.,  1., -1., -0., -0.,  1.,  0., -1.,  1., -1.,  1., -0.,
         1., -1.,  1.,  1., -1., -1., -1., -0.],
       [-1.,  0., -0.,  0., -1.,  0.,  1.,  1.,  0., -1., -0., -1.,  1.,
         1.,  0.,  1., -1.,  1., -0., -0., -1.,  0., -1.,  1.,  0., -1.,
        -0.,  1.,  1.,  1., -0.,  0.,  1., -0.,  1.,  1., -1., -1.,  0.,
         1.,  1.,  1., -1., -1., -1., -1., -0.],
       [ 1.,  1., -0.,  0.,  1.,  1., -0.,  0., -1.,  0., -0., -1.,  1.,
         1.,  0., -1.,  0., -1.,  0.,  0.,  0., -1., -1.,  1.,  0., -1.,
         0.,  1., -1., -1., -1.,  1., -0.,  1.,  1.,  1.,  1.,  0., -0.,
        -1.,  1., -0., -0.,  1.,  1.,  1., -1.],
       [-1., -1.,  0., -0.,  1.,  1.,  0.,  0.,  1., -0., -0., -1., -1.,
        -1., -0.,  1., -0., -1.,  0.,  0.,  0., -1.,  1., -1.,  0.,  1.,
         0., -1.,  1.,  1., -1.,  1.,  0., -1.,  1.,  1., -1.,  0., -0.,
        -1.,  1., -0.,  0., -1.,  1., -1.,  1.],
       [ 1.,  1.,  0.,  0.,  1.,  1., -0., -0.,  1., -0.,  0.,  1., -1.,
         1.,  0., -1., -0., -1., -0.,  0.,  0.,  1.,  1., -1., -0.,  1.,
        -0., -1.,  1., -1., -1., -1., -0.,  1., -1., -1.,  1., -0., -0.,
        -1.,  1., -0., -0.,  1., -1.,  1.,  1.],
       [ 1.,  1., -0.,  0., -1., -1.,  0., -0.,  1., -0., -0.,  1., -1.,
         1.,  0., -1., -0.,  1., -0., -0., -0.,  1., -1., -1.,  0.,  1.,
        -0.,  1., -1., -1.,  1.,  1.,  0., -1.,  1.,  1., -1.,  0.,  0.,
         1., -1.,  0.,  0.,  1., -1., -1., -1.],
       [-1.,  1.,  0.,  0.,  1., -1.,  0., -0., -1., -0., -0.,  1.,  1.,
         1., -0.,  1.,  0.,  1.,  0.,  0., -0.,  1., -1.,  1.,  0., -1.,
         0.,  1., -1., -1., -1.,  1., -0., -1.,  1.,  1., -1.,  0., -0.,
        -1., -1.,  0.,  0.,  1., -1., -1., -1.],
       [-1., -1.,  0., -0.,  1.,  1., -0., -0., -1.,  0., -0.,  1.,  1.,
        -1.,  0.,  1.,  0., -1.,  0.,  0.,  0.,  1., -1.,  1.,  0., -1.,
         0.,  1., -1.,  1., -1., -1.,  0., -1., -1., -1., -1.,  0., -0.,
        -1.,  1.,  0.,  0., -1., -1., -1., -1.],
       [-1., -1., -0.,  0., -1., -1., -0.,  0.,  1.,  0., -0., -1., -1.,
        -1.,  0.,  1., -0.,  1.,  0.,  0., -0., -1., -1., -1.,  0.,  1.,
        -0.,  1., -1.,  1.,  1., -1., -0.,  1., -1., -1.,  1., -0., -0.,
         1., -1.,  0.,  0., -1.,  1.,  1., -1.],
       [ 1., -0., -1., -1., -1.,  0., -0.,  0.,  0.,  0.,  1.,  1., -1.,
         1.,  1., -1.,  0., -1.,  1., -1.,  0., -0.,  1.,  1.,  1., -1.,
        -1., -1.,  1.,  1.,  0., -0.,  0., -0., -1.,  1., -1., -0., -1.,
        -1.,  1.,  0.,  0.,  1., -1., -1.,  0.],
       [ 1.,  0., -1., -1., -1., -0.,  0.,  0., -0., -0.,  1.,  1.,  1.,
         1.,  1., -1., -0.,  1., -1.,  1.,  0.,  0.,  1.,  1.,  1., -1.,
        -1., -1., -1.,  1.,  0., -0.,  0.,  0.,  1.,  1., -1.,  0.,  1.,
         1.,  1., -0.,  0., -1.,  1., -1.,  0.],
       [ 1., -0., -1.,  1.,  1., -0.,  0.,  0., -0.,  0.,  1.,  1., -1.,
        -1.,  1.,  1., -0.,  1., -1., -1., -0.,  0., -1., -1., -1., -1.,
         1., -1.,  1.,  1.,  0., -0.,  0.,  0.,  1.,  1., -1.,  0., -1.,
        -1., -1., -0., -0., -1., -1.,  1.,  0.],
       [ 1., -0., -1., -1.,  1., -0., -0.,  0.,  0., -0., -1., -1.,  1.,
        -1.,  1.,  1.,  0., -1.,  1., -1.,  0., -0.,  1., -1., -1., -1.,
         1.,  1.,  1.,  1., -0., -0., -0., -0., -1.,  1.,  1.,  0.,  1.,
         1., -1., -0.,  0.,  1., -1., -1.,  0.],
       [-1.,  0.,  1., -1., -1.,  0., -0., -0., -0.,  0., -1., -1., -1.,
        -1.,  1., -1.,  0., -1.,  1., -1., -0., -0.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1., -0., -0.,  0., -0., -1.,  1.,  1., -0., -1.,
        -1., -1.,  0.,  0.,  1., -1., -1., -0.],
       [ 1., -0., -1.,  1.,  1.,  0., -0., -0.,  0., -0.,  1.,  1.,  1.,
        -1.,  1.,  1.,  0., -1.,  1.,  1., -0., -0., -1., -1., -1., -1.,
         1., -1., -1.,  1.,  0., -0., -0., -0., -1.,  1., -1., -0.,  1.,
         1., -1.,  0.,  0.,  1.,  1.,  1., -0.],
       [-1.,  0.,  1., -1.,  1., -0., -0., -0., -0., -0.,  1.,  1.,  1.,
        -1., -1.,  1.,  0.,  1., -1., -1.,  0., -0.,  1., -1., -1.,  1.,
         1., -1.,  1., -1.,  0.,  0., -0.,  0.,  1., -1., -1.,  0.,  1.,
         1., -1., -0., -0., -1., -1., -1., -0.]], dtype=np.float32), np.array([[-0., -1.,  0.,  0., -0., -1.,  1., -0.,  0., -0., -1.,  0., -0.,
        -1.,  0.,  0., -1.,  1.,  1.,  1., -0.,  1.,  0.,  0., -0.,  0.,
         1., -1., -0.,  0., -0., -0.,  1., -0., -0., -0., -0., -1.,  0.,
        -0.,  0., -0., -0., -0.,  1.,  0.,  1.],
       [-0.,  1.,  0., -0.,  0.,  1.,  1., -0., -0., -0., -1., -1.,  0.,
         0.,  0., -0., -1., -0.,  1.,  1., -0., -1., -0.,  0., -0., -0.,
         1.,  0., -1.,  0.,  0.,  0.,  1.,  0., -0., -0., -0., -1., -0.,
         0.,  1., -0., -0.,  1.,  0.,  0., -1.],
       [-1., -1., -1., -1.,  0., -0., -1., -1.,  0., -1., -1.,  0.,  0.,
         0., -1.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  1.,  0., -0., -0.,
         0., -0.,  0.,  1., -1.,  1., -0., -1., -1., -0., -1.,  1.,  1.,
         1.,  0.,  1., -1.,  0., -0., -0., -1.],
       [-0.,  1., -1., -1., -0.,  1., -1.,  1.,  1., -0.,  0.,  0., -0.,
         1.,  0.,  1.,  1., -1., -1., -1., -1., -1.,  0.,  0.,  1., -1.,
        -1., -0.,  0., -0., -0.,  0., -1.,  1., -0.,  0., -0.,  0., -0.,
         0.,  0., -1.,  0., -0., -1., -1.,  0.],
       [ 0.,  1.,  0., -0., -1.,  0.,  1., -0., -1.,  1.,  0., -0., -1.,
         0., -1.,  0., -0., -0.,  1., -0.,  1., -0., -0.,  0., -1., -0.,
         0.,  0.,  0.,  0.,  1., -1.,  0., -0., -0.,  1., -0.,  0.,  1.,
         0.,  0., -0.,  1.,  1.,  0.,  0.,  0.],
       [-0.,  1.,  0., -0.,  0., -0.,  1., -0., -1.,  1., -0.,  0., -0.,
         0.,  1.,  0.,  0.,  0., -1., -0.,  1., -0.,  0.,  1.,  1., -0.,
         0., -0., -0., -1.,  1., -1.,  0.,  0.,  1., -0., -0., -0., -1.,
        -1.,  0.,  0.,  1.,  0.,  0., -0.,  0.],
       [-0., -1.,  1., -1.,  0., -1., -1., -1., -1.,  0.,  0.,  0., -0.,
        -1.,  0., -1., -1.,  1.,  1., -1., -1., -1.,  0., -0., -1.,  1.,
         1., -0., -0.,  0., -0.,  0.,  1.,  1., -0., -0., -0.,  0.,  0.,
         0.,  0.,  1.,  0.,  0., -1., -1.,  0.],
       [ 0.,  1.,  0., -0.,  1., -0.,  1., -0.,  1., -1.,  0.,  0.,  1.,
         0., -1., -0.,  0.,  0.,  1.,  0.,  1., -0.,  0., -0.,  1., -0.,
         0.,  0.,  0., -0., -1., -1., -0.,  0., -0.,  1.,  0., -0., -1.,
         0., -0.,  0., -1.,  1.,  0.,  0., -0.],
       [-0., -1., -0., -0., -0., -0.,  1.,  0., -1., -1.,  0.,  0., -0.,
         0., -1.,  0.,  0., -0.,  1.,  0.,  1.,  0., -0.,  1.,  1.,  0.,
        -0.,  0.,  0.,  1.,  1.,  1., -0.,  0., -1.,  0.,  0.,  0., -1.,
        -1.,  0.,  0., -1., -0.,  0.,  0., -0.],
       [ 0., -0.,  1.,  1.,  0.,  0., -0.,  1., -1.,  0., -1.,  0., -0.,
         0.,  0.,  1., -0., -0.,  0.,  0.,  1.,  0., -0.,  0.,  1.,  1.,
         0., -1.,  0.,  0., -0., -0.,  0., -1.,  0.,  0., -0., -1., -0.,
         0., -0.,  1.,  0., -0., -0.,  1.,  1.],
       [ 0., -0., -0., -0.,  1., -1., -0.,  0., -1., -1., -1., -1., -1.,
        -0., -1., -0., -1., -0.,  0., -1.,  1., -1., -0.,  0.,  1.,  0.,
        -1.,  0.,  1., -0., -1., -1., -1.,  0., -0.,  1.,  0.,  1.,  1.,
         0., -1., -0.,  1., -0., -0.,  0.,  1.],
       [-1.,  0., -1., -1., -0., -0.,  0., -1.,  1.,  0.,  1.,  0.,  0.,
        -0.,  0., -0., -0.,  0.,  0., -0., -1.,  0.,  1., -1., -1., -0.,
        -0.,  0.,  0., -0., -0., -0.,  0.,  1.,  0.,  0.,  1.,  1.,  0.,
         0., -0., -1., -0.,  0., -0.,  0., -1.],
       [ 0.,  1., -0., -0., -0.,  0.,  0.,  1., -0., -0., -0., -0., -0.,
         1.,  0.,  1.,  1., -0., -0.,  0.,  0., -0.,  0., -0.,  1.,  0.,
        -1., -0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0.,  0.,  0., -0.,  0., -0., -0.,  0.],
       [-0.,  0., -0., -0., -1.,  1., -0.,  0., -0.,  1., -0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -1.,  0.,
         1.,  0.,  0.,  0.,  1., -0.,  0.,  0., -0., -0., -0., -1., -0.,
        -0.,  1., -0., -0.,  0.,  0.,  0.,  0.],
       [-1., -1., -1.,  0., -0., -0., -0.,  0.,  0., -1.,  0., -0., -0.,
         0., -1.,  0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,
         0.,  0.,  0.,  1.,  0., -0.,  0.,  0.,  0., -0.,  0.,  1.,  0.,
         0.,  0., -0.,  0., -0., -0., -0., -0.],
       [-0., -1.,  0.,  0., -0.,  1.,  1.,  0.,  0., -0.,  1.,  0., -0.,
        -1., -0., -0., -1., -1., -1.,  1., -0.,  1., -0., -0.,  0.,  0.,
         1.,  1., -0.,  0.,  0., -0., -1.,  0.,  0., -0., -0.,  1.,  0.,
        -0., -0.,  0., -0.,  0.,  1., -0., -1.],
       [-0.,  1., -0.,  0.,  0., -1.,  1., -0.,  0.,  0., -1., -1.,  0.,
         0.,  0.,  0., -1., -0.,  1., -1., -0., -1., -0., -0.,  0., -0.,
        -1., -0.,  1., -0.,  0.,  0., -1., -0.,  0., -0., -0.,  1., -0.,
        -0., -1.,  0.,  0.,  1.,  0., -0.,  1.],
       [-1., -1., -1., -1., -0.,  0., -1., -1.,  0., -1.,  1., -0., -0.,
        -0., -1.,  0., -0., -0., -1.,  0.,  0., -0.,  1., -0.,  0., -0.,
        -0.,  0.,  0.,  1., -1., -1.,  0.,  1.,  1., -0.,  1.,  1.,  1.,
         1.,  0., -1.,  1., -0., -0., -0., -1.],
       [-0., -0.,  1., -1.,  0., -0., -0., -1., -1.,  0., -1., -0.,  0.,
        -0.,  0., -1.,  0., -0.,  0.,  0., -1., -0.,  0.,  0., -1.,  1.,
        -0., -1.,  0., -0., -0., -0.,  0.,  1., -0.,  0., -0., -1., -0.,
        -0.,  0.,  1.,  0.,  0., -0., -1.,  1.],
       [-0., -0., -0.,  0., -1.,  1.,  0.,  0., -1.,  1.,  1.,  1., -1.,
        -0.,  1., -0.,  1.,  0., -0., -1., -1.,  1.,  0.,  0., -1., -0.,
         1., -0.,  1., -0.,  1.,  1., -1.,  0.,  0., -1., -0., -1.,  1.,
        -0.,  1., -0.,  1.,  0., -0., -0.,  1.],
       [-1.,  0., -1.,  1.,  0., -0., -0.,  1., -1., -0.,  1., -0.,  0.,
        -0.,  0.,  0.,  0., -0., -0., -0.,  1., -0., -1.,  1.,  1.,  0.,
        -0., -0.,  0.,  0.,  0.,  0., -0.,  1.,  0., -0.,  1.,  1., -0.,
        -0., -0., -1.,  0., -0., -0., -0.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 3
p = 7
rank = 47

verify_tensor_support_rank_decomposition(decomposition_337, n, m, p, rank)

## Rank-54 support rank decomposition of <3,3,8> over Z


In [ ]:
#@title Data

decomposition_338 = (np.array([[ 0.,  1., -0., -0.,  0., -0.,  0.,  1., -0., -0., -1., -0.,  1.,
        -1.,  1., -0.,  0.,  0.,  1.,  0.,  0., -1.,  0., -1.,  1.,  0.,
        -0., -1.,  0., -0.,  0., -1., -1.,  0., -1., -0., -1., -0.,  0.,
         1., -0., -1.,  0.,  1., -1., -1.,  1., -1.,  0., -0.,  0., -0.,
        -0., -0.],
       [ 1.,  0., -1.,  0., -0., -1.,  1.,  0.,  0., -1., -0., -0., -1.,
         1., -1.,  0.,  1., -0.,  0.,  1.,  1., -0., -0., -0., -1., -0.,
        -1., -0.,  1., -1.,  0., -0., -0.,  1.,  1.,  0.,  0.,  0., -0.,
         0.,  0.,  1.,  1.,  0., -0.,  1.,  0., -0.,  0., -0., -0.,  0.,
        -1., -1.],
       [-0.,  0.,  0., -1.,  1.,  0., -0., -1., -1., -1., -0.,  0.,  0.,
         1., -1.,  0., -0.,  1., -1., -0., -0., -0.,  1., -0., -1.,  1.,
         0., -0., -0.,  0., -1., -0., -0.,  1.,  1., -1., -0., -1., -0.,
         0., -1., -0., -0.,  0., -0.,  1.,  0., -0.,  0., -1.,  1., -1.,
         0.,  0.],
       [ 0.,  1., -0., -0.,  0.,  0.,  0., -0.,  1., -0.,  0., -0.,  1.,
         0., -0.,  1.,  0.,  0.,  1.,  0., -1.,  0., -1.,  0., -0.,  0.,
         1., -1.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,  0., -1., -0.,  0.,
        -0.,  0.,  0., -1.,  1., -1.,  0.,  1.,  0.,  1.,  1.,  0., -0.,
        -0., -0.],
       [-0.,  0.,  0.,  0.,  1., -1., -0.,  0.,  0., -1.,  1.,  0.,  0.,
         1., -1.,  0.,  1., -0.,  0.,  1.,  1., -0., -0.,  1., -1., -0.,
         0., -0.,  1.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0.,  1.,
        -1., -1.,  1., -0.,  0.,  0.,  1., -1., -0.,  0.,  0.,  1., -1.,
        -1.,  0.],
       [-0., -1., -1.,  0., -0.,  0., -0.,  0.,  0., -1.,  1., -0., -1.,
         1., -1., -1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,  0., -0.,
        -1.,  1.,  1., -1., -1., -0., -0., -0., -0.,  0.,  1.,  0.,  1.,
        -1., -1.,  1.,  1., -1., -0.,  1., -1., -0.,  0., -1.,  1., -1.,
        -1.,  0.],
       [ 0.,  0.,  0., -1., -0.,  0., -0., -1., -1.,  0., -0.,  0.,  0.,
         1., -1., -1., -0., -0., -0., -0.,  1.,  1.,  1.,  1., -1., -0.,
         0.,  1., -0., -1.,  0., -0.,  1., -0.,  1.,  0., -0.,  0., -0.,
        -1.,  0.,  1.,  1.,  0., -0., -0.,  0.,  1.,  0., -1., -0.,  0.,
        -1., -0.],
       [ 1.,  0., -1.,  0.,  1.,  0.,  1.,  0.,  0.,  0.,  1.,  1., -1.,
        -0.,  0.,  0.,  1., -0.,  0., -0., -0., -0., -0., -0.,  0.,  1.,
        -1.,  0., -0.,  0.,  0., -0., -0.,  1., -0.,  0.,  0.,  0.,  1.,
        -1., -1., -0.,  1., -1.,  0., -0., -1., -0.,  0.,  0., -0., -1.,
         0.,  0.],
       [-0., -1., -1., -1.,  1.,  0.,  1., -1., -1.,  0.,  1., -0., -1.,
         1.,  0., -1.,  1., -0.,  0.,  1.,  1.,  1.,  1.,  1., -1.,  1.,
        -1.,  1., -0., -1.,  0., -0.,  1.,  1.,  1., -1., -0.,  0.,  1.,
        -1.,  0.,  1.,  1., -1., -0., -0., -1., -0., -0.,  0., -0., -1.,
        -1.,  0.]], dtype=np.float32), np.array([[ 0., -1., -0.,  1.,  0.,  0., -0.,  1.,  1.,  0.,  0.,  0., -0.,
        -0., -0., -1.,  0., -1., -1.,  0.,  0., -1.,  0., -0.,  0.,  0.,
         0.,  1.,  0., -0.,  1.,  1.,  0.,  0., -0.,  1.,  0., -0.,  0.,
         0., -0.,  0., -0.,  0., -1.,  0.,  0.,  1.,  0., -1.,  0.,  0.,
         0., -0.],
       [ 0.,  0., -0., -1., -0.,  0.,  0., -1., -1., -1.,  0.,  0., -0.,
        -0.,  1., -0., -0.,  0.,  0.,  1.,  1., -0., -0., -0.,  0.,  0.,
         0.,  0.,  0.,  0., -0.,  1.,  0., -0.,  0., -1.,  0.,  0.,  0.,
        -0.,  0.,  1.,  0.,  0., -1., -1.,  0.,  1., -1.,  0., -0., -0.,
         1.,  0.],
       [-0.,  0.,  1., -0.,  0.,  0., -0., -0., -0.,  1.,  1.,  0.,  1.,
         0., -1., -0., -0., -1., -1.,  0.,  0., -0.,  0.,  0.,  0.,  0.,
        -1., -0.,  0., -0.,  1., -0.,  0., -0., -0., -0., -0.,  0.,  1.,
        -1.,  0.,  0.,  1.,  0.,  0.,  1.,  1., -0.,  1., -1., -0., -0.,
         0.,  0.],
       [-0., -1., -0., -0.,  0., -0.,  0., -0.,  0., -1.,  0., -0., -0.,
        -0.,  1., -1.,  0., -1., -1., -1., -1., -1.,  0.,  0., -0., -0.,
         0.,  1.,  0., -0.,  1., -0., -0., -0., -0., -0., -0.,  0.,  0.,
        -0., -0., -1.,  0., -0.,  0., -1.,  0., -0., -1., -1., -0., -0.,
        -1., -0.],
       [-0.,  1., -0., -1.,  0.,  0.,  0., -1., -1.,  0.,  0.,  0., -0.,
        -0., -0.,  1.,  0., -1., -1., -0., -0.,  1.,  0., -0., -0.,  0.,
         0., -1.,  0.,  0.,  1.,  1.,  0.,  0.,  0., -1.,  0., -0.,  0.,
         0.,  0., -0., -0., -0., -1.,  0.,  0.,  1.,  0., -1.,  0., -0.,
        -0., -0.],
       [ 0.,  1., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -1.,
         0.,  0.,  1.,  0.,  0., -0.,  1.,  1.,  1.,  0.,  0., -0.,  0.,
         1., -1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  1.,
        -1.,  0.,  1., -1., -0., -0., -0.,  1.,  0.,  0.,  0., -0., -0.,
         1.,  0.],
       [ 0.,  0.,  1.,  1.,  0.,  0., -0.,  1.,  1., -0.,  1.,  0.,  1.,
        -0.,  0.,  0.,  0., -1., -1., -1., -1.,  0.,  0.,  0.,  0.,  0.,
        -1., -0.,  0., -0.,  1.,  1.,  0., -0., -0.,  1.,  0.,  0.,  1.,
        -1., -0., -1.,  1.,  0., -1., -0.,  1.,  1.,  0., -1.,  0., -0.,
        -1., -0.],
       [-0.,  0., -1., -0.,  0.,  0.,  0., -0., -0.,  1.,  1.,  0., -1.,
         0., -1., -0., -0.,  1.,  1., -0., -0., -0., -0.,  0.,  0.,  0.,
         1.,  0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
        -1.,  0., -0., -1., -0., -0.,  1.,  1.,  0.,  1.,  1.,  0., -0.,
        -0.,  0.],
       [-0.,  1.,  0.,  0., -1.,  0., -0., -1.,  0.,  0.,  0.,  1., -1.,
         0., -0., -1., -0.,  0., -0., -0.,  1., -0.,  0., -0., -1., -1.,
         0., -0.,  0.,  1., -0., -0.,  0.,  1., -1., -1.,  0.,  0., -0.,
        -0.,  0.,  0.,  1.,  1.,  0.,  0., -1.,  0., -0.,  0.,  0., -0.,
        -0.,  0.],
       [-1.,  0.,  0.,  0., -0.,  1., -0.,  1.,  0.,  1., -0., -1.,  0.,
         0., -0.,  0.,  0.,  1.,  0.,  0.,  0., -1., -0., -1.,  1., -0.,
         0., -0., -0.,  0., -0.,  0.,  0.,  0.,  1., -0., -0.,  0.,  0.,
        -1., -1.,  1.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -1., -0.,
        -0.,  1.],
       [-1.,  0., -1.,  0.,  1.,  1.,  0., -0.,  0.,  1.,  0.,  0., -0.,
         0., -0.,  0., -1.,  1., -0., -1., -0.,  0.,  0.,  0., -0.,  1.,
        -0., -0., -0., -0.,  0.,  0.,  0., -1., -0.,  1.,  0., -0., -1.,
        -0., -1., -0., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -1., -0.,
         0.,  1.],
       [ 1.,  1.,  0., -0., -1., -1.,  0.,  0.,  0.,  1.,  0., -0., -1.,
        -0.,  0., -1.,  0.,  1.,  0.,  0.,  1.,  1., -0.,  1.,  0., -1.,
        -0., -0.,  0.,  1.,  0., -0.,  0.,  1.,  0., -1., -0., -0.,  0.,
         1., -1., -1.,  1.,  1.,  0., -0., -1., -0., -0., -0., -1., -0.,
        -0., -1.],
       [-0.,  1.,  0.,  0., -1.,  0., -0.,  1.,  0., -0.,  0., -1., -1.,
        -0., -0.,  1., -0., -0., -0.,  0., -1.,  0.,  0.,  0.,  1., -1.,
        -0.,  0.,  0., -1., -0.,  0., -0.,  1.,  1., -1.,  0.,  0., -0.,
         0.,  0., -0., -1.,  1.,  0.,  0., -1.,  0., -0.,  0.,  0., -0.,
         0.,  0.],
       [ 0.,  1.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  0., -0.,  0., -1.,
         0., -0.,  1.,  1.,  0.,  0.,  1., -1., -1.,  0., -1., -0.,  0.,
        -0.,  0.,  0., -1.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,
        -1., -0.,  1., -1.,  1.,  0.,  0., -1.,  0., -0.,  0., -0., -0.,
        -0.,  0.],
       [ 0.,  0., -1.,  0.,  1., -0.,  0.,  1., -0., -0.,  0., -1., -0.,
        -0., -0., -0., -1., -0., -0., -1.,  0., -1., -0., -1.,  1.,  1.,
        -0., -0.,  0.,  0., -0.,  0., -0., -1.,  1.,  1.,  0.,  0., -1.,
        -1.,  0.,  1.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0.],
       [-1.,  0.,  1.,  0.,  1.,  1., -0.,  0., -0., -1.,  0.,  0., -0.,
         0., -0., -0.,  1., -1.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  1.,
        -0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0.,  1.,  0.,  0.,  1.,
         0.,  1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0.,  1.,  0.,
        -0.,  1.],
       [-0.,  1.,  0., -0.,  0.,  0.,  0.,  1., -1., -0.,  0.,  0.,  0.,
         0.,  0., -1.,  0.,  1., -1., -0.,  0.,  1.,  1.,  0.,  0.,  0.,
         0., -0., -0., -0., -0.,  0.,  1.,  0., -0., -1.,  1.,  1., -0.,
        -0.,  0., -0., -0., -0.,  1.,  0.,  0., -1.,  0.,  1.,  0., -0.,
         0.,  0.],
       [ 0.,  0., -0., -0.,  0., -1., -0.,  1.,  0., -1., -0., -0.,  0.,
        -1., -1.,  0.,  0., -1., -0., -1.,  0.,  1., -0.,  0., -1.,  0.,
         0.,  0., -1., -0., -0., -0.,  1., -0.,  0.,  1.,  0., -1., -0.,
        -0.,  0.,  1.,  0., -0.,  0., -0.,  0., -1., -0., -0., -0., -0.,
         0.,  0.],
       [-1.,  0.,  1.,  0., -1., -1.,  1., -0., -0., -1.,  0.,  0.,  0.,
         0.,  0.,  0., -0., -1., -0., -1., -0., -0.,  0., -0.,  0.,  0.,
         0., -0., -1.,  0., -0., -0., -0.,  1.,  0.,  1.,  0., -1.,  1.,
         0., -1., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  1.,
         0.,  0.],
       [ 0.,  1.,  0., -0., -0.,  1., -0., -0., -1.,  1.,  0., -0.,  0.,
        -1., -1., -1.,  0., -0., -1.,  1.,  0., -0.,  1.,  0., -1., -0.,
        -0., -0.,  1., -0., -0.,  0., -0., -0.,  0.,  0.,  1., -0.,  0.,
        -0.,  0.,  1., -0.,  0.,  1., -0.,  0.,  0.,  0.,  1., -0.,  0.,
         0., -0.],
       [ 0., -1., -0.,  0., -0.,  0., -0.,  1., -1.,  0., -0., -0.,  0.,
         0.,  0., -1.,  0., -1.,  1.,  0.,  0.,  1.,  1., -0.,  0., -0.,
        -0., -0., -0., -0.,  0., -0.,  1.,  0., -0.,  1., -1., -1.,  0.,
        -0.,  0., -0., -0.,  0., -1.,  0.,  0., -1.,  0.,  1.,  0.,  0.,
         0.,  0.],
       [ 1.,  1., -1., -0., -1.,  0., -1.,  0.,  1.,  0.,  0.,  0.,  0.,
         1.,  1.,  1.,  0.,  1., -1.,  0.,  0.,  0., -1., -0.,  1.,  0.,
        -0.,  0.,  0.,  0., -0.,  0.,  0., -1., -0., -1.,  1.,  1.,  1.,
         0., -1., -1., -0., -0.,  1., -0.,  0., -0., -0., -1., -0.,  1.,
         0.,  0.],
       [ 1.,  0., -1.,  0.,  1., -0., -1., -1., -0., -0.,  0.,  0.,  0.,
         1.,  1., -0.,  0., -0., -0., -0., -0., -1.,  0., -0.,  1., -0.,
        -0., -0., -0.,  0.,  0.,  0., -1., -1., -0.,  0.,  0., -0., -1.,
        -0.,  1., -1.,  0., -0., -0.,  0.,  0.,  1.,  0.,  0., -0., -1.,
         0.,  0.],
       [-1.,  0.,  1., -0.,  1.,  1.,  1.,  0.,  0.,  1.,  0., -0., -0.,
         0.,  0.,  0., -0., -1., -0.,  1.,  0.,  0., -0.,  0.,  0., -0.,
         0.,  0.,  1.,  0., -0., -0.,  0.,  1.,  0.,  1.,  0., -1., -1.,
         0.,  1., -0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0., -1.,
         0., -0.]], dtype=np.float32), np.array([[ 0.,  0., -0.,  1.,  0., -0., -1., -1.,  0.,  1., -1.,  0.,  1.,
         1., -0., -1., -1.,  0.,  1., -0.,  0., -0.,  1.,  1.,  0.,  1.,
         1.,  1.,  1., -1.,  1.,  1., -1.,  1., -1.,  0., -1., -1.,  1.,
         0.,  0.,  1.,  0.,  1.,  0.,  1., -0., -0., -0., -0., -1.,  1.,
         1., -1.],
       [-0.,  0.,  1., -1.,  1.,  1.,  1.,  0.,  1., -0., -1.,  0.,  0.,
        -1.,  0.,  0.,  1., -1., -0., -0., -1., -1., -1.,  1., -1., -1.,
         1., -1.,  1., -1.,  1.,  1.,  1.,  0., -1.,  0., -1., -1., -0.,
        -0.,  0.,  0.,  0., -1.,  1., -0., -1.,  0.,  1., -0., -1., -1.,
        -1., -1.],
       [ 1., -1., -0., -1.,  0., -0.,  1., -0.,  0.,  0.,  1., -1., -0.,
         1.,  1.,  0.,  1., -0., -0., -1.,  0.,  0.,  1., -1.,  0.,  1.,
         1.,  1.,  1., -1.,  1., -1.,  1.,  0., -0.,  1., -1.,  1.,  0.,
        -1.,  1.,  0., -1.,  1.,  0., -1., -0.,  1.,  0.,  1.,  1.,  1.,
        -1., -1.],
       [ 0.,  0.,  0., -1.,  0., -0., -1.,  1.,  0., -1.,  1., -0., -1.,
         1., -0., -1.,  1., -0., -1., -0.,  0.,  0.,  1.,  1., -0.,  1.,
        -1.,  1., -1., -1., -1.,  1.,  1.,  1.,  1.,  0.,  1., -1., -1.,
        -0.,  0.,  1.,  0., -1., -0., -1.,  0.,  0.,  0., -0.,  1., -1.,
         1., -1.],
       [ 0., -0.,  1., -1., -1., -1.,  1.,  0.,  1.,  0., -1.,  0., -0.,
        -1.,  0., -0.,  1., -1.,  0., -0.,  1.,  1., -1., -1., -1.,  1.,
         1.,  1., -1.,  1.,  1., -1., -1.,  0., -1., -0.,  1., -1., -0.,
        -0.,  0.,  0.,  0., -1., -1., -0., -1.,  0.,  1., -0., -1.,  1.,
         1.,  1.],
       [ 1.,  1., -0., -1.,  0., -0.,  1.,  0., -0.,  0.,  1.,  1.,  0.,
         1.,  1.,  0., -1.,  0.,  0.,  1.,  0., -0.,  1., -1., -0.,  1.,
         1., -1., -1., -1.,  1.,  1., -1.,  0.,  0.,  1.,  1.,  1.,  0.,
        -1., -1.,  0., -1., -1., -0., -1.,  0., -1.,  0.,  1., -1., -1.,
         1., -1.],
       [-0.,  0.,  0.,  1., -0.,  0.,  1., -1.,  0., -1., -1.,  0.,  1.,
        -1.,  0.,  1., -1., -0., -1., -0.,  0.,  0., -1., -1.,  0., -1.,
         1., -1., -1.,  1., -1.,  1., -1., -1., -1.,  0.,  1., -1.,  1.,
        -0.,  0., -1.,  0.,  1., -0., -1.,  0.,  0., -0.,  0.,  1.,  1.,
        -1., -1.],
       [ 0.,  0.,  1., -1., -1.,  1.,  1.,  0.,  1., -0., -1.,  0., -0.,
        -1.,  0., -0.,  1.,  1.,  0.,  0.,  1.,  1., -1., -1., -1.,  1.,
         1.,  1.,  1.,  1., -1.,  1., -1.,  0., -1.,  0., -1.,  1., -0.,
         0.,  0.,  0.,  0., -1.,  1., -0., -1.,  0., -1., -0.,  1.,  1.,
         1., -1.],
       [ 1., -1.,  0.,  1., -0., -0.,  1.,  0.,  0.,  0., -1., -1.,  0.,
         1.,  1., -0.,  1.,  0., -0., -1.,  0.,  0.,  1.,  1.,  0., -1.,
        -1.,  1.,  1.,  1.,  1.,  1., -1.,  0.,  0., -1., -1., -1.,  0.,
         1., -1., -0.,  1.,  1.,  0., -1.,  0., -1.,  0.,  1., -1., -1.,
        -1., -1.],
       [-0.,  0., -0., -1., -0.,  0.,  1.,  1.,  0., -1.,  1., -0.,  1.,
        -1.,  0., -1.,  1., -0.,  1.,  0.,  0., -0.,  1., -1.,  0., -1.,
         1.,  1., -1., -1.,  1., -1.,  1., -1.,  1.,  0., -1.,  1., -1.,
         0.,  0., -1.,  0.,  1., -0., -1.,  0.,  0., -0., -0.,  1., -1.,
        -1.,  1.],
       [-0.,  0.,  1.,  1.,  1.,  1.,  1.,  0., -1.,  0.,  1.,  0., -0.,
        -1.,  0., -0.,  1., -1., -0.,  0.,  1., -1.,  1.,  1., -1., -1.,
         1., -1.,  1.,  1.,  1., -1.,  1.,  0., -1.,  0.,  1., -1.,  0.,
         0.,  0., -0.,  0.,  1., -1.,  0.,  1., -0., -1.,  0., -1., -1.,
         1., -1.],
       [-1., -1.,  0.,  1., -0.,  0., -1.,  0.,  0., -0., -1.,  1., -0.,
        -1., -1.,  0., -1., -0., -0.,  1., -0.,  0.,  1.,  1.,  0., -1.,
         1.,  1., -1., -1.,  1.,  1., -1., -0.,  0., -1., -1., -1., -0.,
         1., -1.,  0., -1.,  1., -0.,  1.,  0., -1., -0.,  1., -1., -1.,
         1.,  1.],
       [ 0.,  0.,  0.,  1.,  0.,  0.,  1., -1.,  0., -1.,  1.,  0., -1.,
         1.,  0., -1.,  1., -0., -1., -0.,  0.,  0.,  1.,  1.,  0., -1.,
        -1.,  1., -1., -1., -1., -1., -1., -1., -1., -0.,  1.,  1., -1.,
         0.,  0.,  1.,  0., -1.,  0., -1.,  0., -0.,  0., -0.,  1., -1.,
         1.,  1.],
       [ 0., -0., -1., -1., -1., -1., -1.,  0.,  1.,  0.,  1.,  0.,  0.,
        -1.,  0.,  0., -1.,  1.,  0., -0., -1., -1., -1.,  1., -1.,  1.,
        -1., -1., -1., -1., -1., -1.,  1.,  0., -1.,  0.,  1.,  1.,  0.,
         0.,  0.,  0.,  0.,  1., -1.,  0.,  1., -0., -1.,  0.,  1.,  1.,
        -1.,  1.],
       [-1.,  1., -0.,  1., -0.,  0., -1.,  0.,  0., -0.,  1., -1.,  0.,
         1.,  1., -0., -1.,  0., -0.,  1.,  0.,  0.,  1., -1.,  0., -1.,
         1., -1., -1., -1.,  1., -1.,  1., -0.,  0., -1.,  1., -1.,  0.,
        -1., -1., -0., -1., -1.,  0., -1.,  0.,  1., -0.,  1., -1., -1.,
         1.,  1.],
       [-0., -0., -0.,  1.,  0.,  0.,  1., -1., -0., -1.,  1.,  0.,  1.,
         1.,  0.,  1.,  1.,  0.,  1., -0., -0.,  0., -1.,  1.,  0., -1.,
         1., -1., -1.,  1.,  1., -1., -1., -1., -1.,  0., -1.,  1., -1.,
        -0., -0.,  1.,  0.,  1.,  0., -1., -0.,  0., -0.,  0.,  1., -1.,
         1.,  1.],
       [ 0., -0., -1.,  1., -1., -1., -1.,  0., -1.,  0., -1.,  0.,  0.,
        -1.,  0., -0., -1.,  1., -0., -0.,  1., -1.,  1.,  1., -1.,  1.,
        -1., -1., -1.,  1., -1.,  1.,  1., -0., -1.,  0., -1.,  1., -0.,
         0.,  0.,  0., -0., -1.,  1., -0., -1.,  0.,  1., -0.,  1.,  1.,
         1.,  1.],
       [-1., -1.,  0.,  1.,  0.,  0., -1.,  0., -0.,  0.,  1., -1., -0.,
         1.,  1., -0., -1.,  0.,  0.,  1., -0., -0., -1., -1.,  0., -1.,
        -1.,  1., -1.,  1., -1., -1.,  1.,  0.,  0., -1., -1., -1.,  0.,
        -1., -1.,  0.,  1.,  1., -0., -1., -0.,  1.,  0., -1., -1., -1.,
         1.,  1.],
       [ 0., -0.,  0.,  1., -0., -0., -1., -1.,  0., -1.,  1.,  0., -1.,
        -1.,  0.,  1.,  1., -0., -1.,  0.,  0.,  0., -1., -1.,  0.,  1.,
        -1., -1., -1.,  1., -1.,  1., -1.,  1., -1.,  0.,  1., -1., -1.,
         0.,  0., -1., -0., -1., -0., -1., -0.,  0.,  0.,  0.,  1., -1.,
        -1., -1.],
       [ 0.,  0., -1., -1.,  1.,  1., -1.,  0.,  1.,  0.,  1.,  0.,  0.,
        -1.,  0., -0., -1.,  1.,  0., -0.,  1.,  1., -1., -1., -1., -1.,
        -1.,  1.,  1.,  1., -1.,  1., -1.,  0., -1.,  0., -1.,  1.,  0.,
         0.,  0.,  0., -0.,  1.,  1.,  0.,  1.,  0., -1.,  0.,  1., -1.,
         1., -1.],
       [ 1.,  1.,  0., -1.,  0.,  0.,  1., -0., -0.,  0., -1., -1.,  0.,
        -1., -1., -0., -1.,  0.,  0.,  1.,  0.,  0., -1.,  1.,  0.,  1.,
        -1., -1., -1.,  1., -1., -1.,  1.,  0.,  0.,  1.,  1.,  1.,  0.,
         1., -1.,  0.,  1., -1.,  0.,  1.,  0.,  1., -0., -1., -1., -1.,
         1., -1.],
       [-0.,  0., -0., -1., -0., -0.,  1.,  1.,  0.,  1.,  1.,  0.,  1.,
        -1.,  0., -1.,  1.,  0., -1.,  0.,  0., -0.,  1., -1.,  0., -1.,
         1.,  1.,  1., -1., -1.,  1.,  1., -1.,  1.,  0.,  1., -1., -1.,
         0.,  0., -1.,  0.,  1., -0.,  1., -0.,  0., -0.,  0., -1., -1.,
        -1., -1.],
       [ 0., -0., -1., -1., -1.,  1., -1., -0.,  1., -0., -1.,  0.,  0.,
         1., -0., -0., -1., -1., -0.,  0., -1.,  1., -1., -1.,  1.,  1.,
        -1.,  1.,  1., -1.,  1., -1., -1., -0.,  1.,  0.,  1., -1., -0.,
         0., -0.,  0.,  0., -1., -1., -0., -1., -0., -1.,  0., -1.,  1.,
        -1., -1.],
       [-1.,  1.,  0., -1., -0., -0., -1., -0.,  0., -0.,  1., -1.,  0.,
        -1., -1., -0.,  1.,  0.,  0., -1.,  0.,  0.,  1., -1.,  0.,  1.,
        -1., -1.,  1.,  1.,  1.,  1., -1., -0.,  0.,  1.,  1.,  1.,  0.,
        -1., -1.,  0.,  1., -1.,  0.,  1.,  0., -1., -0.,  1., -1., -1.,
        -1.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 3
p = 8
rank = 54

verify_tensor_support_rank_decomposition(decomposition_338, n, m, p, rank)

## Rank-36 support rank decomposition of <3,4,4> over Z


In [ ]:
#@title Data

decomposition_344 = (np.array([[ 1., -1.,  1., -1.,  1., -0., -1.,  1., -1., -0., -1.,  1., -1.,
         1.,  0., -1.,  1.,  1.,  1.,  1.,  1.,  0.,  1.,  1., -1.,  1.,
        -1., -1.,  1.,  1., -1.,  1.,  1.,  1., -1.,  1.],
       [-1., -1.,  1.,  1., -1., -0., -1.,  1.,  1., -0.,  1., -1.,  1.,
         1., -0., -1., -1., -1., -1., -1.,  1., -0.,  1.,  1.,  1.,  1.,
        -1., -1., -1., -1., -1., -1.,  1.,  1., -1., -1.],
       [-1.,  1., -1.,  1., -1.,  0., -1., -1.,  1.,  0.,  1.,  1.,  1.,
        -1., -0.,  1., -1., -1.,  1., -1., -1.,  0., -1., -1.,  1., -1.,
        -0.,  1.,  0., -1.,  1., -1., -1., -1.,  1.,  1.],
       [ 1.,  1.,  1., -1.,  1.,  0.,  1., -1., -1.,  0., -1.,  1., -1.,
        -1.,  0.,  1.,  1.,  1.,  1.,  1., -1.,  0., -1.,  1., -1., -1.,
        -0.,  1.,  0., -1.,  1.,  1., -1.,  1.,  1.,  1.],
       [ 1.,  1., -1.,  1.,  1.,  1., -1., -1.,  1.,  2.,  1., -1.,  1.,
        -0., -1.,  1.,  1., -0., -1., -1., -1.,  1.,  1., -1., -1.,  1.,
         0., -1., -0.,  1.,  1., -1.,  1., -1.,  1., -1.],
       [ 1., -1.,  1., -1.,  1., -1.,  1.,  1., -1., -2.,  1., -1., -1.,
         0., -1., -1.,  1., -0., -1., -1.,  1.,  1., -1.,  1., -1.,  1.,
        -0., -1., -0.,  1., -1., -1.,  1.,  1., -1., -1.],
       [ 1., -1.,  1., -1.,  1., -1.,  1.,  1., -1.,  0., -1., -1., -1.,
         0.,  0., -1.,  1., -0., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,
        -0., -1., -0.,  1., -1.,  1.,  1.,  1., -1., -1.],
       [-1.,  1., -1.,  1.,  1.,  1., -1., -1.,  1., -0.,  1., -1.,  1.,
        -0.,  0., -1.,  1.,  0., -1., -1., -1.,  1.,  1., -1., -1.,  1.,
         0., -1., -0.,  1., -1., -1.,  1., -1., -1., -1.],
       [ 1.,  1., -1., -0.,  1.,  0., -1.,  0., -1.,  0., -0., -1.,  1.,
         0.,  0., -1.,  0.,  0., -0., -1.,  1.,  0.,  1.,  1., -1.,  1.,
         0., -0.,  0.,  1., -0.,  1.,  1., -0.,  1.,  1.],
       [-1.,  1., -1.,  0., -1.,  0., -1.,  0.,  1., -0.,  0.,  1.,  1.,
         0., -0., -1., -0., -0.,  0.,  1.,  1., -0.,  1.,  1.,  1.,  1.,
         0.,  0., -0., -1., -0., -1., -1., -0.,  1., -1.],
       [ 1., -1.,  1., -0.,  1.,  0.,  1.,  0., -1., -0., -0., -1., -1.,
        -0.,  0., -1.,  0., -0., -0.,  1.,  1., -0.,  1.,  1., -1.,  1.,
        -0., -0., -0.,  1., -0.,  1.,  1.,  0., -1., -1.],
       [-1.,  1., -1.,  0., -1.,  0., -1.,  0.,  1., -0.,  0., -1.,  1.,
         0., -0., -1.,  0., -0., -0., -1.,  1.,  0.,  1., -1., -1.,  1.,
         0., -0., -0.,  1., -0., -1.,  1., -0., -1., -1.]], dtype=np.float32), np.array([[-0., -0.,  1.,  1.,  0., -0.,  1., -0., -1., -0.,  1.,  0., -0.,
        -0.,  1.,  1., -0., -0.,  1.,  0., -0., -0.,  0., -0., -0., -1.,
        -1., -1., -0.,  0.,  1., -1.,  0., -1., -0., -1.],
       [ 1., -0.,  1.,  1.,  0., -0., -0., -0., -0., -0.,  1.,  1., -1.,
        -0.,  1.,  0., -0., -0.,  1.,  0.,  0., -0.,  0., -0., -0., -0.,
        -1., -1.,  0., -0.,  1., -1.,  1., -1., -1., -0.],
       [ 1., -0.,  1.,  1.,  0., -0., -0.,  0., -0.,  0.,  1., -1., -1.,
        -0.,  1., -0., -0.,  0., -1.,  0.,  0., -0., -0., -0., -0.,  0.,
        -1.,  1., -0., -0., -1., -1., -1., -1.,  1.,  0.],
       [-0., -0.,  1.,  1., -0., -0.,  1., -0., -1., -0.,  1., -0., -0.,
        -0.,  1., -1.,  0., -0., -1.,  0.,  0., -0.,  0., -0., -0.,  1.,
        -1.,  1., -0., -0., -1., -1., -0., -1.,  0.,  1.],
       [ 0.,  0., -0., -1., -0., -0.,  0., -0.,  1.,  1., -1.,  1.,  0.,
        -0.,  0.,  1.,  0., -0.,  1., -0., -0., -0.,  0.,  1., -0., -1.,
        -0., -1.,  1.,  1.,  1.,  1.,  0.,  1., -0., -0.],
       [-0., -0.,  1.,  1.,  0.,  0.,  0., -0., -0., -1.,  1., -1., -1.,
         0., -0., -1., -0.,  0., -1.,  1.,  0.,  0.,  1., -0., -0.,  0.,
         0.,  1., -1.,  0., -1., -0., -1., -1.,  0.,  0.],
       [-0., -0.,  1.,  1.,  0., -0., -0.,  0., -0.,  1.,  1.,  1., -1.,
        -0., -0.,  1., -0., -0.,  1.,  1., -0., -0., -1., -0.,  0., -0.,
         0., -1.,  1.,  0.,  1., -0.,  1., -1., -0., -0.],
       [-0., -0., -0.,  1., -0., -0., -0.,  0., -1.,  1.,  1.,  1., -0.,
        -0., -0.,  1.,  0., -0.,  1.,  0., -0., -0.,  0., -1.,  0., -1.,
         0., -1.,  1.,  1.,  1., -1.,  0., -1., -0., -0.],
       [-0.,  0., -1., -0., -1., -2., -1.,  1., -0., -1.,  1., -0., -0.,
         0., -0., -1.,  1.,  0., -1.,  0.,  1., -1., -0., -0.,  0.,  0.,
         1.,  0.,  0.,  0., -1., -1., -0.,  1.,  0.,  1.],
       [-0.,  1., -1.,  0., -0.,  0.,  0., -1., -0., -1.,  1., -1., -0.,
        -2., -0., -1., -1.,  1., -1.,  1., -0., -0.,  1.,  0., -1.,  0.,
         1.,  0., -0., -0., -1., -0., -0.,  1., -0.,  0.],
       [-0., -1.,  1.,  0., -0., -0., -0.,  1., -0.,  1.,  1., -1., -0.,
         2., -0.,  1., -1.,  1., -1.,  1., -0., -0., -1., -0., -1., -0.,
        -1., -0., -0., -0.,  1., -0., -0., -1.,  0.,  0.],
       [-0., -0.,  1.,  0., -1.,  2.,  1., -1., -0.,  1.,  1., -0., -0.,
        -0., -0.,  1.,  1.,  0., -1.,  0., -1., -1.,  0., -0.,  0.,  0.,
        -1.,  0.,  0.,  0.,  1., -1., -0., -1., -0.,  1.],
       [ 0., -0.,  0.,  0.,  1.,  0., -0.,  1., -0.,  0., -1., -1., -0.,
        -0., -1., -1., -1.,  0., -1., -0.,  1.,  0.,  0., -1., -0.,  0.,
         0.,  0., -1., -1., -1.,  1., -0., -1.,  0., -0.],
       [-1., -1.,  1.,  0.,  0.,  0.,  0.,  1., -0.,  0., -1., -1., -0.,
        -0., -1., -0., -1.,  0., -1.,  0.,  0.,  0., -0., -0., -1.,  0.,
         0.,  0., -1., -0., -1.,  1., -0., -1.,  1.,  0.],
       [ 1., -1.,  1., -0., -0.,  0., -0.,  1.,  0.,  0.,  1.,  1., -0.,
        -0.,  1., -0.,  1., -0.,  1., -0.,  0., -0., -0., -0.,  1., -0.,
         0., -0.,  1., -0., -1., -1.,  0., -1.,  1., -0.],
       [-0., -0., -0.,  0., -1.,  0., -0.,  1., -0.,  0.,  1.,  1.,  0.,
        -0.,  1., -1.,  1., -0.,  1.,  0.,  1., -0.,  0., -1.,  0.,  0.,
         0., -0.,  1.,  1., -1., -1.,  0., -1.,  0.,  0.]], dtype=np.float32), np.array([[ 0.,  1., -1.,  1., -0.,  1.,  1., -1.,  0.,  0., -0.,  1.,  1.,
         0.,  0., -0., -1.,  0., -1., -0.,  0., -2.,  0.,  0., -1., -0.,
        -0., -1., -0.,  1., -0.,  0.,  1., -1., -0., -0.],
       [ 1., -0.,  0.,  0., -1., -1.,  0., -0.,  1.,  1., -0.,  0.,  0.,
         0.,  2., -1., -0., -0., -0., -0.,  1.,  2.,  1.,  0., -0.,  1.,
        -0.,  0.,  0., -0.,  0.,  1., -0.,  0.,  0.,  0.],
       [ 0.,  1., -1.,  1., -1.,  0.,  0., -1.,  1., -0., -0.,  1.,  1.,
         0.,  0., -0., -1.,  0., -1., -0.,  1.,  0., -0.,  1., -1., -1.,
        -0., -1.,  0.,  0.,  0.,  0.,  1., -1.,  0., -1.],
       [-0.,  1., -1.,  0., -0.,  0.,  1., -0.,  0.,  0., -0.,  1.,  1.,
        -1., -0.,  0., -0.,  2., -0., -0., -0., -0.,  0.,  0., -1., -0.,
         2., -0., -2.,  1.,  0.,  0.,  1., -0.,  0., -0.],
       [ 1.,  0., -0.,  1., -1.,  0.,  0., -1.,  1., -0., -1.,  0.,  0.,
         1., -0., -1., -1., -2.,  0., -0.,  1.,  0.,  1.,  0., -0.,  1.,
         0.,  1., -0.,  0.,  1.,  1., -0., -0.,  0.,  0.],
       [-0., -1.,  0., -1.,  1., -0., -0.,  1., -1.,  0.,  1., -0., -1.,
        -0.,  0.,  1.,  1., -0.,  0.,  1., -1., -0., -0., -0.,  1., -1.,
         0., -1., -0.,  0., -1., -1.,  1.,  0., -1.,  0.],
       [-0.,  1., -1.,  0.,  0.,  0.,  1., -0.,  0., -0.,  0., -1.,  1.,
        -1., -0.,  0.,  0., -2.,  0., -0., -0.,  0., -0.,  0.,  1.,  0.,
         2.,  0.,  2., -1., -0., -0., -1., -0.,  0.,  0.],
       [-1.,  0., -0., -1.,  1.,  0.,  0., -1., -1., -0.,  1., -0., -0.,
         1.,  0., -1.,  1.,  2.,  0.,  0.,  1., -0.,  1., -0.,  0.,  1.,
         0.,  1.,  0., -0.,  1., -1., -0.,  0.,  0.,  0.],
       [ 0., -1.,  0.,  1., -1., -0., -0.,  1.,  1.,  0., -1.,  0.,  1.,
        -0., -0.,  1., -1.,  0., -0., -1., -1.,  0., -0., -0., -1., -1.,
         0., -1., -0.,  0., -1.,  1.,  1.,  0., -1., -0.],
       [-0.,  1., -1.,  1.,  0.,  1.,  1., -1.,  0.,  0., -0., -1.,  1.,
         0., -0., -0.,  1., -0.,  1., -0.,  0.,  2.,  0.,  0.,  1.,  0.,
        -0.,  1.,  0., -1.,  0., -0., -1., -1.,  0.,  0.],
       [ 1., -0.,  0.,  0., -1.,  1., -0.,  0.,  1., -1., -0.,  0.,  0.,
        -0.,  2.,  1., -0., -0., -0., -0., -1.,  2., -1., -0., -0., -1.,
         0., -0.,  0., -0., -0.,  1.,  0.,  0., -0.,  0.],
       [ 0.,  1., -1.,  1.,  1., -0., -0., -1.,  1.,  0.,  0., -1.,  1.,
         0.,  0., -0.,  1., -0.,  1.,  0.,  1., -0.,  0.,  1.,  1.,  1.,
        -0.,  1., -0., -0.,  0.,  0., -1., -1., -0.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 4
p = 4
rank = 36

verify_tensor_support_rank_decomposition(decomposition_344, n, m, p, rank)

## Rank-45 support rank decomposition of <3,4,5> over Z


In [ ]:
#@title Data

decomposition_345 = (np.array([[-1.,  1., -0., -1.,  1., -1., -1.,  0., -1., -0., -0., -0.,  1.,
          0., -0., -1.,  1., -1.,  0.,  0.,  0.,  0.,  1.,  1.,  1., -1.,
         -1.,  0.,  1.,  0.,  1., -1.,  1., -0.,  0.,  1., -1.,  0.,  0.,
          1.,  0.,  0.,  1.,  1., -1.],
        [-1.,  1., -0.,  1., -1.,  1., -1.,  0.,  1.,  0.,  0.,  0.,  1.,
          0., -0., -1.,  1., -1., -0., -0.,  0.,  0., -1., -1.,  1.,  1.,
         -1.,  0.,  1.,  0., -1.,  1.,  1.,  0.,  0., -1.,  1., -0.,  0.,
          1., -0., -0., -1.,  1.,  1.],
        [-1., -1., -0., -1.,  1., -1., -1.,  0.,  1.,  0.,  0.,  0.,  1.,
          0., -0., -1., -1.,  1.,  0.,  0., -0., -0.,  1., -1.,  1.,  1.,
          1., -0.,  1.,  0., -1.,  1.,  1.,  0., -0.,  1., -1., -0.,  0.,
         -1.,  0.,  0., -1., -1., -1.],
        [ 1.,  1.,  0., -1.,  1., -1.,  1., -0.,  1.,  0., -0., -0., -1.,
          0.,  0.,  1.,  1., -1., -0.,  0., -0., -0.,  1., -1., -1.,  1.,
         -1.,  0., -1., -0., -1.,  1., -1.,  0.,  0.,  1., -1., -0., -0.,
          1., -0., -0., -1.,  1., -1.],
        [ 0., -0., -0., -1., -0., -1., -0., -1.,  0., -1., -1.,  1.,  1.,
         -0.,  0.,  1.,  0., -1., -1., -1., -1.,  1., -0., -0., -0.,  1.,
         -1.,  1.,  0.,  1.,  1.,  0.,  1.,  1., -1.,  0.,  1., -1.,  0.,
         -0., -1.,  1., -0., -1., -0.],
        [ 0., -0., -0.,  1.,  0., -1., -0., -1., -0.,  1., -1.,  1.,  1.,
          0.,  0.,  1.,  0., -1.,  1.,  1., -1., -1.,  0., -0., -0., -1.,
          1., -1.,  0.,  1., -1.,  0., -1.,  1., -1., -0., -1.,  1.,  0.,
         -0.,  1.,  1.,  0., -1., -0.],
        [-0., -0.,  0., -1.,  0.,  1.,  0.,  1.,  0.,  1.,  1., -1.,  1.,
          0., -0., -1.,  0.,  1., -1.,  1., -1.,  1.,  0.,  0.,  0.,  1.,
         -1., -1., -0., -1., -1.,  0., -1.,  1., -1., -0., -1., -1.,  0.,
         -0.,  1.,  1., -0., -1.,  0.],
        [ 0.,  0.,  0., -1.,  0., -1., -0., -1.,  0.,  1., -1.,  1., -1.,
         -0.,  0.,  1.,  0., -1., -1.,  1.,  1.,  1.,  0.,  0., -0.,  1.,
         -1., -1.,  0.,  1., -1.,  0., -1., -1.,  1., -0., -1., -1.,  0.,
         -0.,  1., -1., -0.,  1.,  0.],
        [ 1., -1., -1., -0., -1., -0.,  0.,  1.,  1., -1., -0.,  1.,  0.,
         -1.,  1.,  0.,  1., -0.,  0.,  1., -1.,  1., -1.,  1., -1.,  0.,
         -0., -0.,  1.,  1.,  0., -0.,  0., -0.,  1., -0.,  0., -1., -1.,
          0., -1.,  1., -1.,  0., -1.],
        [ 1., -1., -1., -0.,  1., -0., -0., -1., -1.,  1., -0.,  1., -0.,
          1., -1., -0., -1.,  0.,  0.,  1.,  1.,  1., -1.,  1.,  1.,  0.,
          0.,  0., -1., -1.,  0.,  0., -0.,  0., -1., -0.,  0.,  1., -1.,
         -0., -1.,  1., -1.,  0., -1.],
        [-1., -1., -1.,  0.,  1.,  0., -0., -1.,  1.,  1., -0., -1., -0.,
         -1., -1., -0.,  1., -0.,  0., -1.,  1., -1., -1.,  1., -1.,  0.,
         -0.,  0., -1.,  1.,  0., -0., -0., -0.,  1.,  0., -0., -1.,  1.,
         -0., -1.,  1.,  1.,  0.,  1.],
        [-1., -1., -1.,  0., -1.,  0.,  0.,  1., -1., -1., -0., -1.,  0.,
          1.,  1.,  0., -1.,  0.,  0., -1., -1., -1., -1.,  1.,  1.,  0.,
          0.,  0.,  1., -1.,  0., -0.,  0., -0., -1.,  0., -0.,  1.,  1.,
          0., -1.,  1.,  1.,  0.,  1.]], dtype=np.float32),
 np.array([[-0., -0.,  1., -1.,  0., -0.,  1.,  0., -0., -0., -1., -0.,  1.,
          1., -1., -0.,  1.,  1.,  1., -0.,  1., -1.,  0.,  1.,  0.,  0.,
         -0.,  1., -1., -1.,  1., -1.,  0.,  1., -0., -1.,  0.,  0.,  1.,
          1., -1., -0., -0.,  0.,  1.],
        [ 0., -0., -1.,  0.,  0.,  0.,  1., -1.,  0.,  0., -1.,  0., -0.,
          1., -1.,  1.,  1.,  0., -1., -1., -0., -0.,  0., -1., -0., -1.,
         -0., -1., -1.,  0., -0.,  1.,  0.,  1.,  1.,  1.,  1., -0., -1.,
          1.,  0., -0., -0., -1., -1.],
        [-0.,  0., -1., -0., -0.,  0., -1., -0., -0.,  0.,  1.,  0.,  0.,
         -1.,  1., -1., -0.,  0., -1.,  0., -1.,  1.,  1.,  0.,  1., -1.,
         -0., -1., -0.,  1., -0.,  1.,  0., -1., -0.,  1.,  1.,  0., -1.,
         -1.,  1.,  0., -1.,  1.,  0.],
        [-1., -1.,  1.,  0., -1.,  0.,  1.,  0.,  1., -1., -1., -1.,  0.,
         -1., -1.,  1., -0.,  0.,  1.,  0., -0., -0., -0.,  0., -0.,  1.,
         -0., -1.,  0.,  0.,  0., -1.,  0., -1., -0.,  1.,  1., -1., -1.,
         -1.,  0.,  1., -0.,  1.,  0.],
        [ 0.,  0., -1., -0.,  0.,  1., -1., -0.,  0.,  1.,  1.,  1.,  0.,
          1.,  1.,  0.,  1.,  0., -1.,  0., -0., -0.,  0., -1.,  0.,  0.,
         -1.,  1.,  1., -0.,  0.,  1.,  1.,  1., -0., -1.,  0.,  1.,  1.,
          1.,  0., -1.,  0., -0.,  1.],
        [ 0.,  0., -1.,  1., -0.,  0.,  1.,  0.,  0., -0., -1.,  0.,  1.,
          1., -1., -0.,  1.,  1., -1., -0.,  1.,  1.,  0., -1.,  0.,  0.,
         -0., -1., -1., -1., -1.,  1.,  0.,  1., -0.,  1., -0.,  0., -1.,
          1.,  1., -0., -0.,  0., -1.],
        [-0., -0., -1.,  0.,  0.,  0., -1.,  1., -0., -0.,  1.,  0., -0.,
         -1.,  1., -1., -1., -0., -1., -1.,  0., -0., -0., -1., -0., -1.,
         -0., -1.,  1., -0., -0.,  1., -0., -1., -1.,  1.,  1., -0., -1.,
         -1.,  0., -0., -0.,  1., -1.],
        [ 0.,  0.,  1.,  0., -0., -0., -1., -0.,  0.,  0.,  1.,  0., -0.,
         -1.,  1., -1., -0.,  0.,  1., -0., -1., -1., -1.,  0.,  1.,  1.,
          0.,  1., -0.,  1.,  0., -1.,  0., -1.,  0., -1., -1., -0.,  1.,
         -1., -1.,  0.,  1.,  1., -0.],
        [ 1.,  1., -1., -0., -1., -0., -1., -0.,  1., -1.,  1.,  1., -0.,
         -1., -1., -1.,  0.,  0.,  1., -0., -0.,  0., -0.,  0., -0.,  1.,
          0., -1., -0., -0., -0., -1., -0.,  1.,  0.,  1.,  1., -1.,  1.,
          1., -0., -1.,  0., -1.,  0.],
        [-0., -0.,  1.,  0., -0., -1., -1., -0., -0.,  1., -1., -1.,  0.,
          1.,  1.,  0.,  1., -0., -1.,  0.,  0., -0., -0.,  1., -0.,  0.,
         -1.,  1.,  1.,  0.,  0., -1.,  1., -1.,  0.,  1.,  0.,  1., -1.,
          1., -0.,  1., -0., -0., -1.],
        [ 0.,  0., -1., -1., -0.,  0.,  1.,  0.,  0.,  0.,  1., -0.,  1.,
         -1., -1., -0., -1., -1.,  1., -0.,  1., -1., -0., -1., -0.,  0.,
          0., -1., -1.,  1., -1.,  1., -0.,  1., -0., -1.,  0.,  0.,  1.,
         -1.,  1.,  0., -0.,  0.,  1.],
        [-0.,  0., -1., -0., -0., -0., -1.,  1., -0., -0.,  1.,  0.,  0.,
          1.,  1., -1.,  1., -0., -1.,  1.,  0.,  0., -0., -1., -0., -1.,
         -0.,  1.,  1., -0., -0.,  1., -0.,  1.,  1., -1., -1.,  0.,  1.,
          1., -0., -0., -0., -1.,  1.],
        [-0., -0., -1., -0.,  0.,  0., -1., -0.,  0., -0.,  1.,  0., -0.,
         -1., -1., -1., -0., -0.,  1.,  0.,  1., -1.,  1.,  0.,  1.,  1.,
         -0., -1., -0.,  1., -0., -1., -0.,  1., -0.,  1.,  1.,  0.,  1.,
          1.,  1., -0.,  1., -1., -0.],
        [ 1., -1.,  1.,  0.,  1.,  0., -1., -0.,  1.,  1.,  1.,  1.,  0.,
         -1.,  1., -1.,  0., -0.,  1., -0., -0.,  0., -0., -0., -0.,  1.,
         -0.,  1., -0., -0., -0., -1., -0., -1., -0., -1., -1., -1.,  1.,
         -1.,  0.,  1., -0.,  1., -0.],
        [-0., -0.,  1.,  0.,  0.,  1., -1., -0., -0.,  1.,  1.,  1.,  0.,
         -1.,  1.,  0., -1.,  0.,  1., -0., -0.,  0.,  0.,  1., -0., -0.,
          1.,  1.,  1., -0.,  0., -1.,  1., -1.,  0., -1., -0., -1.,  1.,
         -1.,  0.,  1.,  0.,  0.,  1.],
        [-0., -0.,  1.,  1.,  0.,  0.,  1.,  0., -0., -0.,  1.,  0.,  1.,
         -1., -1., -0., -1., -1., -1., -0.,  1.,  1., -0.,  1., -0.,  0.,
         -0.,  1., -1.,  1.,  1., -1.,  0.,  1., -0.,  1., -0., -0., -1.,
         -1., -1.,  0.,  0.,  0., -1.],
        [ 0., -0., -1.,  0.,  0., -0.,  1., -1., -0., -0., -1., -0.,  0.,
         -1., -1.,  1., -1.,  0., -1.,  1.,  0.,  0.,  0., -1.,  0., -1.,
          0.,  1., -1., -0.,  0.,  1., -0., -1., -1., -1., -1.,  0.,  1.,
         -1.,  0., -0., -0.,  1.,  1.],
        [ 0., -0., -1., -0.,  0.,  0.,  1.,  0.,  0.,  0., -1.,  0.,  0.,
          1.,  1.,  1., -0., -0.,  1.,  0., -1., -1.,  1., -0., -1.,  1.,
         -0., -1.,  0., -1., -0., -1., -0., -1., -0.,  1.,  1., -0.,  1.,
         -1.,  1.,  0.,  1.,  1.,  0.],
        [ 1., -1.,  1.,  0., -1., -0., -1., -0., -1., -1.,  1.,  1., -0.,
          1., -1., -1., -0., -0., -1., -0.,  0., -0.,  0.,  0., -0., -1.,
          0., -1., -0., -0.,  0.,  1.,  0., -1.,  0.,  1.,  1.,  1.,  1.,
         -1.,  0.,  1., -0.,  1.,  0.],
        [ 0.,  0., -1., -0., -0., -1., -1., -0.,  0.,  1., -1., -1.,  0.,
         -1.,  1.,  0., -1.,  0.,  1.,  0.,  0., -0., -0., -1.,  0.,  0.,
          1.,  1.,  1., -0.,  0.,  1.,  1.,  1.,  0.,  1.,  0., -1., -1.,
         -1.,  0., -1., -0.,  0., -1.]], dtype=np.float32),
 np.array([[-0., -0., -0.,  0., -0.,  1., -1.,  0., -0.,  0., -1.,  0.,  0.,
         -0., -0.,  1.,  0., -1.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,
          0., -1., -0.,  0.,  1., -0.,  1.,  0., -0., -1., -1.,  0., -0.,
          0., -0.,  0., -0., -0.,  0.],
        [ 0.,  0.,  0.,  1.,  0., -1.,  1., -0., -0., -0.,  1.,  0.,  1.,
         -0., -0., -1., -0.,  0.,  0.,  0., -0., -0., -0., -0., -0., -0.,
         -0.,  1.,  0.,  0.,  0., -0., -1., -0., -0.,  1.,  1.,  0.,  0.,
         -0., -0.,  0.,  0.,  0.,  0.],
        [-1.,  0.,  0.,  0., -1., -0., -1., -1., -0.,  1.,  1.,  1., -0.,
          0.,  0., -0., -0.,  0., -0.,  1., -0.,  0.,  1.,  0.,  1.,  0.,
          0.,  1.,  1.,  1., -0.,  0., -0., -0.,  0., -1., -0., -0.,  0.,
         -0., -1., -0., -0., -0.,  1.],
        [ 1.,  0.,  0., -1., -0.,  1.,  0., -0.,  1., -0., -1.,  0., -0.,
         -1.,  0.,  1., -1., -1., -1., -0.,  0., -0.,  0., -0., -1., -1.,
          1.,  0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0.,  0.,  1.,
          0.,  0.,  0., -1., -0., -1.],
        [ 0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,  1.,  1.,  0.,
         -1., -0., -0.,  0.,  0.,  1.,  1., -0., -1.,  0.,  0., -0.,  0.,
         -0.,  0.,  0.,  1., -0.,  0., -0.,  0., -1.,  0., -0.,  1.,  1.,
         -0.,  0., -0.,  0., -0.,  0.],
        [ 0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  0., -1., -0., -0.,  0.,
         -0., -1.,  0., -0.,  0., -0., -1.,  1.,  0., -0., -0., -0., -0.,
          0., -1., -0., -0.,  0.,  0., -0.,  1.,  1.,  0.,  0., -0.,  0.,
          0.,  1.,  1., -0.,  0., -0.],
        [-0.,  1.,  1.,  0.,  0., -0.,  0., -0., -1., -0.,  0., -0.,  0.,
          1., -0., -0.,  1., -0.,  0.,  0.,  0.,  0.,  1.,  1.,  1.,  0.,
          0.,  0.,  0.,  0.,  0., -1., -0., -0., -0.,  0.,  0.,  0., -0.,
         -1., -0., -0., -0.,  0.,  0.],
        [-0.,  0., -0., -1., -0.,  1., -1., -1., -0.,  1., -0.,  1., -1.,
         -0.,  1.,  1., -0., -0., -0.,  1., -1., -1.,  0., -0., -0.,  0.,
         -0.,  0.,  0., -0.,  0.,  0.,  1., -0., -0., -1., -1.,  0.,  1.,
          0., -0.,  0., -0.,  0.,  0.],
        [-1.,  0.,  0.,  0., -1., -0., -1.,  0., -0., -0.,  0., -0.,  0.,
         -0., -1., -0.,  0., -0., -0., -0., -0.,  0.,  1., -0.,  1., -0.,
          0., -0.,  1., -0., -0., -0., -0., -0., -0., -1.,  0., -0., -1.,
          0.,  0., -0.,  0., -0.,  1.],
        [-1.,  1.,  1., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  0.,
          0., -0., -0.,  0.,  0.,  0., -0.,  0., -0.,  1.,  1., -0.,  0.,
         -0.,  0.,  0., -0.,  0., -1., -0., -0.,  0., -1.,  0., -0., -1.,
          0., -0., -0.,  1.,  0.,  1.],
        [-0., -0.,  0.,  0.,  0.,  0.,  1.,  1., -0., -1., -0.,  0.,  1.,
          1., -1., -1., -0.,  1.,  0., -0.,  1., -0., -0., -0.,  0., -0.,
         -1.,  0., -0., -1.,  0.,  0., -1.,  0.,  1., -0.,  0., -1.,  0.,
          1., -0.,  0., -0.,  1., -0.],
        [-0., -0.,  1., -0., -1., -0., -0., -0., -1.,  0.,  0.,  0.,  0.,
         -0.,  0.,  0.,  0., -0.,  0., -0., -0., -0.,  1.,  1., -0.,  0.,
         -0.,  0.,  0.,  0.,  0., -1., -0., -0., -0., -1.,  0.,  0., -1.,
         -0., -0., -0.,  1., -0.,  1.],
        [-0., -0.,  0., -1., -0.,  0., -0., -0., -0.,  0., -0., -0., -0.,
         -0., -0., -0.,  0., -0., -1., -0.,  0., -0.,  0.,  0.,  0., -1.,
          1., -1.,  0., -0.,  1., -1.,  1., -0.,  0., -1., -1.,  0.,  0.,
          0., -0., -0.,  0., -0.,  0.],
        [ 0.,  0., -0., -0., -0., -0.,  1., -0., -0.,  0.,  1.,  0.,  1.,
         -0., -0., -1.,  0.,  1., -0., -0.,  0., -0.,  0., -0., -0.,  0.,
         -1., -0.,  0., -0., -0., -0., -1., -1., -0.,  0., -0.,  0., -0.,
          1., -0.,  0., -0.,  1.,  0.],
        [ 0.,  0.,  1., -0., -1.,  0.,  0.,  1., -1.,  0., -1., -1.,  0.,
          1., -1., -0., -0.,  0.,  0., -0.,  1., -0.,  1.,  1., -0.,  0.,
          0.,  0., -0., -1.,  0., -1., -0.,  1.,  1., -1.,  0.,  0., -1.,
          0., -0.,  1.,  1.,  0.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 4
p = 5
rank = 45

verify_tensor_support_rank_decomposition(decomposition_345, n, m, p, rank)

## Rank-60 support rank decomposition of <3,4,7> over Z


In [ ]:
#@title Data

decomposition_347 = (np.array([[-0., -0., -0., -1., -0.,  1., -1., -0., -0., -1.,  1., -1.,  1.,
        -1.,  1.,  1., -1.,  1., -1., -0.,  1.,  0.,  0., -1., -1.,  0.,
        -1., -1.,  1.,  0.,  1., -1.,  1.,  1., -0.,  1.,  0., -1.,  0.,
        -0.,  1.,  0., -0., -0., -1., -0., -1., -1.,  1., -1.,  0., -1.,
        -1., -1., -0.,  0., -0.,  1., -1., -0.],
       [-0., -0., -0., -1., -0.,  1.,  1., -0.,  0.,  1.,  1.,  1.,  1.,
         1., -1., -1.,  1., -1., -1., -0.,  1.,  0., -0., -1.,  1.,  0.,
        -1.,  1.,  1., -0., -1., -1., -1., -1., -0.,  1.,  0., -1.,  0.,
        -0., -1., -0., -0.,  0.,  1.,  0., -1., -1.,  1.,  1., -0., -1.,
         1.,  1., -0.,  0., -0.,  1., -1.,  0.],
       [-0.,  0.,  0., -1.,  0., -1.,  1., -0., -0., -1., -1., -1.,  1.,
         1., -1.,  1., -1.,  1., -1., -0., -1.,  0., -0., -1., -1., -0.,
        -1.,  1., -1., -0., -1.,  1., -1., -1., -0.,  1., -0.,  1.,  0.,
         0.,  1., -0.,  0.,  0.,  1., -0.,  1., -1.,  1., -1., -0., -1.,
        -1.,  1.,  0., -0.,  0., -1.,  1., -0.],
       [-0.,  0.,  0., -1.,  0., -1., -1., -0.,  0.,  1., -1.,  1.,  1.,
        -1.,  1., -1.,  1., -1., -1., -0., -1.,  0., -0., -1.,  1., -0.,
        -1., -1., -1.,  0.,  1.,  1.,  1.,  1., -0.,  1., -0.,  1.,  0.,
         0., -1., -0.,  0., -0., -1.,  0.,  1., -1.,  1.,  1.,  0., -1.,
         1., -1.,  0.,  0.,  0., -1.,  1.,  0.],
       [-1., -1., -1.,  0., -1.,  1.,  0., -1., -0.,  1., -0.,  0., -1.,
         0.,  1.,  0.,  1.,  0., -1., -1., -0., -0.,  1.,  1.,  0., -1.,
        -1., -1., -1.,  1., -0.,  1., -1., -1., -1., -0.,  1.,  0.,  1.,
        -1., -0.,  1.,  1., -1., -0., -1.,  0.,  0.,  0., -1.,  0., -0.,
        -1., -0.,  0.,  1.,  1.,  0., -1., -1.],
       [ 1.,  1.,  1., -0.,  1.,  1.,  0.,  1., -0.,  1., -0.,  0.,  1.,
         0.,  1.,  0., -1.,  0., -1., -1.,  0., -0.,  1.,  1.,  0.,  1.,
         1.,  1., -1.,  1., -0., -1., -1.,  1.,  1.,  0.,  1.,  0., -1.,
         1., -0.,  1.,  1., -1., -0., -1., -0., -0., -0., -1., -0.,  0.,
         1., -0., -0.,  1., -1.,  0.,  1., -1.],
       [-1.,  1.,  1.,  0.,  1.,  1., -0., -1.,  0., -1., -0.,  0.,  1.,
        -0., -1.,  0.,  1.,  0.,  1.,  1., -0.,  0.,  1.,  1.,  0., -1.,
        -1., -1.,  1., -1.,  0.,  1., -1.,  1., -1., -0.,  1.,  0.,  1.,
         1., -0.,  1., -1.,  1.,  0., -1., -0.,  0.,  0., -1., -0., -0.,
         1.,  0.,  0., -1., -1.,  0.,  1., -1.],
       [-1.,  1.,  1.,  0.,  1., -1.,  0., -1., -0.,  1.,  0., -0.,  1.,
         0.,  1., -0.,  1., -0., -1., -1.,  0.,  0., -1., -1., -0., -1.,
        -1., -1., -1.,  1., -0.,  1.,  1.,  1., -1., -0., -1., -0.,  1.,
         1.,  0., -1.,  1., -1., -0.,  1., -0.,  0.,  0.,  1.,  0., -0.,
         1., -0.,  0.,  1., -1.,  0.,  1.,  1.],
       [-0.,  0., -1., -1., -1.,  0., -1., -1.,  1., -0., -1.,  1.,  0.,
         1., -0., -0., -0., -1., -0.,  1.,  0., -1.,  1., -0., -1., -1.,
         0., -0.,  0., -0.,  1., -0.,  0.,  0.,  1., -1., -1., -1., -1.,
         1.,  1., -1.,  1.,  1.,  1., -1., -1., -1., -1.,  0., -1.,  0.,
        -0.,  0.,  1.,  1., -1., -1.,  0.,  0.],
       [-0.,  0., -1.,  1.,  1., -0., -1., -1.,  1., -0.,  1., -1., -0.,
        -1., -0., -0., -0., -1.,  0., -1.,  0.,  1., -1.,  0.,  1.,  1.,
        -0., -0.,  0., -0., -1.,  0.,  0.,  0.,  1., -1.,  1., -1.,  1.,
         1.,  1., -1., -1.,  1.,  1., -1.,  1., -1.,  1.,  0., -1., -0.,
        -0.,  0., -1.,  1.,  1., -1.,  0., -0.],
       [-0., -0., -1., -1.,  1.,  0.,  1.,  1., -1.,  0., -1.,  1., -0.,
        -1., -0.,  0.,  0.,  1.,  0., -1., -0.,  1., -1.,  0.,  1., -1.,
        -0., -0., -0.,  0.,  1., -0.,  0.,  0.,  1., -1., -1., -1.,  1.,
        -1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,  1., -0., -1., -0.,
         0.,  0.,  1., -1., -1.,  1.,  0.,  0.],
       [-0.,  0.,  1., -1.,  1., -0., -1., -1.,  1., -0., -1.,  1., -0.,
        -1.,  0., -0., -0., -1.,  0., -1.,  0.,  1., -1.,  0.,  1., -1.,
        -0.,  0.,  0., -0.,  1., -0., -0., -0., -1.,  1., -1.,  1.,  1.,
         1., -1., -1.,  1., -1., -1.,  1.,  1., -1.,  1.,  0.,  1., -0.,
        -0., -0.,  1.,  1., -1., -1.,  0.,  0.]], dtype=np.float32), np.array([[ 1., -1., -0., -0., -0., -1., -0.,  0.,  1., -0.,  1., -0., -0.,
         0., -0., -1., -0., -1., -1.,  1., -1., -1., -0.,  0.,  0.,  0.,
        -0., -1., -0.,  1.,  0., -0.,  0.,  0.,  1., -0.,  1.,  0., -0.,
        -1., -0., -0.,  0.,  0.,  1., -0., -0., -0.,  1., -0.,  1.,  1.,
        -1.,  1.,  1., -0.,  0., -0.,  0.,  1.],
       [-1., -1.,  0., -0., -0.,  0.,  0.,  0., -1.,  0., -1.,  0., -0.,
         0., -1.,  1., -0.,  1.,  0., -0.,  1., -1.,  0.,  0., -0.,  0.,
        -1.,  0., -0., -1.,  0.,  0., -0.,  0., -0., -0., -0., -0., -1.,
        -0.,  0.,  1.,  0.,  1.,  1., -0.,  0., -0.,  1.,  1.,  1.,  1.,
         0.,  1., -1.,  0., -1., -0.,  1.,  1.],
       [-1., -1.,  0., -0., -1.,  0.,  0.,  0.,  1.,  1., -1.,  0., -1.,
         0.,  0., -1., -0., -1., -0.,  0.,  1., -1., -0.,  0., -0.,  1.,
        -0.,  0., -0.,  1.,  0.,  1., -1.,  0., -0.,  0., -0., -0., -0.,
         0.,  0., -0.,  0., -0., -1., -1., -0.,  0.,  1., -0., -1.,  1.,
         0., -1., -1.,  1.,  0., -0.,  0., -1.],
       [-1.,  1.,  0., -0., -0.,  0., -0., -0., -1., -0., -0.,  1.,  0.,
        -1., -1., -1.,  0., -0.,  0., -1., -1.,  1.,  0., -0., -0.,  0.,
        -1.,  0., -0., -1., -0., -0.,  0., -0., -1.,  1., -1.,  0., -0.,
         1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -1., -1.,  1.,
        -0.,  1., -1.,  0.,  0.,  1., -1., -1.],
       [ 1., -1.,  1., -0., -0.,  0., -0., -1.,  1., -0., -0.,  0.,  0.,
        -0., -1., -1., -0., -0.,  0., -0.,  1.,  1.,  1.,  0., -1.,  0.,
         1., -0., -0., -1., -1.,  0.,  0., -0.,  0.,  0.,  0.,  1.,  0.,
        -0.,  0.,  0.,  1.,  0.,  0.,  0., -0., -1., -0., -1.,  1., -1.,
         0.,  1., -1., -0., -0., -0.,  1., -1.],
       [ 1.,  1., -0.,  1.,  1., -0.,  1., -0., -1.,  0.,  0., -0., -0.,
         0., -1.,  1.,  0.,  0.,  0.,  0., -1.,  1.,  0., -0.,  0., -1.,
         1., -0.,  0., -1.,  0., -0.,  0., -0.,  0., -0.,  0., -0.,  0.,
         0., -1.,  0., -0., -0.,  0.,  1.,  1., -0., -0.,  1.,  1., -1.,
        -0.,  1.,  1., -1., -0., -0., -1.,  1.],
       [ 1., -1.,  1.,  0.,  0., -0.,  0., -1.,  1.,  0., -1., -0., -0.,
        -0., -0., -1., -1., -1.,  0., -0.,  1.,  1.,  1., -1.,  0., -0.,
         0., -0., -1., -1.,  0., -0.,  0.,  1.,  0., -0.,  0.,  0., -0.,
        -0., -0., -0.,  1.,  0.,  1.,  0., -0., -0., -1., -0.,  1., -1.,
        -0.,  1., -1., -0.,  0., -0.,  0., -1.],
       [-1.,  1.,  0., -0., -0., -1., -0.,  0., -1., -0.,  1., -0.,  0.,
        -0.,  0.,  1.,  0.,  1., -1.,  1., -1., -1., -0.,  0.,  0., -0.,
        -0.,  1., -0.,  1.,  0.,  0., -0., -0., -1., -0.,  1.,  0.,  0.,
         1., -0.,  0.,  0., -0., -1.,  0., -0.,  0.,  1.,  0., -1.,  1.,
         1., -1.,  1.,  0., -0.,  0., -0.,  1.],
       [ 1.,  1., -0.,  0.,  0., -0., -0., -0., -1.,  0.,  1.,  0.,  0.,
         0., -1.,  1., -0.,  1.,  0.,  0., -1.,  1.,  0., -0., -0., -0.,
         1.,  0.,  0., -1., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  1.,
         0., -0.,  1.,  0.,  1.,  1., -0.,  0., -0., -1.,  1.,  1., -1.,
         0.,  1.,  1.,  0.,  1.,  0., -1.,  1.],
       [ 1.,  1., -0., -0.,  1., -0.,  0., -0.,  1.,  1.,  1.,  0.,  1.,
         0.,  0., -1., -0., -1., -0.,  0., -1.,  1.,  0., -0., -0., -1.,
         0.,  0., -0.,  1., -0., -1., -1., -0., -0., -0.,  0., -0.,  0.,
         0.,  0., -0., -0., -0., -1., -1.,  0.,  0., -1., -0., -1., -1.,
         0., -1.,  1.,  1., -0., -0., -0., -1.],
       [-1.,  1.,  0., -0., -0.,  0.,  0.,  0., -1.,  0.,  0., -1.,  0.,
         1.,  1.,  1.,  0.,  0.,  0.,  1., -1., -1., -0., -0.,  0.,  0.,
        -1., -0., -0.,  1.,  0., -0., -0., -0., -1.,  1.,  1.,  0.,  0.,
         1., -0.,  0., -0.,  0., -0., -0.,  0., -0.,  0.,  1., -1.,  1.,
        -0., -1.,  1., -0.,  0.,  1., -1.,  1.],
       [-1.,  1., -1., -0.,  0.,  0., -0.,  1., -1., -0., -0.,  0., -0.,
        -0., -1., -1.,  0., -0.,  0., -0., -1.,  1.,  1., -0., -1., -0.,
        -1.,  0., -0., -1., -1.,  0., -0., -0.,  0., -0., -0., -1., -0.,
        -0.,  0., -0.,  1.,  0.,  0., -0.,  0.,  1.,  0., -1., -1.,  1.,
        -0.,  1., -1.,  0.,  0., -0., -1., -1.],
       [-1., -1.,  0., -1., -1.,  0.,  1.,  0., -1.,  0., -0., -0.,  0.,
         0., -1.,  1.,  0.,  0.,  0.,  0.,  1., -1., -0., -0., -0.,  1.,
        -1.,  0., -0., -1.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0., -0.,
         0., -1., -0., -0.,  0.,  0.,  1., -1.,  0.,  0.,  1.,  1.,  1.,
         0.,  1., -1., -1.,  0.,  0.,  1.,  1.],
       [-1.,  1., -1., -0., -0., -0.,  0.,  1., -1.,  0., -1.,  0.,  0.,
        -0.,  0.,  1.,  1.,  1.,  0., -0.,  1.,  1.,  1., -1., -0., -0.,
         0.,  0., -1., -1.,  0., -0., -0., -1.,  0., -0.,  0., -0.,  0.,
        -0.,  0.,  0.,  1., -0., -1.,  0., -0.,  0., -1., -0., -1., -1.,
         0., -1., -1.,  0., -0.,  0., -0., -1.],
       [-1., -1., -0.,  0.,  0.,  1., -0., -0.,  1., -0., -1., -0., -0.,
        -0.,  0., -1., -0., -1., -1.,  1.,  1., -1., -0.,  0., -0., -0.,
        -0.,  1.,  0.,  1.,  0.,  0., -0., -0., -1., -0., -1.,  0., -0.,
        -1., -0., -0., -0., -0., -1., -0., -0.,  0.,  1., -0., -1.,  1.,
        -1., -1., -1., -0.,  0., -0.,  0., -1.],
       [-1.,  1., -0.,  0., -0., -0.,  0., -0., -1.,  0.,  1.,  0., -0.,
         0.,  1.,  1., -0.,  1.,  0., -0., -1., -1.,  0.,  0., -0.,  0.,
        -1., -0.,  0.,  1., -0., -0.,  0.,  0.,  0.,  0.,  0., -0., -1.,
         0.,  0.,  1.,  0., -1., -1., -0.,  0., -0.,  1.,  1., -1.,  1.,
         0., -1.,  1.,  0.,  1.,  0., -1.,  1.],
       [ 1., -1., -0., -0., -1., -0.,  0.,  0.,  1.,  1.,  1.,  0., -1.,
        -0., -0., -1.,  0., -1.,  0., -0., -1., -1.,  0.,  0., -0., -1.,
        -0., -0., -0.,  1., -0., -1.,  1.,  0.,  0.,  0.,  0.,  0., -0.,
         0., -0., -0., -0.,  0.,  1.,  1., -0.,  0.,  1.,  0.,  1.,  1.,
         0.,  1.,  1.,  1.,  0., -0.,  0.,  1.],
       [ 1.,  1.,  0.,  0.,  0., -0., -0.,  0., -1., -0.,  0., -1., -0.,
        -1., -1.,  1., -0.,  0., -0., -1., -1.,  1.,  0., -0.,  0., -0.,
         1.,  0.,  0., -1., -0.,  0.,  0., -0.,  1., -1.,  1.,  0.,  0.,
         1.,  0.,  0.,  0., -0.,  0., -0., -0., -0., -0.,  1.,  1., -1.,
        -0.,  1.,  1., -0.,  0.,  1., -1.,  1.],
       [ 1.,  1., -1.,  0.,  0.,  0.,  0., -1.,  1.,  0.,  0.,  0., -0.,
         0.,  1., -1.,  0., -0.,  0.,  0., -1.,  1.,  1.,  0., -1.,  0.,
         1.,  0., -0.,  1.,  1., -0., -0.,  0., -0.,  0., -0., -1.,  0.,
        -0.,  0.,  0., -1., -0., -0., -0.,  0., -1., -0., -1., -1., -1.,
        -0., -1.,  1.,  0., -0., -0., -1., -1.],
       [-1.,  1., -0., -1.,  1., -0.,  1., -0., -1., -0., -0., -0.,  0.,
         0., -1., -1., -0., -0.,  0.,  0., -1.,  1.,  0.,  0.,  0.,  1.,
        -1., -0., -0., -1.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,
        -0.,  1., -0., -0.,  0.,  0., -1.,  1., -0.,  0., -1., -1.,  1.,
        -0.,  1., -1., -1., -0., -0., -1., -1.],
       [-1., -1.,  1., -0.,  0., -0.,  0.,  1., -1.,  0., -1., -0., -0.,
         0., -0.,  1.,  1.,  1., -0., -0.,  1., -1., -1.,  1., -0.,  0.,
        -0., -0., -1., -1.,  0., -0.,  0.,  1., -0.,  0.,  0.,  0., -0.,
        -0., -0., -0.,  1.,  0.,  1., -0.,  0.,  0.,  1.,  0.,  1.,  1.,
        -0.,  1., -1.,  0.,  0., -0.,  0.,  1.],
       [ 1.,  1.,  0., -0.,  0.,  1.,  0.,  0., -1.,  0., -1.,  0.,  0.,
         0., -0.,  1., -0.,  1., -1.,  1.,  1., -1.,  0., -0., -0.,  0.,
        -0., -1.,  0.,  1.,  0., -0.,  0., -0.,  1., -0., -1., -0., -0.,
         1.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  1.,  0.,  1.,  1.,
         1.,  1., -1.,  0., -0.,  0., -0., -1.],
       [-1.,  1.,  0.,  0., -0., -0., -0.,  0.,  1., -0.,  1., -0., -0.,
         0., -1., -1.,  0., -1.,  0., -0., -1., -1., -0.,  0.,  0., -0.,
        -1.,  0.,  0., -1., -0., -0.,  0.,  0., -0., -0.,  0., -0., -1.,
         0., -0., -1., -0.,  1.,  1.,  0.,  0., -0.,  1., -1.,  1.,  1.,
        -0.,  1.,  1., -0.,  1.,  0., -1., -1.],
       [-1.,  1.,  0., -0.,  1.,  0.,  0., -0.,  1.,  1., -1., -0.,  1.,
        -0., -0., -1.,  0., -1.,  0., -0.,  1.,  1.,  0.,  0., -0.,  1.,
         0., -0., -0.,  1., -0.,  1.,  1.,  0.,  0., -0., -0., -0.,  0.,
         0., -0., -0.,  0.,  0.,  1.,  1.,  0., -0., -1., -0.,  1., -1.,
         0.,  1., -1.,  1., -0., -0., -0.,  1.],
       [-1., -1., -0., -0., -0., -0., -0., -0.,  1.,  0.,  0., -1., -0.,
        -1., -1.,  1., -0.,  0.,  0., -1.,  1.,  1., -0.,  0.,  0.,  0.,
        -1.,  0.,  0., -1.,  0., -0.,  0.,  0., -1.,  1.,  1., -0., -0.,
        -1.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0.,  1., -1.,  1.,
        -0.,  1.,  1., -0.,  0., -1.,  1.,  1.],
       [ 1.,  1., -1., -0.,  0.,  0., -0., -1.,  1., -0., -0.,  0., -0.,
        -0., -1.,  1.,  0.,  0.,  0., -0., -1., -1., -1.,  0.,  1., -0.,
         1., -0., -0., -1., -1., -0.,  0., -0., -0.,  0.,  0., -1.,  0.,
         0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -1., -0.,  1., -1., -1.,
         0.,  1., -1.,  0.,  0., -0., -1.,  1.],
       [-1.,  1., -0., -1.,  1., -0., -1.,  0.,  1., -0., -0., -0.,  0.,
        -0.,  1.,  1.,  0.,  0.,  0., -0., -1.,  1., -0., -0.,  0.,  1.,
        -1.,  0.,  0.,  1., -0.,  0.,  0.,  0., -0.,  0.,  0., -0., -0.,
         0., -1.,  0., -0., -0., -0.,  1.,  1., -0.,  0.,  1.,  1.,  1.,
         0., -1., -1.,  1., -0., -0., -1.,  1.],
       [-1., -1.,  1.,  0.,  0.,  0., -0.,  1., -1., -0.,  1., -0., -0.,
        -0., -0.,  1.,  1.,  1.,  0., -0., -1.,  1.,  1., -1., -0., -0.,
         0., -0.,  1.,  1.,  0.,  0., -0.,  1.,  0., -0., -0.,  0.,  0.,
        -0., -0.,  0., -1.,  0.,  1., -0., -0.,  0., -1.,  0.,  1., -1.,
        -0.,  1.,  1.,  0., -0., -0.,  0., -1.]], dtype=np.float32), np.array([[-1.,  1.,  0., -0., -0.,  1.,  0., -0.,  0., -1.,  0.,  0., -1.,
        -0., -1.,  1.,  1., -0., -1.,  0.,  1.,  0.,  0., -1., -0., -0.,
         1., -1.,  1.,  1.,  0.,  1.,  1.,  1.,  0., -0., -0., -0.,  0.,
        -0., -0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,  1.,
        -1.,  1.,  0., -0., -0., -0.,  1., -1.],
       [ 1., -1., -0.,  0.,  0.,  1., -0.,  0.,  0.,  1., -0., -0.,  1.,
        -0.,  1., -1., -1.,  0., -1.,  0., -1., -0., -0.,  1.,  0.,  0.,
        -1., -1., -1., -1., -0., -1., -1., -1.,  0., -0.,  0.,  0., -0.,
         0.,  0., -0.,  0., -0.,  0., -0., -0., -0., -0., -1.,  0., -1.,
        -1., -1.,  0.,  0.,  0.,  0., -1.,  1.],
       [ 1., -1., -1.,  1.,  1.,  0., -1.,  1.,  0.,  0.,  1., -1.,  0.,
         1., -0.,  1.,  0., -1.,  0., -1.,  1., -0., -1., -0.,  1.,  1.,
         0., -0.,  0., -1., -1., -0., -0.,  0., -1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1., -1.,  1., -1., -1., -1., -1.,  1.,  0.,  0.,  1.,
        -0.,  1.,  0.,  1., -1.,  1., -0.,  1.],
       [ 1.,  1.,  0.,  1., -0.,  1., -1.,  0.,  1.,  1.,  1.,  1., -1.,
         1.,  1.,  0., -1., -1.,  1.,  0.,  0., -1., -0., -1., -1., -0.,
        -1.,  1., -1., -1., -1., -1.,  1.,  1.,  0.,  1.,  0., -1., -0.,
        -0., -1.,  0.,  0., -0.,  1., -0.,  1., -1., -1.,  1.,  1., -0.,
        -1., -0.,  1., -0., -0., -1.,  1., -1.],
       [ 1.,  1.,  1.,  0., -1., -0.,  0.,  1., -1., -0., -0.,  0.,  0.,
        -0.,  0.,  0.,  0.,  0., -0., -1.,  0.,  1.,  1.,  0., -0.,  1.,
        -0., -0.,  0., -1., -0.,  0., -0., -0., -1.,  0., -1., -0., -1.,
        -1., -0.,  1., -1., -1.,  0.,  1.,  0., -0., -0.,  0., -1., -0.,
         0., -0., -1.,  1., -1., -0.,  0., -1.],
       [ 1.,  1.,  1.,  0., -1., -0., -0.,  1., -1.,  0., -0.,  0.,  0.,
         0., -0., -0., -0.,  0., -0., -1.,  0.,  1.,  1.,  0., -0.,  1.,
        -0.,  0.,  0., -1., -0.,  0., -0., -0., -1.,  0., -1., -0.,  1.,
        -1., -0., -1., -1.,  1., -0.,  1.,  0., -0.,  0., -0., -1.,  0.,
        -0.,  0., -1.,  1.,  1., -0.,  0., -1.],
       [-1., -1., -0., -0.,  0.,  1., -0., -0.,  0.,  1., -0., -0.,  1.,
        -0.,  1.,  1.,  1., -0.,  1.,  0., -1.,  0.,  0., -1.,  0., -0.,
         1., -1., -1., -1., -0.,  1.,  1., -1.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0., -0.,  0.,  0., -0.,  0.,  0.,  1.,  0.,  1.,
         1., -1., -0.,  0.,  0.,  0., -1., -1.],
       [ 1.,  1.,  0.,  0., -0., -1., -0.,  0., -0.,  1.,  0., -0.,  1.,
         0., -1., -1., -1.,  0., -1., -0.,  1.,  0.,  0.,  1.,  0.,  0.,
        -1.,  1.,  1.,  1., -0.,  1.,  1.,  1.,  0., -0., -0., -0., -0.,
        -0.,  0., -0., -0.,  0., -0., -0.,  0., -0., -0., -1., -0., -1.,
        -1.,  1., -0.,  0., -0., -0.,  1.,  1.],
       [ 1.,  1.,  1.,  1., -1.,  0.,  1.,  1.,  0., -0., -1., -1.,  0.,
        -1.,  0.,  1.,  0., -1.,  0.,  1., -1.,  0., -1., -0.,  1.,  1.,
         0., -0., -0.,  1.,  1.,  0.,  0., -0., -1.,  1.,  1., -1.,  1.,
        -1.,  1.,  1.,  1., -1.,  1., -1.,  1., -1.,  1.,  0.,  0.,  1.,
         0., -1., -0., -1.,  1., -1.,  0.,  1.],
       [-0., -0., -0.,  1., -0.,  0., -1., -0.,  1.,  0.,  1.,  1.,  0.,
        -1., -0.,  1., -0., -1., -0., -0.,  1.,  1.,  0.,  0.,  1., -0.,
         0., -0., -0.,  0., -1., -0., -0., -0., -0., -1.,  0.,  1.,  0.,
         0.,  1., -0.,  0.,  0., -1.,  0., -1., -1.,  1.,  0., -1.,  1.,
        -0.,  1.,  1., -0., -0., -1.,  0., -0.],
       [ 0., -0.,  1.,  0., -1.,  1., -0., -1.,  1.,  1.,  0., -0.,  1.,
         0.,  1., -1., -1.,  0., -1., -1., -1.,  1.,  1.,  1.,  0., -1.,
        -1., -1., -1.,  0., -0., -1., -1., -1., -1.,  0.,  1., -0., -1.,
         1.,  0., -1.,  1., -1.,  0.,  1.,  0., -0., -0., -1., -1., -1.,
        -1., -1.,  1., -1.,  1.,  0., -1.,  0.],
       [ 0., -0., -0.,  1., -0., -0., -1., -0.,  1.,  0.,  1., -1.,  0.,
         1., -0.,  1., -0., -1., -0.,  0.,  1.,  1., -0.,  0.,  1., -0.,
         0., -0., -0.,  0., -1., -0., -0., -0., -0.,  1.,  0.,  1.,  0.,
         0.,  1.,  0.,  0.,  0., -1.,  0., -1., -1.,  1.,  0., -1.,  1.,
         0.,  1.,  1., -0., -0.,  1.,  0.,  0.],
       [-0.,  0., -0.,  1., -0.,  0.,  1.,  0., -1., -0.,  1.,  1.,  0.,
        -1.,  0., -1.,  0.,  1., -0., -0.,  1.,  1.,  0.,  0.,  1., -0.,
         0.,  0., -0., -0., -1., -0.,  0.,  0.,  0.,  1.,  0., -1.,  0.,
        -0., -1., -0.,  0., -0.,  1., -0., -1.,  1.,  1., -0.,  1.,  1.,
         0., -1.,  1.,  0.,  0.,  1., -0., -0.],
       [ 0., -0., -1., -0., -1.,  1., -0.,  1., -1., -1.,  0., -0.,  1.,
         0., -1.,  1.,  1., -0., -1., -1., -1.,  1.,  1.,  1., -0., -1.,
        -1.,  1., -1., -0.,  0., -1.,  1.,  1.,  1., -0.,  1.,  0., -1.,
        -1.,  0.,  1.,  1.,  1., -0., -1.,  0., -0., -0.,  1.,  1., -1.,
         1.,  1.,  1.,  1.,  1., -0., -1., -0.],
       [ 0., -0., -0.,  1., -0.,  0.,  1.,  0., -1., -0.,  1.,  1.,  0.,
        -1.,  0., -1.,  0.,  1., -0., -0.,  1.,  1.,  0.,  0., -1., -0.,
         0., -0., -0., -0.,  1., -0.,  0., -0., -0.,  1.,  0.,  1.,  0.,
         0., -1., -0.,  0., -0.,  1., -0., -1., -1.,  1., -0.,  1.,  1.,
        -0., -1.,  1.,  0., -0.,  1., -0.,  0.],
       [ 0., -0., -0.,  1.,  0.,  0.,  1.,  0., -1., -0.,  1.,  1., -0.,
         1., -0., -1.,  0.,  1.,  0.,  0.,  1., -1., -0., -0., -1., -0.,
        -0., -0., -0.,  0., -1., -0., -0., -0., -0., -1.,  0.,  1., -0.,
        -0.,  1., -0.,  0.,  0., -1.,  0.,  1.,  1., -1., -0., -1., -1.,
         0.,  1.,  1.,  0.,  0.,  1., -0.,  0.],
       [ 0.,  0., -1.,  0., -1., -1., -0., -1.,  1.,  1., -0.,  0.,  1.,
         0., -1., -1., -1.,  0., -1., -1.,  1.,  1.,  1.,  1., -0.,  1.,
        -1.,  1.,  1.,  0., -0.,  1.,  1.,  1.,  1., -0., -1.,  0., -1.,
         1.,  0., -1., -1.,  1., -0., -1.,  0.,  0., -0., -1.,  1., -1.,
        -1.,  1., -1., -1., -1.,  0.,  1.,  0.],
       [-0., -0., -0., -1.,  0., -0., -1., -0., -1.,  0.,  1.,  1., -0.,
         1., -0., -1., -0.,  1., -0., -0.,  1., -1.,  0.,  0., -1., -0.,
        -0.,  0.,  0.,  0., -1., -0., -0.,  0.,  0., -1., -0.,  1., -0.,
         0., -1., -0., -0.,  0., -1.,  0., -1.,  1., -1., -0., -1., -1.,
        -0.,  1.,  1., -0., -0.,  1.,  0., -0.],
       [-1.,  1.,  0., -0., -0., -1., -0., -0., -0.,  1., -0., -0., -1.,
        -0.,  1., -1.,  1.,  0.,  1., -0.,  1.,  0., -0.,  1.,  0., -0.,
         1., -1., -1., -1., -0.,  1., -1.,  1.,  0.,  0.,  0., -0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  1.,
        -1., -1., -0.,  0., -0., -0.,  1.,  1.],
       [-1.,  1.,  0., -0., -0., -1.,  0., -0., -0.,  1., -0., -0., -1.,
        -0.,  1., -1., -1.,  0.,  1.,  0.,  1., -0., -0., -1.,  0., -0.,
         1., -1.,  1., -1.,  0.,  1., -1., -1., -0.,  0., -0., -0.,  0.,
        -0.,  0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -1., -0.,  1.,
        -1., -1., -0.,  0., -0., -0.,  1.,  1.],
       [-1.,  1.,  1., -1., -1.,  0., -1., -1.,  0.,  0., -1., -1., -0.,
         1., -0.,  1., -0., -1., -0., -1., -1., -0., -1.,  0.,  1., -1.,
        -0.,  0.,  0., -1., -1., -0., -0.,  0.,  1., -1.,  1., -1., -1.,
        -1.,  1.,  1., -1.,  1., -1., -1.,  1.,  1., -1.,  0., -0., -1.,
         0.,  1., -0.,  1.,  1., -1.,  0.,  1.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 3
m = 4
p = 7
rank = 60

verify_tensor_support_rank_decomposition(decomposition_347, n, m, p, rank)

## Rank-59 support rank decomposition of <4,4,5> over Z


In [ ]:
#@title Data

decomposition_445 = (np.array([[-1., -1.,  1., -1.,  1., -0.,  1.,  1., -1., -0., -0., -1., -0.,
        -0.,  0., -1., -1.,  0.,  1.,  0.,  0., -1.,  1.,  0.,  1.,  1.,
         0.,  0., -0.,  0., -0., -0.,  1.,  0.,  0.,  1., -1.,  1.,  0.,
         1., -1., -1.,  0.,  0.,  1.,  0.,  0.,  0., -1.,  0., -1., -1.,
        -0., -1., -1.,  0., -1., -1., -0.],
       [ 1., -1., -1.,  1.,  1., -0.,  1., -1.,  1., -0., -0.,  1., -0.,
        -0.,  0.,  1.,  1., -0., -1.,  0.,  0.,  1., -1.,  0.,  1., -1.,
        -0.,  0., -0., -0., -0., -0.,  1.,  0.,  0.,  1.,  1., -1.,  0.,
        -1., -1., -1., -0., -0.,  1.,  0.,  0.,  0.,  1.,  0., -1., -1.,
         0.,  1., -1., -0.,  1., -1., -0.],
       [ 1.,  0.,  1.,  1.,  1., -0., -1., -1., -1.,  0., -0.,  1., -0.,
         0., -0.,  0., -0.,  0., -1.,  0., -0., -1.,  1.,  0.,  1.,  1.,
         0.,  0.,  0., -0., -0., -0.,  1., -0.,  0., -0., -0.,  1.,  0.,
         1.,  1., -1.,  0.,  0., -1., -0., -0., -0., -1., -0., -1., -1.,
         0., -1.,  0., -0.,  1.,  1., -0.],
       [ 1.,  0.,  1.,  1., -1.,  0.,  1., -1., -1., -0.,  0.,  1.,  0.,
        -0.,  0.,  0.,  0.,  0., -1.,  0.,  0., -1.,  1., -0., -1.,  1.,
        -0., -0.,  0.,  0., -0.,  0., -1.,  0., -0.,  0.,  0.,  1., -0.,
         1., -1.,  1.,  0., -0.,  1.,  0.,  0.,  0., -1., -0.,  1.,  1.,
         0., -1., -0.,  0.,  1., -1.,  0.],
       [ 0., -1.,  1.,  1., -1., -1., -1., -0., -1.,  1.,  1., -1., -0.,
         1., -1., -0., -1.,  1.,  0., -1., -0., -0., -0., -1.,  1., -1.,
         1., -1., -0.,  1.,  1.,  1.,  1., -0.,  1.,  0.,  1., -1.,  1.,
         1., -1.,  1.,  0., -1.,  1., -1., -1.,  1.,  0.,  1., -1.,  0.,
        -1., -0.,  1.,  1., -0.,  0., -0.],
       [ 0.,  1.,  1., -1.,  1.,  1., -1.,  0., -1.,  1., -1., -1.,  0.,
        -1.,  1.,  0., -1., -1., -0., -1., -0., -0., -0.,  1., -1.,  1.,
        -1., -1.,  0.,  1.,  1., -1., -1.,  0., -1., -0.,  1., -1., -1.,
        -1., -1.,  1., -0., -1., -1., -1., -1.,  1.,  0., -1., -1.,  0.,
        -1.,  0., -1., -1.,  0., -0.,  0.],
       [ 0.,  0.,  1.,  1., -1.,  1., -1., -0.,  1., -1., -1., -1.,  0.,
        -1., -1., -1.,  0.,  1., -0.,  1.,  0.,  0., -0.,  1., -1.,  1.,
         1.,  1.,  0.,  1., -1., -1., -1., -0.,  1., -1.,  0.,  1.,  1.,
         1.,  1., -1., -0., -1.,  1.,  1., -1.,  1.,  0.,  1., -1.,  0.,
        -1.,  0., -0., -1., -0.,  0.,  0.],
       [-0., -0.,  1., -1.,  1., -1., -1., -0.,  1., -1.,  1., -1., -0.,
         1.,  1., -1.,  0., -1., -0.,  1., -0.,  0.,  0., -1.,  1., -1.,
        -1.,  1., -0.,  1., -1.,  1.,  1.,  0., -1.,  1., -0.,  1., -1.,
        -1.,  1., -1., -0., -1., -1.,  1., -1.,  1., -0., -1., -1.,  0.,
        -1., -0., -0.,  1., -0.,  0., -0.],
       [-0., -1.,  1.,  1., -1.,  0., -1.,  1., -1.,  1., -0., -1.,  0.,
        -0.,  0., -0., -1., -0., -1.,  0., -0., -0.,  1., -1.,  1., -1.,
        -0., -1.,  1., -0., -0.,  1.,  1., -0., -0.,  0.,  1., -1.,  1.,
         1., -1.,  1., -1., -1.,  1.,  0.,  0.,  0.,  0.,  1., -1.,  0.,
        -1.,  1.,  1., -0., -0.,  0., -0.],
       [-0.,  1.,  1.,  1.,  1., -0.,  1., -1.,  1., -1., -0.,  1.,  0.,
         0.,  0.,  0., -1.,  0., -1., -0.,  0.,  0.,  1., -1.,  1., -1.,
         0., -1.,  1.,  0.,  0., -1., -1., -0., -0., -0.,  1., -1., -1.,
         1.,  1., -1., -1.,  1.,  1., -0., -0.,  0., -0.,  1.,  1.,  0.,
        -1.,  1., -1.,  0., -0.,  0., -0.],
       [ 0., -0.,  1., -1., -1., -0.,  1., -1., -1.,  1.,  0.,  1., -0.,
        -0., -0., -1.,  0.,  0.,  1.,  0., -0.,  0., -1., -1.,  1., -1.,
        -0.,  1.,  1., -0.,  0., -1., -1., -0.,  0., -1.,  0.,  1.,  1.,
        -1., -1.,  1.,  1.,  1., -1., -0., -0., -0.,  0., -1.,  1.,  0.,
        -1.,  1., -0.,  0.,  0.,  0.,  0.],
       [-0., -0., -1.,  1., -1.,  0.,  1., -1., -1.,  1.,  0.,  1., -0.,
         0.,  0.,  1., -0.,  0., -1.,  0., -0.,  0.,  1.,  1., -1.,  1.,
         0., -1., -1.,  0., -0., -1., -1.,  0., -0., -1.,  0., -1.,  1.,
         1., -1.,  1., -1.,  1.,  1.,  0., -0.,  0.,  0.,  1.,  1., -0.,
         1., -1., -0.,  0.,  0.,  0., -0.],
       [ 0., -1.,  1., -1.,  1.,  0.,  1., -0., -1.,  0.,  1., -1.,  1.,
         1., -1., -1., -1.,  0.,  0., -1., -1.,  0., -0., -0.,  1.,  1.,
         1.,  0., -0.,  1.,  0.,  0.,  1., -1., -1.,  1., -1.,  1., -0.,
         1., -1., -1., -0.,  0.,  1.,  1., -0., -1., -1., -0., -1., -1.,
         0.,  0., -1., -1., -1., -1.,  1.],
       [ 0.,  1.,  1.,  1., -1., -0.,  1.,  0.,  1.,  0.,  1.,  1.,  1.,
        -1., -1., -1., -1., -0., -0.,  1.,  1., -0., -0.,  0.,  1., -1.,
        -1., -0., -0.,  1., -0.,  0., -1., -1.,  1., -1., -1.,  1.,  0.,
        -1., -1., -1.,  0., -0.,  1.,  1., -0., -1.,  1.,  0., -1., -1.,
         0., -0.,  1.,  1.,  1., -1., -1.],
       [ 0.,  0.,  1., -1.,  1.,  0.,  1.,  0.,  1., -0., -1., -1., -1.,
         1., -1.,  0.,  0.,  0.,  0.,  1.,  1.,  0., -0., -0., -1., -1.,
        -1., -0.,  0., -1., -0., -0.,  1., -1., -1.,  0., -0.,  1., -0.,
        -1., -1.,  1.,  0.,  0.,  1., -1., -0., -1.,  1.,  0.,  1.,  1.,
         0., -0.,  0.,  1., -1., -1.,  1.],
       [-0.,  0.,  1.,  1., -1., -0.,  1.,  0., -1., -0., -1.,  1., -1.,
        -1., -1.,  0.,  0.,  0.,  0., -1., -1.,  0., -0.,  0., -1.,  1.,
         1., -0.,  0., -1., -0.,  0., -1., -1.,  1.,  0.,  0.,  1.,  0.,
         1., -1.,  1.,  0., -0.,  1., -1., -0., -1., -1.,  0.,  1.,  1.,
         0.,  0., -0., -1.,  1., -1., -1.]], dtype=np.float32), np.array([[ 1., -0.,  0.,  0., -0., -1., -0.,  1.,  1., -1.,  1., -1., -1.,
        -0., -1., -0.,  0.,  1.,  0.,  1.,  1.,  1.,  0., -1.,  1.,  0.,
        -0.,  0., -1.,  0.,  1., -0., -0.,  1.,  0., -0.,  0., -0., -0.,
         0., -0., -0., -1.,  1.,  1.,  0.,  1.,  0., -1.,  1.,  0., -1.,
        -0.,  0., -0.,  0., -1.,  1., -1.],
       [ 1.,  1., -0., -0., -0., -1.,  0.,  0., -0., -0.,  0.,  0.,  1.,
        -1., -0., -1., -0., -1.,  0., -0.,  1., -1.,  1., -0., -0.,  0.,
         1.,  1.,  1.,  1., -1., -1.,  0.,  1., -0., -1., -1.,  0.,  1.,
         0., -0., -0., -1., -0., -0., -0.,  1.,  0.,  0., -0., -0.,  0.,
        -1.,  0.,  0.,  0., -0., -0.,  1.],
       [ 1.,  0.,  0.,  0.,  0., -1.,  0.,  0.,  0., -0., -0., -0., -1.,
         0.,  0., -1.,  1., -1., -1.,  0.,  1., -1., -0.,  0., -0., -0.,
        -0.,  1.,  1.,  0.,  1.,  1., -0., -1.,  1.,  1.,  0., -0., -1.,
        -0., -0., -0., -1.,  0., -0., -1., -1.,  1.,  0.,  0.,  0., -0.,
        -1.,  1., -1., -1., -0.,  0.,  1.],
       [-1.,  0.,  0.,  1., -0., -1., -1.,  0., -0., -1., -0.,  0., -1.,
         0.,  0., -0.,  0., -1.,  1., -0.,  1.,  1.,  0., -1., -0., -1.,
         0.,  0., -1.,  0.,  1., -0., -0., -1.,  1.,  0., -0.,  0.,  0.,
         0.,  0.,  1.,  1., -1.,  0., -1., -1.,  1., -1., -1.,  0., -1.,
         0., -1.,  0., -1.,  1., -1.,  1.],
       [ 1.,  0., -0.,  0., -0.,  1.,  0., -0., -0., -1.,  0.,  0., -1.,
         1., -0.,  0.,  0.,  1.,  0., -0., -1., -1.,  1.,  1.,  0., -0.,
        -1., -0.,  1., -1.,  1.,  0., -0., -1.,  0.,  0.,  0., -0.,  0.,
         1., -1., -0., -1., -1.,  0.,  0., -1., -0.,  1.,  1., -1., -1.,
         0., -0., -0., -0., -1., -1., -1.],
       [-1., -0., -0.,  0., -0., -1.,  0., -1., -1.,  1.,  1.,  1., -1.,
         0., -1., -0., -0.,  1., -0., -1., -1., -1.,  0., -1.,  1.,  0.,
         0., -0., -1.,  0., -1.,  0., -0.,  1., -0., -0., -0., -0., -0.,
        -0., -0., -0., -1., -1.,  1.,  0., -1.,  0.,  1.,  1.,  0., -1.,
        -0., -0.,  0.,  0.,  1.,  1.,  1.],
       [ 1., -1.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,  0.,  1.,
         1.,  0., -1., -0.,  1.,  0.,  0., -1., -1.,  1., -0.,  0., -0.,
        -1.,  1.,  1.,  1., -1.,  1.,  0.,  1.,  0.,  1., -1.,  0., -1.,
        -0.,  0., -0., -1., -0., -0.,  0.,  1., -0.,  0.,  0., -0., -0.,
        -1.,  0.,  0.,  0.,  0.,  0., -1.],
       [ 1., -0.,  0.,  0., -0.,  1., -0.,  0.,  0., -0., -0.,  0., -1.,
        -0.,  0., -1.,  1.,  1., -1.,  0., -1., -1., -0.,  0., -0., -0.,
         0.,  1.,  1.,  0.,  1., -1., -0., -1., -1., -1.,  0.,  0.,  1.,
         0., -0.,  0., -1.,  0., -0., -1., -1.,  1., -0., -0., -0.,  0.,
        -1.,  1.,  1.,  1.,  0.,  0., -1.],
       [ 1., -0., -0., -1.,  0.,  1., -1.,  0.,  0., -1.,  0.,  0., -1.,
        -0.,  0.,  0., -0.,  1., -1.,  0., -1., -1., -0.,  1.,  0.,  1.,
         0., -0.,  1.,  0.,  1.,  0.,  0., -1., -1.,  0., -0., -0., -0.,
        -0.,  0.,  1., -1., -1., -0., -1., -1.,  1.,  1.,  1.,  0., -1.,
         0.,  1., -0.,  1., -1., -1., -1.],
       [ 1., -0., -0.,  0., -0.,  1., -0., -0.,  0.,  1.,  0.,  0.,  1.,
         1., -0.,  0.,  0.,  1.,  0., -0., -1., -1.,  1.,  1.,  0.,  0.,
        -1.,  0.,  1.,  1., -1.,  0., -0.,  1.,  0., -0.,  0.,  0., -0.,
         1.,  1.,  0., -1.,  1., -0., -0.,  1.,  0.,  1.,  1.,  1.,  1.,
         0., -0.,  0.,  0., -1.,  1., -1.],
       [-1.,  0., -0.,  0., -0., -1.,  0., -1.,  1., -1.,  1.,  1., -1.,
         0.,  1.,  0.,  0., -1.,  0.,  1.,  1.,  1.,  0., -1.,  1.,  0.,
        -0., -0., -1.,  0.,  1., -0.,  0., -1., -0., -0., -0.,  0.,  0.,
         0.,  0., -0.,  1., -1., -1.,  0., -1., -0., -1., -1.,  0., -1.,
         0.,  0., -0.,  0.,  1., -1.,  1.],
       [ 1.,  1., -0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,  0.,  1.,
         1., -0., -1., -0., -1., -0., -0.,  1.,  1., -1., -0.,  0.,  0.,
         1., -1.,  1.,  1.,  1.,  1.,  2., -1.,  0., -1., -1., -1.,  1.,
         0., -0., -0.,  1., -0.,  0., -0.,  1., -0.,  0.,  0.,  0., -0.,
        -1., -0.,  0.,  0.,  0.,  0., -1.],
       [-1.,  0.,  1., -0.,  1.,  1.,  0., -0.,  0., -0.,  0., -0.,  1.,
         0.,  0.,  1., -1., -1.,  1.,  0., -1., -1., -0., -0., -0., -0.,
         0., -1.,  1.,  0., -1., -1.,  0., -1.,  1., -1.,  0., -0., -1.,
         0.,  0., -0.,  1.,  0.,  0.,  1., -1.,  1., -0.,  0.,  0., -0.,
        -1.,  1.,  1.,  1.,  0., -0.,  1.],
       [ 1.,  0.,  0., -1.,  0., -1.,  1., -0., -0., -1., -0.,  0., -1.,
         0., -0.,  0., -0.,  1., -1., -0.,  1.,  1., -0., -1., -0., -1.,
         0., -0., -1.,  0.,  1., -0.,  0.,  1., -1., -0., -0., -0., -0.,
         0.,  0.,  1., -1.,  1., -0., -1.,  1., -1., -1.,  1., -0., -1.,
        -0., -1., -0., -1., -1.,  1., -1.],
       [ 1.,  0., -0.,  0.,  0.,  1., -0., -0., -0., -1.,  0.,  0.,  1.,
         1., -0.,  0.,  0., -1.,  0., -0.,  1.,  1., -1.,  1.,  0.,  0.,
         1.,  0.,  1.,  1.,  1., -0.,  0., -1.,  0.,  0.,  0.,  0.,  0.,
        -1., -1., -0.,  1.,  1.,  0.,  0.,  1.,  0., -1., -1.,  1.,  1.,
         0., -0.,  0., -0., -1., -1., -1.],
       [ 1.,  0.,  0., -0., -0., -1.,  0.,  1., -1.,  1.,  1., -1., -1.,
         0.,  1., -0.,  0., -1., -0., -1., -1., -1.,  0., -1.,  1.,  0.,
         0., -0., -1., -0., -1., -0.,  0., -1.,  0., -0.,  0.,  0.,  0.,
         0.,  0., -0.,  1.,  1., -1.,  0.,  1., -0.,  1., -1., -0., -1.,
         0., -0., -0.,  0., -1., -1., -1.],
       [-1.,  1.,  0., -0.,  0.,  1.,  0., -0.,  0., -0., -0.,  0., -1.,
         1., -0.,  1., -0., -1.,  0., -0.,  1., -1.,  1.,  0.,  0., -0.,
         1.,  1., -1., -1., -1.,  1.,  2.,  1., -0., -1.,  1.,  1.,  1.,
         0., -0., -0., -1., -0., -0.,  0., -1.,  0.,  0., -0.,  0.,  0.,
         1.,  0.,  0.,  0.,  0.,  0., -1.],
       [-1., -0.,  1.,  0., -1., -1.,  0.,  0.,  0.,  0., -0.,  0.,  1.,
         0.,  0.,  1., -1.,  1.,  1., -0.,  1., -1.,  0., -0., -0.,  0.,
        -0., -1.,  1.,  0., -1.,  1.,  0., -1., -1.,  1., -0., -0.,  1.,
         0., -0.,  0.,  1., -0.,  0.,  1., -1.,  1., -0.,  0.,  0., -0.,
        -1.,  1., -1., -1., -0.,  0., -1.],
       [ 1., -0.,  0., -1.,  0., -1., -1.,  0.,  0.,  1.,  0.,  0.,  1.,
        -0.,  0., -0.,  0.,  1., -1.,  0.,  1.,  1.,  0., -1.,  0., -1.,
         0., -0., -1.,  0., -1., -0.,  0., -1., -1., -0.,  0., -0., -0.,
        -0., -0., -1., -1., -1.,  0.,  1., -1.,  1., -1.,  1.,  0.,  1.,
         0., -1., -0., -1., -1., -1., -1.],
       [ 1., -0., -0.,  0., -0.,  1.,  0., -0.,  0.,  1.,  0.,  0., -1.,
         1.,  0., -0.,  0., -1.,  0.,  0.,  1.,  1., -1.,  1.,  0., -0.,
         1.,  0.,  1., -1., -1., -0., -0.,  1.,  0.,  0., -0.,  0.,  0.,
        -1.,  1.,  0.,  1., -1., -0., -0., -1.,  0., -1., -1., -1., -1.,
        -0., -0.,  0.,  0., -1.,  1., -1.]], dtype=np.float32), np.array([[ 0.,  0.,  0., -0.,  0.,  1., -1.,  0., -1.,  1., -1.,  1.,  0.,
         1., -1.,  0., -0.,  1.,  0.,  0.,  1., -0.,  0., -0.,  0.,  0.,
         1.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  1.,  0., -0., -0., -0.,
         0.,  1.,  1.,  0.,  1., -0.,  0.,  0., -0.,  0.,  0., -1.,  1.,
        -0., -0.,  0.,  1.,  0., -1.,  1.],
       [ 0., -0.,  0.,  0.,  0.,  1.,  0., -1., -0., -0.,  0.,  1.,  0.,
         1.,  0., -0.,  0.,  1., -0., -1.,  1.,  0., -0.,  0.,  0.,  0.,
         1., -0.,  0., -0.,  0.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  0.,
        -0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,
        -0., -0., -0.,  1.,  0.,  0.,  1.],
       [ 0., -0.,  0.,  0., -0., -1.,  1.,  1.,  0., -1.,  0., -1.,  0.,
        -1., -0.,  0.,  0., -1.,  0.,  1., -1., -0., -0.,  0.,  1., -0.,
        -1.,  0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,  0.,  0.,  0.,
        -0., -1., -1.,  0., -1.,  1., -0.,  0.,  0., -0.,  0.,  1., -1.,
        -0.,  0., -0., -1., -0.,  1., -1.],
       [-0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0., -0., -1., -0.,  0.,
         1., -1.,  0.,  0.,  1.,  0.,  0.,  1., -0., -0.,  0.,  0., -0.,
         1., -0.,  0., -0.,  0., -0., -0.,  0.,  1., -0., -0., -0., -0.,
        -0.,  0., -0.,  0., -0., -0., -0., -0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0., -0.,  1.,  0.,  0.,  1.],
       [ 1.,  1.,  0., -1.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,
        -0., -0., -0., -0.,  1., -1.,  0., -0., -0., -1., -0.,  0.,  0.,
         1., -0., -0., -0., -0., -1.,  1., -0.,  1.,  1., -0., -0., -0.,
         1.,  0.,  0.,  1., -0., -0.,  0.,  0., -0.,  0., -1., -0.,  0.,
        -0., -0., -1., -0.,  1.,  0.,  1.],
       [ 0.,  0.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0., -0., -0., -0.,
        -0., -0.,  1.,  1., -0., -0., -0.,  1.,  1., -1., -1., -0.,  1.,
         1., -1., -1.,  0., -0., -0., -0.,  0., -0.,  0., -1., -0.,  0.,
         1.,  0.,  0.,  0., -0., -0.,  0., -0.,  0.,  1.,  0.,  0., -0.,
         0.,  1., -0.,  1.,  0., -0., -0.],
       [-0., -1., -0.,  0.,  0., -1., -0., -0.,  0.,  0.,  0.,  0.,  0.,
         0., -0.,  0.,  0.,  0.,  0.,  0., -1., -1.,  1.,  1.,  0., -1.,
        -1., -0.,  1., -0.,  0., -1., -0., -0.,  0.,  1., -0., -0.,  0.,
        -1.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -1.,  0., -0., -0.,
         0., -1., -1., -1.,  0., -0., -0.],
       [ 1., -0.,  0., -1., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0.,  1.,  1.,  1., -1.,  0., -0., -0., -1., -0.,  0.,  0.,
         1., -1.,  0.,  0., -0.,  0., -0., -0.,  1., -0.,  1.,  2., -0.,
         1.,  0., -0.,  1.,  0., -0., -0.,  0., -0., -0., -1.,  0.,  0.,
         0., -0.,  0.,  0.,  1.,  0.,  1.],
       [-0., -1.,  0., -1.,  2., -0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
         0.,  0., -0., -0.,  1., -1.,  0.,  1.,  1., -1., -0., -0., -0.,
         1., -0.,  0.,  0.,  0., -0.,  0.,  0.,  1.,  1., -0., -0., -1.,
         1.,  0., -0.,  1., -0.,  0., -0.,  0.,  0.,  1., -1.,  0.,  0.,
        -0., -0.,  1.,  0., -0., -0., -0.],
       [ 0.,  0., -0.,  0.,  0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,
        -0., -0.,  1.,  1.,  1.,  0., -0.,  1.,  1., -1.,  0., -0.,  1.,
         1.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0., -1., -0.,  0.,
         1., -0.,  0.,  1.,  0., -0., -0., -0.,  0.,  1., -1., -0., -0.,
        -1.,  1.,  0.,  1., -0., -0., -0.],
       [-0., -1.,  0.,  0., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,
         0., -0., -0., -0.,  1.,  0., -0.,  1.,  1., -1.,  0.,  0.,  1.,
         1., -0.,  0.,  0., -0.,  0.,  0.,  0., -0.,  1.,  0.,  0., -1.,
         1.,  0.,  0.,  1., -0., -0.,  0., -0., -0.,  1., -1.,  0., -0.,
         0.,  1., -1.,  1.,  0.,  0., -0.],
       [ 0., -0.,  2.,  1.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0., -0.,
         0.,  0.,  1., -1., -1.,  1.,  0., -1., -1.,  1., -0., -0.,  0.,
        -1.,  0., -0.,  0., -0.,  0., -0.,  0., -1.,  0., -1.,  0.,  0.,
        -1.,  0.,  0., -1.,  0.,  0., -0.,  0., -0., -1.,  1.,  0., -0.,
        -1.,  0.,  0.,  0.,  0., -0.,  0.],
       [ 0., -0.,  2., -0., -0., -1., -0.,  1.,  1., -1.,  1., -1.,  1.,
        -1.,  0.,  1., -1., -1.,  1.,  0., -1., -1.,  1.,  0., -0., -0.,
        -1.,  0., -0., -1., -0.,  0., -0.,  0., -1.,  0., -1., -0.,  0.,
         0., -1., -1., -1.,  0.,  1.,  0., -1.,  1.,  0., -0., -0.,  0.,
        -1.,  0.,  0., -1.,  0.,  1., -1.],
       [ 0., -0.,  0., -0., -0., -1., -0.,  1., -0.,  0.,  1., -1.,  1.,
        -1., -0., -1., -1., -1., -0.,  0., -1., -1.,  1., -0.,  0., -1.,
        -1., -0., -0., -1., -0., -0.,  0., -0., -1.,  0.,  1.,  0., -0.,
        -1.,  0., -0., -1., -0., -0., -1., -1., -0., -1.,  1.,  0., -0.,
         1., -1.,  0., -1.,  0., -0., -1.],
       [ 0., -0.,  0.,  0., -0., -1.,  1.,  1.,  1., -1.,  1., -1.,  1.,
        -1.,  0., -1., -1., -1., -0., -0., -1., -1.,  1.,  0.,  0.,  0.,
        -1., -0., -0., -1., -0., -0., -0.,  0., -1.,  0.,  1.,  0.,  0.,
        -0., -1.,  0., -1.,  0.,  1., -1., -1., -0.,  0.,  0., -0.,  0.,
         1., -1., -0., -1., -0.,  1., -1.],
       [ 0., -0., -2., -1.,  0.,  1., -0., -1.,  0., -0., -1.,  1., -1.,
         1., -0., -1.,  1.,  1., -1.,  0.,  1.,  1., -1.,  0., -0., -0.,
         1.,  0., -0.,  1.,  0.,  0.,  0.,  0.,  1., -0.,  1.,  0., -0.,
         1., -0., -0.,  1.,  0.,  0., -0.,  1., -1.,  1., -1.,  0.,  0.,
         1., -0.,  0.,  1., -0.,  0.,  1.],
       [ 1.,  0.,  0.,  0.,  0., -0., -1.,  1.,  0., -0., -0.,  0.,  0.,
         0.,  1.,  1.,  1.,  0., -1.,  0., -0., -0., -1.,  0., -0.,  0.,
        -0., -1.,  0., -1.,  0.,  0.,  0.,  1., -0., -0.,  1.,  2.,  0.,
        -0., -0.,  0.,  1.,  1., -1.,  0., -1.,  1.,  0.,  0., -1., -0.,
        -0.,  0.,  0., -0., -0., -1., -0.],
       [-0., -0., -0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0.,  1.,
         0., -0.,  1.,  1.,  0.,  0.,  1.,  0.,  1., -1., -1.,  0.,  1.,
        -0., -1., -1., -1.,  1.,  0.,  0., -0.,  0.,  0., -1., -0., -0.,
         1.,  0., -0., -0.,  0., -0., -1.,  0., -0.,  1.,  0.,  0., -0.,
         0.,  1., -0., -0.,  0., -0.,  0.],
       [ 0., -0.,  0.,  0.,  0.,  0., -0.,  0., -1.,  1.,  1., -0.,  1.,
         0., -0.,  1.,  1.,  0.,  0.,  1., -0.,  1., -1.,  0., -1., -0.,
         0., -1., -1., -1.,  1., -0.,  0.,  0.,  0.,  0., -1., -0., -0.,
        -0., -0.,  1., -0., -0.,  0., -1.,  0.,  0., -0., -0., -1.,  1.,
         0.,  1., -0., -0.,  0.,  0., -0.],
       [ 1., -0., -0., -1., -0.,  0.,  0.,  1., -0.,  0., -0., -1., -0.,
         0.,  1.,  1.,  1., -0., -1., -0., -0., -0., -1.,  0.,  0., -0.,
        -0., -1.,  0., -1., -0., -0.,  0.,  1., -0.,  0.,  1.,  2., -0.,
         1.,  0.,  0.,  1.,  0.,  0.,  0., -1.,  1.,  0., -1.,  0.,  0.,
         0., -0.,  0.,  0.,  1.,  0., -0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 4
m = 4
p = 5
rank = 59

verify_tensor_support_rank_decomposition(decomposition_445, n, m, p, rank)

## Rank-72 support rank decomposition of <4,4,6> over Z


In [ ]:
#@title Data

decomposition_446 = (np.array([[-0.,  1.,  0.,  0.,  0., -1., -1., -0., -1., -0., -0., -1., -1.,
        -0.,  1.,  1.,  1., -1., -1., -1.,  1., -0.,  0.,  1.,  0., -0.,
        -0.,  1.,  0.,  0., -1., -1., -1., -1., -0.,  1.,  1.,  0.,  0.,
        -0.,  1.,  0.,  0., -0.,  0., -1., -0.,  1.,  0., -0., -0., -0.,
         1.,  1., -0., -1., -0.,  1., -0.,  1., -1., -0.,  0., -0., -0.,
         0., -0.,  0.,  0.,  0., -1.,  1.],
       [-0., -0., -1., -1., -1., -1.,  1., -0., -0.,  1.,  1.,  0.,  0.,
        -1.,  0., -0., -0., -0.,  0.,  0., -0.,  1.,  1.,  0., -1.,  1.,
         1., -0.,  1.,  0.,  0.,  0., -0.,  0., -1., -1., -0., -1.,  0.,
         1.,  1.,  1.,  1.,  0.,  1., -0., -0.,  0.,  1.,  0., -0., -1.,
         0.,  1., -0., -1., -0., -0., -0., -0.,  1., -1.,  0.,  1., -1.,
         1., -1., -1., -1.,  1.,  0., -1.],
       [ 1., -0., -1., -1.,  1.,  1.,  1.,  0.,  0., -1., -0.,  0.,  0.,
        -1.,  0., -0.,  0.,  0.,  0.,  0., -0.,  1.,  1., -0., -1., -1.,
         1., -0.,  1., -0., -0., -0.,  0., -0., -1.,  1.,  0.,  1.,  0.,
         1.,  1.,  1.,  1., -1.,  0., -0.,  0.,  0.,  1., -0.,  0., -1.,
        -0., -1.,  1., -1., -1., -0.,  0.,  0.,  1., -1.,  0.,  1., -1.,
         0., -0., -1., -1.,  1.,  0.,  1.],
       [ 0., -1., -0., -0., -0.,  1., -1.,  1., -1., -0.,  0., -1.,  1.,
        -0., -0.,  0., -1.,  0.,  1., -1., -1., -0., -0., -0.,  0.,  0.,
        -0.,  1.,  0., -1.,  1., -0., -1., -1., -0.,  1., -1., -0., -1.,
        -0., -1., -0., -0.,  0.,  0.,  0.,  1.,  1., -0.,  1.,  1.,  0.,
         0.,  1., -0., -1.,  0., -1., -1.,  1.,  1., -0., -1.,  0.,  0.,
         0.,  0., -0., -0., -0.,  0., -1.],
       [-0., -1.,  0., -0.,  0.,  1., -1., -0.,  1., -0., -0.,  1., -1.,
        -0.,  1.,  1., -1.,  1.,  1., -1., -1.,  0.,  0., -1., -0., -0.,
        -0., -1., -0., -0., -1.,  1.,  1., -1.,  0., -1.,  1., -0.,  0.,
         0.,  1.,  0.,  0., -0., -0., -1., -0.,  1.,  0.,  0.,  0., -0.,
        -1., -1., -0., -1., -0.,  1.,  0.,  1., -1.,  0.,  0.,  0., -0.,
         0., -0., -0.,  0.,  0., -1., -1.],
       [-0.,  0.,  1.,  1., -1.,  1.,  1.,  0., -0., -1.,  1.,  0., -0.,
         1.,  0., -0.,  0., -0.,  0.,  0., -0., -1.,  1., -0., -1., -1.,
         1., -0.,  1.,  0., -0., -0.,  0.,  0.,  1., -1., -0., -1., -0.,
         1., -1.,  1., -1., -0., -1., -0.,  0.,  0.,  1.,  0., -0., -1.,
         0.,  1., -0., -1.,  0.,  0., -0., -0., -1.,  1.,  0.,  1.,  1.,
        -1., -1.,  1.,  1.,  1.,  0.,  1.],
       [ 1., -0.,  1.,  1., -1.,  1., -1.,  0., -0., -1., -0.,  0., -0.,
        -1., -0.,  0., -0., -0., -0.,  0.,  0., -1.,  1., -0.,  1., -1.,
         1., -0.,  1.,  0.,  0., -0., -0., -0.,  1., -1., -0., -1., -0.,
         1.,  1.,  1., -1.,  1.,  0.,  0., -0., -0., -1., -0.,  0., -1.,
         0.,  1., -1.,  1., -1., -0.,  0.,  0.,  1.,  1.,  0.,  1., -1.,
         0.,  0.,  1.,  1.,  1., -0.,  1.],
       [ 0., -1., -0., -0., -0., -1., -1., -1.,  1., -0., -0., -1.,  1.,
         0.,  0.,  0.,  1., -0., -1., -1., -1., -0.,  0.,  0., -0.,  0.,
         0.,  1., -0.,  1.,  1.,  0.,  1.,  1., -0., -1.,  1., -0.,  1.,
        -0., -1., -0., -0., -0.,  0.,  0.,  1., -1.,  0.,  1.,  1., -0.,
        -0., -1., -0., -1.,  0.,  1., -1.,  1.,  1., -0.,  1., -0., -0.,
         0., -0., -0., -0.,  0., -0.,  1.],
       [-0.,  1.,  0., -0.,  0., -1.,  1., -0., -1.,  0., -0., -1.,  1.,
         0., -1.,  1., -1., -1., -1.,  1., -1.,  0.,  0., -1., -0.,  0.,
        -0., -1.,  0., -0., -1., -1.,  1.,  1., -0.,  1., -1.,  0., -0.,
         0., -1.,  0.,  0.,  0.,  0.,  1.,  0.,  1.,  0., -0., -0.,  0.,
        -1., -1., -0., -1., -0.,  1., -0.,  1., -1., -0.,  0., -0.,  0.,
         0.,  0.,  0.,  0., -0., -1., -1.],
       [ 0., -0., -1., -1., -1., -1., -1.,  0.,  0., -1.,  1., -0.,  0.,
        -1.,  0.,  0., -0., -0., -0., -0., -0., -1.,  1.,  0., -1.,  1.,
        -1., -0.,  1., -0.,  0., -0.,  0.,  0.,  1., -1.,  0.,  1., -0.,
        -1., -1.,  1.,  1., -0., -1., -0.,  0.,  0., -1., -0., -0., -1.,
        -0., -1., -0., -1.,  0.,  0.,  0., -0.,  1.,  1.,  0., -1.,  1.,
         1.,  1.,  1., -1., -1.,  0.,  1.],
       [ 1., -0., -1., -1., -1.,  1., -1., -0., -0., -1., -0., -0., -0.,
        -1., -0., -0.,  0., -0.,  0., -0., -0.,  1.,  1.,  0., -1.,  1.,
        -1.,  0.,  1., -0.,  0., -0., -0.,  0.,  1., -1.,  0.,  1.,  0.,
        -1.,  1.,  1., -1.,  1., -0.,  0., -0.,  0., -1.,  0., -0.,  1.,
         0., -1.,  1., -1.,  1.,  0., -0.,  0., -1.,  1.,  0., -1.,  1.,
         0., -0.,  1., -1.,  1.,  0., -1.],
       [-0.,  1., -0.,  0.,  0., -1., -1.,  1., -1.,  0.,  0., -1.,  1.,
        -0., -0.,  0., -1.,  0., -1.,  1., -1.,  0.,  0.,  0.,  0., -0.,
        -0., -1.,  0.,  1., -1.,  0.,  1.,  1.,  0.,  1., -1.,  0.,  1.,
         0.,  1.,  0.,  0., -0., -0., -0., -1.,  1., -0.,  1., -1., -0.,
         0., -1., -0.,  1.,  0.,  1., -1.,  1.,  1.,  0., -1., -0.,  0.,
        -0., -0., -0.,  0.,  0.,  0., -1.],
       [ 0., -1.,  0.,  0., -0.,  1.,  1.,  0.,  1.,  0., -0.,  1.,  1.,
         0., -1.,  1.,  1.,  1.,  1.,  1.,  1.,  0., -0.,  1.,  0.,  0.,
         0.,  1., -0.,  0., -1.,  1., -1.,  1.,  0., -1., -1.,  0., -0.,
         0., -1., -0.,  0.,  0.,  0.,  1.,  0.,  1., -0.,  0., -0.,  0.,
         1.,  1., -0., -1., -0.,  1., -0.,  1., -1.,  0.,  0., -0.,  0.,
         0.,  0., -0.,  0.,  0., -1.,  1.],
       [-0., -0.,  1.,  1., -1.,  1., -1.,  0., -0.,  1.,  1., -0.,  0.,
         1., -0., -0.,  0., -0.,  0.,  0.,  0.,  1., -1.,  0., -1., -1.,
        -1.,  0.,  1., -0., -0., -0.,  0., -0., -1., -1.,  0.,  1.,  0.,
         1.,  1.,  1., -1., -0.,  1.,  0., -0.,  0., -1., -0., -0., -1.,
         0., -1., -0., -1., -0.,  0.,  0., -0., -1., -1.,  0., -1., -1.,
        -1.,  1.,  1., -1., -1.,  0., -1.],
       [ 1., -0., -1.,  1.,  1.,  1.,  1., -0.,  0., -1.,  0., -0., -0.,
        -1.,  0., -0.,  0., -0., -0., -0., -0., -1.,  1., -0.,  1.,  1.,
        -1.,  0., -1.,  0., -0., -0.,  0.,  0., -1.,  1., -0., -1., -0.,
        -1.,  1.,  1.,  1., -1., -0., -0., -0., -0.,  1.,  0.,  0.,  1.,
         0.,  1., -1.,  1.,  1.,  0., -0.,  0., -1.,  1., -0.,  1.,  1.,
         0.,  0., -1.,  1.,  1., -0., -1.],
       [ 0., -1., -0., -0., -0., -1.,  1.,  1., -1., -0., -0.,  1., -1.,
        -0.,  0., -0., -1., -0., -1., -1.,  1., -0., -0.,  0.,  0.,  0.,
        -0.,  1.,  0.,  1.,  1.,  0.,  1.,  1.,  0.,  1., -1., -0.,  1.,
         0., -1., -0., -0.,  0.,  0.,  0.,  1.,  1., -0., -1.,  1., -0.,
        -0., -1., -0., -1., -0.,  1.,  1., -1., -1.,  0., -1.,  0., -0.,
        -0., -0., -0.,  0.,  0., -0., -1.]], dtype=np.float32), np.array([[-0., -1., -0., -0.,  0.,  0.,  0., -0., -1., -0.,  0.,  1.,  1.,
         0.,  0.,  0.,  1., -1.,  1., -1., -1., -0.,  0.,  0., -0.,  0.,
         0., -1., -0., -0.,  1.,  0.,  1.,  1.,  0., -0., -1.,  0., -1.,
         0.,  0.,  0.,  0.,  0., -0.,  1.,  1.,  1., -0., -1., -0., -0.,
        -1., -0., -0., -0., -0.,  1., -0., -1.,  0.,  0.,  1.,  0.,  0.,
         0.,  0.,  0., -0., -0., -1.,  0.],
       [ 0.,  1., -0.,  0., -0., -0., -0.,  0., -1., -0., -0., -1., -1.,
        -0.,  0., -0.,  1.,  1.,  1.,  1.,  1.,  0.,  0., -0.,  0., -0.,
        -0.,  1.,  0.,  0., -1., -0.,  1.,  1., -0.,  0., -1., -0., -1.,
         0.,  0.,  0.,  0.,  0., -0.,  1., -1.,  1., -0.,  1.,  0.,  0.,
         1., -0., -0., -0.,  0.,  1.,  0.,  1., -0.,  0.,  1., -0., -0.,
        -0., -0., -0., -0.,  0., -1., -0.],
       [ 1., -0., -0.,  0., -0.,  1.,  1., -1., -0., -0.,  1.,  0.,  0.,
        -0.,  1.,  1., -0.,  0., -0.,  0.,  0., -0.,  0., -1.,  0.,  0.,
        -0.,  0., -0.,  1.,  0.,  1.,  0., -0., -0., -1., -0., -0.,  0.,
        -0.,  1., -0., -0.,  1.,  1.,  0.,  0., -0., -0., -0., -1., -0.,
         0.,  1.,  1., -1.,  1., -0.,  1., -0.,  1.,  0.,  0.,  0.,  0.,
        -1., -1., -0.,  0.,  0.,  0., -1.],
       [-0.,  1.,  0.,  0.,  0., -0.,  0.,  0., -1., -0.,  0.,  1.,  1.,
        -0., -0., -0.,  1., -1., -1.,  1., -1., -0.,  0., -0.,  0., -0.,
        -0.,  1.,  0., -0., -1.,  0., -1., -1., -0.,  0., -1., -0.,  1.,
         0.,  0., -0., -0.,  0.,  0., -1., -1.,  1.,  0., -1.,  0., -0.,
         1.,  0.,  0., -0., -0., -1., -0., -1.,  0., -0.,  1.,  0.,  0.,
         0.,  0.,  0., -0.,  0., -1.,  0.],
       [ 1.,  0.,  0., -0., -0.,  1.,  1., -1., -0., -0.,  1., -0.,  0.,
        -0., -1., -1.,  0., -0., -0., -0., -0.,  0.,  0.,  1., -0., -0.,
        -0.,  0., -0.,  1., -0., -1.,  0.,  0.,  0., -1., -0.,  0., -0.,
         0.,  1.,  0.,  0.,  1.,  1., -0.,  0., -0.,  0., -0., -1.,  0.,
        -0.,  1.,  1., -1.,  1.,  0.,  1.,  0.,  1., -0.,  0.,  0.,  0.,
        -1., -1., -0.,  0.,  0., -0., -1.],
       [-0., -1., -0.,  0., -0.,  0.,  0.,  0., -1.,  0.,  0., -1., -1.,
         0., -0.,  0.,  1., -1., -1., -1.,  1.,  0., -0., -0., -0., -0.,
         0., -1., -0., -0.,  1.,  0., -1., -1.,  0., -0., -1., -0.,  1.,
         0., -0.,  0., -0., -0.,  0.,  1.,  1.,  1., -0.,  1., -0.,  0.,
         1.,  0.,  0., -0., -0., -1.,  0.,  1., -0., -0.,  1., -0.,  0.,
         0.,  0., -0.,  0., -0.,  1., -0.],
       [-0., -0.,  0.,  1.,  1., -0.,  0.,  0.,  0., -1., -0.,  0.,  0.,
        -1.,  0., -0.,  0., -0., -0.,  0., -0., -1.,  1.,  0.,  1.,  1.,
        -1., -0., -0., -0., -0., -0., -0.,  0.,  1., -0.,  0., -1.,  0.,
         1., -0., -1., -1.,  0.,  0.,  0.,  0., -0.,  1., -0.,  0.,  1.,
         0., -0., -0., -0., -0., -0., -0., -0., -0.,  0.,  0.,  0.,  1.,
        -0.,  0., -1., -1., -1., -0., -0.],
       [-0., -0., -0., -1.,  1., -0.,  0.,  0.,  0., -1.,  0.,  0., -0.,
        -1.,  0.,  0., -0.,  0.,  0., -0., -0.,  1.,  1., -0., -1., -1.,
         1., -0.,  0., -0.,  0.,  0.,  0., -0.,  1., -0.,  0.,  1., -0.,
        -1.,  0., -1., -1.,  0., -0.,  0., -0.,  0.,  1., -0., -0., -1.,
         0.,  0., -0.,  0.,  0., -0., -0., -0., -0., -0.,  0.,  0., -1.,
        -0.,  0., -1.,  1., -1., -0., -0.],
       [-1.,  0., -0., -0.,  0., -1.,  1., -1.,  0.,  0., -1.,  0., -0.,
         0.,  1., -1., -0., -0.,  0., -0.,  0., -0.,  0., -1.,  0.,  0.,
         0.,  0., -0., -1., -0., -1., -0.,  0.,  0., -1.,  0., -0.,  0.,
         0., -1.,  0.,  0.,  1.,  1.,  0., -0.,  0.,  0., -0.,  1., -0.,
         0., -1., -1.,  1.,  1., -0.,  1.,  0.,  1.,  0.,  0., -0., -0.,
         1., -1.,  0.,  0., -0., -0., -1.],
       [ 0.,  0.,  0.,  1.,  1., -0.,  0.,  0.,  0.,  1., -0.,  0.,  0.,
         1.,  0.,  0.,  0.,  0., -0.,  0.,  0., -1., -1.,  0.,  1., -1.,
         1., -0., -0.,  0., -0.,  0.,  0., -0.,  1., -0., -0., -1., -0.,
        -1., -0.,  1., -1., -0.,  0.,  0., -0.,  0.,  1.,  0., -0., -1.,
         0., -0.,  0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -1.,
        -0.,  0., -1., -1.,  1., -0.,  0.],
       [-1., -0., -0.,  0., -0., -1.,  1., -1., -0.,  0.,  1., -0., -0.,
        -0.,  1., -1., -0., -0., -0.,  0.,  0., -0., -0., -1.,  0., -0.,
         0.,  0., -0., -1., -0., -1.,  0.,  0.,  0., -1., -0.,  0.,  0.,
         0., -1.,  0.,  0.,  1., -1.,  0., -0.,  0.,  0., -0.,  1.,  0.,
         0., -1., -1.,  1.,  1., -0.,  1.,  0.,  1.,  0.,  0., -0.,  0.,
        -1.,  1.,  0., -0.,  0., -0., -1.],
       [-0.,  0., -0.,  1., -1., -0., -0.,  0.,  0., -1.,  0.,  0., -0.,
        -1.,  0., -0.,  0., -0., -0.,  0.,  0., -1.,  1.,  0.,  1., -1.,
         1.,  0., -0.,  0.,  0., -0., -0.,  0., -1., -0., -0., -1.,  0.,
        -1.,  0., -1.,  1.,  0., -0., -0.,  0., -0., -1.,  0., -0., -1.,
         0., -0., -0., -0., -0., -0., -0.,  0., -0., -0., -0., -0., -1.,
        -0.,  0.,  1., -1., -1., -0., -0.],
       [-0.,  0.,  1., -1., -1.,  0.,  0.,  0., -0.,  1., -0.,  0., -0.,
        -1.,  0., -0., -0., -0., -0.,  0., -0.,  1.,  0.,  0.,  1., -1.,
         1.,  0.,  1.,  0.,  0.,  0.,  0., -0., -1., -0., -0.,  1., -0.,
         0., -0.,  1.,  1., -0., -0.,  0., -0.,  0.,  1., -0.,  0., -1.,
         0.,  0.,  0., -0.,  0., -0., -0., -0., -0., -1.,  0., -1.,  1.,
        -0.,  0., -0., -0.,  1.,  0., -0.],
       [ 0.,  0., -1.,  1., -1., -0., -0.,  0.,  0.,  1.,  0., -0.,  0.,
         1., -0.,  0., -0.,  0.,  0., -0., -0.,  1.,  0., -0.,  1.,  1.,
        -1., -0.,  1., -0., -0., -0., -0., -0., -1.,  0., -0., -1., -0.,
        -0., -0.,  1., -1.,  0., -0., -0., -0., -0., -1., -0., -0., -1.,
         0.,  0.,  0., -0., -0.,  0., -0., -0.,  0., -1., -0.,  1.,  1.,
        -0., -0.,  0.,  0., -1.,  0., -0.],
       [-1., -0.,  0.,  0., -0., -1., -1., -1.,  0.,  0.,  1.,  0.,  0.,
        -0., -1.,  1., -0., -0.,  0., -0., -0.,  0.,  0., -1.,  0.,  0.,
        -0.,  0.,  0., -1.,  0., -1.,  0.,  0.,  0., -1.,  0., -0.,  0.,
        -0.,  1., -0., -0., -1.,  1.,  0., -0.,  0., -0.,  0., -1., -0.,
         0., -1.,  1., -1.,  1., -0., -1., -0., -1., -0.,  0., -0.,  0.,
         1.,  1.,  0.,  0., -0., -0., -1.],
       [ 0.,  0., -1.,  1., -1., -0.,  0., -0.,  0., -1.,  0.,  0.,  0.,
         1., -0., -0., -0., -0.,  0.,  0., -0., -1.,  0., -0.,  1.,  1.,
         1., -0.,  1.,  0.,  0., -0.,  0.,  0.,  1., -0., -0.,  1.,  0.,
         0.,  0.,  1., -1., -0., -0., -0., -0., -0.,  1.,  0., -0., -1.,
         0., -0., -0., -0.,  0., -0.,  0.,  0., -0.,  1., -0., -1., -1.,
         0.,  0.,  0.,  0.,  1., -0., -0.],
       [-1.,  0., -0.,  0., -0.,  1.,  1.,  1.,  0.,  0., -1., -0.,  0.,
        -0.,  1., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,
         0., -0., -0.,  1., -0.,  1., -0., -0., -0.,  1.,  0.,  0., -0.,
        -0., -1.,  0., -0., -1., -1.,  0., -0., -0.,  0., -0.,  1., -0.,
        -0.,  1.,  1.,  1.,  1., -0.,  1., -0.,  1.,  0.,  0.,  0.,  0.,
        -1., -1.,  0.,  0.,  0., -0.,  1.],
       [ 0.,  0., -1.,  1.,  1.,  0., -0., -0., -0.,  1.,  0.,  0., -0.,
         1., -0., -0., -0., -0., -0., -0., -0.,  1.,  0., -0., -1.,  1.,
         1.,  0., -1.,  0., -0., -0., -0., -0., -1.,  0., -0.,  1., -0.,
        -0., -0., -1., -1.,  0., -0., -0.,  0., -0.,  1.,  0.,  0.,  1.,
        -0., -0., -0., -0.,  0., -0.,  0.,  0.,  0., -1., -0., -1.,  1.,
         0.,  0.,  0., -0.,  1.,  0.,  0.],
       [ 0., -1.,  0.,  0., -0., -0.,  0.,  0., -1., -0.,  0.,  1., -1.,
         0.,  0., -0., -1., -1., -1., -1., -1.,  0.,  0.,  0., -0., -0.,
        -0., -1.,  0.,  0., -1.,  0.,  1.,  1., -0., -0., -1.,  0., -1.,
         0., -0.,  0., -0.,  0.,  0.,  1.,  1.,  1., -0., -1., -0., -0.,
        -1., -0.,  0.,  0.,  0.,  1., -0., -1., -0.,  0.,  1., -0.,  0.,
         0.,  0., -0., -0.,  0., -1., -0.],
       [ 0.,  1., -0.,  0., -0.,  0., -0.,  0.,  1., -0., -0., -1., -1.,
        -0., -0., -0.,  1.,  1.,  1., -1.,  1.,  0., -0., -0.,  0.,  0.,
        -0.,  1.,  0.,  0., -1., -0., -1.,  1., -0.,  0., -1., -0., -1.,
        -0.,  0.,  0., -0., -0., -0.,  1., -1.,  1., -0.,  1.,  0., -0.,
         1.,  0., -0., -0.,  0.,  1.,  0., -1., -0.,  0.,  1., -0.,  0.,
        -0.,  0.,  0., -0.,  0., -1., -0.],
       [-1.,  0., -0., -0., -0., -1.,  1.,  1., -0.,  0.,  1.,  0.,  0.,
        -0.,  1.,  1.,  0., -0., -0., -0.,  0., -0., -0.,  1., -0., -0.,
         0., -0., -0., -1., -0., -1., -0.,  0., -0., -1.,  0.,  0.,  0.,
         0., -1.,  0., -0.,  1., -1.,  0., -0., -0., -0., -0., -1., -0.,
        -0.,  1.,  1., -1., -1., -0.,  1., -0., -1., -0., -0.,  0., -0.,
         1., -1., -0.,  0., -0.,  0.,  1.],
       [ 0., -1., -0., -0.,  0.,  0.,  0.,  0., -1.,  0., -0.,  1.,  1.,
        -0.,  0.,  0.,  1., -1., -1.,  1.,  1., -0., -0., -0., -0., -0.,
         0.,  1.,  0., -0., -1.,  0., -1., -1.,  0., -0.,  1., -0.,  1.,
        -0.,  0.,  0., -0.,  0.,  0., -1., -1.,  1.,  0., -1.,  0.,  0.,
         1., -0.,  0., -0., -0.,  1., -0., -1.,  0.,  0.,  1., -0., -0.,
         0., -0.,  0., -0.,  0., -1.,  0.],
       [ 1.,  0., -0.,  0., -0.,  1., -1.,  1.,  0., -0., -1., -0.,  0.,
        -0., -1., -1., -0.,  0.,  0., -0.,  0.,  0.,  0., -1., -0.,  0.,
         0., -0.,  0., -1.,  0.,  1., -0., -0.,  0.,  1.,  0., -0.,  0.,
         0.,  1.,  0.,  0., -1.,  1., -0.,  0., -0.,  0., -0., -1.,  0.,
         0., -1., -1.,  1.,  1.,  0.,  1., -0.,  1., -0., -0., -0., -0.,
        -1.,  1.,  0., -0.,  0., -0., -1.],
       [-0.,  1.,  0., -0.,  0., -0.,  0., -0.,  1.,  0., -0., -1.,  1.,
        -0., -0.,  0., -1.,  1.,  1.,  1., -1.,  0.,  0.,  0.,  0.,  0.,
         0., -1., -0.,  0., -1., -0.,  1., -1.,  0.,  0.,  1., -0., -1.,
        -0.,  0., -0., -0., -0., -0., -1., -1.,  1.,  0., -1.,  0., -0.,
        -1., -0., -0., -0., -0.,  1., -0., -1.,  0.,  0., -1.,  0.,  0.,
         0.,  0.,  0., -0.,  0., -1.,  0.]], dtype=np.float32), np.array([[ 0.,  0., -1., -0., -1., -0., -0., -0.,  0., -1., -0., -0., -1.,
         0., -0.,  0.,  1.,  1., -1.,  0., -0.,  0.,  1., -0., -0.,  1.,
        -0., -0.,  1.,  0., -1., -0., -0., -0.,  0.,  0., -0.,  1.,  1.,
         1.,  0., -0.,  0.,  0.,  0., -1.,  1.,  0.,  0., -1.,  0.,  0.,
        -1., -0., -0.,  0.,  0.,  0.,  0., -0., -0.,  1., -1., -1., -0.,
        -0., -0.,  1.,  1.,  0.,  1.,  0.],
       [-0., -0., -1.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,
        -1.,  0., -0.,  1.,  1., -1., -0., -0.,  0.,  1., -0., -1., -0.,
        -0.,  0., -1., -0.,  1., -0., -0.,  0.,  0., -0.,  0.,  0.,  1.,
         1., -0., -0.,  0., -0.,  0.,  1., -1.,  0.,  1.,  1.,  0., -0.,
        -1.,  0.,  0., -0., -0., -0., -0.,  0.,  0.,  1., -1.,  1.,  1.,
         0.,  0., -1., -1., -0., -1.,  0.],
       [ 0., -0.,  1.,  0., -0., -0., -0., -0.,  0., -0.,  0.,  0.,  1.,
        -0., -0., -0., -1.,  1., -1., -0.,  0., -1., -1.,  0.,  0.,  0.,
         0., -0., -1.,  0., -1., -0., -0.,  0., -0.,  0.,  0.,  0.,  1.,
         1., -0., -0.,  1.,  0., -0.,  1.,  1.,  0.,  0.,  1.,  0.,  1.,
         1.,  0.,  0.,  0., -0., -0., -0.,  0.,  0.,  1.,  1., -1., -0.,
        -0., -0.,  1., -1., -1.,  1.,  0.],
       [ 0., -0., -1., -1.,  0.,  0.,  0., -0.,  0.,  0., -0., -0., -1.,
         0., -0., -0., -1.,  1., -1., -0.,  0., -0.,  1.,  0.,  0., -0.,
        -1.,  0., -1., -0.,  1., -0., -0., -0.,  1.,  0., -0., -0.,  1.,
        -1., -0.,  1., -0.,  0., -0., -1., -1., -0.,  0., -1., -0., -0.,
         1., -0., -0.,  0., -0., -0.,  0., -0., -0., -1.,  1., -1., -0.,
        -0., -0.,  1., -1., -0., -1., -0.],
       [ 0.,  0., -1., -0.,  1., -0.,  0., -0.,  1.,  1.,  0., -0.,  0.,
        -0., -0.,  0.,  0., -1., -0., -1.,  0., -0., -1.,  0.,  0.,  1.,
        -0., -0., -1., -0.,  0.,  0., -1., -0., -0., -0., -0.,  1., -1.,
         1.,  0.,  0.,  0., -0.,  0., -1.,  1.,  0., -0., -1.,  0., -0.,
         1.,  0.,  0., -0., -0.,  0.,  0.,  1.,  0., -1.,  1., -1.,  0.,
        -0.,  0., -1.,  1.,  0.,  1., -0.],
       [-0., -0., -1.,  0., -0., -0., -0., -0.,  1.,  0., -0.,  0., -0.,
        -1.,  0.,  0.,  0., -1., -0.,  1., -0., -0.,  1.,  0.,  1.,  0.,
        -0.,  0.,  1., -0., -0.,  0., -1.,  0.,  0., -0., -0., -0., -1.,
        -1.,  0., -0.,  0., -0.,  0.,  1., -1., -0.,  1.,  1., -0.,  0.,
         1., -0., -0., -0.,  0.,  0., -0., -1.,  0., -1.,  1.,  1., -1.,
        -0.,  0., -1.,  1., -0., -1.,  0.],
       [-0., -0.,  1., -0.,  0., -0.,  0.,  0., -1., -0.,  0.,  0., -0.,
        -0., -0.,  0.,  0.,  1.,  0., -1., -0.,  1., -1.,  0., -0.,  0.,
         0., -0.,  1., -0., -0.,  0., -1.,  0., -0.,  0.,  0., -0., -1.,
        -1., -0.,  0.,  1., -0., -0., -1.,  1.,  0.,  0.,  1.,  0., -1.,
         1., -0.,  0.,  0., -0., -0., -0., -1., -0., -1., -1., -1.,  0.,
         0., -0.,  1.,  1., -1., -1.,  0.],
       [ 0., -0., -1., -1., -0., -0.,  0.,  0., -1., -0., -0., -0., -0.,
         0.,  0.,  0., -0.,  1.,  0.,  1.,  0.,  0., -1.,  0.,  0., -0.,
        -1.,  0.,  1., -0., -0., -0., -1.,  0., -1., -0.,  0., -0., -1.,
        -1.,  0., -1.,  0.,  0.,  0.,  1., -1.,  0.,  0., -1., -0.,  0.,
         1.,  0., -0., -0., -0.,  0.,  0.,  1.,  0.,  1., -1., -1.,  0.,
         0.,  0., -1., -1.,  0.,  1., -0.],
       [ 1.,  0.,  0.,  0., -0.,  0.,  0., -1.,  0.,  0., -1., -0., -0.,
         0., -1., -1., -0.,  0.,  0., -0., -0., -0.,  0.,  1.,  0., -0.,
        -0.,  0.,  0., -1., -0.,  1., -0.,  0.,  0.,  1., -0.,  0., -0.,
         0., -1., -0.,  0., -1.,  1., -0.,  0., -0., -0.,  0.,  1., -0.,
         0., -1., -1., -0.,  1., -0.,  1., -0.,  1., -0., -0.,  0., -0.,
         1.,  1., -0., -0., -0.,  0.,  0.],
       [ 1., -0., -0.,  0., -0.,  0., -0.,  1., -0., -0.,  1., -0., -0.,
         0., -1., -1., -0., -0., -0.,  0.,  0., -0.,  0., -1.,  0., -0.,
        -0.,  0., -0.,  1.,  0., -1.,  0.,  0.,  0., -1.,  0., -0.,  0.,
         0., -1.,  0.,  0.,  1.,  1.,  0.,  0.,  0., -0.,  0.,  1.,  0.,
        -0.,  1.,  1.,  0.,  1.,  0.,  1.,  0.,  1., -0.,  0.,  0., -0.,
         1., -1., -0.,  0., -0., -0.,  0.],
       [ 1.,  0., -0.,  0., -0., -0.,  0.,  1., -0., -0.,  1., -0., -0.,
         0., -1.,  1., -0., -0., -0., -0., -0., -0.,  0.,  1.,  0., -0.,
         0., -0., -0., -1., -0., -1., -0.,  0.,  0., -1.,  0.,  0., -0.,
         0., -1.,  0.,  0.,  1.,  1.,  0.,  0., -0., -0., -0.,  1.,  0.,
         0., -1., -1., -0., -1.,  0., -1., -0., -1.,  0.,  0.,  0., -0.,
        -1.,  1., -0., -0.,  0.,  0., -0.],
       [-1., -0., -0., -0.,  0., -0., -0.,  1., -0., -0.,  1., -0.,  0.,
        -0.,  1., -1.,  0., -0., -0., -0., -0.,  0., -0.,  1.,  0.,  0.,
        -0., -0.,  0., -1.,  0., -1., -0.,  0.,  0., -1.,  0.,  0.,  0.,
        -0.,  1., -0.,  0.,  1., -1., -0., -0., -0., -0.,  0., -1., -0.,
         0., -1., -1., -0.,  1.,  0.,  1.,  0.,  1.,  0.,  0., -0.,  0.,
         1.,  1., -0., -0.,  0., -0.,  0.],
       [-0.,  1., -1.,  0.,  1.,  0., -0.,  0.,  0., -1., -0., -0., -0.,
         0., -0., -0.,  0.,  1., -0., -0., -1.,  0.,  1.,  0.,  0.,  1.,
        -0.,  0., -1., -0.,  0.,  0.,  0.,  0.,  0.,  0., -1., -1., -1.,
         1., -0., -0.,  0.,  0., -0.,  1., -1.,  0., -0., -1., -0., -0.,
         1.,  0.,  0., -0.,  0., -1.,  0.,  0.,  0.,  1., -1.,  1., -0.,
         0., -0., -1., -1., -0.,  1.,  0.],
       [ 0.,  1., -1.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -0.,  0.,
        -1., -0., -0.,  0.,  1.,  0., -0., -1.,  0.,  1.,  0.,  1., -0.,
        -0.,  0.,  1.,  0., -0., -0., -0., -0., -0.,  0.,  1., -0.,  1.,
         1., -0.,  0.,  0.,  0., -0., -1., -1., -0., -1., -1.,  0.,  0.,
         1.,  0., -0., -0.,  0.,  1.,  0., -0., -0.,  1.,  1., -1.,  1.,
        -0.,  0.,  1.,  1.,  0., -1.,  0.],
       [ 0., -1., -1.,  0., -0.,  0.,  0.,  0.,  0., -0., -0.,  0., -0.,
        -0.,  0., -0.,  0., -1.,  0.,  0., -1.,  1., -1.,  0.,  0., -0.,
         0., -0., -1.,  0., -0.,  0.,  0.,  0.,  0., -0., -1., -0.,  1.,
         1.,  0., -0., -1., -0.,  0.,  1.,  1.,  0.,  0., -1.,  0.,  1.,
         1., -0., -0.,  0., -0.,  1.,  0., -0.,  0., -1., -1., -1.,  0.,
         0., -0., -1.,  1., -1., -1.,  0.],
       [-0.,  1.,  1.,  1.,  0., -0., -0.,  0.,  0.,  0.,  0.,  0., -0.,
         0.,  0., -0., -0.,  1.,  0., -0.,  1., -0.,  1., -0.,  0.,  0.,
        -1.,  0., -1.,  0., -0., -0., -0.,  0., -1., -0., -1., -0.,  1.,
        -1.,  0.,  1.,  0., -0.,  0.,  1., -1., -0., -0.,  1., -0.,  0.,
        -1.,  0., -0.,  0.,  0.,  1., -0.,  0., -0.,  1., -1., -1., -0.,
        -0.,  0., -1.,  1., -0., -1., -0.],
       [ 1., -0., -0., -0., -0., -1., -1., -1.,  0., -0., -1., -0.,  0.,
        -0., -1., -1., -0.,  0.,  0., -0., -0., -0., -0.,  1.,  0., -0.,
        -0.,  0.,  0., -1.,  0.,  1., -0., -0., -0., -0., -0.,  0., -0.,
         0.,  0., -0., -0., -1.,  1.,  0.,  0., -0., -0.,  0.,  1.,  0.,
         0.,  0., -1.,  1.,  1., -0.,  1.,  0., -0., -0.,  0.,  0.,  0.,
         1.,  1., -0.,  0., -0., -0., -1.],
       [-1., -0., -0., -0., -0.,  1., -1.,  1., -0., -0., -1., -0.,  0.,
        -0., -1., -1.,  0., -0.,  0., -0.,  0., -0.,  0., -1., -0., -0.,
        -0.,  0., -0.,  1., -0., -1.,  0., -0., -0., -0.,  0.,  0.,  0.,
         0.,  0.,  0., -0., -1., -1., -0.,  0., -0.,  0.,  0.,  1.,  0.,
        -0.,  0., -1.,  1., -1., -0.,  1., -0., -0., -0.,  0., -0., -0.,
        -1.,  1., -0.,  0., -0., -0.,  1.],
       [-1.,  0.,  0.,  0.,  0.,  1., -1., -1.,  0., -0.,  1., -0., -0.,
        -0., -1.,  1., -0., -0.,  0., -0.,  0., -0.,  0.,  1., -0., -0.,
         0.,  0.,  0.,  1.,  0., -1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        -0.,  0., -0., -0., -1.,  1.,  0., -0., -0., -0.,  0., -1.,  0.,
         0.,  0.,  1., -1.,  1.,  0.,  1., -0.,  0., -0., -0., -0., -0.,
        -1.,  1.,  0.,  0., -0.,  0., -1.],
       [ 1.,  0., -0., -0., -0., -1., -1.,  1., -0., -0.,  1.,  0.,  0.,
         0., -1.,  1.,  0.,  0., -0.,  0., -0., -0., -0., -1., -0.,  0.,
        -0.,  0., -0., -1.,  0.,  1.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
        -0., -0., -0.,  0., -1., -1., -0., -0.,  0., -0.,  0., -1.,  0.,
        -0., -0.,  1., -1., -1., -0.,  1., -0.,  0., -0., -0., -0.,  0.,
         1.,  1., -0., -0.,  0., -0.,  1.],
       [-0., -0., -1.,  0., -1., -0.,  0., -0.,  0.,  1.,  0.,  1.,  0.,
         0.,  0., -0.,  0.,  1.,  0.,  0.,  0., -0., -1.,  0., -0.,  1.,
        -0., -1.,  1.,  0., -0.,  0., -0.,  1., -0., -0., -0., -1.,  1.,
         1., -0.,  0., -0., -0., -0., -1., -1.,  1., -0., -1., -0., -0.,
         1.,  0.,  0.,  0., -0., -0.,  0.,  0., -0., -1.,  1.,  1.,  0.,
         0.,  0.,  1., -1.,  0., -1.,  0.],
       [-0.,  0.,  1., -0.,  0.,  0.,  0., -0., -0., -0., -0., -1.,  0.,
         1., -0., -0.,  0., -1., -0.,  0., -0.,  0., -1., -0.,  1., -0.,
        -0.,  1.,  1.,  0.,  0.,  0., -0.,  1.,  0., -0., -0.,  0.,  1.,
         1., -0.,  0., -0.,  0., -0., -1.,  1.,  1.,  1.,  1.,  0.,  0.,
        -1.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  1.,  1.,  1.,  1.,
        -0.,  0., -1.,  1., -0., -1., -0.],
       [-0., -0.,  1., -0.,  0., -0., -0., -0.,  0.,  0.,  0.,  1.,  0.,
         0.,  0.,  0.,  0.,  1.,  0., -0.,  0.,  1.,  1., -0., -0., -0.,
        -0.,  1., -1., -0., -0., -0., -0., -1., -0.,  0.,  0., -0., -1.,
         1.,  0., -0.,  1.,  0.,  0.,  1.,  1.,  1.,  0., -1.,  0.,  1.,
        -1.,  0.,  0., -0., -0.,  0.,  0., -0., -0., -1.,  1.,  1., -0.,
        -0.,  0.,  1.,  1.,  1., -1., -0.],
       [-0., -0., -1., -1., -0.,  0.,  0.,  0., -0.,  0., -0.,  1.,  0.,
         0.,  0.,  0., -0.,  1.,  0.,  0.,  0.,  0.,  1., -0.,  0., -0.,
         1.,  1., -1.,  0.,  0., -0., -0.,  1., -1.,  0.,  0.,  0.,  1.,
         1.,  0.,  1.,  0.,  0., -0., -1.,  1., -1., -0., -1.,  0., -0.,
        -1., -0., -0., -0.,  0., -0.,  0., -0.,  0.,  1., -1.,  1., -0.,
        -0., -0., -1., -1., -0.,  1.,  0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 4
m = 4
p = 6
rank = 72

verify_tensor_support_rank_decomposition(decomposition_446, n, m, p, rank)

## Rank-94 support rank decomposition of <4,4,8> over Z


In [ ]:
#@title Data

decomposition_448 = (np.array([[ 1.,  0., -1., -1.,  0.,  0., -0.,  1.,  1.,  0.,  1.,  1., -0.,
        -0.,  1., -0., -0., -0., -0., -0., -0., -0.,  0., -0., -0.,  0.,
         0.,  0.,  0.,  1.,  0.,  1., -0.,  0., -1., -0., -0., -0., -0.,
         0.,  0.,  1., -0.,  1.,  0.,  0., -0.,  0.,  1., -0.,  0., -0.,
         0.,  0., -0., -1.,  0., -0.,  0., -0., -1., -0., -0.,  0., -0.,
         0.,  0., -1.,  0., -0., -1.,  1., -0., -0., -0.,  0.,  1.,  1.,
        -0., -0.,  1.,  0., -0.,  1.,  0., -1., -0., -0., -0.,  0., -0.,
        -0.,  0.,  0.],
       [-1.,  1., -0., -0.,  1.,  0.,  0., -1., -1.,  1.,  0., -1., -0.,
         0.,  0., -0., -0.,  0.,  0., -1.,  0.,  0.,  0., -0.,  0.,  0.,
         0.,  0.,  1.,  0.,  1., -1.,  0., -0., -0., -0., -1.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,  0., -0.,  1.,  0.,
        -0., -0., -0.,  1.,  0.,  0., -0.,  0., -0., -1., -0., -0.,  0.,
        -0., -0., -0., -0.,  0.,  1., -1., -0.,  0.,  0., -0., -1., -1.,
        -1., -0., -1.,  1.,  0.,  0.,  0.,  1., -0.,  0.,  0., -0.,  0.,
         0.,  1., -0.],
       [-0.,  0., -1., -1.,  0., -0.,  0.,  1.,  1., -1., -0.,  1.,  0.,
         1.,  1., -0.,  0.,  0.,  0., -0., -0.,  0.,  0.,  0.,  1.,  0.,
        -0.,  0., -1.,  1.,  0., -0.,  0.,  0.,  0., -0., -0., -0.,  0.,
        -0., -0., -0.,  0., -0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,
         0.,  0.,  0., -1.,  0., -0.,  0.,  1., -1., -0., -0., -0.,  1.,
         0.,  0.,  0., -1., -0.,  0., -0., -0.,  1., -0., -0., -0.,  1.,
         1., -0.,  1., -1.,  0.,  1., -0.,  0., -0., -0., -0., -0., -0.,
        -0., -1.,  0.],
       [-0., -1.,  0., -1., -1., -0.,  0., -0.,  1.,  0.,  1.,  1.,  0.,
         1., -0., -0., -0.,  0., -0.,  1., -0., -0.,  0., -0.,  1.,  0.,
         0.,  0., -1., -0.,  0., -0.,  0., -0., -1.,  0.,  1., -0.,  0.,
         0., -0.,  1.,  0., -0., -0., -0., -0., -0.,  1.,  0.,  0., -0.,
        -0., -0.,  0., -1.,  0., -0.,  0.,  1.,  0.,  1.,  0.,  0., -0.,
        -0.,  0., -1., -1., -0.,  0.,  1., -0.,  1., -0., -0., -0.,  1.,
        -0., -0.,  1.,  0.,  0., -0., -0.,  0.,  0.,  0.,  0., -0.,  1.,
        -0.,  0., -0.],
       [ 0., -0., -0.,  0., -0., -0., -1., -0., -0.,  0.,  0.,  0., -0.,
         0., -0.,  0.,  1., -1.,  1., -0.,  0.,  0.,  0.,  0., -0.,  1.,
         1.,  1.,  0., -0., -0., -0.,  1., -0.,  0.,  1.,  0.,  0., -1.,
         0.,  1.,  0.,  0., -0., -0.,  1., -1.,  1., -0.,  0., -0., -1.,
        -1.,  1., -0.,  0.,  1., -0.,  1.,  0., -0.,  0., -0.,  0.,  0.,
         0.,  0.,  0., -0., -0.,  0.,  0., -1., -0., -0., -0., -0.,  0.,
         0.,  0., -0., -0., -1., -0.,  1., -0., -0., -1.,  1., -0.,  0.,
        -0.,  0., -0.],
       [-0.,  0., -0., -0.,  0.,  0., -0., -0.,  0., -0.,  0.,  0.,  0.,
        -0.,  0., -0., -1., -0., -1., -0., -0.,  1.,  1.,  1.,  0., -1.,
        -1.,  0.,  0.,  0.,  0., -0., -1., -1.,  0., -1., -0., -0., -0.,
        -0.,  0., -0.,  1.,  0.,  0., -1., -0., -1.,  0.,  1., -0.,  1.,
        -0.,  0.,  0., -0., -1., -0.,  0.,  0., -0.,  0., -1., -0., -0.,
         1.,  1., -0., -0.,  0.,  0., -0., -0., -0.,  0., -1.,  0.,  0.,
        -0.,  1., -0.,  0.,  1., -0.,  0., -0.,  0., -0., -1.,  0.,  0.,
        -1., -0.,  0.],
       [ 0., -0.,  0., -0.,  0.,  1.,  0.,  0., -0.,  0., -0.,  0.,  1.,
         0.,  0.,  0.,  1., -1., -0., -0.,  0., -1.,  0., -1., -0.,  1.,
        -0.,  0., -0., -0., -0.,  0., -0.,  1., -0., -0., -0., -1.,  0.,
        -1.,  1.,  0.,  0.,  0.,  1.,  1., -1.,  1., -0.,  0., -0.,  0.,
         0., -0.,  1., -0.,  1.,  0.,  1., -0.,  0.,  0., -0.,  0., -0.,
        -1., -1.,  0., -0., -0., -0.,  0., -1., -0.,  0.,  1.,  0.,  0.,
         0.,  0.,  0.,  0., -1.,  0., -0.,  0., -0., -1., -0., -0.,  0.,
        -0.,  0., -0.],
       [ 0.,  0., -0.,  0.,  0.,  1., -1., -0.,  0., -0., -0., -0.,  1.,
        -0., -0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  0., -1.,  0.,  1.,
        -0.,  1.,  0., -0., -0., -0.,  1., -0., -0., -0., -0., -1., -1.,
        -1.,  1., -0., -1., -0.,  1.,  1.,  0., -0.,  0., -1.,  0.,  0.,
         0.,  1., -0., -0.,  1.,  0., -0., -0., -0., -0.,  1., -1., -0.,
         0.,  0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,
        -0., -1., -0., -0., -1., -0.,  1.,  0., -0.,  0., -0., -0., -0.,
         1.,  0., -0.],
       [-0.,  0., -0.,  0., -0., -0., -1., -0., -0.,  0., -0., -0., -0.,
         0., -0., -1.,  1., -1.,  1., -0., -0.,  0., -0.,  0., -0.,  1.,
        -0., -0.,  0.,  0., -0., -0.,  1., -0., -0.,  1.,  0., -1., -1.,
         0.,  1.,  0., -1.,  0., -0., -0.,  0.,  1.,  0.,  0., -0., -1.,
        -1.,  1., -0., -0.,  1., -0.,  1.,  0.,  0., -0.,  0.,  0.,  0.,
        -1.,  0., -0.,  0., -0.,  0., -0., -1., -0., -0., -0.,  0., -0.,
        -0.,  0.,  0., -0., -1., -0.,  1., -0., -0., -1.,  1., -0., -0.,
        -0., -0., -0.],
       [ 0.,  0., -0.,  0., -0.,  1., -1., -0.,  0., -0., -0.,  0.,  1.,
         0., -0., -1., -0., -1., -0.,  0., -0., -1.,  0.,  0., -0., -0.,
        -0.,  1., -0., -0.,  0.,  0., -0., -0.,  0.,  1., -0., -1.,  0.,
        -1.,  1.,  0.,  0.,  0., -0., -0., -1., -0., -0.,  0.,  0.,  0.,
        -1.,  1.,  1., -0.,  1.,  0., -0.,  0.,  0., -0.,  1., -1., -0.,
         0.,  0., -0.,  0., -0.,  0.,  0., -1.,  0.,  0., -0.,  0.,  0.,
         0.,  0., -0., -0.,  0.,  0.,  1.,  0.,  1., -1., -0.,  1., -0.,
        -0.,  0.,  1.],
       [ 0.,  0.,  0., -0.,  0., -1., -0., -0.,  0., -0., -0.,  0., -1.,
        -0.,  0., -0., -1.,  1.,  0., -0., -0.,  1., -0.,  1., -0., -1.,
        -0.,  0., -0., -0.,  0.,  0.,  0., -1., -0.,  0.,  0.,  1., -0.,
        -0., -1., -0., -0., -0., -1., -1.,  1., -1., -0., -0.,  0.,  1.,
        -0., -1., -1.,  0., -1., -0., -1., -0.,  0., -0.,  0., -0.,  0.,
         1.,  1., -0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0., -0.,
         0.,  1.,  0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,
         0.,  0., -1.],
       [-0., -0., -0., -0., -0.,  0., -0., -0.,  0., -0.,  0., -0., -1.,
        -0., -0., -0., -1.,  1.,  0.,  0., -0., -0., -0.,  1., -0.,  0.,
         0., -1.,  0.,  0.,  0.,  0., -1.,  0., -0.,  0.,  0.,  1.,  1.,
         1., -1., -0.,  1.,  0., -1., -1., -0.,  0., -0., -0., -0., -0.,
        -0., -1.,  0.,  0., -1.,  0.,  0.,  0., -0., -0., -1.,  1., -0.,
        -0.,  1.,  0., -0., -0.,  0., -0., -0.,  0.,  0.,  0., -0.,  0.,
        -0.,  1., -0., -0.,  1.,  0., -1.,  0.,  0., -0., -1., -1.,  0.,
        -1., -0.,  0.],
       [ 1.,  0., -1., -1., -1., -0.,  0.,  1.,  1., -1.,  1.,  1.,  0.,
        -0.,  1.,  0., -0.,  0.,  0., -0.,  1., -0.,  0.,  0.,  1.,  0.,
        -0.,  0.,  0., -0.,  0.,  1., -0., -0.,  0., -0., -0.,  0.,  0.,
        -0.,  0.,  1., -0.,  1., -0.,  0., -0., -0.,  1., -0.,  0.,  0.,
        -0.,  0.,  0., -1.,  0., -0., -0., -0., -1., -0.,  0.,  0., -0.,
        -0.,  0., -1.,  0., -0., -1.,  1., -0., -0., -0.,  0., -0., -0.,
        -0.,  0.,  1.,  0.,  0.,  1.,  0., -1.,  0.,  0.,  0., -0., -0.,
         0.,  0.,  0.],
       [ 0., -0., -0., -0.,  1., -0.,  0., -1.,  0.,  1.,  0., -1.,  0.,
         0.,  0., -0., -0.,  0.,  0., -1.,  0.,  0.,  0.,  0.,  0., -0.,
        -0., -0.,  1.,  0.,  1., -1., -0., -0., -0., -0., -1.,  0., -0.,
         0.,  0., -1., -0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  1.,  0.,
        -0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  1., -1., -0., -0.,  0.,
        -0., -0., -0.,  1.,  0.,  1., -1.,  0.,  0., -1., -0., -1., -1.,
        -1., -0., -1.,  1.,  0.,  0.,  0.,  1.,  0.,  0., -0.,  0.,  0.,
        -0., -0., -0.],
       [-1.,  1., -0., -0.,  1.,  0., -0.,  0.,  0., -0., -1., -1.,  0.,
         0.,  0., -0.,  0., -0.,  0., -1., -1.,  0., -0., -0.,  0.,  0.,
        -0., -0., -0.,  0.,  1.,  0., -0., -0.,  1., -0.,  0.,  0., -0.,
         0.,  0., -1., -0., -1., -0., -0., -0.,  0.,  0.,  0.,  1.,  0.,
        -0., -0., -0., -0., -0., -1.,  0., -1., -0., -1., -0., -0.,  0.,
        -0., -0.,  1., -0., -1.,  1., -1.,  0.,  0., -1.,  0., -1.,  0.,
         0.,  0.,  0., -0.,  0., -1.,  0.,  1., -0., -0., -0.,  0., -1.,
         0., -0.,  0.],
       [-0., -1., -1., -1., -1., -0.,  0., -0.,  1.,  0., -0.,  1., -0.,
        -0., -0., -0.,  0., -0.,  0.,  1., -0.,  0., -0., -0.,  1.,  0.,
         0., -0., -1., -0.,  0., -0., -0., -0., -1.,  0.,  1., -0., -0.,
        -0.,  0.,  1., -0., -0.,  0., -0.,  0., -0.,  1., -0.,  0., -0.,
         0., -0.,  0.,  0., -0., -0.,  0.,  1.,  0., -0.,  0., -0., -0.,
         0.,  0., -1., -1.,  1.,  0.,  1., -0.,  1., -0.,  0., -0.,  1.,
        -0., -0.,  1., -1., -0., -0.,  0., -1.,  0.,  0., -0.,  0.,  1.,
        -0.,  0., -0.]], dtype=np.float32), np.array([[ 0., -0., -0.,  1., -1.,  0., -0.,  1., -0., -1., -0., -0., -1.,
        -0., -1., -1., -1.,  0., -1., -1.,  1., -0.,  1., -1., -1.,  0.,
        -1., -1., -1., -1., -1., -0., -1., -1.,  1.,  0., -0.,  1.,  0.,
        -0.,  1.,  0., -1.,  1., -0.,  1.,  1., -1.,  0., -0., -0., -0.,
        -1.,  0.,  1., -0., -0.,  1.,  0.,  0., -0.,  0., -0., -1.,  1.,
        -1., -0.,  1., -0.,  0.,  1., -1.,  1., -1.,  0., -0.,  1.,  1.,
         1.,  0., -1., -0., -0., -0.,  1.,  0., -1.,  0.,  0.,  0., -1.,
         1., -0.,  0.],
       [ 0., -0.,  0., -1., -1., -0.,  0., -1.,  0.,  1., -0., -0.,  1.,
         0.,  1.,  1.,  1., -0.,  1., -1.,  1.,  0., -1.,  1.,  1.,  0.,
         1., -1.,  1.,  1., -1., -0., -1., -1.,  1., -0., -0., -1.,  0.,
         0., -1.,  0., -1.,  1., -0., -1.,  1., -1., -0., -0., -0., -0.,
         1., -0.,  1.,  0.,  0.,  1., -0.,  0.,  0.,  0.,  0., -1., -1.,
        -1.,  0.,  1., -0.,  0.,  1., -1.,  1.,  1., -0.,  0.,  1., -1.,
        -1., -0.,  1., -0., -0., -0.,  1.,  0.,  1., -0.,  0.,  0., -1.,
         1.,  0., -0.],
       [ 0., -0.,  0., -1.,  1., -0., -0.,  1., -0., -1.,  0., -0.,  1.,
         0.,  1., -1.,  1., -0., -1.,  1.,  1., -0.,  1.,  1.,  1., -0.,
        -1., -1., -1.,  1.,  1., -0., -1.,  1.,  1.,  0., -0., -1.,  0.,
         0., -1.,  0., -1.,  1., -0., -1., -1.,  1.,  0., -0.,  0.,  0.,
        -1.,  0., -1., -0., -0.,  1.,  0., -0.,  0., -0.,  0., -1., -1.,
         1., -0.,  1.,  0., -0., -1.,  1., -1.,  1.,  0.,  0., -1.,  1.,
         1., -0., -1.,  0., -0., -0.,  1., -0., -1., -0., -0., -0., -1.,
         1., -0.,  0.],
       [-0.,  0., -0., -1.,  1.,  0.,  0., -1., -0.,  1., -0., -0., -1.,
         0., -1.,  1.,  1., -0., -1.,  1., -1., -0.,  1.,  1.,  1.,  0.,
        -1.,  1., -1., -1., -1.,  0., -1.,  1.,  1., -0., -0.,  1.,  0.,
         0.,  1., -0., -1., -1., -0., -1.,  1.,  1.,  0., -0.,  0.,  0.,
         1., -0.,  1.,  0., -0., -1., -0.,  0., -0., -0., -0.,  1.,  1.,
         1.,  0.,  1.,  0.,  0.,  1.,  1.,  1.,  1.,  0., -0.,  1.,  1.,
        -1.,  0., -1.,  0., -0.,  0., -1.,  0.,  1., -0.,  0., -0., -1.,
         1., -0.,  0.],
       [ 0.,  0.,  0.,  1.,  1.,  0., -0.,  1.,  0., -1., -0.,  0.,  1.,
         0.,  1.,  1., -1.,  0., -1.,  1., -1., -0.,  1., -1., -1.,  0.,
        -1.,  1.,  1.,  1., -1., -0., -1., -1.,  1., -0.,  0., -1.,  0.,
         0., -1.,  0., -1., -1., -0.,  1., -1., -1., -0., -0.,  0.,  0.,
         1., -0., -1., -0.,  0., -1.,  0., -0.,  0., -0.,  0.,  1., -1.,
        -1., -0.,  1., -0., -0.,  1.,  1., -1., -1., -0.,  0.,  1., -1.,
         1., -0.,  1.,  0.,  0.,  0., -1.,  0.,  1., -0., -0., -0., -1.,
         1.,  0., -0.],
       [ 0.,  0.,  0.,  1.,  1., -0., -0., -1., -0.,  1.,  0., -0.,  1.,
        -0.,  1., -1., -1., -0.,  1.,  1.,  1., -0., -1., -1., -1., -0.,
         1.,  1., -1.,  1., -1., -0., -1.,  1., -1.,  0., -0., -1.,  0.,
         0., -1., -0., -1.,  1., -0.,  1.,  1.,  1.,  0.,  0.,  0., -0.,
        -1.,  0.,  1.,  0.,  0.,  1.,  0., -0.,  0., -0.,  0.,  1., -1.,
         1.,  0., -1.,  0., -0.,  1.,  1.,  1., -1.,  0.,  0.,  1.,  1.,
        -1.,  0., -1., -0., -0., -0., -1.,  0., -1.,  0.,  0., -0.,  1.,
         1.,  0.,  0.],
       [-0., -0.,  0., -1., -1., -0.,  0.,  1., -0., -1., -0., -0.,  1.,
         0.,  1.,  1., -1., -0., -1., -1., -1., -0.,  1., -1.,  1., -0.,
        -1., -1., -1.,  1., -1., -0.,  1.,  1., -1., -0., -0., -1., -0.,
        -0., -1., -0.,  1., -1.,  0.,  1.,  1.,  1., -0.,  0.,  0., -0.,
         1., -0.,  1.,  0.,  0., -1., -0., -0., -0.,  0., -0., -1., -1.,
         1.,  0., -1., -0.,  0.,  1., -1.,  1.,  1., -0.,  0.,  1.,  1.,
         1.,  0., -1., -0.,  0.,  0.,  1.,  0.,  1., -0., -0.,  0.,  1.,
        -1., -0.,  0.],
       [-0.,  0.,  0.,  1., -1.,  0.,  0., -1., -0.,  1., -0., -0., -1.,
        -0.,  1.,  1., -1.,  0.,  1., -1., -1.,  0., -1., -1., -1., -0.,
         1., -1., -1.,  1.,  1.,  0., -1.,  1.,  1., -0., -0.,  1., -0.,
        -0.,  1.,  0., -1., -1.,  0.,  1., -1.,  1., -0.,  0.,  0.,  0.,
         1.,  0., -1.,  0., -0., -1.,  0., -0.,  0.,  0., -0., -1., -1.,
         1.,  0.,  1.,  0.,  0., -1., -1., -1., -1.,  0.,  0., -1.,  1.,
        -1.,  0., -1., -0., -0.,  0.,  1., -0.,  1., -0.,  0.,  0., -1.,
         1., -0., -0.],
       [-1., -1.,  0.,  1., -0., -0., -0., -1.,  1., -0.,  0., -0.,  1.,
         0.,  1.,  0.,  1., -0.,  1.,  1., -0., -1.,  1., -1., -0., -0.,
         0.,  0., -1.,  0., -1., -0.,  1., -1.,  0., -1., -0., -0.,  1.,
         0.,  1.,  1., -0.,  1., -1.,  0., -0.,  1.,  0.,  0., -0.,  0.,
        -1.,  0., -1.,  0.,  1.,  1.,  1., -0., -1., -0., -1.,  1., -1.,
         0., -0., -1., -1., -0.,  1.,  1.,  1., -1., -1.,  0., -0., -0.,
        -1., -0., -1.,  0.,  0., -0.,  1.,  0.,  1., -0., -0.,  0.,  1.,
         1., -1.,  0.],
       [ 1.,  1.,  0.,  1.,  0.,  0.,  0., -1.,  1., -0., -0., -0., -1.,
        -0.,  1., -0., -1., -0., -1., -1.,  0., -1., -1.,  1.,  0., -0.,
        -0., -0., -1.,  0.,  1.,  0.,  1., -1., -0.,  1., -0.,  0.,  1.,
         0., -1., -1., -0., -1.,  1.,  0., -0.,  1.,  0., -0.,  0.,  0.,
         1., -0., -1.,  0., -1., -1.,  1., -0., -1.,  0., -1.,  1., -1.,
        -0.,  0.,  1., -1., -0., -1., -1.,  1., -1.,  1., -0.,  0., -0.,
        -1.,  0., -1.,  0.,  0.,  0.,  1., -0., -1.,  0., -0.,  0., -1.,
         1., -1., -0.],
       [ 1.,  1.,  0.,  1., -0., -0., -0.,  1.,  1., -0., -0., -0., -1.,
         0., -1., -0., -1., -0.,  1., -1.,  0.,  1.,  1.,  1., -0., -0.,
         0.,  0.,  1.,  0., -1.,  0.,  1.,  1.,  0., -1.,  0.,  0.,  1.,
         0., -1.,  1., -0., -1.,  1.,  0.,  0., -1.,  0.,  0., -0., -0.,
        -1., -0.,  1.,  0., -1.,  1., -1.,  0.,  1., -0., -1.,  1., -1.,
        -0.,  0., -1.,  1.,  0., -1.,  1., -1.,  1., -1.,  0., -0., -0.,
        -1.,  0., -1.,  0.,  0.,  0.,  1.,  0.,  1.,  0., -0.,  0., -1.,
         1., -1., -0.],
       [-1.,  1.,  0.,  1., -0.,  0.,  0., -1.,  1.,  0., -0., -0., -1.,
         0.,  1., -0.,  1., -0., -1., -1., -0., -1.,  1.,  1., -0.,  0.,
         0., -0.,  1., -0.,  1.,  0.,  1., -1.,  0.,  1.,  0., -0.,  1.,
         0.,  1.,  1., -0.,  1.,  1.,  0., -0., -1.,  0.,  0.,  0.,  0.,
         1.,  0., -1.,  0.,  1., -1., -1., -0., -1., -0.,  1., -1.,  1.,
         0.,  0., -1.,  1., -0.,  1.,  1., -1.,  1.,  1.,  0.,  0., -0.,
         1.,  0., -1., -0., -0.,  0.,  1.,  0.,  1., -0., -0.,  0., -1.,
        -1.,  1.,  0.],
       [-1.,  1.,  0., -1.,  0., -0.,  0.,  1., -1., -0.,  0.,  0.,  1.,
        -0., -1., -0., -1., -0., -1., -1., -0.,  1.,  1., -1.,  0., -0.,
         0.,  0., -1., -0.,  1., -0.,  1.,  1.,  0.,  1., -0.,  0.,  1.,
         0., -1.,  1.,  0.,  1., -1., -0.,  0.,  1.,  0.,  0.,  0., -0.,
         1., -0.,  1., -0., -1., -1.,  1., -0.,  1.,  0.,  1., -1., -1.,
        -0., -0., -1., -1.,  0.,  1.,  1.,  1., -1.,  1.,  0., -0., -0.,
        -1., -0.,  1.,  0.,  0.,  0.,  1.,  0.,  1., -0., -0.,  0., -1.,
        -1., -1.,  0.],
       [-1.,  1.,  0.,  1.,  0.,  0., -0.,  1.,  1.,  0.,  0., -0.,  1.,
        -0., -1.,  0., -1., -0.,  1., -1., -0., -1., -1., -1.,  0.,  0.,
        -0.,  0., -1., -0., -1.,  0.,  1., -1.,  0., -1., -0.,  0.,  1.,
         0., -1., -1.,  0.,  1., -1., -0.,  0., -1.,  0., -0., -0., -0.,
        -1., -0., -1., -0., -1.,  1., -1.,  0.,  1., -0.,  1., -1.,  1.,
         0., -0.,  1., -1.,  0.,  1., -1., -1., -1., -1., -0., -0., -0.,
         1., -0., -1., -0.,  0., -0.,  1., -0., -1.,  0., -0., -0., -1.,
        -1.,  1., -0.],
       [-1., -1., -0.,  1., -0., -0.,  0.,  1.,  1.,  0.,  0., -0., -1.,
         0., -1.,  0.,  1.,  0.,  1.,  1.,  0.,  1., -1.,  1.,  0., -0.,
        -0.,  0.,  1., -0.,  1., -0.,  1.,  1., -0., -1.,  0., -0.,  1.,
         0.,  1., -1., -0.,  1.,  1.,  0., -0.,  1.,  0., -0.,  0., -0.,
        -1.,  0.,  1.,  0.,  1., -1.,  1.,  0.,  1., -0.,  1., -1., -1.,
        -0.,  0.,  1.,  1., -0.,  1., -1.,  1.,  1.,  1., -0.,  0.,  0.,
        -1.,  0., -1., -0., -0., -0.,  1.,  0., -1., -0., -0., -0.,  1.,
        -1., -1., -0.],
       [ 1., -1.,  0.,  1., -0., -0.,  0.,  1.,  1.,  0.,  0.,  0.,  1.,
        -0., -1., -0.,  1.,  0., -1.,  1., -0.,  1., -1., -1.,  0.,  0.,
         0.,  0., -1.,  0.,  1.,  0.,  1.,  1.,  0.,  1.,  0., -0.,  1.,
         0.,  1.,  1., -0., -1., -1., -0.,  0., -1.,  0., -0., -0.,  0.,
         1.,  0.,  1., -0.,  1., -1., -1., -0.,  1.,  0., -1.,  1.,  1.,
        -0., -0., -1., -1., -0., -1.,  1., -1., -1.,  1., -0.,  0., -0.,
         1., -0., -1., -0.,  0., -0.,  1.,  0., -1., -0., -0.,  0.,  1.,
         1.,  1., -0.],
       [ 0., -0.,  0.,  1., -0., -0.,  0.,  1.,  0., -0., -0., -1.,  1.,
        -0.,  1., -0., -1., -0.,  1.,  1.,  0.,  0.,  1.,  1., -0., -0.,
         0., -0.,  1., -0., -1.,  1.,  1.,  1., -0.,  0., -1.,  0., -0.,
        -1.,  1., -0., -0., -1., -0.,  0., -0., -1., -1., -0.,  1.,  1.,
         1., -1., -1., -0., -0., -1., -0.,  1.,  0.,  0., -0., -1., -1.,
        -0.,  0.,  1.,  0.,  0.,  1.,  1.,  1., -1., -0., -1., -0.,  0.,
         1., -1.,  1., -0., -1., -1., -1.,  0., -1.,  1.,  0.,  0., -1.,
         1.,  0.,  1.],
       [ 0.,  0.,  0.,  1.,  0., -0., -0.,  1., -0.,  0., -0.,  1., -1.,
         0., -1.,  0.,  1.,  0., -1.,  1.,  0., -0.,  1.,  1.,  0., -0.,
         0., -0.,  1., -0.,  1.,  1., -1.,  1., -0., -0., -1., -0.,  0.,
         1.,  1.,  0.,  0.,  1., -0.,  0., -0.,  1., -1., -0., -1., -1.,
         1., -1.,  1.,  0., -0., -1.,  0., -1., -0., -0.,  0.,  1., -1.,
        -0., -0.,  1., -0.,  0.,  1., -1.,  1.,  1., -0., -1.,  0., -0.,
        -1., -1., -1., -0.,  1.,  1., -1., -0.,  1.,  1.,  0., -0.,  1.,
         1.,  0., -1.],
       [-0.,  0.,  0., -1., -0., -0.,  0.,  1., -0.,  0., -0.,  1.,  1.,
         0.,  1.,  0., -1., -0., -1.,  1., -0., -0.,  1., -1., -0., -0.,
        -0.,  0.,  1.,  0.,  1.,  1.,  1.,  1.,  0., -0., -1.,  0., -0.,
        -1., -1., -0., -0., -1., -0., -0., -0.,  1.,  1.,  0., -1., -1.,
         1.,  1.,  1., -0.,  0.,  1., -0.,  1.,  0., -0., -0., -1.,  1.,
        -0., -0., -1.,  0., -0.,  1., -1.,  1., -1.,  0., -1., -0.,  0.,
        -1.,  1., -1., -0., -1., -1.,  1.,  0.,  1.,  1., -0.,  0., -1.,
        -1.,  0., -1.],
       [ 0.,  0., -0., -1., -0., -0.,  0., -1., -0., -0.,  0.,  1.,  1.,
        -0., -1.,  0.,  1., -0.,  1.,  1.,  0., -0., -1.,  1.,  0.,  0.,
        -0.,  0.,  1., -0., -1., -1., -1., -1., -0.,  0., -1.,  0.,  0.,
        -1., -1.,  0.,  0.,  1.,  0.,  0., -0., -1.,  1., -0.,  1.,  1.,
         1.,  1.,  1.,  0.,  0., -1.,  0.,  1.,  0., -0.,  0., -1., -1.,
        -0.,  0., -1.,  0.,  0., -1., -1.,  1., -1.,  0.,  1., -0., -0.,
         1., -1., -1., -0.,  1.,  1.,  1., -0.,  1.,  1.,  0., -0., -1.,
         1.,  0., -1.],
       [ 0., -0.,  0.,  1.,  0.,  0., -0.,  1.,  0., -0.,  0.,  1.,  1.,
         0., -1., -0.,  1., -0., -1., -1., -0.,  0., -1., -1., -0., -0.,
        -0., -0., -1.,  0., -1.,  1., -1., -1.,  0.,  0.,  1.,  0.,  0.,
        -1.,  1.,  0.,  0.,  1.,  0., -0.,  0.,  1., -1.,  0.,  1., -1.,
         1., -1., -1., -0., -0.,  1., -0.,  1.,  0.,  0., -0., -1.,  1.,
        -0., -0.,  1.,  0., -0.,  1., -1.,  1., -1.,  0.,  1., -0., -0.,
         1.,  1., -1.,  0.,  1.,  1., -1.,  0., -1.,  1.,  0., -0., -1.,
        -1., -0.,  1.],
       [-0., -0., -0., -1., -0.,  0.,  0.,  1., -0., -0.,  0., -1., -1.,
        -0., -1., -0., -1., -0., -1., -1.,  0.,  0., -1.,  1., -0., -0.,
        -0.,  0., -1., -0.,  1.,  1.,  1., -1.,  0.,  0.,  1., -0., -0.,
         1., -1., -0., -0.,  1., -0.,  0., -0.,  1.,  1., -0., -1., -1.,
         1.,  1., -1., -0.,  0., -1., -0.,  1., -0.,  0.,  0.,  1., -1.,
         0.,  0., -1., -0.,  0.,  1.,  1.,  1., -1., -0.,  1., -0.,  0.,
        -1., -1.,  1.,  0., -1.,  1.,  1.,  0., -1.,  1.,  0., -0., -1.,
         1.,  0.,  1.],
       [-0., -0.,  0.,  1.,  0.,  0., -0., -1., -0., -0., -0.,  1., -1.,
        -0.,  1., -0., -1.,  0.,  1., -1.,  0.,  0., -1., -1.,  0.,  0.,
         0.,  0., -1., -0.,  1., -1.,  1., -1., -0., -0.,  1., -0., -0.,
         1.,  1.,  0., -0., -1.,  0., -0., -0., -1., -1.,  0., -1.,  1.,
         1., -1.,  1.,  0., -0., -1.,  0.,  1.,  0.,  0., -0.,  1., -1.,
         0.,  0.,  1.,  0.,  0., -1., -1.,  1., -1., -0.,  1.,  0., -0.,
        -1.,  1., -1.,  0., -1., -1., -1., -0.,  1.,  1., -0., -0., -1.,
        -1.,  0., -1.],
       [-0., -0., -0., -1.,  0.,  0.,  0.,  1., -0.,  0.,  0.,  1., -1.,
        -0.,  1.,  0.,  1., -0.,  1., -1.,  0.,  0.,  1., -1., -0.,  0.,
        -0., -0., -1., -0., -1.,  1., -1.,  1.,  0.,  0.,  1., -0.,  0.,
         1., -1.,  0.,  0., -1.,  0., -0.,  0., -1.,  1.,  0.,  1.,  1.,
         1.,  1., -1.,  0.,  0., -1., -0., -1., -0.,  0.,  0.,  1., -1.,
        -0., -0., -1.,  0.,  0.,  1., -1.,  1.,  1., -0., -1., -0., -0.,
         1.,  1., -1.,  0.,  1., -1.,  1., -0., -1.,  1.,  0., -0.,  1.,
        -1.,  0.,  1.],
       [ 0., -0.,  1., -1., -0.,  1., -1.,  1.,  0.,  0., -1.,  0.,  1.,
         1., -1., -0., -1., -1.,  1., -1., -0.,  0.,  1.,  1., -0.,  1.,
        -0., -0., -1., -0.,  1., -0., -1., -1., -0., -0., -0.,  0., -0.,
        -0.,  1., -0.,  0.,  1.,  0.,  0.,  0.,  1., -0., -1.,  0., -0.,
         1.,  0.,  1.,  1., -0., -1.,  0., -0.,  0., -1., -0.,  1., -1.,
         0., -1., -1.,  0., -1.,  1.,  1., -1., -1., -0., -0., -0., -0.,
        -1., -0.,  1., -1., -0.,  0.,  1., -1., -1.,  0., -1.,  1., -1.,
        -1., -0.,  0.],
       [ 0., -0., -1.,  1., -0.,  1.,  1., -1., -0., -0., -1.,  0.,  1.,
        -1.,  1.,  0., -1.,  1.,  1., -1., -0., -0., -1., -1., -0.,  1.,
         0., -0.,  1., -0.,  1., -0., -1.,  1., -0.,  0.,  0., -0., -0.,
         0., -1., -0.,  0.,  1.,  0., -0., -0.,  1.,  0.,  1., -0.,  0.,
        -1.,  0.,  1., -1.,  0., -1., -0., -0., -0., -1.,  0.,  1.,  1.,
         0.,  1., -1., -0., -1.,  1.,  1.,  1.,  1., -0., -0.,  0.,  0.,
         1., -0., -1.,  1.,  0., -0., -1., -1., -1., -0., -1.,  1., -1.,
         1., -0.,  0.],
       [ 0., -0.,  1., -1.,  0., -1., -1., -1.,  0., -0.,  1.,  0., -1.,
         1., -1.,  0.,  1.,  1.,  1., -1.,  0.,  0.,  1., -1.,  0., -1.,
         0., -0.,  1.,  0.,  1., -0., -1.,  1., -0., -0.,  0., -0.,  0.,
        -0., -1.,  0.,  0., -1.,  0., -0., -0., -1., -0., -1.,  0.,  0.,
         1., -0., -1., -1.,  0.,  1., -0., -0.,  0., -1.,  0.,  1., -1.,
        -0.,  1.,  1.,  0.,  1.,  1.,  1.,  1., -1., -0.,  0.,  0., -0.,
         1., -0., -1.,  1., -0., -0.,  1., -1., -1., -0., -1.,  1.,  1.,
        -1.,  0., -0.],
       [ 0.,  0.,  1., -1., -0., -1., -1.,  1.,  0., -0., -1.,  0., -1.,
        -1., -1.,  0., -1.,  1., -1.,  1.,  0.,  0., -1.,  1., -0.,  1.,
         0.,  0.,  1., -0., -1.,  0.,  1., -1.,  0.,  0.,  0.,  0.,  0.,
        -0., -1., -0.,  0.,  1., -0.,  0.,  0.,  1., -0.,  1.,  0., -0.,
         1., -0., -1.,  1., -0.,  1.,  0.,  0., -0.,  1.,  0.,  1.,  1.,
        -0., -1., -1., -0.,  1.,  1.,  1.,  1.,  1., -0.,  0., -0.,  0.,
         1.,  0.,  1.,  1., -0., -0.,  1., -1., -1., -0.,  1.,  1.,  1.,
         1.,  0., -0.],
       [-0., -0.,  1., -1.,  0., -1.,  1.,  1.,  0.,  0.,  1.,  0., -1.,
        -1., -1., -0., -1.,  1.,  1., -1.,  0.,  0.,  1.,  1., -0.,  1.,
        -0.,  0.,  1.,  0.,  1., -0., -1., -1., -0., -0., -0., -0.,  0.,
         0., -1.,  0., -0., -1., -0., -0., -0.,  1., -0., -1., -0., -0.,
        -1.,  0., -1.,  1.,  0., -1., -0., -0.,  0., -1., -0., -1.,  1.,
         0., -1.,  1., -0., -1., -1., -1.,  1.,  1.,  0.,  0.,  0.,  0.,
         1.,  0.,  1.,  1., -0., -0., -1.,  1.,  1., -0., -1., -1., -1.,
        -1., -0., -0.],
       [-0.,  0., -1.,  1., -0., -1.,  1.,  1.,  0.,  0.,  1.,  0., -1.,
         1.,  1., -0., -1., -1., -1.,  1., -0., -0.,  1., -1.,  0.,  1.,
        -0., -0.,  1., -0., -1., -0.,  1.,  1., -0.,  0.,  0., -0.,  0.,
         0.,  1.,  0., -0., -1., -0., -0.,  0.,  1.,  0., -1.,  0.,  0.,
        -1.,  0., -1.,  1., -0., -1., -0., -0., -0.,  1.,  0.,  1., -1.,
         0.,  1.,  1.,  0., -1.,  1.,  1., -1., -1.,  0., -0., -0.,  0.,
         1., -0.,  1.,  1.,  0.,  0., -1., -1., -1.,  0.,  1.,  1., -1.,
        -1., -0., -0.],
       [-0., -0., -1.,  1.,  0., -1., -1.,  1.,  0.,  0.,  1.,  0., -1.,
        -1.,  1.,  0., -1., -1.,  1., -1.,  0., -0., -1., -1., -0.,  1.,
         0.,  0., -1., -0.,  1., -0., -1.,  1.,  0.,  0., -0., -0.,  0.,
        -0.,  1., -0.,  0., -1., -0., -0.,  0.,  1.,  0.,  1., -0.,  0.,
         1., -0., -1.,  1., -0.,  1.,  0.,  0.,  0., -1., -0., -1.,  1.,
         0.,  1.,  1., -0.,  1.,  1.,  1., -1.,  1., -0.,  0.,  0.,  0.,
        -1., -0.,  1., -1., -0.,  0.,  1., -1.,  1.,  0., -1., -1.,  1.,
         1.,  0., -0.],
       [-0.,  0., -1.,  1.,  0.,  1., -1.,  1.,  0.,  0., -1., -0.,  1.,
         1.,  1., -0., -1.,  1., -1., -1., -0., -0.,  1., -1.,  0.,  1.,
         0.,  0.,  1., -0.,  1., -0.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,
         0., -1.,  0., -0.,  1.,  0.,  0.,  0.,  1., -0., -1.,  0., -0.,
         1., -0.,  1.,  1.,  0.,  1.,  0., -0.,  0., -1., -0., -1., -1.,
         0.,  1., -1.,  0.,  1., -1., -1.,  1., -1., -0.,  0.,  0.,  0.,
         1.,  0.,  1.,  1., -0.,  0.,  1.,  1.,  1.,  0.,  1., -1.,  1.,
        -1., -0.,  0.]], dtype=np.float32), np.array([[ 1., -1., -1., -1., -1., -0.,  0.,  1., -1., -1.,  1., -1., -0.,
        -1., -0., -0., -0.,  0.,  0., -0., -1.,  0., -0.,  0.,  1.,  0.,
         0., -0., -1., -1., -1., -1.,  0.,  0., -1., -0.,  1., -0.,  0.,
        -0.,  0.,  1.,  0.,  1.,  0., -0., -0.,  0.,  1., -0., -1., -0.,
         0.,  0.,  0.,  1., -0., -0., -0.,  1.,  1.,  1., -0., -0., -1.,
         0., -0.,  0.,  1., -1., -0., -1., -0., -0., -1.,  0.,  1.,  1.,
        -0., -0., -0.,  1., -0., -1., -0.,  1.,  0., -0.,  0., -0.,  1.,
        -0., -1.,  0.],
       [-0., -0.,  0., -0., -0., -1., -1., -0.,  0., -0., -0., -0., -0.,
         0., -0.,  1.,  0.,  1.,  0., -0.,  0.,  1., -1.,  1.,  0., -1.,
         1.,  1.,  0.,  0.,  0.,  0.,  1., -0., -0., -1., -0., -1.,  1.,
        -1., -1., -0.,  1.,  0.,  1., -1.,  1.,  1., -0., -1., -0., -1.,
        -1., -1.,  1., -0.,  1., -0., -1., -0.,  0.,  0., -1.,  1., -0.,
         1.,  1.,  0., -0., -0., -0.,  0.,  0.,  0., -0.,  1.,  0.,  0.,
        -0.,  1., -0.,  0., -1., -0., -0., -0.,  0.,  1., -1.,  1., -0.,
         0.,  0.,  1.],
       [-0., -0.,  0., -0.,  0., -1., -1.,  0.,  0.,  0.,  0.,  0.,  1.,
        -0., -0.,  1., -1.,  1., -1., -0.,  0.,  1.,  0., -0., -0., -1.,
         1.,  1.,  0., -0.,  0., -0., -0.,  1.,  0., -1.,  0., -1.,  1.,
        -1.,  0.,  0.,  1., -0.,  1., -1.,  1., -0., -0., -1., -0., -1.,
         0., -1., -0., -0.,  1.,  0., -1.,  0.,  0.,  0., -1., -0.,  0.,
         1.,  1., -0., -0., -0., -0., -0., -1., -0., -0.,  1.,  0.,  0.,
         0.,  1., -0., -0., -1., -0.,  1., -0.,  1.,  1., -1.,  1., -0.,
        -1., -0.,  1.],
       [-1.,  1., -1.,  0., -1., -0.,  0.,  0.,  1., -1.,  1.,  1.,  0.,
        -1.,  1., -0.,  0., -0., -0., -1., -1., -0., -0.,  0.,  1., -0.,
        -0.,  0.,  0., -1.,  0.,  1., -0.,  0., -1., -0., -1.,  0.,  0.,
        -0., -0., -1., -0., -0., -0.,  0.,  0., -0., -1., -0.,  1.,  0.,
        -0.,  0.,  0.,  1., -0.,  1.,  0., -1., -1.,  1., -0.,  0.,  0.,
        -0.,  0.,  1., -1., -1.,  1., -0., -0., -1.,  1., -0.,  1.,  1.,
         1., -0.,  1.,  1.,  0.,  1.,  0.,  1., -0., -0., -0.,  0.,  0.,
        -0.,  1., -0.],
       [-1.,  1., -1., -1.,  1.,  0.,  0.,  1., -1., -1., -1.,  1.,  0.,
        -1., -0.,  0.,  0., -0., -0., -0.,  1.,  0.,  0., -0.,  1., -0.,
        -0., -0., -1., -1.,  1., -1.,  0.,  0.,  1.,  0.,  1., -0.,  0.,
         0., -0., -1.,  0., -1.,  0., -0., -0., -0.,  1.,  0.,  1.,  0.,
        -0.,  0., -0.,  1.,  0.,  0., -0., -1.,  1., -1.,  0., -0., -1.,
        -0., -0.,  0.,  1.,  1.,  0.,  1.,  0.,  0.,  1.,  0., -1.,  1.,
        -0.,  0., -0.,  1., -0.,  1.,  0., -1.,  0.,  0., -0., -0., -1.,
        -0., -1.,  0.],
       [ 0.,  0.,  0., -0., -0., -1.,  1., -0.,  0.,  0.,  0.,  0., -0.,
         0., -0., -1., -0., -1., -0.,  0., -0.,  1.,  1., -1., -0., -1.,
        -1.,  1.,  0.,  0.,  0.,  0.,  1., -0., -0.,  1., -0.,  1.,  1.,
        -1.,  1., -0.,  1.,  0., -1.,  1.,  1.,  1.,  0.,  1., -0., -1.,
         1.,  1.,  1., -0., -1., -0., -1.,  0.,  0.,  0., -1.,  1., -0.,
         1., -1., -0.,  0.,  0., -0., -0., -0., -0.,  0., -1.,  0.,  0.,
        -0., -1., -0., -0., -1.,  0., -0., -0., -0., -1., -1.,  1.,  0.,
         0., -0.,  1.],
       [ 0.,  0.,  0., -0., -0.,  1., -1., -0.,  0., -0., -0., -0., -1.,
         0.,  0., -1.,  1.,  1.,  1.,  0., -0.,  1.,  0., -0.,  0.,  1.,
        -1.,  1.,  0., -0.,  0.,  0.,  0.,  1.,  0.,  1.,  0.,  1.,  1.,
         1., -0., -0.,  1.,  0., -1.,  1.,  1.,  0., -0., -1.,  0.,  1.,
        -0., -1., -0.,  0., -1.,  0., -1.,  0., -0.,  0., -1., -0.,  0.,
         1.,  1., -0.,  0.,  0.,  0., -0., -1., -0.,  0.,  1.,  0., -0.,
        -0.,  1., -0.,  0.,  1., -0.,  1.,  0., -1.,  1.,  1., -1.,  0.,
        -1., -0., -1.],
       [-1.,  1.,  1., -0., -1., -0., -0.,  0., -1.,  1.,  1., -1.,  0.,
         1., -1.,  0.,  0.,  0., -0., -1., -1., -0., -0.,  0., -1.,  0.,
        -0.,  0., -0.,  1.,  0.,  1., -0., -0., -1., -0., -1., -0.,  0.,
         0., -0., -1., -0.,  0., -0., -0.,  0.,  0., -1.,  0., -1.,  0.,
         0.,  0., -0., -1., -0.,  1., -0.,  1.,  1.,  1., -0., -0.,  0.,
        -0., -0.,  1.,  1., -1.,  1.,  0.,  0.,  1.,  1., -0.,  1., -1.,
        -1.,  0., -1., -1.,  0., -1.,  0.,  1., -0.,  0.,  0.,  0., -0.,
        -0., -1.,  0.],
       [-1.,  1., -1., -1., -1.,  0.,  0., -1., -1.,  1., -1., -1.,  0.,
        -1., -0., -0., -0.,  0.,  0.,  0.,  1., -0., -0., -0.,  1., -0.,
         0.,  0.,  1., -1., -1.,  1., -0.,  0.,  1., -0., -1.,  0., -0.,
         0., -0.,  1., -0., -1., -0., -0.,  0.,  0.,  1., -0., -1., -0.,
        -0., -0., -0., -1., -0.,  0.,  0., -1., -1.,  1.,  0., -0., -1.,
        -0., -0.,  0., -1.,  1., -0., -1., -0.,  0., -1., -0.,  1., -1.,
         0., -0.,  0., -1.,  0.,  1.,  0.,  1., -0., -0., -0., -0., -1.,
        -0., -1.,  0.],
       [ 0.,  0., -0.,  0., -0.,  1., -1.,  0., -0., -0.,  0.,  0.,  0.,
        -0.,  0.,  1., -0., -1., -0.,  0.,  0., -1., -1., -1.,  0.,  1.,
         1.,  1., -0., -0.,  0., -0.,  1.,  0.,  0., -1.,  0.,  1.,  1.,
        -1.,  1.,  0.,  1., -0., -1.,  1., -1., -1.,  0., -1., -0.,  1.,
        -1.,  1., -1., -0., -1.,  0.,  1., -0.,  0., -0., -1.,  1.,  0.,
        -1., -1.,  0.,  0., -0.,  0.,  0., -0., -0., -0.,  1., -0.,  0.,
         0., -1., -0., -0., -1., -0., -0.,  0., -0.,  1., -1.,  1.,  0.,
         0.,  0., -1.],
       [ 0.,  0.,  0., -0., -0., -1.,  1.,  0., -0., -0., -0., -0.,  1.,
         0.,  0., -1., -1.,  1.,  1., -0.,  0.,  1.,  0., -0.,  0., -1.,
        -1., -1.,  0., -0.,  0.,  0.,  0.,  1.,  0.,  1., -0., -1., -1.,
        -1.,  0., -0., -1.,  0.,  1., -1.,  1., -0.,  0.,  1., -0.,  1.,
        -0.,  1., -0., -0.,  1., -0., -1., -0., -0., -0.,  1.,  0.,  0.,
         1.,  1., -0., -0.,  0.,  0., -0., -1., -0., -0.,  1.,  0.,  0.,
         0., -1., -0.,  0., -1., -0., -1., -0., -1.,  1.,  1., -1., -0.,
         1., -0., -1.],
       [-1.,  1., -1.,  0., -1.,  0., -0.,  0., -1.,  1., -1., -1.,  0.,
        -1.,  1., -0., -0., -0., -0., -1.,  1., -0., -0., -0.,  1.,  0.,
        -0.,  0., -0., -1.,  0.,  1.,  0., -0.,  1.,  0., -1., -0.,  0.,
         0.,  0.,  1.,  0.,  0., -0., -0., -0.,  0.,  1., -0., -1., -0.,
        -0., -0.,  0., -1., -0., -1., -0., -1., -1.,  1.,  0.,  0., -0.,
        -0., -0., -1., -1.,  1.,  1.,  0., -0., -1., -1., -0.,  1., -1.,
        -1.,  0., -1., -1.,  0.,  1.,  0.,  1.,  0.,  0.,  0., -0.,  0.,
         0., -1.,  0.],
       [ 1.,  1., -1., -1., -1.,  0.,  0.,  1., -1., -1.,  1., -1.,  0.,
         1.,  0., -0.,  0., -0., -0.,  0., -1.,  0., -0.,  0.,  1.,  0.,
        -0.,  0.,  1.,  1.,  1., -1., -0., -0.,  1.,  0., -1.,  0., -0.,
         0.,  0.,  1., -0.,  1., -0.,  0.,  0.,  0.,  1.,  0.,  1., -0.,
        -0.,  0., -0.,  1., -0.,  0.,  0., -1.,  1., -1.,  0.,  0.,  1.,
        -0., -0.,  0., -1.,  1.,  0., -1., -0.,  0.,  1., -0., -1., -1.,
        -0.,  0.,  0., -1., -0., -1.,  0.,  1., -0., -0., -0.,  0., -1.,
        -0.,  1., -0.],
       [-0., -0.,  0.,  0., -0.,  1., -1.,  0.,  0.,  0.,  0., -0., -0.,
        -0.,  0.,  1.,  0., -1.,  0.,  0.,  0., -1.,  1.,  1.,  0., -1.,
        -1.,  1., -0., -0.,  0.,  0., -1., -0.,  0., -1., -0.,  1., -1.,
        -1.,  1.,  0., -1., -0.,  1., -1., -1.,  1., -0.,  1.,  0., -1.,
        -1.,  1., -1., -0., -1.,  0., -1.,  0.,  0.,  0., -1.,  1., -0.,
         1.,  1.,  0.,  0., -0., -0.,  0., -0., -0., -0., -1.,  0., -0.,
         0.,  1., -0., -0.,  1.,  0.,  0.,  0.,  0.,  1.,  1.,  1., -0.,
         0., -0., -1.],
       [ 0., -0., -0.,  0., -0., -1.,  1.,  0., -0.,  0., -0., -0.,  1.,
         0., -0., -1.,  1.,  1., -1., -0.,  0., -1.,  0.,  0.,  0.,  1.,
         1., -1.,  0.,  0.,  0.,  0., -0., -1., -0., -1., -0., -1., -1.,
        -1., -0., -0.,  1., -0.,  1.,  1.,  1.,  0., -0., -1., -0., -1.,
        -0.,  1., -0.,  0., -1., -0., -1., -0., -0., -0., -1.,  0., -0.,
        -1., -1., -0., -0., -0.,  0., -0., -1.,  0.,  0., -1., -0., -0.,
         0.,  1.,  0.,  0.,  1., -0., -1., -0., -1.,  1., -1., -1., -0.,
        -1., -0., -1.],
       [-1., -1., -1., -0.,  1.,  0.,  0.,  0.,  1.,  1.,  1.,  1.,  0.,
         1.,  1., -0.,  0.,  0., -0.,  1.,  1.,  0.,  0., -0., -1., -0.,
        -0., -0.,  0., -1., -0.,  1., -0., -0., -1.,  0.,  1.,  0., -0.,
         0.,  0., -1.,  0.,  0., -0., -0.,  0.,  0., -1., -0., -1.,  0.,
        -0.,  0.,  0.,  1.,  0., -1.,  0.,  1., -1., -1., -0.,  0.,  0.,
         0.,  0.,  1.,  1.,  1.,  1., -0.,  0.,  1., -1., -0.,  1.,  1.,
        -1., -0.,  1., -1., -0.,  1., -0.,  1.,  0., -0.,  0., -0.,  0.,
        -0., -1., -0.],
       [-1., -1., -1., -1.,  1., -0., -0.,  1., -1., -1., -1.,  1., -0.,
         1.,  0., -0., -0.,  0., -0.,  0.,  1.,  0., -0., -0.,  1.,  0.,
         0.,  0.,  1.,  1., -1., -1.,  0., -0., -1., -0., -1.,  0., -0.,
        -0.,  0., -1., -0., -1., -0., -0., -0., -0.,  1., -0., -1.,  0.,
         0.,  0.,  0.,  1.,  0.,  0., -0.,  1.,  1.,  1., -0., -0.,  1.,
         0.,  0.,  0., -1., -1.,  0.,  1., -0., -0., -1.,  0.,  1., -1.,
         0., -0.,  0., -1., -0.,  1.,  0., -1., -0.,  0.,  0.,  0.,  1.,
        -0.,  1.,  0.],
       [ 0.,  0., -0.,  0.,  0.,  1.,  1.,  0., -0., -0.,  0., -0., -0.,
        -0., -0., -1.,  0., -1.,  0., -0., -0., -1., -1.,  1.,  0., -1.,
         1., -1., -0., -0., -0.,  0.,  1., -0., -0.,  1.,  0.,  1.,  1.,
         1.,  1., -0.,  1., -0.,  1., -1., -1.,  1., -0., -1.,  0., -1.,
         1.,  1., -1., -0., -1.,  0., -1., -0., -0., -0.,  1., -1., -0.,
         1.,  1.,  0., -0.,  0.,  0., -0.,  0.,  0., -0.,  1., -0., -0.,
         0.,  1.,  0., -0., -1.,  0.,  0.,  0.,  0., -1., -1., -1.,  0.,
         0.,  0., -1.],
       [ 0.,  0.,  0., -0., -0.,  1.,  1., -0., -0., -0.,  0.,  0., -1.,
         0.,  0., -1., -1., -1., -1., -0., -0.,  1.,  0., -0.,  0., -1.,
         1., -1., -0.,  0., -0., -0., -0.,  1.,  0., -1.,  0.,  1., -1.,
         1., -0., -0.,  1.,  0., -1., -1., -1., -0., -0., -1., -0., -1.,
        -0.,  1.,  0.,  0.,  1.,  0.,  1., -0., -0., -0., -1., -0., -0.,
         1.,  1., -0.,  0.,  0., -0., -0.,  1., -0., -0.,  1., -0.,  0.,
        -0.,  1., -0., -0., -1.,  0., -1., -0., -1., -1., -1., -1.,  0.,
        -1.,  0., -1.],
       [-1., -1.,  1., -0.,  1., -0.,  0.,  0., -1., -1.,  1., -1., -0.,
        -1., -1., -0., -0.,  0.,  0.,  1.,  1.,  0., -0., -0.,  1.,  0.,
        -0.,  0.,  0.,  1., -0.,  1., -0., -0., -1.,  0.,  1., -0., -0.,
         0., -0., -1., -0., -0., -0.,  0.,  0., -0., -1.,  0.,  1., -0.,
        -0., -0., -0., -1., -0., -1.,  0., -1.,  1., -1., -0., -0., -0.,
         0.,  0.,  1., -1.,  1.,  1., -0.,  0., -1., -1., -0.,  1., -1.,
         1., -0., -1.,  1.,  0., -1.,  0.,  1., -0.,  0.,  0.,  0.,  0.,
         0.,  1., -0.],
       [ 1.,  1., -1., -1.,  1., -0.,  0., -1., -1.,  1.,  1.,  1., -0.,
         1.,  0., -0.,  0., -0.,  0., -0., -1.,  0., -0., -0.,  1.,  0.,
        -0.,  0., -1.,  1., -1.,  1.,  0., -0.,  1.,  0.,  1.,  0.,  0.,
        -0.,  0., -1., -0.,  1.,  0., -0., -0., -0.,  1.,  0., -1., -0.,
        -0., -0., -0., -1.,  0.,  0.,  0., -1., -1.,  1.,  0., -0.,  1.,
         0., -0., -0.,  1.,  1.,  0.,  1.,  0.,  0., -1., -0.,  1.,  1.,
         0., -0.,  0.,  1., -0., -1.,  0., -1., -0., -0.,  0., -0., -1.,
         0.,  1., -0.],
       [ 0.,  0.,  0., -0.,  0.,  1.,  1.,  0.,  0., -0., -0., -0., -0.,
         0., -0., -1.,  0.,  1.,  0., -0., -0., -1., -1., -1.,  0., -1.,
         1.,  1.,  0., -0., -0.,  0., -1.,  0., -0.,  1.,  0., -1., -1.,
        -1., -1.,  0., -1.,  0., -1.,  1., -1.,  1., -0., -1.,  0., -1.,
         1., -1., -1., -0.,  1.,  0., -1.,  0.,  0., -0., -1.,  1., -0.,
         1., -1.,  0., -0.,  0.,  0.,  0., -0., -0.,  0.,  1.,  0.,  0.,
         0., -1.,  0., -0.,  1.,  0.,  0., -0.,  0., -1.,  1.,  1.,  0.,
        -0., -0., -1.],
       [-0., -0.,  0., -0.,  0.,  1.,  1., -0.,  0.,  0.,  0., -0., -1.,
         0.,  0.,  1., -1.,  1.,  1.,  0., -0., -1., -0., -0.,  0., -1.,
        -1., -1.,  0.,  0.,  0.,  0.,  0., -1., -0.,  1.,  0.,  1., -1.,
         1.,  0.,  0.,  1.,  0., -1., -1.,  1., -0., -0., -1.,  0.,  1.,
        -0.,  1.,  0., -0.,  1., -0., -1.,  0.,  0., -0., -1., -0., -0.,
        -1., -1., -0.,  0.,  0.,  0.,  0., -1., -0.,  0., -1.,  0., -0.,
         0.,  1., -0.,  0., -1., -0., -1., -0.,  1.,  1.,  1.,  1., -0.,
        -1.,  0.,  1.],
       [-1., -1.,  1.,  0.,  1.,  0., -0., -0.,  1.,  1., -1.,  1.,  0.,
        -1., -1.,  0., -0., -0., -0.,  1., -1.,  0.,  0.,  0.,  1.,  0.,
         0., -0.,  0.,  1.,  0.,  1.,  0.,  0.,  1., -0.,  1.,  0., -0.,
        -0.,  0.,  1., -0., -0., -0., -0.,  0., -0.,  1.,  0., -1.,  0.,
        -0., -0.,  0.,  1.,  0.,  1.,  0., -1.,  1., -1.,  0.,  0., -0.,
         0.,  0., -1., -1., -1.,  1.,  0., -0., -1.,  1., -0.,  1.,  1.,
        -1., -0.,  1., -1., -0., -1.,  0.,  1., -0.,  0.,  0., -0.,  0.,
        -0., -1., -0.],
       [-1.,  1.,  1.,  1., -1.,  0.,  0.,  1.,  1., -1., -1., -1., -0.,
         1.,  0.,  0., -0., -0., -0.,  0.,  1.,  0.,  0., -0., -1.,  0.,
        -0.,  0., -1.,  1., -1., -1., -0., -0.,  1.,  0.,  1., -0., -0.,
         0.,  0.,  1., -0., -1.,  0., -0.,  0., -0., -1.,  0., -1.,  0.,
        -0., -0., -0.,  1., -0., -0., -0., -1.,  1.,  1.,  0.,  0.,  1.,
         0.,  0.,  0.,  1.,  1., -0., -1.,  0., -0., -1., -0.,  1.,  1.,
        -0.,  0., -0.,  1.,  0.,  1., -0.,  1., -0., -0., -0.,  0., -1.,
        -0.,  1., -0.],
       [ 0.,  0.,  0., -0., -0., -1.,  1., -0., -0.,  0., -0., -0., -0.,
        -0., -0., -1., -0., -1., -0.,  0., -0.,  1., -1.,  1.,  0.,  1.,
         1.,  1.,  0.,  0.,  0., -0., -1., -0.,  0.,  1., -0.,  1., -1.,
        -1.,  1.,  0., -1., -0.,  1., -1.,  1., -1., -0., -1., -0.,  1.,
         1.,  1.,  1., -0., -1.,  0.,  1., -0.,  0.,  0., -1.,  1., -0.,
        -1.,  1.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  1., -0.,  0.,
        -0.,  1., -0., -0.,  1., -0.,  0.,  0.,  0., -1.,  1.,  1., -0.,
         0.,  0.,  1.],
       [-0.,  0.,  0.,  0., -0.,  1., -1.,  0., -0., -0., -0.,  0., -1.,
         0., -0., -1., -1.,  1., -1.,  0.,  0., -1.,  0.,  0., -0., -1.,
         1.,  1., -0., -0., -0., -0., -0., -1., -0., -1., -0.,  1.,  1.,
         1.,  0.,  0., -1.,  0., -1., -1.,  1., -0.,  0.,  1.,  0., -1.,
         0., -1., -0., -0.,  1.,  0., -1., -0.,  0.,  0.,  1.,  0.,  0.,
        -1., -1.,  0.,  0.,  0.,  0.,  0., -1.,  0., -0., -1., -0.,  0.,
         0., -1.,  0.,  0., -1.,  0.,  1., -0., -1.,  1., -1., -1.,  0.,
         1.,  0., -1.],
       [-1.,  1.,  1., -0., -1.,  0.,  0., -0.,  1., -1., -1.,  1.,  0.,
         1., -1.,  0.,  0.,  0., -0., -1.,  1., -0.,  0., -0., -1., -0.,
        -0.,  0.,  0.,  1., -0.,  1., -0., -0.,  1., -0., -1.,  0., -0.,
         0.,  0.,  1., -0.,  0.,  0.,  0., -0.,  0.,  1.,  0.,  1.,  0.,
         0.,  0., -0.,  1.,  0., -1.,  0.,  1.,  1.,  1.,  0.,  0., -0.,
         0.,  0., -1.,  1.,  1.,  1.,  0.,  0.,  1., -1.,  0.,  1.,  1.,
         1.,  0.,  1.,  1.,  0., -1.,  0.,  1., -0., -0.,  0., -0.,  0.,
         0.,  1., -0.],
       [ 1.,  1.,  1.,  1.,  1., -0., -0.,  1.,  1., -1.,  1.,  1.,  0.,
        -1.,  0., -0.,  0., -0., -0.,  0., -1.,  0.,  0.,  0., -1.,  0.,
        -0.,  0.,  1., -1., -1., -1.,  0., -0.,  1., -0., -1., -0., -0.,
         0., -0., -1., -0.,  1., -0.,  0., -0.,  0., -1.,  0., -1.,  0.,
        -0.,  0., -0.,  1., -0., -0.,  0., -1.,  1.,  1.,  0.,  0., -1.,
         0.,  0., -0., -1.,  1., -0.,  1., -0.,  0., -1.,  0.,  1., -1.,
         0.,  0., -0., -1.,  0., -1.,  0., -1.,  0., -0.,  0., -0., -1.,
         0., -1., -0.],
       [ 0., -0.,  0., -0.,  0., -1., -1., -0.,  0.,  0., -0., -0.,  0.,
         0.,  0.,  1., -0., -1.,  0.,  0., -0.,  1., -1., -1., -0., -1.,
         1., -1.,  0.,  0.,  0.,  0., -1., -0., -0., -1., -0.,  1., -1.,
         1.,  1., -0., -1., -0., -1.,  1.,  1.,  1., -0., -1., -0., -1.,
        -1.,  1.,  1., -0., -1.,  0., -1.,  0.,  0., -0.,  1., -1., -0.,
         1., -1.,  0.,  0., -0.,  0., -0.,  0.,  0., -0.,  1., -0.,  0.,
         0., -1., -0.,  0.,  1., -0.,  0.,  0., -0.,  1.,  1., -1., -0.,
         0.,  0.,  1.],
       [ 0., -0., -0.,  0.,  0.,  1.,  1., -0.,  0., -0., -0.,  0., -1.,
        -0., -0.,  1.,  1.,  1., -1., -0., -0.,  1., -0.,  0.,  0.,  1.,
         1., -1.,  0., -0., -0., -0.,  0.,  1., -0., -1., -0.,  1., -1.,
         1., -0., -0., -1.,  0., -1.,  1.,  1., -0., -0.,  1.,  0., -1.,
         0.,  1., -0., -0., -1.,  0., -1., -0.,  0.,  0.,  1.,  0.,  0.,
         1.,  1.,  0., -0., -0., -0., -0., -1., -0., -0.,  1.,  0.,  0.,
         0., -1., -0., -0.,  1., -0., -1.,  0.,  1.,  1., -1.,  1., -0.,
         1., -0.,  1.],
       [ 1.,  1.,  1., -0., -1.,  0., -0., -0.,  1.,  1.,  1.,  1.,  0.,
        -1., -1., -0.,  0.,  0.,  0., -1.,  1., -0., -0.,  0.,  1., -0.,
        -0.,  0.,  0.,  1.,  0., -1.,  0.,  0., -1.,  0., -1.,  0.,  0.,
        -0.,  0., -1., -0., -0., -0., -0., -0.,  0., -1.,  0., -1., -0.,
         0.,  0.,  0.,  1.,  0., -1., -0., -1.,  1.,  1., -0.,  0., -0.,
        -0., -0.,  1., -1.,  1., -1., -0.,  0., -1., -1., -0., -1.,  1.,
        -1., -0.,  1., -1., -0., -1.,  0., -1., -0.,  0.,  0., -0.,  0.,
         0., -1., -0.]], dtype=np.float32))

In [ ]:
#@title Verification

n = 4
m = 4
p = 8
rank = 94

verify_tensor_support_rank_decomposition(decomposition_448, n, m, p, rank)